# Methods: Symmetric Dual-Encoder Group-Head Regression for Same-Time Sky-to-Science Coefficient Transfer

**Status:** working notebook. §5.5 documents the current default model, a
per-group compressed variant of the architecture described in §6 that
replaced the uncompressed baseline on 2026-07-28, had its compressor,
loss weights and architecture materially revised on 2026-08-11 in response
to the naive-baseline check now recorded in §11.1, and had its per-group
blend head switched from a sigmoid-parametrized logit to a directly-stored
clamped α on the same day after a convergence diagnostic (§12 entry
"direct alpha parametrization + init A/B"). On 2026-08-16 the loss became
a per-coefficient heteroscedastic-Gaussian NLL that consumes the
decomposition's ``COEF_ERR`` uncertainties in compressed score space
(§7 revised, §12 entry "coef_err-weighted loss (2026-08-16)").
§11 lists what remains
incomplete; read it before quoting any number out of this notebook.

## 1. Scientific objective

Given two sky spectra taken *simultaneously* with a science exposure but at
different points on the sky, predict the sky-component coefficients at the
science line of sight.

At each valid row $i$ the model receives

- near-sky coefficients $\mathbf{c}^{(i)}_{near}$ and their context $\mathbf{x}^{(i)}_{ctx,near}$
- far-sky coefficients $\mathbf{c}^{(i)}_{far}$ and their context $\mathbf{x}^{(i)}_{ctx,far}$
- science-pointing context $\mathbf{x}^{(i)}_{ctx,sci}$

and predicts

$$
\hat{\mathbf{c}}^{(i)}_{sci} = f\!\left(\mathbf{c}^{(i)}_{near},\,\mathbf{c}^{(i)}_{far},\,
\mathbf{x}^{(i)}_{ctx,near},\,\mathbf{x}^{(i)}_{ctx,far},\,\mathbf{x}^{(i)}_{ctx,sci}\right).
$$

This is a **transfer** problem, not a forecast: all inputs are contemporaneous
with the target. The framing matters because airglow brightness fluctuates by
tens of percent on ten-minute timescales under gravity-wave forcing, and that
variation is not predictable from ephemeris context. It *is* measured by the two
sky telescopes, so the two sky coefficient vectors carry the information a
context-only regression cannot access. The network's job is only the
*differential* effect of pointing geometry.

Both sky contexts are supplied so the model can form the geometric relationship
between each sky pointing and the science pointing. A version that received only
$\mathbf{x}_{ctx,sci}$ could not do this even in principle.

## 2. Data model and triplet construction

Three decomposition products sharing a common input FITS source: sky-near,
sky-far, and science. The loader enforces

1. numeric coercion and finite-row filtering for coefficient and context matrices;
2. context harmonisation across near, far and science modes;
3. row-index alignment by intersection of valid source rows across all three products;
4. construction of shell-geometry features and folded time features from the timestamp.

Result: aligned $\mathbf{C}_{near},\mathbf{C}_{far},\mathbf{C}_{sci}\in\mathbb{R}^{N\times P}$
and $\mathbf{X}_{ctx,near},\mathbf{X}_{ctx,far},\mathbf{X}_{ctx,sci}\in\mathbb{R}^{N\times Q}$
with shared coefficient- and context-name schemas.

Near/far ordering is by angular separation from the science pointing, not by
telescope identity, so which physical telescope occupies the "near" channel
changes from row to row.

## 3. Filtering and quality control

Applied in this order, so the printed per-filter counts refer to the full
input dataset and only compose via boolean AND on the row-keep mask.

1. optional exclusion of tiles whose science pointing falls within a specified
   angular radius of a listed sky region (default: LMC at $(\alpha,\delta)=(80.894^\circ,-69.756^\circ)$
   and SMC at $(13.187^\circ,-72.829^\circ)$, 10$^\circ$ radius each). This is a
   coarse pre-filter for the target contamination discussed in §11.2 — both Clouds
   combine bright stellar populations, dense H II regions and diffuse ionised gas at
   velocities close enough to airglow to blend with a sky-component fit;
2. per-arm sanity checks (non-negative altitude, airmass ≤ 3, van Rhijn shell factors ≥ 1) applied to near / far / sci simultaneously;
3. optional combined reduced-$\chi^2$ gating across near/far/science decompositions;
4. hard coefficient bounds for selected components;
5. kappa-sigma clipping in concatenated triplet coefficient space;
6. optional thinning, applied LAST so that steps 1–5 report fractions against the
   full pre-thinning dataset rather than a downsampled subset.

## 4. Geometric normalisation

This is the physical core of the current implementation and the part that
changed most recently.

### 4.1 The scaling law

An airglow amplitude observed at zenith angle $z$ relates to the intrinsic
emissivity of the emitting layer by

$$
A_{obs} = A_{int}\; \underbrace{V(z;h)}_{\text{slant path}}\;
\underbrace{10^{-0.4\,k(\lambda)\,[X(z)-1]}}_{\text{extinction}},
$$

with the van Rhijn factor for a thin shell at height $h$ above radius $R_\oplus$

$$
V(z;h) = \left[1 - \left(\frac{R_\oplus}{R_\oplus + h}\right)^{\!2}\sin^2 z\right]^{-1/2},
\qquad V(0;h)=1 .
$$

$V$ saturates toward the horizon (6.0 at $h=87$ km, 3.4 at $h=285$ km) rather
than diverging like $\sec z$, because the observer sits inside a curved shell.
Using plane-parallel airmass instead overstates the enhancement by 4% at
$z=60^\circ$ for a mesospheric layer and 12% for an ionospheric one — a smooth
function of $z$, so it does not average away and would alias into whatever the
network learns as geometry.

### 4.2 Where it is applied

`airglow_geometry_scale()` returns $V\cdot 10^{-0.4k(X-1)}$ per row and per
coefficient, and is applied **in physical units, before the square-root
transform and before robust scaling**. Neither of those operations commutes with
a division by the geometry factor: the scaler subtracts a median, so
$(\sqrt{c}-m)/V \ne \sqrt{c/V}-m$, and under the square root the correct divisor
would be $\sqrt{V}$ rather than $V$. Applying the correction downstream of either
transform produces an error of order $V$ itself.

Consequently:

- sky coefficients are divided by their own geometry factor before encoding;
- the target is divided by the science-pointing factor;
- network outputs are **intrinsic emissivities**, restored to observed
  amplitudes by multiplying by $V(z_{sci};h)\cdot 10^{-0.4k(X_{sci}-1)}$ inside
  `predict_sci_coefficients_default`;
- the `vanrhijn_*` context columns are therefore redundant as network inputs and
  are excluded from the encoders (`ctx_keep_idx`). `alt` and `airmass` are kept,
  because the moon and continuum groups still need them.

Because the reference point is $X=1$ rather than $X=0$, the recovered
"emissivity" is a **zenith-equivalent amplitude**, still carrying one airmass of
extinction. This is self-consistent and cancels in the sky-to-science transfer,
but the quantity is not a physical layer emissivity and should not be
interpreted as one.

`assert_context_is_physical()` guards every call. It checks physical ranges and
cross-checks any `vanrhijn_*` column against van Rhijn recomputed from `alt`.
Those two are computed independently upstream, so disagreement means the context
was transformed on the way in — the failure mode being guarded against is
evaluating geometry on scaler output, where an altitude of $\sim 0.5$ becomes a
zenith angle of $\sim 89.5^\circ$ and every row silently receives a near-horizon
factor of $\sim 6$.

### 4.3 Layer heights

Assigned per group via `GROUP_HEIGHT_FEATURE`: 87 km for `mesospheric`
(OH Meinel bands + O$_2$ atmospheric band) and `continuum` (HO$_2$ / FeO /
O$_2$ Chamberlain bands — mesopause chemiluminescence, not aerosol), 95 km
for `atomic` (K, Na, N I, OI 5577 mesopause metal / metastable), 285 km for
`ionospheric` (OI 6300/6364 red doublet and OI 7774/8446 F-region
recombination). The `moon` and `other` groups receive a factor of exactly
1.0 — scattered moonlight is not thin-shell emission and requires
scattering geometry, not van Rhijn. The `continuum` group name is retained
from an earlier grouping in which HO$_2$ / FeO / O$_2$Ac were treated as
aerosol continuum; they are actually airglow, but they are kept in a
separate coefficient group from `mesospheric` because they use a broadband
basis rather than the OH line-group basis and their compression pipeline
stays sqrt-identity (§5.5).

### 4.4 Per-coefficient wavelengths

Extinction varies strongly across the LVM range, so a single wavelength per
group is not adequate: the `mesospheric` group alone spans OI 5577, Na D and the
OH/O$_2$ bands, and even within the OH Meinel system $k$ varies by nearly a
factor of two between the reddest and bluest bands. `resolve_coef_wavelengths_a()`
therefore assigns a wavelength per coefficient, from, in order of precedence:

1. **basis centroid** — `coef_wavelengths_from_basis()` exploits linearity of the
   decomposition: reconstructing with $\mathbf{c}=\mathbf{e}_j$ isolates basis
   component $j$, whose intensity-weighted centroid is
   $\lambda_{eff}(j)=\sum\lambda|f_j| / \sum|f_j|$. This reads the actual basis,
   which is built from the same PMD population model, LSF and wavelength grid
   the decomposition uses at fit time, so the mesopause rotational Boltzmann
   factor for OH, the exact vibrational populations, and the blend structure
   across overlapping bands are all baked in by construction -- it is the
   definitionally-correct weighting. The same reconstruction pass returns a
   B$^2$-weighted effective extinction $\langle k(\lambda)\rangle_{f_j^2}$ (§4.5),
   so the population-model auxiliary route used until 2026-08-04 is no longer
   needed (§11.5, resolved). Cached to `coef_wavelengths_basis_v2.npz`;
   one reconstruction call per coefficient (~2 s each, so ~15 min cold on
   442 coefficients; ~0 s warm).
2. **coefficient name** — explicit wavelength baked into `COEF_SCHEMA` for
   named species (K I 7699, N I 5199, Na D, OI 5577, OI 6300, OI 7774, OI 8446,
   O$_2$ b-band). Used only for the handful of coefficients whose basis
   support is empty on the current wavelength grid.
3. **group default** — `GROUP_EFFECTIVE_WAVELENGTH_A`, last resort.

Provenance is recorded per coefficient and printed for the airglow groups,
because a silently wrong wavelength becomes a silently wrong extinction
correction.

### 4.5 Effective extinction

$k(\lambda)$ enters as $10^{-0.4k(X-1)}$. Three facts govern how it should be
chosen, and the implementation follows from them.

**Only the ratio matters, so the absolute value barely does.** The
sky-to-science transfer is

$$
R = \frac{V(z_{sci})}{V(z_{sky})}\;10^{-0.4\,k\,(X_{sci}-X_{sky})},
\qquad
\frac{\partial \ln R}{\partial k} = -0.4\ln 10\;\Delta X \approx -0.921\,\Delta X .
$$

The absolute $k$ cancels; only $k$ times the airmass *difference* survives. The
three pointings are simultaneous and share one atmosphere, so a per-night error
$\delta k$ propagates as $0.921\,\delta k\,\Delta X$. With realistic aerosol
variability ($\delta k \sim 0.02$–$0.04$ mag/airmass) and $\Delta X \sim 0.2$
that is a 0.2–0.7% transfer error, an order of magnitude below the gravity-wave
floor. **Per-observation extinction is therefore not modelled, and does not need
to be.** The spectrophotometric telescope would be better spent on the moon and
continuum terms, which depend on aerosol optical depth directly rather than
through $\Delta X$.

**A stellar curve is the wrong curve for an extended source.** This is the part
that does matter, because it is a coherent bias rather than random scatter. For
a star, scattered photons are lost. Airglow is a quasi-uniform source filling the
sky, so photons scattered out of the beam are largely replaced by photons
scattered in from adjacent lines of sight, and the effective attenuation is well
below the single-scattering value. Scattering dominates the total everywhere
airglow matters — Rayleigh plus aerosol is roughly 70–100% of $k$ from 4000 Å to
1 µm at LCO's 2380 m — so a stellar curve over-corrects, most severely in the blue.

**$h$ and $k$ are nearly degenerate.** Over the zenith range LVM observes,
$\ln V(z;h)$ and $X-1$ are collinear to $\rho > 0.995$ for $z \le 60^\circ$. A
height error can masquerade as an extinction error and vice versa. What the data
constrain, and all the model needs, is the product $V\cdot 10^{-0.4k(X-1)}$.

#### The implementation: fit $k_{\rm eff}$, do not model it

`fit_effective_extinction()` performs a Bouguer fit that uses airglow as its own
source. For coefficient $j$ and a simultaneous sky pair,

$$
\ln\!\frac{A_{near}}{A_{far}} - \ln\!\frac{V_{near}}{V_{far}}
= -0.4\ln 10\;k_{\rm eff}\,(X_{near}-X_{far}) + \varepsilon ,
$$

where $\varepsilon$ is the gravity-wave fluctuation between the two lines of
sight — zero mean in the log and uncorrelated with the airmass difference. The
intrinsic emissivity cancels because both arms are the same coefficient at the
same instant. The recovered $k_{\rm eff}$ absorbs the multiple-scattering
correction, the airglow-versus-stellar difference, the site aerosol level and any
residual error in the assumed layer height, none of which has to be modelled
explicitly.

Four details make the estimator trustworthy:

- **Layer height is held fixed.** Because of the degeneracy above, only
  $k_{\rm eff}$ is fitted. Fitting both would be ill-conditioned.
- **Selection is at column level only.** A per-row amplitude threshold is
  selection on the outcome: whichever arm sits at lower airmass has the smaller
  van Rhijn factor and fails the cut unless it carries a positive fluctuation,
  which correlates the retained residual with the sign of $\Delta X$. In testing,
  a 40th-percentile per-row cut inflated the recovered $k_{\rm eff}$ by a factor
  of 1.8. Coefficients are instead kept or dropped whole, via
  `min_positive_fraction`, and `retained_frac` is reported so residual row-level
  loss is visible.
- **Errors are cluster-robust, clustered on rows.** All coefficients in a row
  share one gravity-wave fluctuation, so naive OLS errors are far too small.
- **Stability is reported.** Each bin carries split-half values, and the fitted
  intercept — which should be zero. A significantly nonzero intercept indicates a
  relative throughput offset between the two sky channels rather than anything
  atmospheric.

`resolve_coef_extinction_k()` then interpolates the fitted values in wavelength
over well-constrained bins, falls back to the generic curve elsewhere, and clips
into $[0, k_{\rm generic}]$: the multiple-scattering argument makes the stellar
curve an upper bound, so a fitted value above it signals noise or an unmodelled
gradient, not physics. The resulting per-coefficient array is stored on the
dataset, threaded into `airglow_geometry_scale`, and saved in the training
artifacts so that prediction uses exactly the values training used. Setting
`USE_FITTED_EXTINCTION = False` in the wavelength cell reverts to the generic
curve.

On synthetic pairs with an injected $k_{\rm eff}$ at 40% of the stellar curve,
6000 rows and 12% gravity-wave scatter, the fit recovers a ratio of 0.41 with an
rms pull of 1.4.

**Cost.** The fit is fully vectorised. Because the airmass difference is
constant within a row, the design statistics and the whole cluster-robust meat
matrix reduce to per-row counts and per-row sums, so there is no flattened
$(N \times P)$ array and no per-cluster loop; van Rhijn is evaluated once per
distinct layer height as an $(N,)$ vector rather than as an $(N \times P)$
matrix. Only the robust scale for sigma clipping touches individual elements,
and it is estimated on a row subsample (`clip_sample_rows`). Measured against
the previous implementation this is 8–10x faster and about 9x smaller in peak
allocation, with identical coefficients and errors to floating-point precision.
At 50 000 rows and 419 coefficients the fit takes about 3 s.

**Known confounder.** A systematic horizontal gradient in layer brightness that
correlates with elevation angle at a fixed site would bias $k_{\rm eff}$, since
the estimator cannot distinguish it from extinction. The split-half columns catch
instability but not a persistent gradient. Checking the fit against azimuth, and
across seasons, would test for it.

## 5. Target and feature transforms

Coefficients enter model space via a non-negative square root,
$\mathbf{z}=\sqrt{\max(\mathbf{c},0)}$, applied *after* geometry normalisation.
If amplitudes track photon counts this is variance-stabilising, so squared error
in this space approximates $\chi^2$, and it degrades more gracefully than a log
near zero. The clip at zero does discard the sign of negative coefficients.

A robust (median/IQR) scaler is fit on **training rows only** and applied to
near, far and science coefficients and contexts, with values clipped to $\pm 25$.

## 5.5 Per-group compression (the model this notebook now trains)

The diagnostic of §9.1 said the coefficient space is heavily redundant. On
this dataset it identifies 79 transferable degrees of freedom in the 442
coefficient columns — a 5.6× reduction on the target and, because the
encoder consumes the same per-arm scores, on the encoder input as well.
The default model of the notebook is trained in this compressed
representation. The uncompressed model of §6 is kept only as the parent
class of `DualEncoderGroupHeadMLPCompressed`; its architecture, geometry
handling, splitting and optimiser are reused verbatim, and it is no longer
trained.

The compression is per group, and is not the same operation for every group.
For each group $g$:

1. **Geometry removal.** Coefficients are divided by the airglow geometry
   factor of §4, in physical units, giving zenith-equivalent amplitudes.
2. **Space transform.** The group is mapped into the space where its physical
   variability is linear:
   - `mesospheric` — asinh (multiplicative variation from rotational
     temperature and vibrational populations); since 2026-08-11 also
     "identity" (no PCA rotation, no xarm truncation), because the
     rank-truncated projection kept ML below the B1_near_geo naive
     baseline on OH (see §11.1 and §12),
   - `moon` — sqrt (updated 2026-08-19b from `linear`, briefly `asinh`);
     the only group still using PCA rotation. As of the 2026-08-19b
     tune the default `COMPRESSION_XARM_THRESHOLD = 0.10` retains all 29
     PCs. The intermediate `asinh` variant crushed extreme-blue precision
     (encoder-free compressor round-trip rms $\approx 8.3$ at 3600 Å vs
     $0.005$ under `sqrt`), and the two moon PCs at $|r_{xarm}| \approx 0.20$
     and $0.53$ that the old 0.55 threshold dropped turned out to carry
     the entire extreme-red representation (round-trip rms at 9500-9800 Å
     jumped from $0.06$ to $7.3$ when dropped). Full derivation and the
     A/B numbers are in §12 (2026-08-19b).
   - `atomic` / `continuum` / `ionospheric` / `other` — sqrt "identity"
     (variance-stabilising, no rotation, no truncation).
3. **Column standardisation.** Subtract per-coefficient mean, divide by
   per-coefficient sd, both fit on training rows only, pooled over
   near/far/sci. This is the correlation-matrix parameterisation for the PCA
   that follows.
4. **PCA rotation.** Eigendecomposition of the sample covariance in the
   standardised space, ordered by variance. Rotation only; no truncation at
   this step. As of 2026-08-11 applied only to `moon` (mesospheric moved
   to asinh-identity).
5. **Cross-arm selection.** Retain component $c$ iff $r_{xarm}(c) >$
   `COMPRESSION_XARM_THRESHOLD`$= 0.10$ on held-out nights, where $r_{xarm}$
   is the Pearson correlation of the near-arm score with the far-arm score
   for that component. On this dataset the threshold sits roughly $30$ times
   above the correlation null. Held-out is validation-set only (not
   val + test), so no test-set structure shapes which components survive.
6. **Blend + residual head, per group.** (Cross-arm selection in step 5 now
applies only to `moon`.) Each group head predicts a signed
   residual $\Delta_g$ on top of an explicit learnable convex blend of the
   two sky-arm scores in score space,
   $\hat{\mathbf{s}}_g = \alpha_g\,\mathbf{s}_{near,g} + (1-\alpha_g)\,\mathbf{s}_{far,g} + \Delta_g(\mathbf{h}),$
   with $\alpha_g \in [\varepsilon, 1-\varepsilon]$ a per-group learnable weight
   stored directly (no sigmoid) and projected back into the open interval
   $[\varepsilon, 1-\varepsilon]$ after each optimizer step; initialised at
   $\alpha_g = 0.7$ (near-arm biased). This is the `blend_optim='direct'`
   parametrization adopted 2026-08-11 (§12); the previous $\alpha_g = \sigma(\ell_g)$
   route saturated the blend logit gradient too aggressively and left the joint
   $(\alpha, \Delta_g)$ optimum too far from the init in a 50-epoch budget. The
   loss-minimising point of
   $\Delta_g = 0$ is the interpolation baseline rather than the training
   mean, which the earlier "predict $\hat{\mathbf{s}}_g$ directly" head was
   collapsing to on bright-moon / near-adjacent rows where the science
   pointing sat within a few degrees of the near arm but the model still
   over-predicted the near-arm continuum by roughly a factor of two. The
   failure mode is structural: the encoder fusion
   $[\mathbf{e}_{mean};\mathbf{e}_{diff};|\mathbf{e}_{diff}|;\mathbf{e}_{ctx}]$
   is symmetric-with-sign under near/far exchange, the PCA-kept subspace
   sits closer to the mean than to either arm, and without a residual /
   skip path the model has nowhere to write "output the near arm
   verbatim." Making the blend explicit fixes it. The learned per-group
   $\alpha_g$ is printed at the best epoch of every training run and captured
   per epoch in `artifacts['blend_history']` for convergence diagnostics.
   Non-negativity is restored on the reconstruction path when scores are
   inverse-projected back to physical emissivity and clipped at zero (the
   sqrt-identity groups invert through square and are non-negative by
   construction).

**Selection by correlation, not by variance rank.** PCA orders by variance;
transferability is a different ordering. A shared mode can sit behind an
arm-private one, so truncating at "the first $k$" throws away exactly the
directions that *can* be predicted at a third pointing while keeping the
ones that cannot. Numerically, on this dataset the retained mesospheric
components are not the leading block. The `xarm_keep` array in each fitted
compressor records which components were kept.

**Consequences for the rest of the pipeline.**

- **Loss balance (§7).** A per-group multiplier $m_g$ is exposed on top
  of $1/\sqrt{n_g}$. As of the 2026-08-16 flat-weight retune (§12), the
  shipped default is $m_g = 1$ for every group; the earlier per-group lifts
  ($m_{\rm moon}=3.5$, $m_{\rm continuum}=1.5$, $m_{\rm mesospheric}=3.0$,
  $m_{\rm ionospheric}=1.0$) were tuned against the pre-2026-08-16 unweighted
  SmoothL1 loss and overfit the new heteroscedastic-Gaussian surface. The
  per-group empirical mean-bias calibration below (also generalised on
  2026-08-16) now handles residual group-scale bias without needing a
  loss-side tilt. The multipliers stay exposed on the trainer for future
  A/B sweeps.
- **Smoothness prior.** The moon spline's PCA truncation is itself a
  smoothness prior. `moon_smooth_lambda` is retired.
- **Empirical per-group mean-bias calibration.** Applied to every group
  (widened on 2026-08-16 from the earlier `asinh` / `log`-only gate; the
  gate had left linear/`sqrt` compressor groups -- `moon`, `continuum`,
  `ionospheric`, `atomic` -- with the raw SmoothL1 median-bias, producing
  a $-1.77\%$ ensemble-mean bias on `continuum` that showed up in the
  per-group bias diagnostic). `train_compressed_group_mlp` computes
  naive physical-space predictions on the train $\cup$ val union
  (widened on 2026-08-16 from val only so small groups like `continuum`
  and `atomic` at $n = 3$ have a stable estimator with $\sim\!85\%$ of
  the rows rather than $\sim\!15\%$), then per group measures
  $\text{lift}_g = \overline{y_{true,g}} / \overline{y_{pred,g}^{naive}}$
  and stores it as a uniform per-coefficient scalar in
  `jensen_corrections[g]`. `inverse_group_compressor` multiplies its
  output by that scalar for every group (the mirrored
  `comp['kind'] in ('asinh', 'log')` filter inside the consumer was
  removed on 2026-08-16 so the lift actually reaches linear/`sqrt`
  groups). Value clipped to $[0.5, 2.0]$ as a broken-group guard, plus a
  degeneracy filter that skips groups whose $|\overline{y_{pred,g}}|$ falls
  below $5\%$ of $|\overline{y_{true,g}}|$ or whose true and predicted
  means have opposite sign. Effect on the current 10-seed ensemble: every
  group's absolute mean-coefficient bias sits under $0.05\%$ (was
  $-1.77\%$ for continuum, $-0.61\%$ for mesospheric, $-0.42\%$ for moon
  under the pre-upgrade gated-and-val-only calibration). This replaces the
  earlier analytic Jensen factor $\exp(\sigma_j^2/2)$ from a robust MAD
  estimate capped at $\sigma_{cap} = 0.4$ (2026-08-08 to 2026-08-11); see
  §12 (2026-08-16 calibration generalisation) for the derivation.
- **Predictor is a 10-seed ensemble by default (raised from 4 on 2026-08-12).**
  `mlp_artifacts` contains N=10 members trained at seeds {42..51} with
  identical config, and `predict_sci_coefficients_default` averages the
  per-member physical-space predictions when the ``is_ensemble`` flag is
  set. Ensemble stderr on test `mean_rmse` scales as
  $\sigma_{\text{seed}}/\sqrt{N}$ ($\sigma_{\text{seed}} \approx 0.43$
  from the current 10-seed run, so ~0.14 on test `mean_rmse` $\approx 24.1$,
  i.e. ~$0.6\%$). N was raised from 4 to 10 on 2026-08-12 after the
  item-6 predictive-σ diagnostic (§11.7) showed the 4-seed spread
  under-estimates true error by $\sim 2.5\times$; N=10 tightens both the
  mean-bias floor of §11.1 and the σ-scale estimate of item 6 without
  fixing the under-dispersion factor itself (a proper Gaussian-NLL head
  does; §11.7 remains partially open). Cost is 10× training time (once,
  on the deployment host) and 10× inference time (per call, still sub-
  second on this dataset).
- **Non-negativity.** Softplus in coefficient space is replaced by clipping
  in physical space after the inverse compressor. Small negatives from
  numerical noise do not reach the science pipeline.
- **Out-of-distribution flag.** Reconstruction residual in the fitted basis
  can be reported per row as an OOD indicator for nights that fall outside
  the training span.

**Empirical result (night-held-out test, 1386 rows).** The compressed model
beats the uncompressed baseline on mean RMSE by ~33%, median RMSE by ~26%,
MAE by ~34%, and median per-coefficient Pearson correlation by 0.09 (0.80 →
0.89). Mean per-coefficient correlation drops by 0.025, which is the
expected penalty on coefficients whose truth has weak but nonzero signal in
the discarded subspace and by design collapses to a constant after inverse
projection. Spectrum-space per-row RMSE (evaluated in the batch-RMSE cell)
tracks the coefficient-space improvement.

**Empirical result post-2026-08-16 (10-seed ensemble, night-held-out split, 1173 test rows).**
Test `mean_rmse` = 24.12 (seed-to-seed std $= 0.43$, ensemble stderr
$= 0.14$, i.e. $\sim 0.6\%$ of the aggregate) with `mean_wrmse`
$\approx 17.5$ (per-column weighted RMSE using decomposition `COEF_ERR`
with the per-group sigma floor of §7). Per-group ensemble mean-coefficient
bias, on all 11 624 filtered rows: `moon` $-0.05\%$, `continuum`
$-0.02\%$, `mesospheric` $+0.02\%$, `ionospheric` $+0.03\%$, `atomic`
$+0.02\%$ -- max $|{\rm bias}| = 0.05\%$, an order of magnitude below
the pre-2026-08-16 residual (continuum $-1.77\%$, mesospheric $-0.61\%$,
moon $-0.42\%$). The bias-diagnostic verdict flipped from "investigate
calibration / loss balance" to "N=10 seeds are sufficient for the
group-level unbiasedness §11.1 requires". Comparison against the
strongest naive baseline $B_1 = \mathrm{em}_{near} \cdot G_{sci}$ (§11.1)
is preserved on the same split and remains a clear ML win; the biggest
single change enabling the new bias floor was the calibration
generalisation (train+val, all groups) plus the ungating of the lift
consumer in `inverse_group_compressor`; see §12 (2026-08-16 calibration
generalisation).

## 6. Network architecture

**Terminology.** Three overlapping words that recur throughout §5.5 and
§6–§7:

- **MLP module** — a stack of `Linear -> LayerNorm -> GELU` blocks, built
  by `DualEncoderGroupHeadMLP._make_mlp`. Three of them appear inside
  the network: `sky_encoder`, `ctx_encoder`, `trunk`. The name of the
  class itself, `DualEncoderGroupHeadMLP`, refers to the *architecture as
  a whole* being a multi-layer perceptron; internally it is composed of
  these MLP modules plus the per-group heads.
- **GELU** — Gaussian Error Linear Unit activation,
  $\mathrm{GELU}(x) = x\,\Phi(x)$ where $\Phi$ is the standard normal
  CDF. Smooth alternative to ReLU, used as the non-linearity in every
  MLP module and every head.
- **Trunk** — the specific MLP module that maps the fusion vector
  $\mathbf{z}_{fuse}$ to a shared intermediate representation
  $\mathbf{h}$. Every head sees the same $\mathbf{h}$.
- **Head** — a small per-group *output* subnetwork
  (`Linear -> GELU -> Linear`), one per coefficient group. Each head
  consumes $\mathbf{h}$ and emits that group's prediction (for the
  compressed model of §5.5, only the residual
  $\Delta_g(\mathbf{h})$; the blend baseline is added outside the head).
  Heads do *not* share weights with each other, unlike the sky encoder,
  which is shared between the two arms.

Every phrase below that talks about "the heads emit ...", "the head loss",
"$w_g$", "per-group $\alpha_g$", or "score-space output" means this
per-group Linear-GELU-Linear block, not the whole network.

### 6.1 Symmetric dual encoder

A shared MLP $E_{sky}$ is applied to each sky arm, concatenating that arm's
geometry-normalised coefficients with that arm's own context:

$$
\mathbf{e}_{near}=E_{sky}([\mathbf{z}_{near};\mathbf{x}_{ctx,near}]),\qquad
\mathbf{e}_{far}=E_{sky}([\mathbf{z}_{far};\mathbf{x}_{ctx,far}]).
$$

Weight sharing forces both channels onto a common representation manifold.

### 6.2 Fusion

$$
\mathbf{z}_{fuse} = [\,\mathbf{e}_{mean};\ \mathbf{e}_{diff};\ |\mathbf{e}_{diff}|;\ E_{ctx}(\mathbf{x}_{ctx,sci})\,],
\qquad
\mathbf{e}_{mean}=\tfrac12(\mathbf{e}_{near}+\mathbf{e}_{far}),\quad
\mathbf{e}_{diff}=\mathbf{e}_{near}-\mathbf{e}_{far}.
$$

Note that $\mathbf{e}_{diff}$ is antisymmetric under exchange of the two arms, so
the model is *not* invariant to swapping them. This is intentional given that
near/far is defined by separation, but it forecloses swap augmentation and does
not generalise to more than two reference measurements.

### 6.2b Context features reaching the encoders

`drop_vanrhijn_from_context` (default **True**) controls whether the three
`vanrhijn_87km`, `vanrhijn_95km`, `vanrhijn_285km` columns are hidden from the
encoders. Every airglow group -- `mesospheric`, `atomic`, `ionospheric`, and
(as of 2026-07-30) `continuum` -- has its geometry factor removed analytically
in the loader before anything reaches the encoders, so those columns are
strictly redundant for the coefficient directions they were introduced to
help. Only `moon` still receives an `airglow_geometry_scale` factor of exactly
$1$ and relies on context for its altitude dependence; `alt` and `airmass`
remain in the context in both flag settings and carry that signal, so hiding
`vanrhijn_*` does not disconnect the moon head from altitude. Empirically, at
seed=42, same split, identical hyperparameters, hiding the three columns is
indistinguishable from keeping them: test `mean_rmse` and `median_corr` both
move by less than their single-seed noise, and per-group biases stay sub-0.5%.
That is what should happen once every airglow term is geometry-corrected
analytically -- the KEEP variant's earlier edge came from the encoder using
`vanrhijn_87km` as an ad-hoc van Rhijn correction for HO$_2$ / FeO / O$_2$Ac,
and closing that hole (§4.3) removes the incentive. The three columns are
also pairwise Pearson-correlated at $> 0.998$ over the LVM zenith range, so
they carry $\sim$one altitude signal rather than three per-height signals;
dropping them removes three near-collinear encoder inputs at no measurable
cost. Set the flag to `False` to A/B it.

### 6.3 Trunk and group heads

The shared trunk maps $\mathbf{z}_{fuse}$ to an intermediate representation
$\mathbf{h}$; every per-group head consumes the SAME $\mathbf{h}$ and emits
that group's own output. Concretely a head is a small MLP
$\mathrm{Linear}(d_{trunk} \to d_{head})\to\mathrm{GELU}\to
\mathrm{Linear}(d_{head} \to n_g^{out})$ with $n_g^{out}$ the coefficient count
of group $g$ (parent) or its retained score count (compressed default of
§5.5). Heads do NOT share weights across groups; the sky encoder does
share weights across the two arms (§6.1).

The parent baseline (`DualEncoderGroupHeadMLP`) finishes each head with a
$\mathrm{Softplus}$, giving non-negative emissivity in model space. The
compressed default (`DualEncoderGroupHeadMLPCompressed`) drops that
$\mathrm{Softplus}$ and instead emits a SIGNED residual $\Delta_g$ added to an
explicit per-group learnable convex blend of the two sky-arm scores in score
space,
$\hat{\mathbf{s}}_g = \alpha_g\,\mathbf{s}_{near,g}
+ (1 - \alpha_g)\,\mathbf{s}_{far,g} + \Delta_g(\mathbf{h})$,
with $\alpha_g \in [\varepsilon, 1-\varepsilon]$ a per-group learnable weight
stored directly and clamped in place after each optimizer step, initialised at
$\alpha_g = 0.7$ (near-arm biased; `blend_optim='direct'`, `blend_init_alpha=0.7`).
Non-negativity is restored later, once
scores have been inverse-projected back to physical emissivity, by clipping at
zero (§5.5). The rationale for the blend is documented at length in
§5.5.

At the current default config (adopted 2026-08-11 from a multi-seed sweep)
the shapes are:

- $d_{enc\_out} = 384$ (last layer of `encoder_dims=(768, 384)`), so
  $\mathbf{e}_{near}, \mathbf{e}_{far} \in \mathbb{R}^{B \times 384}$.
- $d_{ctx\_out} = 64$ (`ctx_dims=(64,)`), so
  $\mathbf{e}_{ctx} \in \mathbb{R}^{B \times 64}$.
- $\mathbf{z}_{fuse} = [\mathbf{e}_{mean}; \mathbf{e}_{diff};
  |\mathbf{e}_{diff}|; \mathbf{e}_{ctx}] \in
  \mathbb{R}^{B \times (3 \cdot 384 + 64)} = \mathbb{R}^{B \times 1216}$.
- $d_{trunk} = 160$ (last layer of `trunk_dims=(320, 160)`), so
  $\mathbf{h} \in \mathbb{R}^{B \times 160}$.
- $d_{head} = 384$, so each group head is
  $\mathrm{Linear}(160 \to 384) \to \mathrm{GELU} \to
  \mathrm{Linear}(384 \to n_g^{score})$.
- $n_g^{score}$: `moon` = 28 (PCA + xarm on 29 spline coefficients); all
  other groups are identity so $n_g^{score}$ = raw coefficient count
  (`continuum` 3, `mesospheric` 403, `ionospheric` 4, `atomic` 3). Total
  compressed input width is $S = 441$. `train_compressed_group_mlp` prints
  these at fit time.

Grouping is defined once, in the loaders cell (`COEF_SCHEMA`), and is shared
by the wavelength resolver, the structure-function diagnostics, the per-group
compressor and the model.

### 6.4 Data flow through the compressed model

At prediction time (see `predict_sci_coefficients_default`) the pipeline from
`filtered_triplet` rows to a predicted science coefficient vector is:

1. **Load** physical coefficients and context per row:
   $\mathbf{c}_{near}, \mathbf{c}_{far} \in \mathbb{R}^{B \times P}$ and
   $\mathbf{x}_{near}, \mathbf{x}_{far}, \mathbf{x}_{sci} \in
   \mathbb{R}^{B \times Q}$ (currently $P = 442$, $Q = 31$; the three ecliptic features `ecl_beta_deg`, `ecl_lon_sin`, `ecl_lon_cos` were added 2026-08-19).
2. **Remove geometry per arm** in physical units (`airglow_geometry_scale`,
   §4). Compute
   $G_{arm}[n, j] = V(z_{arm,n}; h_g) \cdot
   10^{-0.4 k_{eff, j} (X_{arm,n} - 1)}$
   per row and coefficient, with $G = 1$ for the `moon` group and $G$ as
   defined in §4 for every airglow group -- including `continuum` at
   $87$ km. Divide: $\mathbf{em}_{arm} = \mathbf{c}_{arm} / G_{arm}$.
3. **Compress per group** (`compress_coefs_to_scores`, §5.5). For each
   group $g$, $\mathbf{em}_{arm,g}$ passes through the group's forward
   transform (asinh for `mesospheric`, sqrt for `moon` and the other
   groups — moon was `linear` before 2026-08-19b, briefly `asinh`; see §12),
   column standardisation with the group's training-set mean / sd, and
   (`moon` / `mesospheric` only) rotation into the group's PCA basis. Retain
   the `xarm_keep_g` components picked by cross-arm correlation on the
   validation split. Concatenate over groups:
   $\mathbf{s}_{arm} \in \mathbb{R}^{B \times S}$,
   $S = \sum_g |\text{xarm\_keep}_g|$.
4. **Robust-scale and clip** the concatenated $\mathbf{s}_{arm}$ and the
   three context matrices with median / IQR scalers fit on training rows
   only, clipping to $\pm 25$. Drop the three `vanrhijn_*` columns from what
   reaches the encoders (`ctx_keep_idx`, §6.2b) so
   $\mathbf{x}_{arm}^{enc} \in \mathbb{R}^{B \times (Q - 3)}$ (currently $31 - 3 = 28$; ecliptic features are kept).
5. **Encode each arm** with the shared sky encoder:
   $\mathbf{e}_{arm} = E_{sky}\!\bigl([\mathbf{s}_{arm};
   \mathbf{x}_{arm}^{enc}]\bigr) \in \mathbb{R}^{B \times 224}$.
6. **Fuse**:
   $\mathbf{e}_{mean} = \tfrac12(\mathbf{e}_{near} + \mathbf{e}_{far})$,
   $\mathbf{e}_{diff} = \mathbf{e}_{near} - \mathbf{e}_{far}$,
   $|\mathbf{e}_{diff}|$, and
   $\mathbf{e}_{ctx} = E_{ctx}(\mathbf{x}_{sci}^{enc}) \in
   \mathbb{R}^{B \times 64}$. Concatenate to
   $\mathbf{z}_{fuse} \in \mathbb{R}^{B \times 736}$.
7. **Trunk**: $\mathbf{h} = \mathrm{trunk}(\mathbf{z}_{fuse}) \in
   \mathbb{R}^{B \times 160}$.
8. **Per-group blend + residual head**: for each group $g$,
   $\Delta_g = \mathrm{head}_g(\mathbf{h}) \in
   \mathbb{R}^{B \times n_g^{score}}$ and
   $\alpha_g \in [\varepsilon, 1-\varepsilon]$ a direct learnable weight
   (`blend_use_direct=True`, projected clamp after each optimizer step); the
   group's scaled-score prediction is
   $\hat{\mathbf{s}}_g = \alpha_g\,\mathbf{s}_{near,g}
   + (1 - \alpha_g)\,\mathbf{s}_{far,g} + \Delta_g$.
   For $g={\rm moon}$ under `moon_alt_conditional_alpha=True` (adopted
   2026-08-12; see §12), $\alpha_{\rm moon}$ is a per-row two-value lookup
   indexed by $\mathrm{sign}(\mathrm{moon\_alt})$ at the science pointing:
   $\alpha_{\rm moon,dn} \approx 0.66$ when the moon is below the horizon
   and $\alpha_{\rm moon,up} \approx 0.75$ when it is above (all four
   ensemble seeds agree; §12, 2026-08-12). All other groups keep a single
   scalar $\alpha_g$.
9. **Inverse robust-scale** the concatenated $\hat{\mathbf{s}}$ back to raw
   score space with the fitted score scaler.
10. **Inverse compressor per group** (`inverse_group_compressor`, §5.5):
    place the retained scores back into their PCA slots, invert the PCA
    rotation (for `moon` / `mesospheric`), for `asinh` / `log` groups clip
    the forward-space value to
    $[y_{train,min} - 0.5,\, y_{train,max} + 0.5]$ to cap $\sinh$ / $\exp$
    blow-up on tail extrapolation, then invert the forward transform
    (square / $\sinh \cdot \text{scale}$ / identity) to intrinsic emissivity
    $\hat{\mathbf{em}} \in \mathbb{R}^{B \times P}$.
11. **Reapply geometry at the science pointing**: compute $G_{sci}$ from
    $\mathbf{x}_{sci}$ the same way as step 2, then
    $\hat{\mathbf{c}}_{sci} = \min\!\big(U_g,\ \max(0,\;\hat{\mathbf{em}} \cdot G_{sci})\big)$.
    The $\max(0, \cdot)$ replaces the parent-baseline Softplus non-negativity
    (which is why the score-space heads can stay linear); the upper cap
    $U_g = 3 \cdot \max_g(\mathbf{c}_{sci,\mathrm{train}})$ per group per coefficient
    (adopted 2026-08-12, item 11 of the closing review; see §12) is a
    defensive runaway guard against asinh-inverse blowups on out-of-envelope
    trunk outputs. On the current dataset the cap fires 4 times out of 5.1M
    coefficient points, all on the sparse-tail mesospheric OH lines with
    near-zero training envelopes; no legitimate prediction reaches it.

At TRAINING time the loss is evaluated at the output of step 8 (before
inverse-scale + inverse-compressor + geometry re-application) against the
compressed target $\mathbf{s}_{sci}$ produced by running steps 1-4 on
`coef_sci` / `ctx_sci`. Steps 9-11 are exercised only at prediction time,
so the loss lives in a space where a fixed per-coefficient error costs the
same at every zenith angle, as designed in §7.

## 7. Loss

**Heteroscedastic-Gaussian NLL on compressed scores (default since 2026-08-16).**
Every science coefficient carries a decomposition 1σ uncertainty stored
in the ``COEF_ERR`` HDU (see the sky-decomposition method note in
``sky_decomp/fit.py``). Treating those as fixed measurement errors of the
target, the training NLL for one row is

$$
\mathcal{L}_i = \tfrac{1}{2}\sum_{j}\!\left[
   \frac{(\hat{c}^{(i)}_{sci,j}-c^{(i)}_{sci,j})^2}{\sigma^{(i)\,2}_{sci,j}}
   + \log \sigma^{(i)\,2}_{sci,j}
\right].
$$

The log-variance term is constant in the model parameters and drops out
of the gradient, so the per-row loss reduces to an inverse-variance
weighted quadratic. In the compressed score space in which the model
actually predicts, the weight for group $g$, per row $i$ and per score
index $j$, is

$$
w^{(i)}_{g,j} = \left(\tilde{\sigma}^{(i)}_{g,j}\right)^{-2},
\qquad
\tilde{\sigma}^{(i)}_{g,j} = \frac{\sigma^{(i)}_{scores,g,j}}
                                  {\mathrm{scale}_{g,j}},
$$

where $\sigma^{(i)}_{scores,g,j}$ is the first-order propagation of
$\sigma^{(i)}_{sci,j}$ through the fixed compressor pipeline
(``compress_coef_err_to_score_sigma`` in cell 17) and
$\mathrm{scale}_{g,j}$ is the RobustScaler column scale that maps raw
compressed scores to the trainer-facing target ``sci_s``. Coefficients
pinned at the ``c ≥ 0`` boundary in the decomposition carry no symmetric
error (NaN in ``COEF_ERR``); they fall through to a per-column floor
``coef_err_sigma_floor_rel[group] * median_finite_sigma`` so they
contribute a bounded, small weight rather than blowing up.  The floor
is per group because the sigma / weight diagnostic (2026-08-16) showed
the mesospheric and atomic groups have $p_{99}/p_{50}$ ratios of $10^3$–$10^6$
in compressed-score space, whereas moon, continuum and ionospheric are
well behaved.  Defaults: ``moon=continuum=ionospheric=0.05``,
``mesospheric=atomic=0.20``.  A scalar is also accepted and broadcast
to every group for A/B sweeps.

The weights $w^{(i)}_{g,j}$ are normalised so the training-set mean per
compressed-score column is 1; that keeps the overall loss scale
comparable to the pre-2026-08-16 unweighted variant and avoids
silently re-tuning learning-rate and early-stopping. Setting
``use_coef_err_weights=False`` (or supplying decomposition files without
a ``COEF_ERR`` HDU) makes ``w_pe`` an all-ones tensor and the loss
reduces exactly to the previous unweighted SmoothL1. To preserve the
robustness of SmoothL1 on the small fraction of rows where the
decomposition sigma is a bad match to the observed residual, the
per-element base loss remains SmoothL1 rather than pure quadratic:

$$
\mathcal{L} = \frac{1}{G}\sum_{g=1}^{G} w_g\,
\overline{ w^{(i)}_{g,\cdot}\odot
\mathrm{SmoothL1}(\hat{\mathbf{s}}_g,\mathbf{s}_g) },
\qquad w_g = m_g\,n_{g,\mathrm{score}}^{-1/2},
$$

where $m_g$ is a per-group multiplier and $\mathbf{s}_g$ is the retained
score vector for group $g$ (§5.5). As of the 2026-08-16 flat-weight retune,
$m_g = 1$ for every group -- the earlier per-group lifts
(historically $m_{\rm moon}=3$, then $3.5$, plus $m_{\rm continuum}=1.5$
and $m_{\rm mesospheric}=3$) were tuned against the pre-2026-08-16
unweighted SmoothL1 and overfit the new heteroscedastic-Gaussian surface.
The residual per-group mean bias that a moon lift was originally introduced
to fight ($\sim +4\%$ on moon under the flat unweighted loss) is now
handled by the empirical mean-bias calibration of §5.5 (train+val fit,
applied to every group as of 2026-08-16), which drives every group's
ensemble bias under $0.05\%$ without touching the loss balance. The
multipliers stay exposed on the training driver as `moon_group_weight`,
`continuum_group_weight`, `mesospheric_group_weight`,
`ionospheric_group_weight`, and are recorded in the training artifacts so
a saved model records the loss balance it was trained under. On this
dataset the per-group score dimensions after PCA + cross-arm selection
sit around `moon` $\approx 20{-}30$ (PCA + xarm truncation) and
`mesospheric` $\approx 30{-}40$ (PCA + xarm truncation); `atomic`,
`continuum`, `ionospheric` are `sqrt`-identity compressors with no PCA
and no truncation, so their score dim equals the raw coefficient count
(3, 3, 4 respectively at the current schema). The exact per-group weights
$w_g = m_g\,n_{g,\mathrm{score}}^{-1/2}$ are printed by
`train_compressed_group_mlp` at fit time. The $1/\sqrt{n_g}$ term still
balances the raw group sizes; $m_g$ is available if a future dataset
reintroduces a persistent under-fit signature on a specific group.
`moon_smooth_lambda` remains retired: it is subsumed by the PCA truncation
on the moon spline (§5.5).
Only components with $r_{xarm} >$ `COMPRESSION_XARM_THRESHOLD`$= 0.10$
Only components with $r_{xarm} >$ `COMPRESSION_XARM_THRESHOLD`$= 0.55$
receive a head output at all, so directions the sky arms cannot transfer to
the science pointing contribute nothing to the loss and are reconstructed as
unpredictable structure. At the current threshold all 29 moon PCs pass
(0.10 is well below the observed minimum $|r_{xarm}| \approx 0.20$); the
tighter historical thresholds of 0.55 and 0.70 dropped PCs 4 and 6, which
carry the extreme-red representation (§12, 2026-08-19b).

**Per-row weighting.** The trainer exposes a per-row weight $w_i$ via
`high_airmass_boost` on the training driver; the shipped default was
$w_i = 1.25$ for $\mathrm{airmass} > 1.5$ from 2026-08-12 until the
2026-08-16 flat-weight retune reverted it to $w_i = 1$ (the earlier
boost was tuned against the unweighted SmoothL1 loss and overfit the
new heteroscedastic-Gaussian surface; §12). Bright-moon and high-F10.7
boosts were tested in the same design and rejected because their renorm
implicitly downweighted the dark and low-airmass rows and regressed the
Q1 mesospheric and moon-down moon biases (§12, 2026-08-12). The hook
stays in place for future dataset-specific tilts.

The loss acts on scores of zenith-equivalent emissivity, not on observed
amplitude, so a given error costs the same at all zenith angles. Per-group
unweighted validation losses are still tracked and printed at the best
epoch to separate "this head is under-trained" from "this head is being
traded away".

## 8. Optimisation and data splitting
validation loss, best-epoch checkpoint restoration.
AdamW, global-norm gradient clipping, patience-based early stopping on
validation loss, best-epoch checkpoint restoration.

### 8.1 Sample split

The split is **night-level and stratified by lunar phase**. Everything downstream
— per-group compressor, robust scalers, model training, sweeps, evaluation —
uses the same partition, computed once and stored on the triplet dictionary.
moon_phase, train_frac=0.8, val_frac=0.1, seed=42, n_bins=10)` does:
**How the split is computed.** `split_indices_by_moon_phase(obstime_mjd,
moon_phase, train_frac=0.8, val_frac=0.1, seed=42, n_bins=10)` does:

1. **Assign each row to a night** — `night_id = floor(mjd - 0.5)`. The $-0.5$
   shift centres the day boundary near solar noon at LCO, so evening twilight
   ($\sim 23$ UT) and morning twilight ($\sim 10$ UT the next calendar day)
   map to the same integer. All rows from one LCO observing night share one
   `night_id`.
2. **Attach one moon phase per night** — median `moon_phase` across the
   exposures of that night (moon phase is an ephemeris quantity, so the
   median just guards against outlier rows).
3. **Sort nights by moon phase** with a fixed-seed random tie-break, then cut
   into `n_bins=10` equal-count quantile bins across the phase range (dark
   → bright).
4. **Assign inside each bin** — within each phase bin, shuffle the nights and
   allocate `train_frac`, `val_frac`, and the remaining fraction to test,
   rounded to whole nights with a guarantee of at least one night per split
   per bin (when the bin has $\geq 3$ nights). Rounding across ten bins gives
   overall fractions close to 80/10/10.
5. **Collect row indices** — every row of a night goes together.

The three arrays are computed once at the top of the compressor cell and stored
as `filtered_triplet['compress_train_idx' | 'compress_val_idx' |
'compress_test_idx']`. The training driver reuses them via `split_indices=(...)`,
so the compressor is fit on exactly the rows the model is trained on. Cross-arm
correlation selection uses `val` only, so neither training-set nor test-set
structure shapes the compressor's retained subspace.

**Why by night, not by row.** Airglow within a single LCO night is autocorrelated
on $\sim 10$-minute timescales via gravity waves (§1). Consecutive exposures
share the same gravity-wave phase draw. A per-row random split would put
near-duplicate rows into train and test, so validation loss would underestimate
generalisation and test-set scores would overstate performance. Night-level
assignment enforces "predict on a night the model has never seen", which is the
correct decision problem at deployment.

**Why stratified by moon phase.** Val and test are small (~10% of nights each).
Under a plain random shuffle of nights, either sub-sample can miss a whole
region of the phase axis by chance — a val set with only dark nights would
under-report bright-time errors, and a test set biased toward bright nights
would overstate them. Moon phase drives the biggest single-context term in the
problem (moon brightness, moon-scattered continuum, sodium D contamination),
so having *every* held-out sub-sample span the full range from dark to full
moon uniformly is the cheapest way to make validation and test error
estimates representative of deployment.

**Filtering is upstream of the split.** LMC/SMC field exclusion, physical sanity,
$\chi^2$, hard-bound, $\kappa\sigma$, and thinning cuts all run before
`split_indices_by_moon_phase`, so the fractions apply to the post-filter row
count.

**What each split is used for.**
- **Train** — fits the per-coefficient `RobustScaler` for scores and context,
  fits the per-group compressor (per-coefficient robust scale from near-arm
  training rows, PCA basis pooled over near/far/sci training rows), and drives
  AdamW updates.
- **Val** — held out from all fits. Evaluated every epoch; the epoch with lowest
  val loss defines the best checkpoint and drives early stopping. Per-group
  unweighted val losses are also tracked for head-balance diagnostics.
- **Test** — held out from all fits *and* from best-epoch selection. Every
  reported test metric (`mean_rmse`, `median_rmse`, `mean_mae`, `median_corr`,
  etc.) is computed here.

## 9. Evaluation

### 9.0 Nomenclature: eRMSE / sRMSE / pRMSE
notebook uses three prefixes and their weighted counterparts:
To keep RMSE labels unambiguous when aggregating over different axes, this
notebook uses three prefixes and their weighted counterparts:

| Symbol | Aggregation axis | Emitted by | Literature name |
|--------|------------------|------------|-----------------|
| **eRMSE** | rows (per coefficient) | `_metric_row` returns `mean_eRMSE`, `median_eRMSE` (mean / median across the per-coefficient vector) | **per-target RMSE** |
| **sRMSE** | coefficients (per row) | `weighted_rmse_per_row` returns a length-$N_{\rm row}$ vector | **per-sample RMSE** |
| **pRMSE** | pixels (per row of reconstructed flux) | `pixel_wrmse_per_row` returns a length-$N_{\rm row}$ vector | (no standard name; sometimes "per-spectrum RMSE") |
| eWRMSE / sWRMSE / pWRMSE | as above, weighted by $1/\sigma^{2}$ | same helpers when `sigma` is provided | — |

The pWRMSE $\sigma$ source is one of (i) `FLUX_SIGMA_TOTAL` HDU in the
decomposition FITS (the LSF-aware propagator output), (ii) on-the-fly
propagation via `reconstruct_component_spectra(coef_err=…)`, or (iii)
floor-only fallback — see §9.1 for the full policy.

consistent two-axis vocabulary for RMSE / WRMSE reporting:
**Terminology (2026-08-17 rename).** The notebook now uses a
consistent two-axis vocabulary for RMSE / WRMSE reporting:

- **sRMSE** ("spectral RMSE") -- per-row error, averaged over
  the coefficient or pixel dimension of a single row. Answers
  *"how far, in flux or coefficient units, is this particular
  row from the target?"*. In the literature this is often called
  per-sample RMSE, per-row RMSE, sample-wise RMSE, or in
  spectroscopy per-spectrum residual RMS.
- **eRMSE** ("ensemble RMSE") -- per-coefficient (or per-pixel)
  error, aggregated over the ensemble of rows. Answers *"how
  well is this particular coefficient / pixel predicted across
  the ensemble?"*. Common alternatives are per-feature RMSE,
  per-coefficient / column-wise RMSE, feature-wise RMSE, or
  pixel-wise RMS.
- **sWRMSE / eWRMSE** -- weighted counterparts, using the
  per-group $\sigma$ floor and per-column normalisation of
  §7 so magnitudes stay comparable to the trainer's
  heteroscedastic-Gaussian loss.

The `_metric_row` helper returns `mean_eRMSE` / `median_eRMSE` /
`mean_eWRMSE` / `median_eWRMSE` / `total_eWRMSE`; the
`weighted_rmse_per_row` and `pixel_wrmse_per_row` helpers return
per-row sWRMSE arrays; the batch RMSE cell exposes
`rmse_subset_results["*_wrmse"]` (per-row pixel sWRMSE keyed by
arm) alongside `["*_rmse"]` (per-row pixel sRMSE).

1. coefficient-space metrics on held-out rows -- plain RMSE, per-coefficient
   Pearson correlation, and (as of 2026-08-16) per-column weighted RMSE
   `mean_wrmse` / `median_wrmse` / `total_wrmse` and per-row `wrmse` for
   the map and correlation cells. Weights match the training loss: $w = 1 /
   \sigma_{\rm eff}^{2}$ with $\sigma_{\rm eff}$ the decomposition
   `COEF_ERR` floored per group by
   `DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP`, per-column normalised so
   $E[w] = 1$ across the reported subset. Missing `COEF_ERR` (older FITS
   files) collapses the weighting to floor-only, so WRMSE reduces to
   plain RMSE; the reporting path is safe against missing sigmas;
2. single-row full-spectrum reconstruction checks;
3. random-subset spectral RMSE (near self-reconstruction, far self-reconstruction,
   predicted science versus observed science);
4. relationship visualisations in context and latent space, including the
   per-row sWRMSE_coef galactic map (2026-08-16, replaced the earlier R² map) and
   sWRMSE_coef-vs-context scatter panels;
5. structure-function diagnostics $D(\Delta\theta)=\langle(\Delta\log A_{resid})^2\rangle$
   for grouped coefficients after geometric normalisation.

Comparing predicted-science reconstruction RMSE against near/far
self-reconstruction RMSE benchmarks against an internal decomposition-fidelity
floor. That floor is a sanity bound, not a competitor: see §11.

### 9.1 Pixel-space pWRMSE alongside spectrum-space (pRMSE) RMSE

Since 2026-08-16 every "spectrum-space RMSE" reporting site (single-row
reconstruction, batch RMSE evaluation, worst-case row table + subplot
titles, and the RMSE dual diagnostic) also reports a pixel-space WRMSE.
The two metrics answer different questions:

- **pRMSE (per-row pixel RMSE)** ($\sqrt{\overline{r^{2}}}$, uniform weights over
  pixels) is a decomposition-fidelity check: how far, in physical flux
  units, does the reconstruction differ from the observation, without any
  noise weighting. It is sensitive to bright-line residuals dominating a
  handful of pixels.
- **pWRMSE (per-row pixel WRMSE)** ($\sqrt{\overline{w \cdot r^{2}}}$ with
  $w = 1/\sigma_{\rm pix}^{2}$ and $\overline{w} = 1$ per row) asks
  whether the residual is within the propagated 1$\sigma$ noise floor of
  the reconstruction: a value near unity means "predicted spectrum sits
  inside the LSF-convolved 1$\sigma$ envelope of the target". Large
  bright-line residuals get down-weighted where the propagated
  $\sigma_{\rm pix}$ is correspondingly large.

**Sigma source (in preference order).** The `pixel_wrmse_per_row` helpers try three
sources for the per-pixel 1$\sigma$ array, so the plumbing degrades
gracefully as the decomposition pipeline matures.

1. **`FLUX_SIGMA_TOTAL` HDU in the decomposition FITS** (planned output
   of the LSF-aware sigma propagator in `sky_decomp/fit.py`; see §12,
   2026-08-16 LSF-aware $\sigma$ propagator). Shape must match the
   corresponding `FLUX` HDU. This is the fast path once the new
   decompositions land: the helper reads the HDU once per file and does
   not re-run the propagator. Alternative HDU name `FLUX_SIGMA` is also
   accepted; the accepted list is `PIXEL_SIGMA_HDU_NAMES`.
2. **On-the-fly propagation from `COEF_ERR`**: when the decomposition
   FITS lacks the sigma HDU but has the `COEF_ERR` HDU (current schema),
   `reconstruct_component_spectra(coef, coef_err=..., lsf=...)` returns
   `sigma_total` via first-order Jacobian propagation through the
   LSF-convolved design matrix. Mathematically identical to route 1
   (both use `SkyDecompBase._components_sigma_from_coef_err`), just
   slower because the propagator runs per row.
3. **Median floor** (fallback of last resort): when neither is
   available, $\sigma_{\rm eff}$ collapses to `PIXEL_SIGMA_FLOOR_REL *
   median(|flux_true|)` per row and pWRMSE reduces to a
   floor-normalised per-row pRMSE.

The batch RMSE cell logs which per-pixel σ source was picked for each of the three
arms at fit time (near/far/sci), so switching schemas is visible in the
run log without inspecting FITS files.

**Reported keys.** `rmse_subset_results` now carries `near_wrmse`,
`far_wrmse`, `sci_wrmse` in addition to the pre-existing `*_rmse`, and
the summary DataFrame has three extra rows
(`*_wrmse`). The worst-case row table adds `sky1_wrmse_pix`,
`sky2_pWRMSE`, `sci_pWRMSE` alongside the coefficient-space
`sci_sWRMSE` already introduced earlier the same day (§12,
2026-08-16 WRMSE reporting).

### 9.2 Effective dimensionality of the coefficient groups

A dedicated cell measures how many degrees of freedom each group actually
carries, on geometry-normalised amplitudes, with the basis fit on training
nights only. Three criteria are reported because they disagree:

- **Variance fraction** — shown only to demonstrate that it is a trap here. Past
  the real structure the leftover variance is noise spread over hundreds of
  directions, so "99.9% of variance" tries to capture the noise.
- **Horn parallel analysis** — eigenvalues compared against a column-shuffled
  null, computed from the *correlation* matrix. Standardisation is essential:
  otherwise the null inherits the column variances, which span a large range
  (a high-excitation OH line is far more sensitive to $T_{\rm rot}$ than a
  low-excitation one), and the method badly under-detects. This is the right
  rank for the **encoder input**.
- **Cross-arm score correlation** — near and far scores projected onto the same
  basis and correlated per component, on held-out nights. A direction
  uncorrelated between two simultaneous lines of sight cannot be predicted at a
  third, so this is the right rank for the **target**.

Two structural points fall out of the second and third criteria. The
transferable components are *not* generally the leading block, because PCA sorts
by variance rather than transferability — the target basis is therefore a subset
selected by correlation, not a truncation at the first $k$. And the transferable
count can *exceed* the Horn rank, because correlating two arms beats the noise
floor of either one alone; the encoder input should be sized by the larger of
the two.

The projection space is `asinh` by default rather than `sqrt`: a change in
rotational temperature is a single linear direction in log space but a curved
manifold in linear space, so log-like coordinates give a more compact and more
interpretable basis. This is deliberately decoupled from the loss, which stays
in the variance-stabilising `sqrt` space. A sensitivity scan reports the Horn
rank under each transform.
per-coefficient weight balance (§7), and per-group caveats. The
recommendation of this cell — proceed with per-group compression using
asinh for the mesospheric group, sqrt for moon (originally linear; changed
2026-08-19b, see §12) and sqrt-identity for the rest, and target components
selected by cross-arm correlation above the current default threshold
(0.10 as of 2026-08-19b) on held-out nights — has been adopted as the
default model of the notebook; see §5.5. Two consequences that were decided deliberately before adopting
a projection: Softplus non-negativity is lost in score space, and PCA
truncation on the moon spline is itself a smoothness prior, making
`moon_smooth_lambda` redundant. Reconstruction residual in the fitted
basis also serves as an out-of-distribution flag.
basis also serves as an out-of-distribution flag.
### 9.3 Solar-activity coupling of the coefficient groups

An empirical check of whether the `f107`, `f107_81d`, `kp` context features
(\S12, 2026-08-04) carry signal the model can use. For each coefficient,
Pearson $r$ is computed between the solar-activity feature at the
observation MJD and $\log$ of that observation's zenith-equivalent
emissivity (target arm, geometry removed via `airglow_geometry_scale`,
\S4). Log space because airglow amplitudes are approximately log-normal;
geometry removed so the correlation isolates intrinsic emissivity
variation rather than airmass + zenith modulation.

Measured on the filtered triplet, $n = 10\,743$ rows spanning MJD
$60177 \to 61086$ ($909$ days, mid-2023 to early-2026 -- the window
covers the ascent and peak of solar cycle 25, so `f107` sweeps
$111 \to 384$ sfu and Kp median is $2.0$, max $8.7$).
$111 \to 384$ sfu and Kp median is $2.0$, max $8.7$).
**Per-group summary:**

| group | $n$ | median $r$ vs `f107` | max $|r|$ vs `f107` | median $|r|$ vs `kp` |
|---|---|---|---|---|
| **ionospheric** | 4 | **+0.174** | 0.246 | 0.057 |
| atomic | 3 | -0.077 | 0.158 | 0.052 |
| mesospheric (OH + O$_2$b) | 403 | -0.055 | 0.197 | 0.011 |
| continuum (HO$_2$ / FeO / O$_2$Ac) | 3 | -0.013 | 0.020 | 0.008 |
| moon spline (null control) | 29 | -0.021 | 0.112 | 0.010 |

Only `ionospheric` carries a group-wide signal, and it does so in the
right direction. F-region O I recombination increases with EUV flux,
F10.7 is the standard EUV proxy, and all four `ionospheric` coefficients
agree in sign:

| coef | line | $r$ vs `f107` | $r$ vs `f107_81d` | $r$ vs `kp` |
|---|---|---|---|---|
| `ATOM_Orc_OI0845` | OI 8446 F-region recombination | **+0.246** | +0.236 | +0.118 |
| `ATOM_Orc_OI0777` | OI 7774 F-region recombination | +0.179 | +0.176 | +0.040 |
| `ATOM_N` | [NI] 5199 F-region metastable | +0.158 | +0.165 | -0.034 |
| `ATOM_Or` | [OI] 6300 / 6364 red doublet | +0.147 | +0.173 | -0.074 |

`atomic` shows a physically-correct negative F10.7 coupling for Na D
(`ATOM_Na`, $r = -0.158$): solar-max heating and photodissociation deplete
mesopause Na, so its column density drops as F10.7 rises. `ATOM_K` follows
the same direction more weakly ($r = -0.077$); `ATOM_Og` (OI 5577, mixed
mesopause + F-region origin) is flat ($r = -0.018$).

`mesospheric` (OH + O$_2$ b-band) shows a systematic but weak negative $r$
vs `f107` (median $-0.055$), consistent with solar-max mesopause heating
reducing OH volume emission rate. Not one individual OH coefficient
exceeds $|r| = 0.20$ -- gravity-wave variability and OH altitude drift
dominate the amplitude budget on the sampled timescales.

`continuum` (HO$_2$ / FeO / O$_2$Ac) is flat against every solar index
($|r| < 0.02$ everywhere), consistent with mesopause chemistry that does
not respond directly to EUV.

`moon` spline coefficients are a null control (nothing physical connects
scattered moonlight to solar activity) and behave as one: median $|r|$ =
0.037, max = 0.112. The small nonzero max reflects the observing-schedule
confound described below.

**Kp is uniformly weak** -- $|r| < 0.12$ everywhere, including
`ionospheric`. LCO's $\sim -19^\circ$ geomagnetic latitude puts it well
below the auroral zone, and Kp's 3-hourly spikiness is largely averaged
out by the median-stack exposures. Kp is retained in the context because
it costs nothing; its main effect will be to be down-weighted by the
encoder's first-layer robust scaling.

**Interpretation for the model.** The `ionospheric` head will pick up
usable signal from `f107` / `f107_81d`; the `atomic` head will likely
gain a small edge predicting Na D via the negative F10.7 coupling;
`mesospheric` and `continuum` heads are expected to gain little. Robust
scaling applied at the encoder input handles that gracefully: a feature
carrying no signal for a given head gets a small effective weight at
fit time, so no hand-designed gating is required to keep the irrelevant
heads from being distracted.

**Caveat.** The observing window covers the ascent and peak of solar
cycle 25, so `f107` covaries with `obstime_year_sin` / `obstime_year_cos`
and with month-of-year systematics already in the context. A partial
correlation residualising F10.7 against the year features would separate
the two contributions; the marginal $r$ values reported here should be
read as upper bounds on the unique F10.7 information.
read as upper bounds on the unique F10.7 information.
## 10. Reproducibility

- deterministic under explicit seeds (split and random subsets);
- artifacts carry scalers, group mapping, resolved per-coefficient wavelengths
  and their provenance, best checkpoint, and resolved hyperparameters;
- RMSE reported in both physical and display-scaled units.
- RMSE reported in both physical and display-scaled units.
## 11. Known limitations and open items
## 11. Known limitations and open items
Ordered by expected impact.

**11.1 Naive baseline (resolved 2026-08-11).** The `naive-baseline` cell
compares the model against three physically-motivated baselines on the same
night-held-out test split:
- `B0_copy_near`: $\hat{c} = c_{near}$ (no physics)
- `B1_near_geo`: $\hat{c} = \mathrm{em}_{near} \cdot G_{sci}$, the near arm
  divided by its own geometry factor and multiplied by the science-pointing
  factor
- `B2_mean_geo`: $\hat{c} = 0.5 (\mathrm{em}_{near} + \mathrm{em}_{far}) G_{sci}$,
  the symmetric geometry-corrected average.

B1 is the strongest (mean_rmse 23.57, median_corr 0.9787). The current
default model 4-seed ensemble beats B1 by **15.3%** on aggregate mean_rmse
and +0.0063 on median_corr; wins 4 of 5 groups (moon, continuum, mesospheric,
atomic). Ionospheric (n=4 F-region coefficients) is the only group where B1
still wins by +18%; a 2.0 lift on `ionospheric_group_weight` and a dedicated
ionospheric encoder-head branch both narrowed the gap but regressed other
groups (see §12), so neither was adopted. The pre-adoption baseline check
showed ML was 40% *below* B1 on mean_rmse; removing PCA on the mesospheric
compressor and adding `mesospheric_group_weight` closed the gap and pushed
ML ahead. That adoption is the biggest single change in §12 for 2026-08-11.

**11.2 The target is contaminated by the science field.** `coef_sci` is fitted to
`FLUX_SCI`, which contains the nebular emission of the target. A sky-component
fit to that spectrum absorbs nebular flux, worst in OI 6300, which is
simultaneously a thermospheric airglow line and a shock/DIG diagnostic. Training
on it teaches the model to over-subtract, most severely in the brightest fields.
The `sci_rmse` metric compounds this: a perfect sky prediction leaves nonzero
residual at every nebular line, and the metric scores that as error. Cheap
diagnostic: if `sci_rmse` ever drops below `near_rmse`/`far_rmse`, the model is
fitting the object. Kinematic separation (sky component fixed at
$v_{topo}=0$, nebular component at the field's systemic velocity) works for
targets with large $|v_{sys}|$ and not at all for local gas.

**11.3 Extinction confounder (open).** The effective-extinction fit of §4.5 is
implemented and used by default, which removes the stellar-curve bias. What
remains open is the confounder noted there: a horizontal brightness gradient
correlated with elevation would be absorbed into $k_{\rm eff}$. Worth testing by
refitting against azimuth and by season, and by refitting per lunation to track
aerosol.

**11.4 Per-coefficient heights (open).** Heights are assigned per group:
$87$ km for the OH Meinel bands and O$_2$ atmospheric band (`mesospheric`)
and the HO$_2$ / FeO / O$_2$Ac continuum (`continuum`); $95$ km for K, Na and
OI 5577 (`atomic`); $285$ km for N I 5199 and OI 6300 / 6364 / 7774 / 8446
(`ionospheric`). Within `atomic` the true emitting layers differ by a few km
(OI 5577 nearer $96$ km, Na D nearer $92$ km, K nearer $90$ km), and that is
the largest remaining within-group height approximation now that `continuum`
sits at $87$ km with `mesospheric` and `ATOM_N` sits at $285$ km with the rest
of the ionospheric block. A per-coefficient height array is a small extension
since the per-coefficient wavelength array already exists, but the $h$–$k$
degeneracy of §4.5 limits how much a finer-grained height correction can
move the transfer -- the fitted $k_{\rm eff}$ already absorbs part of any
height error.

**11.5 Population-model weighting (resolved 2026-08-04).** The
`oh_wavelengths_from_input_file` auxiliary route and `cross_check_wavelengths`
were retired. `resolve_coef_wavelengths_a` now runs the basis centroid path
unconditionally for every coefficient (§4.4). Every OH wavelength therefore
carries the mesopause rotational Boltzmann weighting the basis already
encodes, and the missing-$\exp(-E/kT)$ approximation is gone. With a single
wavelength source there is nothing left to cross-check. See §12
(2026-08-04, OH wavelength single-source) for the change details.

**11.6 Loss is in coefficient space, not spectrum space.** Basis components are
partially degenerate: two coefficient vectors can differ substantially yet
produce near-identical spectra. Since the decomposition is linear ($s=B\mathbf{c}$),
the objective that matches the science requirement is
$(\hat{\mathbf{c}}-\mathbf{c})^\top B^\top\Sigma^{-1}B(\hat{\mathbf{c}}-\mathbf{c})$
with $\Sigma$ the per-pixel variance and nebular-contaminated ranges masked. That
also removes the need for `continuum_group_weight` and fixes the $w_g$ issue in §7.

**11.7 No predictive uncertainty (partially resolved 2026-08-12).** The
model is deterministic, so nothing propagates into the science error
budget. Given that §11.2 prevents verifying the prediction against the
science spectrum in general, calibrated uncertainty is not optional.
Minimum viable version: a second output per group giving $\log\sigma$,
trained under Gaussian NLL, checked with a PIT histogram on
night-held-out rows. **Interim diagnostic (item 6 of the closing review,
2026-08-12; ensemble size raised from 4 to 10 the same day):** the 10-seed
ensemble std across seeds is computed per row per coefficient (floored at
$10^{-3} \cdot \mathrm{MAD}(\mathbf{c}_{sci,\mathrm{train}})$ per
coefficient to avoid $\sigma \to 0$ blowups on dead OH lines), along
with per-group $\sigma_{\mathrm{true}}/\sigma_{\mathrm{pred}}$ ratios,
coverage at $|z|<1.96$, and a 20-bin PIT histogram of $u = \Phi(z)$ on
val plus a val$\to$test transfer sanity check, in a diagnostic cell
after the trainer. The 4-seed baseline diagnostic
($\sigma_{\mathrm{true}}/\sigma_{\mathrm{pred}} \in [1.93, 2.89]$,
aggregate 2.89 on val and 2.48 on test, coverage $|z|<1.96 = 57.4\%$ on
test vs nominal $95\%$, PIT $\chi^2_{20} = 9.5\text{e}5$) drove the
bump to N=10, which shrinks the ensemble stderr by $\sqrt{10/4} \approx
1.58\times$ and tightens both the mean-bias floor of §11.1 and the
σ-scale bolt-on estimate; N=10 does **not** fix the underlying
under-dispersion factor (a proper Gaussian-NLL head does). Verdict:
classic deep-ensemble under-dispersion; the under-estimation factor
transfers cleanly from val to test ($<1\%$ drift), so a single per-group
$\sigma$-scale from val is a defensible bolt-on until a proper
Gaussian-NLL head lands. Per-group scale factors and the $\sigma$ floor
arrays are stored in `mlp_artifacts['predictive_uncertainty']` under keys
`sigma_scale_by_group_val` and `sigma_floor_by_group_train`. Combined
predictive $\sigma$ (aleatoric $\sigma_\mathrm{dec}$ from `COEF_ERR`
added in quadrature with epistemic $\sigma_\mathrm{ens}$ from the seed
spread) is exposed via a helper on the deployed ensemble artifact for
downstream use. See §12, 2026-08-12 (items 6 + 11) for the full context.
downstream use. See §12, 2026-08-12 (items 6 + 11) for the full context.
**11.7b Missing pixel-noise propagation into `COEF_ERR` (open, blocks the §11.7 aleatoric branch).** The `COEF_ERR` produced by the current decomposition pipeline is not a photon-noise-derived error bar: (a) `medians_computation/build_sframe_stack.py` reads only `FLUX`, `SKY`, `LSF`, `SLITMAP` from each `lvmSFrame` file and discards the per-fiber `IVAR` HDU, so the median stack has no ivar sidecar; (b) `decompose_parallel.py::fit_chunk_worker` therefore passes `ivar_row = np.ones_like(flux_row)` (or, for the moon-zodi branch, a 0/1 finite-mask) into `SkyDecomp.fit`, so the active-set covariance is a unit-weight formal covariance rescaled by the aggregate residual RMS via `max(reduced_chi², 1)`. The observed miscalibration in cells 41/42 (aleatoric σ over-estimated ~300× on mesospheric, ~7× on moon, every group flagged UNCALIBRATED) is dominated by this missing propagation; the 2026-08-16 per-column χ² fix in `fit.py` cannot close it because the `max(per_column_χ², 1)` clip refuses to shrink σ below the raw Cramér-Rao floor of a unit-weight fit. **To do**, in order: (i) extend `build_sframe_stack.py`'s `OUTPUT_ARRAYS` with `IVAR_SCI` / `IVAR_SKY_NEAR` / `IVAR_SKY_FAR`, propagating `IVAR` through the faintest-fiber median selector -- for a median of N i.i.d. samples the variance of the median estimator is ~1.57/N × per-sample variance, so `ivar_median ≈ N / (1.57 · Σ 1/ivar_i)` on the selected fibers, applied per pixel; (ii) update `decompose_parallel.py::fit_chunk_worker` to read the new `IVAR_*` HDUs and forward `ivar_row = ivar_from_stack[i]` into `SkyDecomp.fit` instead of `np.ones_like(flux_row)`; (iii) regenerate the `_meta_coef*` decomposition products end-to-end and re-run cells 41/42 of this notebook to re-check calibration. Only then does §11.7's aleatoric branch have a physically defensible σ; the epistemic branch (ensemble spread) is independent and unaffected.

**11.8 Filtering removes the hard cases.** Kappa-sigma clipping runs on
concatenated near+far+**sci** coefficients, so rows extreme in the *target* are
rejected — bright time, high airmass, high solar activity. That raises test scores
and lowers real-world performance. Clip on inputs only and keep the extremes as a
held-out stress set. Similarly, $\chi^2$ gating uses `nanmax` across the three
products, so rows where a sky-only model fits `FLUX_SCI` badly are cut, biasing
the training set toward faint fields.

**11.9 Context gaps.** F10.7 (observed 10.7 cm solar radio flux) and Kp are
now in the context, sourced from GFZ Potsdam's authoritative archive (§12,
2026-08-04). `sci_sep` (per-arm angular separation from the science pointing)
is also in the context. `ew` (per-arm sky-telescope identity: $+1$ for SKYE,
$-1$ for SKYW, $0$ for the sci arm) was added on 2026-08-05, so any
throughput difference between the two sky telescopes can be absorbed by the
shared sky encoder as a linear per-arm offset instead of leaking into a
geometry-correlated systematic. The apparent dusk-vs-dawn degeneracy of
`sun_alt` is broken by `obstime_day_sin` / `obstime_day_cos`, which
distinguish e.g. 3 UT from 9 UT for the same solar altitude, so the model
already has time-of-day information paired with sun altitude. Per-arm
ecliptic coordinates `ecl_beta_deg` / `ecl_lon_sin` / `ecl_lon_cos` were
added on 2026-08-19 so the moon-vs-zodi degeneracy in the `Moon_bs`
continuum spline has a native geometric handle (raw sci COEF sanity check:
shape-space $|r|$ up to $0.34$ against $|\beta|$, partial correlation
unaffected by moon geometry — see §12, 2026-08-19). The one gap
that remains open is a bracketing indicator (whether a science exposure sits
between two sky-arm exposures or at the edge of a sky-arm sequence), which
would let the model account for stronger interpolation confidence in the
interior of a sky-arm sequence than at its ends.

**11.11 Hyperparameter search over the compressed loss (resolved on 2026-07-28).**
A dense 18-config grid centred on the previous default
(encoder_dims ∈ {(320,160),(384,192)}, head_dim ∈ {160,192,224},
lr ∈ {5e-4,7e-4,1e-3}, trunk_dims=(320,160), weight_decay=1e-4,
seed=42, budget=50 epochs/patience 4) was evaluated on the night-held-out
test split. The winner — encoder_dims=(384,192), trunk_dims=(320,160),
head_dim=224, lr=5e-4, weight_decay=1e-4 — improved mean per-coefficient
RMSE from 102.4 to 99.75 (−2.6%), MAE from 74.96 to 71.33 (−4.8%), and
median per-coefficient Pearson correlation from 0.889 to 0.890, converging
at epoch 30 rather than 20 under the same patience. That geometry has been
adopted as `default_dual_group_config`. A prior wide random sweep
(sampled 14 of ~1200 candidates) missed the exact default geometry, and
its local best was 2.6% worse than the current default when retrained at
the default budget; both sweeps are retained in the notebook (gated by
`RUN_SWEEP` / `RUN_DENSE_SWEEP`) as templates for the next dataset. The
improvement was measured at a single seed (42), so a multi-seed rerun is
still open work if a smaller change matters. The stale uncompressed sweep
cell was removed on 2026-07-28. The 2026-07-30 post-audit re-run of the
same dense grid at the corrected pipeline state (§12) picked a different
geometry (`encoder_dims=(448, 224)`, `head_dim=224`, `lr=5\text{e-}4`), which
was then superseded by the 2026-08-08 post-LSF-surface re-run at
`encoder_dims=(512, 256)`, `head_dim=288`, `lr=7\text{e-}4` (the current
default); the 2026-07-28 numbers here are historical.

## 12. Change log

**2026-08-19b — moon compressor final tune (adopted default).**
Three coupled changes on the moon prediction path, all deployed together:
`COMPRESSION_XARM_THRESHOLD` $0.55 \to 0.10$ (keeps all 29 moon PCs; only
the moon group has `use_pca=True`, so no side effects on the other four
groups), `COMPRESSION_TRANSFORM_BY_GROUP['moon']` $\text{linear} \to \text{asinh} \to \text{sqrt}$
(intermediate `asinh` step reverted; see below), and `moon_group_weight`
$3.0 \to 4.0$. Motivation and diagnostic (encoder-free compressor
round-trip on test rows): PC 4 ($|r_{xarm}| = 0.20$, centroid 9644 Å,
red-band power fraction 0.99) and PC 6 ($|r_{xarm}| = 0.53$, centroid
8842 Å, red-band power fraction 0.69) that the old 0.55 threshold dropped
carry essentially the entire extreme-red moon representation — round-trip
rms at 9500-9800 Å is 7.3 when they are dropped and 0.02 when they are
kept. Their low $|r_{xarm}|$ is SNR-limited (moon is faint in the extreme
red), not non-transferability. Separately the `asinh` transform crushed
extreme-blue precision (round-trip rms at 3600-3800 Å $\approx 8.3$ vs
$0.005$ under `sqrt` or `linear`) because it forces the PCA to spend
variance on the tiny asinh-compressed tail values instead of the
large blue-bright coefficients. `sqrt` is milder than `asinh` but still
caps the bright-moon-row loss share; `moon_group_weight = 4.0` then
directs the freed capacity into moon shape fit since the sqrt+all-29-PC
compressor is essentially lossless (per-coef empirical calibration lift
now sits inside $[0.98, 1.02]$ with $0/29$ knots hitting the $[0.7, 1.4]$
clip; the trainer no longer needs a compensating bias correction). Effect
on the sci-arm residuals evaluated by the batch RMSE + per-component
decomposition cells (100-row every10 sample): moon.band_rms.extred
$0.429 \to 0.057$ ($-87\%$), moon.band_bias.extred $+0.141 \to -0.001$
(bias eliminated); moon.band_bias.extblue $+0.066 \to +0.000$; moon.rms
mean $0.167 \to 0.110$ ($-34\%$); moon.rms max $1.25 \to 0.90$ ($-28\%$);
total.rms max $1.59 \to 1.32$ ($-17\%$); ensemble mean_eRMSE goes
$24.30 \to 25.44$ (+5%, expected coefficient-space weighting artefact from
tripling then quadrupling the moon multiplier), ensemble median_corr
stays at 0.983. All six moon-residual bands now have $|\text{bias}| \le 0.008$.

**2026-08-19c/d/e — extreme-blue moon tightening attempts (all rejected).**
The 2026-08-19b tune left the extreme-blue moon rms essentially unchanged
(the bias was eliminated but the per-row rms floor at 3600-3800 Å is
$\approx 0.05$ on typical rows and $\approx 1.3\text{-}2.5$ on a
handful of bright-moon outliers). Three follow-up attempts all failed
to move the metric:
- **(c) Per-knot sigma boost.** Scale `coef_err_sci` down $2\times$ on
  moon spline knots 0-6 (peak wavelengths 3600-4792 Å) so their per-element
  loss weight goes up $4\times$. Diagnostic said the low-blue knots are
  model-limited (residual rms / coef_err_sigma $\approx 2.4$, $R^2 \approx 0.99$),
  suggesting a real headroom. Retrained. Delta: moon.band_rms.extblue
  $0.457 \to 0.481$ ($+5\%$); all bands within $\pm 5\%$.
- **(d) Per-row sigma boost on bright-moon rows.** Scale `coef_err_sci`
  down $2\times$ on the 236 training rows (2%) with `moon_fli` $\ge 0.85$
  and `moon_alt` $> 0$. Retrained. Delta: moon.band_rms.extblue
  $0.457 \to 0.471$ ($+3\%$); essentially no change anywhere.
- **(e) Physics-informed context features.** Added
  `moon_airmass_v` (global saturated $\sec(z_{\rm moon})$),
  `moon_scatter_ray` (per-arm $\mathrm{fli} \cdot (1 + \cos^2\rho)$)
  and `moon_scatter_mie` (per-arm $\mathrm{fli} \cdot \exp(-\rho / 40°)$)
  to each arm's ctx via a new `_augment_triplet_with_moon_scatter` cell
  patterned on the ecliptic augment. Ctx dim $31 \to 34$; retrained.
  Result: training destabilised (seed-to-seed std $0.4 \to 0.95$),
  moon.band_rms.extblue $0.457 \to 0.474$ ($+4\%$), all bands
  $+3\text{-}11\%$, ensemble mean_eRMSE $25.44 \to 25.94$. Reverted.

Root-cause diagnostic across all three: the 0.457 sample-wide
extreme-blue rms is 70% SS from 5 rows out of 100 in the every10 sample;
the median row has extreme-blue rms $0.051$ (excellent). Those 5
outliers are all bright-moon rows (9/10 in the worst 15 have
$\text{moon\_fli} \ge 0.89$, $\text{moon\_alt} \in [21°, 81°]$,
$\text{sci\_moon\_sep} \in [46°, 92°]$), and only 359/1726 filtered
rows (21%) hit those thresholds. Weight tuning cannot fix this because
2% of rows cannot teach the other 98% about a physically-distinct
scattering regime; physics-feature engineering cannot help either
because the raw covariates (`moon_alt`, `moon_sep`, `moon_phase_sin/cos`)
are already in ctx and the network already fits typical rows near the
decomposition noise floor. Escape routes for future work — moon-
specialised sub-network branched on `moon_fli > threshold` with
dedicated capacity, synthetic bright-moon data augmentation via a
Krisciunas-Schaefer forward model, or accept the current level as the
physical limit given available data. No config change adopted from
(c)/(d)/(e); the 2026-08-19b deployed defaults stand.

**2026-08-19 — NaN-pixel resilience in the batch RMSE cell + defensive
`np.max` on the chi2 filter.** Investigation of the worst-15
reconstruction rows (cell 28) surfaced an "arm-excluded" pattern where
6/15 rows had `sky1_pRMSE = NaN` or `sky2_pRMSE = NaN` while the sci
channel was finite. Not a real per-arm exclusion: every affected row
has finite `reduced_chi2` on all three arms, finite decomposition
coefficients, and 99.99% finite flux pixels. Mechanism: raw
`FLUX_SKY_NEAR/FAR/SCI` arrays contain isolated NaN pixels (cosmic-ray
hits + dead-pixel flags emitted by the extraction pipeline) — 704 of
1726 every10 rows have **exactly one** NaN pixel out of 12401, 18 rows
have 2-4, none have 5+. The batch RMSE loop's
`np.mean((flux_recon - flux_true)**2)` propagated that lone NaN to the
whole-row pRMSE. Fixed: switch to `np.nanmean` for `near_rmse`,
`far_rmse`, `sci_rmse` in the batch RMSE cell so an isolated masked
pixel yields an accurate pRMSE over the remaining ~12400 valid pixels
instead of a whole-row NaN. Alongside, flipped
`np.nanmax(chi2_stack, axis=1) → np.max(chi2_stack, axis=1)` in both
`apply_triplet_filters` and the every10 batch filter as a defensive
safety net: currently a no-op (all chi2 values are finite in the
current data), but if any pipeline in the future emits a NaN chi2 on
one arm the whole observation will now drop through the `isfinite`
gate rather than silently passing on the strength of the other two
arms. Verified independently that the LMC/SMC science-field exclusion
in both filter sites already uses `sci_ra`/`sci_dec` only — no fix
needed there. No rows dropped; the metric-fidelity fix affects the ~40%
of every10 rows that had 1 NaN pixel in some arm.

**2026-08-19 — per-arm ecliptic coordinates added to `context_cols` +
input FITS moved to `spline_moon/`.** A new augmentation cell
(`ecliptic-ctx-augment`, inserted right after the filter cell) attaches three
per-arm features to each of `ctx_near`, `ctx_far`, `ctx_sci`:
`ecl_beta_deg` (raw ecliptic latitude in degrees, so the network sees the
zodi $\cos\beta$ amplitude without pre-transform), and the cyclic pair
`ecl_lon_sin` / `ecl_lon_cos` for ecliptic longitude. Coordinates are
computed from each arm's RA/Dec via `astropy.SkyCoord(icrs).transform_to(
BarycentricMeanEcliptic())`. Because the triplet builder in this notebook
attaches only `sci_ra`/`sci_dec`, the augment cell reads `SKY_NEAR_RA/DEC`
and `SKY_FAR_RA/DEC` directly from the meta FITS as a fallback so each arm
gets its own ecliptic vector (falling back to the sci pointing only for
arms whose META RA/Dec is missing). The function is idempotent: pre-existing
ecliptic columns (including two legacy sci-only V1/V2 layouts) are stripped
before the new features are appended, so re-running never accumulates stale
copies. Context width grows from 28 to 31; §6.4 step 1/4 dimension
bookkeeping was updated accordingly ($Q = 31$; $Q - 3 = 28$ after the
`vanrhijn_*` drop). The three call sites that rebuild triplets outside the
main loader are covered:
- the every10 single-row check and batch-RMSE cells now call
  `_augment_triplet_with_ecliptic(e10_triplet, meta_fits_path=EVERY10_INPUT)`
  after `build_triplet_coef_dataset`;
- the full-sky WRMSE map cell strips `ECLIPTIC_FEATURE_NAMES` from its
  `context_columns` before calling the loader (otherwise
  `_build_context_matrix` fails on unknown META columns) and re-augments
  the returned `triplet_full` with `meta_fits_path=INPUT_FITS`.

Ecliptic $\leftrightarrow$ Moon\_bs correlation check (raw sci COEF on
17 210 gated rows before training, done as a sanity gate for feature
usefulness):
- **Raw per-coefficient Pearson $|r|$ is $\le 0.012$** for every one of
  the 29 Moon\_bs coefficients against every ecliptic feature; identical
  to three decimal places across the block because the 29 spline
  amplitudes are near-collinear (one common Moon-brightness scalar
  dominates their linear projection) and the ecliptic features have very
  low overlap with that scalar.
- **After per-row L2 normalisation the SHAPE of the spline correlates
  clearly with ecliptic geometry**: median $|r_{shape}| \approx 0.24$ for
  `ecl_lon_sin` and $\approx 0.19$ for `ecl_lon_cos`, growing along the
  spline index and peaking at Moon\_bs27–28 (the red end, where the zodi
  continuum has its Rayleigh-Jeans tail); mid-spline knots pick up the
  latitude signal with $|r_{shape}|$ up to $0.34$ against `|ecl\_beta|`.
- **Row-summed spline amplitude tracks $|\beta|$ / $\cos\beta$**:
  $r(\log\Sigma\,\text{Moon\_bs},\ \cos\beta) = +0.18$,
  $r(\log\Sigma\,\text{Moon\_bs},\ |\beta|) = -0.17$, i.e. rows near
  the ecliptic plane sit noticeably higher in Moon\_bs total, which is the
  expected zodi signature bleeding into the spline that currently soaks up
  both scattered moonlight and zodiacal continuum.
- **Partial-correlation control**: residualising both sides on
  `moon_alt + moon_sep + airmass + sun_alt + moon_phase` (the moon /
  geometry covariates already available in context) leaves the ecliptic
  raw-amplitude correlations essentially unchanged (partial $|pr|$ within
  $\pm 0.001$ of raw $|r|$), so the ecliptic information is **independent**
  of the moon-geometry channel the network already had, and is exactly the
  kind of signal a per-arm feature can plausibly help disentangle
  zodi-vs-moon in the Moon\_bs spline.

Verdict: the shape-space $|r|$ up to $0.34$ and the amplitude $r \approx
\pm 0.18$ against $\cos\beta$ / $|\beta|$ are large enough that the
feature is not just noise, and small enough that it will not dominate the
loss the way `moon_alt` or `airmass` do — the encoder can pick it up
adaptively via robust-scaled input weights. A/B against the pre-ecliptic
default is deferred to the next multi-seed run of the deployed default;
until then the flag is on-by-default (no opt-out kwarg — set
`filtered_triplet['ctx_names']` back to the 28-feature layout by editing
the augment cell if a comparison run is needed). Alongside the ecliptic
port, the notebook's input FITS references now point at the `spline_moon/`
subdirectory (files there use suffix `_lsf_surface_iterative`, matching the
existing `_DECOMP_SUFFIX`) so downstream cells read the new decomposition
without any per-cell path edits; `WAVELENGTH_CACHE` moved to
`spline_moon/coef_wavelengths_basis_v3.npz` so the new basis writes its own
cache and can't silently return centroids from the parent-directory files.

**2026-08-16 — pixel-space WRMSE plumbing everywhere spectrum-space RMSE lives.**
Added a small helper cell (`load_pixel_sigma_if_available` +
`pixel_wrmse_per_row` + `PIXEL_SIGMA_HDU_NAMES` +
`PIXEL_SIGMA_FLOOR_REL`) with the three-tier sigma-source policy of §9.0,
then threaded a pixel-space WRMSE column through every existing
spectrum-RMSE reporting site: the batch RMSE eval (`rmse_subset_results`
gains `near_wrmse` / `far_wrmse` / `sci_wrmse`; the summary DataFrame
gets three extra rows; the log line names the sigma source used per
arm), the worst-case row table + subplot titles (adds
`sky1_pWRMSE` / `sky2_pWRMSE` / `sci_pWRMSE`, retitles the
coefficient-space one to `sci_sWRMSE`), the single-row full-spectrum
check (WRMSE printed alongside RMSE for near / far / sci and included
in the plot title), and the RMSE dual diagnostic (percentile table +
stats block + a new histogram panel). The helper reads
`FLUX_SIGMA_TOTAL` from the decomposition FITS when present (that HDU
is the planned output of the LSF-aware propagator that will ship with
the next round of decompositions) and otherwise runs the propagator
on-the-fly from `COEF_ERR`; both routes call
`SkyDecompBase._components_sigma_from_coef_err` and produce identical
sigmas. The `_fast_reconstruct` helper inside the batch RMSE cell was
extended to accept `coef_err=` so this in-loop propagation reuses the
already-built LSF-convolved design matrices and adds negligible cost
per row. All reporting paths fall back gracefully to floor-only
weighting when neither the FITS HDU nor `COEF_ERR` is available, so
nothing breaks on the current-schema decomposition files.

**2026-08-16 — empirical calibration generalised (train+val, all groups).**
Two independent gates were silently discarding most of the per-group
mean-bias correction: `train_compressed_group_mlp` filtered the calibration
loop to `comp['kind'] in ('asinh', 'log')`, and
`inverse_group_compressor` mirrored the same filter on the consumer side.
Together they left `moon` (`linear`), `continuum` / `ionospheric` /
`atomic` (`sqrt`) with the raw SmoothL1 median-bias, so the per-group
bias diagnostic reported ensemble `continuum` $= -1.77\%$, `moon` $=
$-0.42\%$, `mesospheric` $= -0.61\%$ on all 11 624 filtered rows.
Both gates were removed on 2026-08-16, and the calibration fit set was
widened from val only to train $\cup$ val (test excluded) so small
groups (`continuum`, `atomic` at $n = 3$) get $\sim\!85\%$ of the rows
rather than $\sim\!15\%$. A tightened degeneracy filter skips groups
with sign-flipped or near-zero means. The $[0.5, 2.0]$ multiplicative
clip stays as a broken-group guard; observed lifts are all within
$[0.98, 1.03]$. Effect on the current 10-seed ensemble: every group's
ensemble bias sits under $0.05\%$ and the diagnostic's auto-verdict
flipped from "investigate calibration / loss balance" to "N=10 seeds are
sufficient". The loss-side per-group multipliers (`moon_group_weight`
etc.) were **not** re-tilted -- they stay at the flat 2026-08-16 retune
defaults because the calibration alone closes the bias floor without
perturbing the LR / WD sweep that landed those values. §5.5 and §7
updated accordingly.

**2026-08-16 — LSF-aware σ propagator (`sky_decomp/fit.py`).**
`SkyDecompBase._components_sigma_from_coef_err` propagates per-coefficient
$1\sigma$ to per-pixel $1\sigma$ for every reconstructed component via
first-order (Jacobian) propagation:
$\sigma^2_{\rm comp}(\lambda) = \sum_j M[j,\lambda]^2 \cdot \sigma^2_{c,j}$,
exact because the reconstruction $\text{flux} = M^\top c$ is linear in
the coefficients. `reconstruct_component_spectra` gained an optional
`coef_err=` kwarg; when supplied, the returned dict carries a
`sigma[<name>]` array for every component plus a `sigma_total`
(quadrature sum over `{oh, moon, diffuse, atom, orc, o2}` -- `diffuse`
is already the quadrature of `ho2`/`feo`/`o2ac` from a single coefficient
each). Both paths use the identical `mats` bundle that
`_components_from_coef` uses so the LSF convolution stays consistent
between mean and $\sigma$. NaN sigmas are treated as zero contribution
(consistent with the missing-uncertainty fallback used elsewhere in the
notebook). Seven new unit tests under `sky_decomp/tests/test_sigma_flux_propagation.py`
cover: analytical single-column propagation on a scalar coefficient,
diffuse quadrature, matrix (`oh`, `atom`, `orc`) propagation, NaN
passthrough, shape validation, `sigma_total` consistency, and BLAS
benchmark parity vs the reference einsum path (the direct `(m*m).T @ err²`
form is faster than `einsum` for typical shapes and is the shipped
implementation; the docstring records the benchmark result to prevent
future einsum re-introductions). Downstream: this is the hook for
phase-2 pixel-space WRMSE once pipeline-produced per-pixel σ becomes
available -- plug in the ensemble seed-spread as `coef_err` to get a
per-pixel epistemic σ, or `COEF_ERR` for aleatoric σ.

**2026-08-16 — WRMSE reporting (galactic map + worst-rows + trainer).**
Added the per-row WRMSE helper `weighted_rmse_per_row` (companion to the
per-column `_per_column_wrmse` already used by `_metric_row`) and wired
it through the reporting cells: `_metric_row` emits `mean_wrmse`,
`median_wrmse`, `total_wrmse` when sigma is available; the trainer
driver, naive-baseline comparison and per-context-slice metrics pass
`sigma=coef_err_sci_all[...]` and the per-group floor dict. The
galactic map cells and the correlation / scatter panels were switched
from R² to WRMSE (`R^2` map removed; colorbar, hover text and per-region
summaries all report WRMSE now). The worst-case rows report gained a
per-row `sci_wrmse` column alongside the existing `sky1_rmse` /
`sky2_rmse` / `sci_rmse`. All WRMSE code paths gracefully fall back to
floor-only weighting when `coef_err_sci` is missing (older FITS files
without a `COEF_ERR` HDU), so the reporting collapses to plain RMSE
in that case rather than erroring. §9 updated.

**2026-08-16 — coef_err-weighted loss (default).**
The decomposition (``sky_decomp/fit.py`` + ``lsf_surface_iterative.py``)
now writes a ``COEF_ERR`` HDU that carries the active-set 1σ posterior
uncertainty for every coefficient it fits. The FITS layout mirrors
``COEF`` column-for-column; older decomposition files without the HDU
remain readable (missing values become NaN). ``read_decomp_dataset``,
``_load_decomp_with_row_index``, ``build_triplet_coef_dataset`` and
``apply_triplet_filters`` now propagate a per-row ``coef_err_sci``
(plus ``coef_err_near`` / ``coef_err_far`` for symmetry) through the
loaders end-to-end, and cell 18 exposes them as
``coef_err_{near,far,sci}_all`` globals.

The training loss (``§7``) has changed accordingly: the default is
now the heteroscedastic-Gaussian NLL derived in §7. In the code this
is implemented by the new helper ``compress_coef_err_to_score_sigma``
(first-order Jacobian propagation of σ through the geometry step,
nonlinear per-column transform, RobustScaler centering and the
kept-basis projection), followed by a per-column floor and
training-set mean normalisation so the weighted loss magnitude stays
comparable to the previous unweighted variant. The trainer
``train_compressed_group_mlp`` gained two keywords
``use_coef_err_weights=True`` and
``coef_err_sigma_floor_rel=DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP`` (also mirrored in
``default_dual_group_config``); both are logged in the training
artifacts so a saved model records whether it was trained with the
weights. Fallback: setting ``use_coef_err_weights=False`` or feeding
the trainer a filtered triplet without ``coef_err_sci`` collapses the
per-element weight to 1 and reproduces the pre-2026-08-16 loss
exactly.

Follow-up (2026-08-16, same day): a first A/B of the naive default
(scalar floor=0.05, single seed, 15 epochs) showed the weighted loss
cut ``best_val_loss`` in half but degraded test-set physical-space
RMSE by +10% overall (moon +37%, continuum +26%, atomic +15%,
mesospheric +10%, ionospheric -3%).  Root cause: the ``mesospheric``
and ``atomic`` groups have $p_{99}/p_{50}$ sigma ratios of $10^3$-$10^6$
in compressed-score space and a global 5% floor left the loss
dominated by the tiny high-precision tail.  The floor was therefore
made per group: ``moon = continuum = ionospheric = 0.05`` (unchanged),
``mesospheric = atomic = 0.20``.  These defaults live in the module
constant ``DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP`` and are mirrored in
``default_dual_group_config``.  The trainer still accepts a scalar
(broadcast to every group) for future A/B sweeps.

Follow-up (2026-08-16, session end -- retunes at the flat-weight defaults).
After reverting the per-group hand-tuned multipliers and airmass boost
to 1.0 (they were tuned against the pre-2026-08-16 unweighted loss and
overfit the new heteroscedastic-Gaussian surface), a LR / weight-decay
sweep and a trunk / head sweep at seed=42 delivered the following default
changes to ``default_dual_group_config``: ``weight_decay`` $10^{-4} \to 3.3 \times 10^{-5}$,
``patience`` $16 \to 8$ (best-epoch was consistently 18-22 of 30 across
the sweep -- patience=4 was cutting early on the new loss surface),
``head_dim`` $384 \to 192$ (halves per-group heads and improves test
RMSE by ~3% with cleaner convergence).  A parallel trunk = (640, 320)
variant gave a marginally better single-seed RMSE (269.1 vs 270.2) but
did **not** combine additively with head/2 (the two are substitutes,
not complements) and cost 21% more compute per epoch, so head_dim=192
was the shipped choice.  See the sweep cells at the tail of this
notebook for full tables and val curves.  Also new in this session: a
weighted-RMSE report, per-group $\chi^2_\nu$ (which confirms the
sigma-calibration diagnostic quantitatively: $\chi^2_\nu \approx 0.0016$
for mesospheric on the current best model, i.e. the decomposition sigma
over-estimates by ~25x), weighted-$R^2$ and weighted mean-bias, plus a
combined-predictive-sigma helper (aleatoric $\sigma_\mathrm{dec}$ +
epistemic $\sigma_\mathrm{ens}$ in quadrature) for the deployed
ensemble artifact.

**2026-08-16 -- decomposition per-column $\chi^2$ inflation (``fit.py``).**
The active-set posterior 1$\sigma$ estimator ``SkyDecomp._coef_err_active_set``
historically inflated the covariance diagonal by
``max(reduced_chi², 1)`` -- a single scalar shared by every fitted
coefficient in the row.  The MLP sigma-calibration diagnostic (§7)
showed the miscalibration is strongly column-dependent (mesospheric
$\chi^2_\nu \approx 0.0016$, moon $\chi^2_\nu \approx 0.68$), so a scalar
inflation systematically over-inflates well-fit columns and under-inflates
poorly fit ones.  The estimator now accepts a per-column ``per_column_chi2``
vector; when supplied, the inflation becomes ``max(per_column_chi2[j], 1)``
per column.  A new static helper
``SkyDecomp._per_column_chi2_from_residuals`` computes it from the fit
residuals weighted by each column's spectral support
$w_j(i) = |A(i,j)| / \max_i |A(i,j)|$: for emission-line columns the
weight collapses to the few pixels around the line centre, for continuum
spline columns it spreads across the arm.  ``_fit_design`` passes the
vector through automatically; ``per_column_chi2=None`` restores the exact
pre-2026-08-16 behaviour so old callers stay bit-identical.  Seven new
unit tests in ``sky_decomp/tests/test_fit_per_column_chi2.py`` cover the
hand formula, the zero-support fallback, the good-mask restriction, and
the end-to-end inflation semantics inside ``_coef_err_active_set``.
Expected downstream effect once the pipeline regenerates ``COEF_ERR``:
mesospheric sigma should drop $\sim 5\times$ (chi2_nu $\to \sim 0.04$ rather than
0.0016), and the MLP sigma-calibration diagnostic will move much closer
to the ideal $\chi^2_\nu \approx 1$ line.


**2026-08-12 (items 6 + 11: predictive-σ diagnostic + physical-space runaway guard).**
Two review-tier items deployed alongside the v3 training recipe (next entry).
(a) **Item 11 -- physical-space cap on the inference pipeline.**
`expand_scores_to_coefs` gained a `coef_upper_bound` kwarg
(`{group: per-coefficient upper-bound array}`) applied after the existing
$\max(0, \cdot)$ clip. `train_compressed_group_mlp` computes
$U_g = 3 \cdot \max_g(\mathbf{c}_{sci,\mathrm{train}})$ per group per
coefficient and stores it in the artifact under `'coef_upper_bound'`;
`predict_sci_coefficients_default` routes it through automatically for
both single-model and ensemble artifacts, and the ensemble assembler
copies it to the top-level `mlp_artifacts` for introspection. Purpose:
catch asinh-inverse runaways on out-of-envelope trunk outputs without
touching training. Verification on train + val + test (5.14M coefficient
points) shows the cap fires 4 times total ($0.00008\%$), all on the sparse
tail of mesospheric OH lines whose training envelopes are near zero; no
legitimate prediction reaches the cap. Aggregate test `mean_rmse`
unchanged at $19.18$. §6.4 step 11 updated to reflect the two-sided clip.
(b) **Item 6 -- predictive uncertainty from ensemble spread.** A new
diagnostic cell after the trainer computes $\sigma_{ij} =
\mathrm{std}_s(\hat{c}_{s,ij})$ across the 4 ensemble seeds per row per
coefficient (floored at $10^{-3} \cdot \mathrm{MAD}(\mathbf{c}_{sci,\mathrm{train}})$
per coefficient to prevent $\sigma \to 0$ blowups on dead OH lines),
reports per-group $\sigma_{\mathrm{true}}/\sigma_{\mathrm{pred}}$
(RMS-based, robust), median$|z|$, MAD-based robust $\mathrm{std}(z)$,
coverage at $|z|<1.00$ and $<1.96$, and a 20-bin PIT histogram of
$u = \Phi(z)$ on val plus a val$\to$test transfer sanity check. Findings:
moon $2.32\times$, continuum $1.93\times$, mesospheric $2.89\times$,
ionospheric $2.35\times$, atomic $2.11\times$; aggregate $2.89$ on val,
$2.48$ on test. Coverage $|z|<1.96 = 57.4\%$ vs nominal $95\%$; MAD-robust
$\mathrm{std}(z)$ transfers $2.34 \to 2.33$ val$\to$test; PIT
$\chi^2_{20} = 9.5\text{e}5$ vs uniform target $\chi^2_{19} \approx 19$.
Verdict: classic deep-ensemble under-dispersion. Since the under-estimation
factor transfers cleanly from val to test, a single per-group
$\sigma$-scale bolt-on from val closes most of the gap without any
retraining. Per-group scale factors and $\sigma$ floors stored in
`mlp_artifacts['predictive_uncertainty']`; item 6 partially resolves
§11.7 (the full Gaussian-NLL head is still out of scope for a bolt-on).

**2026-08-12 (moon_alt-conditional α + high-airmass row weight; v1/v2 post-hoc calibration retired).**
Two coordinated training-time changes to attack the per-slice bias floor that
remained after the 2026-08-11 mean-bias calibration. Two earlier post-hoc
calibration cells (v1 stratified lifts and v2 4×2 shrunk empirical-Bayes
grids) were removed at adoption because they failed to transfer from val to
test in sub-100-row bins.
(a) **`moon_alt_conditional_alpha=True`.** `DualEncoderGroupHeadMLPCompressed`
now allocates two learnable blend weights for the `moon` group, indexed by
$\mathrm{sign}(\mathrm{moon\_alt})$ at the science pointing (0 = moon
down, 1 = moon up). The per-row 2-value lookup replaces the single scalar
$\alpha_{\rm moon}$ for that group; all other groups keep one scalar. All
four ensemble seeds converge to $\alpha_{\rm moon,dn} \approx 0.66$ and
$\alpha_{\rm moon,up} \approx 0.74$–$0.75$ (gap $+0.08$ to $+0.10$), a
physically-sensible pattern: the near arm tracks the science pointing better
when the moon is above the horizon (both share the moon's illumination
gradient) and less well when it is below.
(b) **`high_airmass_boost=1.25`.** SmoothL1 gains a per-row weight $w_i$
up-weighting rows with airmass $> 1.5$ by $1.25\times$, normalised so
$\overline{w}_{\rm train} = 1$. Cuts the airmass$>1.5$ RMSE gap from $+19\%$
(naive) to $+16.8\%$ and holds airmass$>1.5$ max$|$bias$|$ at $2.2\%$. Two
other boosts were tested in the same design and rejected: `bright_moon_boost`
(illumination $> 0.5$) and `high_f107_boost` (top-decile F10.7), each at
$1.3\times$. Both cut aggregate max$|$bias$|$ from $1.04\%$ to $0.47\%$ but
regressed Q1 mesospheric ($1.46\% \to 2.97\%$) and moon-down moon
($0.68\% \to 1.92\%$) because the renorm implicitly downweighted the dark
and low-airmass rows.
The `sample_weight_config` dict wrapper, the two rejected boost branches, and
the `_sw_conf` indirection were removed from `train_compressed_group_mlp` at
adoption; the trainer's row-weight block is now a single airmass check
exposing `high_airmass_boost` as a scalar kwarg. Ensemble metrics at the new
default (§11.1): mean_rmse $19.18$ (parity with the naive geometry baseline),
median_corr $0.9850$, aggregate max_abs_bias $1.06\%$ (moon). §6.4 step 8 and
§7 updated to reflect the two changes; the artifact `config` dict now records
`moon_alt_conditional_alpha` and `high_airmass_boost` for reproducibility.

**2026-08-11 (pre-fit alpha rejected + moon compression revisited).** Two follow-up
diagnostics on the current defaults, both leaving the config unchanged.
(a) **Pre-fit alpha (`blend_optim='prefit_freeze'` and `'prefit_warmstart'`).**
The trainer gained a closed-form pre-fit path: solve
$\alpha_g^\star = \langle s_{near}{-}s_{far},\, s_{sci}{-}s_{far}\rangle / \|s_{near}{-}s_{far}\|^2$
per group on training scores in scaled score space, then either freeze the
value or use it as an init for AdamW. 4-seed A/B against the current `direct`
baseline (`blend_init_alpha=0.7`): ensemble mean_rmse 19.87 (direct) vs 20.40
(prefit_freeze, +2.7%, ~1.5$\sigma$) vs 20.17 (prefit_warmstart, +1.5%,
~0.8$\sigma$). The marginal OLS $\alpha_g^\star$ is systematically higher
than the joint-training $\alpha$ for the small groups: continuum saturates
at 0.999 (vs joint 0.74), ionospheric at 0.974 (vs joint 0.84). Interpretation:
with only 3-4 continuum/ionospheric scores, near-arm alone tracks sci almost
perfectly on training rows, so the marginal $\alpha^\star$ collapses to
"use near arm only". But the joint $(\alpha, \Delta_g)$ optimum wants both
arms in the blend so the shared trunk representation $h$ can encode both
near/far signals, which the residual head then draws on. That coupling only
exists when $\alpha$ is trained jointly with the head; freezing at $\alpha^\star$
breaks it. `prefit_warmstart` partially recovers as AdamW drifts $\alpha$
back toward the joint optimum (continuum 0.999 -> 0.904; ionospheric
0.974 -> 0.958 at seed=42), but 50 epochs is not enough. Result: pre-fit
modes retained in the trainer as $\{'\text{prefit\_freeze}', '\text{prefit\_warmstart}'\}$
for future diagnostic use but NOT adopted; `direct + blend_init_alpha=0.7`
remains the default. Confirms that the residual head is doing meaningful
group-coupling work, not merely absorbing scale.
(b) **Moon compression revisit.** The historical `pca_rank_diagnostic`
estimate of "$\sim$4 transferable moon directions" no longer applies. On the
current pipeline (post-LSF-surface, post-B$^2$-weighted extinction, post-
`ew` context feature) all 29 moon spline directions have cross-arm correlation
$|xarm| \in [0.541, 0.925]$ -- a broad continuum with no natural gap. The
default `COMPRESSION_XARM_THRESHOLD = 0.55` retains 28 of 29, dropping only
the one direction at 0.541. 4-seed A/B against the current threshold: retaining
all 29 without PCA rotation (`use_pca=False`) is $+2.8\%$ aggregate ($\sim 1.6\sigma$)
because moon spline coefficients are strongly correlated across adjacent knots
and the head predicts them better in the decorrelated PCA basis. Aggressive
truncation (`top_4_by_xarm`) ties the current threshold on aggregate ($-0.04$,
$\sim 0.1\sigma$) but is $3\times$ worse on moon-only RMSE ($0.86$ vs $0.27$)
and drops `median_corr` by $0.005$. A single-component variant
(`xarm > 0.99`, keeps 1 of 29) is $+1.5\%$ aggregate ($\sim 0.8\sigma$).
Confirms the current threshold is a real optimum, not a leftover, and that
moon-only RMSE (which drives downstream moon-subtraction quality) is what
the compression level trades off, not aggregate mean_rmse. `RUN_MOON_COMPRESSION_AB`
-guarded cell retained in the notebook for future re-verification if the
pipeline changes materially.

**2026-08-11 (direct alpha parametrization + alpha_init A/B).** Follow-up to
the same-day mesospheric / geometry adoption below. Diagnostic starting point:
with `blend_init_alpha=0.5` (uniform, previously 0.7 near-arm biased), the
sigmoid parametrization $\alpha_g = \sigma(\ell_g)$ moves $\alpha$ by only
+0.02 to +0.06 by the best epoch. That is trainable but shallow -- $\sigma'(0) = 0.25$
damps the blend-logit gradient by a factor of 4 at init, and the residual head
$\Delta_g(\mathbf{h})$ absorbs most of the near/far mismatch, leaving little
signal for $\ell_g$. A four-way A/B of optimizer variants at seed=42 tested:
(i) `default` (single AdamW group, weight_decay applied to $\ell_g$); (ii)
`no_wd` (blend params in a separate group with weight_decay=0); (iii) `lr10x`
(blend params at 10x the base LR); and (iv) `direct` ($\alpha_g$ stored
directly with no sigmoid, projected clamp to $[10^{-3}, 1 - 10^{-3}]$ after each
opt.step()). Result: (i) `no_wd` is a numerical no-op (WD on 5 parameters was
not the throttle; identical $\alpha$ trajectory to `default`); (ii) `lr10x`
moves $\alpha$ by +0.20 but regresses test mean_rmse by +2.16 because early
lock-in of $\alpha$ (best epoch 8 vs 14) breaks joint training with the
residual head; (iii) `direct` gives $\alpha$ effectively 4x more gradient
signal (no $\sigma'$ damping) without disturbing the base LR, and is the only
variant that both moves $\alpha$ further AND improves test metrics
(single-seed mean_rmse 21.62 -> 21.56 and median_corr 0.9827 -> 0.9830 at
$\alpha_{init} = 0.5$). A follow-up 3-way A/B on `blend_init_alpha` at 4 seeds
each under `direct` mode revealed that the joint $(\alpha, \Delta_g)$
optimum depends strongly on the init: 4-seed ensemble mean_rmse landed at
**20.88** for init=0.5, **19.87** for init=0.7 (winner), and **20.62** for
init=0.8. Init=0.8 does NOT come down toward 0.7; mesospheric $\alpha$
barely moves across the three inits (0.68 / 0.69 / 0.71), continuum locks near
its init (0.60 / 0.75 / 0.80), and ionospheric drifts UP (0.70 / 0.88 / 0.92).
$\alpha$ is therefore a slow-varying prior whose init sets the neighbourhood
of the joint minimum found in ~30 epochs of AdamW, not a parameter that
converges to a unique test-loss minimum. Adopted `blend_optim='direct'` with
`blend_init_alpha=0.7` as the compressed trainer defaults; the class gained a
`blend_use_direct` flag and the trainer a `blend_optim` kwarg with four
supported modes ('default', 'no_wd', 'lr10x', 'direct'). Per-epoch $\alpha$
snapshots are now captured in `artifacts['blend_history']` for diagnostics.
Direct + $\alpha_{init} = 0.7$ gives 4-seed ensemble mean_rmse 19.87 vs 19.96
under the previous sigmoid + $\alpha_{init} = 0.7$ (−0.4%), so this replaces
the earlier default at parity-or-better on the aggregate metric with
substantially better $\alpha$ convergence diagnostics and physically sensible
per-group $\alpha$ ordering (ionospheric 0.88 > atomic 0.86 > continuum 0.75
> moon 0.70 $\approx$ mesospheric 0.69). §5.5 and §6.4 updated to reflect the
new parametrization. -- **Confidence check (same day, follow-up):** the
$\alpha_{init}=0.7$ selection above rested on a 3-point grid ({0.5, 0.7, 0.8}) at
4 seeds each, so a dense grid at seed=42 and an 8-seed check at the winner and its
neighbours were added. (a) A dense grid $\alpha_{init} \in \{0.40, 0.50, 0.55,
0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90\}$ at seed=42 only produced a
non-unimodal curve with mean_rmse spread of 4.28 across the grid -- ~5x the
per-seed std observed in multi-seed runs, dominated by early-stopping timing
(best_epoch collapses from 13-18 for $\alpha \leq 0.70$ to 8 for $\alpha \geq 0.75$).
At seed=42 alone $\alpha_{init}=0.65$ (mean_rmse=20.81) beats 0.70 (21.47), but this
is seed noise: no other seed reproduces it. (b) 8-seed ensembles at
$\alpha_{init} \in \{0.60, 0.70, 0.75\}$ give 19.98 / **19.79** / 20.52. Per-seed
std $\sim 0.72$, so $\sigma_{ensemble} \approx 0.72/\sqrt{8} \approx 0.25$: the
0.70 vs 0.75 gap of 0.72 is a ~2.4$\sigma$ real preference, but the 0.70 vs 0.60
gap of 0.19 is only ~0.6$\sigma$ (a statistical tie). The optimum is a broad plateau
in $\alpha_{init} \in [0.60, 0.70]$; regressions kick in below 0.55 or above 0.75.
(c) A long-budget check at $n_{epochs}=200$, $patience=40$ for
$\alpha_{init} \in \{0.5, 0.7, 0.8\}$ at seed=42 refined the "$\alpha$ is a prior"
claim above: mesospheric $\alpha$ *does* converge to $\sim 0.66$ regardless of
init (per-group spread 0.022), because the mesospheric group is 403 of 441 input
scores and dominates the gradient signal; continuum and ionospheric remain
init-dependent (spread 0.16 and 0.17); moon and atomic partially converge (spread
0.08 and 0.09). Best epoch is unchanged from the 50-epoch runs (8, 14, 17), so the
extra budget did not improve any per-group $\alpha$ or any test metric. Combined
verdict: $\alpha_{init}=0.70$ is the defensible point estimate but the surface is
flat enough that anything in $[0.60, 0.70]$ is within noise; 0.70 kept as the
default for continuity with the historic sigmoid+0.7 baseline.

**2026-08-11 (mesospheric no-PCA + per-group mesospheric weight + multi-seed geometry).**
Three-part revision driven by the naive-baseline check now recorded in §11.1. Before
the change, `B1_near_geo = em_near * G_sci` beat the compressed default by 40% on
aggregate mean_rmse; the shortfall was almost entirely on the mesospheric (403 OH
coefficients) group. Root cause: the score-space blend + residual head is bounded
above by the rank of the retained target subspace, and the PCA + xarm truncation
was keeping only 28 of 403 mesospheric directions; at $\alpha_g \to 1$ the model
output was em_near projected onto that 28-dim subspace, not em_near itself. B1
bypasses the compressor entirely, so it always wins on the truncated 375 directions.
(a) `COMPRESSION_USE_PCA_BY_GROUP['mesospheric']` was flipped from `True` to `False`,
so the mesospheric compressor is now asinh-identity (all 403 scores kept, no rotation,
no truncation). Mesospheric RMSE 36.06 → 21.27 (−41%); aggregate mean_rmse
32.97 → 21.22 (beats B1 by 10%). Lowering the xarm threshold to 0.35 (28 → 230
retained) as a diagnostic closed only ~40% of the gap and regressed the small groups,
confirming that even 230 kept components was lossy relative to raw em_near.
(b) `train_compressed_group_mlp` gained `mesospheric_group_weight` (default 3.0) to
compensate for the $1/\sqrt{403}$ shrinkage — without the lift, mesospheric per-
coefficient loss weight would be ~800× smaller than continuum's. `ionospheric_group_weight`
was added at the same time (default 1.0); a 2.0 experiment closed the ionospheric gap
(5.14 → 4.70) but regressed mesospheric (+15%) and was reverted. A dedicated
ionospheric encoder+trunk+head branch (`DualEncoderGroupHeadMLPCompressedIonHead`)
closed the ionospheric gap fully (5.14 → 4.04 ≈ B1's 4.01) but cost 3% on mesospheric
and 11% on atomic; aggregate mean_rmse went up 0.5, so it was also not adopted.
(c) A 24-config dense sweep at the new 441-dim compressor input
(encoder ∈ {(512,256), (768,384), (1024,512)}, trunk ∈ {(320,160), (512,256)},
head ∈ {288, 384}, lr ∈ {5e-4, 7e-4}) picked `encoder_dims=(1024, 512)`,
`trunk_dims=(512, 256)`, `head_dim=384` at seed=42 with test mean_rmse 19.48. A
3-seed rerun of the top-5 configs at seeds 43-45 showed that seed=42 winner had by
far the highest cross-seed variance (std 1.15 vs 0.75 for the runner-up); its 19.48
was seed noise. The multi-seed winner is a smaller geometry, `encoder_dims=(768, 384)`,
`trunk_dims=(320, 160)`, `head_dim=384`, `lr=7e-4`, with a 4-seed mean mean_rmse of
21.07 ± 0.75 vs 21.43 ± 1.15 for the seed=42 winner. Adopted the multi-seed winner as
`default_dual_group_config` because ensembling collapses to a lower mean_rmse
(19.96 vs 20.18 at the previous adoption) with lower ensemble max |bias|
(1.05% vs 1.20%) and roughly a third less seed variance. `verify_config` was updated
to match. §5.5, §6.4, §7 and §11.1 all updated to reflect the new defaults; §11.11
supplemented below with the multi-seed adoption rationale.

**2026-08-08 (asinh Jensen bias correction, mesospheric under-prediction resolved).** A 4-seed sanity check on the 2026-08-08 dense-sweep default showed the mesospheric group under-predicting by $-1.04\%$ to $-1.96\%$ at every seed, ensemble mean $-1.45\%$ -- a systematic bias, not seed noise. Root cause: the compression pipeline's `asinh` forward transform combined with the PCA + xarm truncation (403 -> 28 retained scores) leaves a per-coefficient asinh-space residual with nonzero variance, and because $\sinh$ is convex on positive arguments, the naive inverse $x = \text{scale} \cdot \sinh(\hat{y})$ under-predicts the true conditional mean by a Jensen factor $E[\cosh(\delta_j)] \approx \exp(\sigma_j^2 / 2)$. `inverse_group_compressor` gained an optional `jensen_correction` per-coefficient argument that multiplies the naive inverse (asinh/log only; sqrt/linear inverses are not multiplicatively biased). `train_compressed_group_mlp` now fits those factors on val rows after training completes by estimating a robust MAD-based sigma $\sigma_j = 1.4826 \cdot \mathrm{MAD}(\delta_j)$, capping at $\sigma_{cap} = 0.4$ (so per-coefficient lift is at most $\exp(0.08) = 1.083$, i.e. $\leq 8.3\%$; the cap guards against heavy-tailed weak-signal coefficients whose empirical mean$(\cosh(\delta))$ is dominated by outliers rather than a well-defined variance -- the naive empirical estimator gave max factors of $\sim 3700$ before capping), and stores the per-coefficient factors in `artifacts['jensen_corrections']`. `predict_sci_coefficients_default` and `expand_scores_to_coefs` auto-route them. Empirically the mesospheric per-coefficient sigmas are concentrated at the cap (mean multiplicative lift 2.3\%, median 0.4\%, p95 8.3\%). Group-level effect: ensemble mesospheric bias $-1.45\% \to +0.52\%$, per-seed mesospheric range $[-1.96, -1.04]\% \to $[-0.05, +0.97]\%$; max ensemble $|$bias$|$ across all five groups collapses from $1.45\%$ (mesospheric) to $0.65\%$ (continuum, which is sqrt-identity and unaffected by the correction). Cost: `test_mean_rmse` $30.53 \to 30.72$ ($+0.6\%$), `median_corr` and `mean_corr` unchanged. The cap value $0.4$ was tuned by a two-point bracket ($1.0$ gave $+2.9\%$ mesospheric overshoot; $0.4$ nulls the bias); it corresponds to a maximum allowable asinh-space per-coefficient residual sigma of $0.4$, meaning coefficients with wider residuals are treated as "unpredictable beyond training-mean shrinkage" and get a fixed $1.083$ lift rather than the runaway empirical value. §5.5 gained a new bullet in "Consequences for the rest of the pipeline" documenting the mechanism. `sqrt` groups (continuum, atomic, ionospheric) and `linear` groups (moon) are not corrected: their inverses are $y^2$ and identity respectively; while $y^2$ also has a Jensen-style shrinkage bias in principle, its coefficients are not PCA-truncated (n=3-4 sqrt-identity compressors), so the residual variance is small and the observed group biases are already sub-$1\%$.

**2026-08-08 (post-LSF-surface dense sweep, new default config).** After adopting the wavelength-dependent LSF surface decomposition (2026-08-07) and loosening the `feo` / `atom_k` hard clips to $[0, 10]$ (retaining 11,494 filtered rows), the 48-config dense sweep was re-run at the corrected pipeline state (same night-held-out split as `mlp_artifacts`, seed=42, n_epochs=50, patience=16, batch_size=256; grid: encoder_dims $\in \{(384,192), (448,224), (512,256), (576,288)\}$, head_dim $\in \{192, 224, 256, 288\}$, lr $\in \{3\text{e-}4, 5\text{e-}4, 7\text{e-}4\}$, trunk=(320,160), wd=1e-4). New winner: `encoder_dims=(512, 256)`, `head_dim=288`, `lr=7\text{e-}4` -- improved test `mean_rmse` 31.13 $\to$ 30.53 ($-1.9\%$) and `median_corr` 0.9639 $\to$ 0.9648 ($+0.0009$) versus the previous default (`encoder_dims=(448, 224)`, `head_dim=224`, `lr=5\text{e-}4`), converging at epoch 19 rather than 16. Adopted as the new `default_dual_group_config`; loss multipliers `moon_group_weight=3.5`, `continuum_group_weight=1.5` and `patience=16` are unchanged. Landscape: all four top configs use lr=7e-4 (previously 5e-4 was preferred), consistent with the larger encoder tier having more directions to descend along per step; the (576, 288) tier does not beat (512, 256), so the capacity ceiling for the current compressed input width (67 scores + 22 context) is reached at (512, 256). Two A/B verdicts were re-checked at the new default and both retained: `drop_vanrhijn_from_context=False` wins by $+2.5\%$ mean_rmse; $(m_{moon}, m_{cont.}) = (3.5, 1.5)$ wins over the previous $(3.0, 1.0)$ by $+0.9\%$ mean_rmse (though 3.0/1.0 has slightly better moon and continuum bias magnitudes and $+0.001$ median_corr -- worth revisiting under multi-seed). Direction of the geometry shift toward a larger encoder + higher LR mirrors the physically-cleaner training signal from the wavelength-dependent LSF surface (the Gaussian LSF absorbed a per-wavelength shape systematic the surface path removes). §6.4 dimension bookkeeping and the §11.11 current-default parenthetical were updated to match, and the verify-cell config was refreshed to the new default. Incidental observation: pca_rank now reports mesospheric transferable dimensions $220 \to 35$ on this larger + cleaner dataset (Horn cutoff also dropped to 19), so the compressed input width is 67 vs 60 previously -- the transferable-mode contraction is consistent with the LSF surface removing low-variance shape residuals that had been showing up as PCA components in earlier runs.

**2026-08-07 (LSF-surface aware reconstruction).** The visual and batch reconstruction cells now read a fitted wavelength-dependent LSF from the decomposition FITS `LSF_COEF` / `LSF_KNOTS` / `LSF_META` extensions produced by `sky_decomp.lsf_surface_iterative` when available, instead of the Gaussian sigma derived from the input FITS `LSF_SCI` column. Two new helpers in the reconstruction-helpers cell -- `load_lsf_state_if_available` and `reconstruct_with_lsf` -- do the dispatch: `reconstruct_with_lsf` accepts either an `LSFSurfaceState` (routes through `SkyDecompLSFSurfaceIterative._assemble_refined_matrices`, which convolves stick matrices with the fitted per-wavelength 11-tap B-spline kernel via `apply_lsf_surface`) or a per-pixel Gaussian sigma (routes through the existing `reconstruct_component_spectra`). The per-arm reconstruction sites (single-row visualisation cell + batch RMSE cell) now try to load the surface state from each arm's own decomposition FITS at the requested input-row index, and fall back to the Gaussian LSF_SCI derived sigma if the extensions are absent. Backward-compatible: on decompositions produced by the legacy `sky_decomp.fit.SkyDecomp` (no LSF surface written) every reconstruction transparently uses the previous Gaussian path. `coef_wavelengths_from_basis` is not switched over -- basis-centroid positions are invariant under a symmetric convolution kernel, so the wavelength / effective-extinction cell is unaffected. Motivation: the red arm's LSF varies noticeably in shape across each channel (the surface B-spline captures asymmetry and per-wavelength width the single-sigma Gaussian cannot), so reconstructions of new decompositions will more faithfully reproduce the observed line profiles. No config change; simply re-run the reconstruction cells and they will pick up the surface state from compatible decomp files.

**2026-08-05 (per-arm telescope identity `ew` added to context).** A new context feature `ew` was added to `context_cols`: $+1$ for SKYE, $-1$ for SKYW, $0$ for the sci arm (which does not have an E/W identity). Resolved at context-build time by `_resolve_context_feature` from the existing `SKY_NEAR_LABEL` / `SKY_FAR_LABEL` META columns (§2). Motivation: the near / far assignment shuffles between SKYE and SKYW row-by-row based on angular separation from the science pointing, so any per-telescope throughput offset would otherwise enter as a geometry-correlated systematic in the sky-to-science transfer (§11.9 flagged this). Making the identity available lets the shared sky encoder absorb any offset as a linear correction. `context_cols` grew from 27 to 28 features; the encoder-input width in §6.4 goes from $Q - 3 = 24$ to $Q - 3 = 25$ (the three `vanrhijn_*` columns are still dropped at the encoder boundary under `drop_vanrhijn_from_context=True`). Rebuild the triplet and re-run the geometry / structure-function / compressor / trainer cells; this is a context-schema change, so `mlp_artifacts` fit on the 27-feature schema will not load. Closes the remaining telescope-identity item in §11.9.

**2026-08-04 (OH wavelength single-source: population-model route retired, §11.5 resolved).** `oh_wavelengths_from_input_file` and `cross_check_wavelengths` were deleted from the loaders cell, and `resolve_coef_wavelengths_a` lost its `input_wavelengths_a` parameter. The wavelength / extinction cell now calls `coef_wavelengths_from_basis` for every coefficient (previously restricted to the population-model-unresolved subset plus a 40-coefficient cross-check sample), returning both the intensity-weighted centroid AND the B$^2$-weighted effective extinction from the same reconstruction pass. The disk-cache filename was bumped from `coef_wavelengths_basis.npz` to `coef_wavelengths_basis_v2.npz` so any pre-existing partial cache is invalidated on first run of this update. Motivation: the population-model route weighted each OH group's centroid by $A_{ij} g_i$, ignoring the rotational Boltzmann factor $\exp(-E_{rot}/kT_{rot})$ at mesopause $T_{rot} \sim 200\,$K. High-$J$ lines were therefore over-counted and centroids drifted by a few $\text{\AA}$ (a few tens of $\text{\AA}$ in the worst compact-band cases) relative to the physically correct intensity weighting the basis carries by construction. Effect on the sky-to-science transfer was negligible ($\Delta \ln R \lesssim 2 \times 10^{-5}$ per 10 $\text{\AA}$ centroid shift, well below the gravity-wave floor and single-seed training noise), so this is a physical-correctness fix rather than a model-quality one -- but with the basis route already providing B$^2$-weighted extinction as of 2026-07-30, the population-model auxiliary path stopped earning its complexity. Re-run the wavelength / extinction cell (this triggers a one-time full basis rebuild; each `reconstruct_component_spectra` call is ~2 s, so ~15 min for 442 coefficients on this dataset; subsequent runs load the `.npz` cache in ~0 s). Then re-run the compressor and trainer cells.

**2026-08-04 (solar-activity coupling documented in §9.2).** Per-coefficient Pearson $r$ between the three new solar-activity features and $\log$ of the zenith-equivalent target-arm emissivity was measured on the filtered triplet ($n = 10\,743$ rows, MJD $60177 \to 61086$, covering the ascent and peak of solar cycle 25) and added as §9.2 to the methods cell. Summary: `ionospheric` is the only group that correlates with F10.7 as a group (median $r = +0.174$), driven by the four F-region recombination coefficients (OI 8446 leads at $r = +0.246$). `atomic` shows a physically-correct negative Na D coupling ($r = -0.158$). `mesospheric`, `continuum`, and `moon` are at the noise floor. Kp is uniformly weak (LCO's $\sim -19^\circ$ geomagnetic latitude, sub-auroral). This is a documentation-only change; no code or config was modified.

**2026-08-04 (solar-activity context features from GFZ).** Three new context features -- `f107` (observed 10.7 cm solar radio flux at the closest daily grid point, sfu), `f107_81d` (81-day running mean of the observed F10.7, sfu), and `kp` (Kp value at the 3-hour window containing the observation, 0-9 scale) -- are now fetched from GFZ Potsdam's authoritative daily archive (`https://kp.gfz.de/app/files/Kp_ap_Ap_SN_F107_since_1932.txt`) and cached locally under `solar_activity_cache/`. The raw text file is re-downloaded if missing or older than 7 days, then parsed and stored as an `.npz` for fast reload; the parsed table is also memoised at module scope (`_SOLAR_ACTIVITY_TABLE`) so a single kernel session pays the parse cost once. A new module-level `SOLAR_ACTIVITY_FEATURES` set is recognised by `_resolve_context_feature`, which dispatches to `_solar_activity_lookup` (linear interpolation for daily F10.7, nearest-lower-window lookup for 3-hourly Kp; both return NaN for out-of-range MJDs so any downstream row would be dropped by the finite-context filter rather than silently getting a boundary value). `context_cols` in the starter data-load cell grew from 24 to 27 features. Motivation: OI 6300 / 6364 (`ionospheric` group at 285 km) is F-region recombination and its intrinsic amplitude tracks EUV flux, for which F10.7 is the standard proxy; Kp gates auroral and short-timescale storm forcing that also modulates 285 km emission. `f107_81d` supplies the $\sim 3$-solar-rotation baseline separate from the $\sim 27$-day active-region rotation modulation carried by `f107`. Rebuild the triplet and re-run the geometry / structure-function / compressor / trainer cells for downstream state to reflect the expanded context ($Q = 24 \to 27$, so $\mathbf{z}_{fuse}$ dimension shifts accordingly at fit time). Closes the corresponding item in §11.9.

**2026-07-30 (post-audit dense-sweep re-run, new default config).** After the audit-follow-up fixes brought the per-group biases to sub-0.5%, the 36-config dense sweep was re-run at the corrected pipeline state (same night-held-out split as `mlp_artifacts`, seed=42, n_epochs=50, patience=4, batch_size=256). New winner: `encoder_dims=(448, 224)`, `head_dim=224`, `lr=5e-4`, `trunk_dims=(320, 160)`, `weight_decay=1e-4` -- improved test `mean_rmse` 56.27 $\to$ 55.85 (-0.42) and `median_corr` 0.9209 $\to$ 0.9225 (+0.0016) versus the previous default (`encoder_dims=(320, 160)`, `head_dim=192`, `lr=1.5e-3`). Adopted as the new `default_dual_group_config`; `moon_group_weight=3.5`, `continuum_group_weight=1.5` are preserved (they had no measurable effect at these low bias levels but were kept as-is until a multi-seed check). Direction of the shift is physically sensible: the cleaner training signal after the continuum-airglow fix + B$^2$-weighted extinction supports a larger model at a slower LR without overfitting; the previous default was the smaller / faster geometry the noisier pre-fix signal preferred. `RUN_DENSE_SWEEP` flipped back to False after adoption and the sweep-cell comment updated to reflect the new winner.

**2026-07-30 (audit follow-up: ATOM_N reclass, continuum fallback, val-only xarm, broadband B^2-weighted extinction).** Four small fixes from a systematic pipeline audit after the continuum reclassification. (a) `ATOM_N` (N I 5199 A, [N I] $^2$D $\to$ $^4$S forbidden doublet) moved from `atomic` (95 km) to `ionospheric` (285 km): the nightglow [N I] 5199 emission is thermospheric F-region recombination, physically the same class as OI 6300 / 6364. Small numerical impact but corrects the physical grouping; per-group sizes on the standard product go `atomic=3, ionospheric=4`. (b) A `'continuum': 7500.0` fallback was added to `GROUP_EFFECTIVE_WAVELENGTH_A` so the extinction path degrades to a mid-band value if the basis-centroid route ever fails to resolve HO2 / FeO / O2Ac; in practice the basis route always resolves them, so the fallback is defensive. (c) Per-group compressor cross-arm selection now uses `_split_held = _split_va` (val only). The previous val+test held-out set let test-set correlations shape which score directions survived the `r_xarm > 0.55` cut, which is preprocessing leakage; the 25$\sigma$ threshold makes the numerical effect small but the protocol is now rigorous, and §8.1 was updated. (d) `coef_wavelengths_from_basis` was extended to compute a B$^2$-weighted effective extinction $k_{eff}(j) = \int f_j^2 k(\lambda) d\lambda / \int f_j^2 d\lambda$ in the same reconstruction pass that produces the centroid wavelength, and `resolve_coef_extinction_k` now accepts it via `coef_basis_k_generic` as an override to `_lco_extinction_k(\lambda_{centroid})`. Motivation: for a broadband basis component the least-squares decomposition amplitude picks up $\langle f(\lambda) \rangle_{f^2}$, so $\langle k \rangle_{f^2}$ is the right effective extinction, not the point value at the centroid. For narrow-line components (OH, atomic, ionospheric) $|f_j|^2$ is sharply peaked and both agree; the correction only affects HO2 / FeO / O2Ac. The cache format gained a `k_eff_a` array; an older cache without it is auto-invalidated on first read. Rerun the wavelength / extinction cell, then the compressor, then the trainer.

**2026-07-30 (continuum reclassified as mesopause airglow at 87 km).** `AIRGLOW_GROUPS` and `GROUP_HEIGHT_FEATURE` in the loaders cell now put the `continuum` group -- HO$_2$ (H+O$_2$+M chemiluminescence, ~87 km), FeO (Fe+O$_3$ orange arc, ~85-90 km) and O$_2$Ac (O+O+M -> O$_2$ Chamberlain bands, ~90-95 km) -- at 87 km, alongside the OH Meinel bands. Motivation: all three are mesopause chemiluminescence, not aerosol or Rayleigh continuum, and their sky-to-sci transfer follows the same van Rhijn slant-path law as OH. Previously they were treated as V=1 non-airglow, which forced the encoders to learn the airmass dependence of a thin-shell emitter from context features alone -- a coherent systematic the network was measurably absorbing (see the drop-vanrhijn A/B below, where the KEEP variant relied on the vanrhijn_* columns to fake the correction for continuum, ionospheric and moon at the encoder input). The group boundary is preserved because HO$_2$ / FeO / O$_2$Ac are broadband basis components rather than line-group templates, so their compression pipeline stays sqrt-identity with no PCA, but the geometric normalisation is now shared with `mesospheric` via `airglow_geometry_scale`. §4.3 and the group-layer comment block in the loaders cell are updated. Downstream cells must be re-run in order: (a) the wavelength / extinction cell, so that `HO2` / `FeO` / `O2Ac` join the airglow_idx that receives extinction; (b) the compressor cell, because the training-set range of these three coefficients changes when their amplitudes are divided by V(87 km) $\in$ [1, 6]; (c) the default training cell.

**2026-07-30 (drop_vanrhijn_from_context A/B).** Trained the compressed default at seed=42 with `drop_vanrhijn_from_context` False and True, identical split / hyperparameters, on the pre-continuum-fix schema. KEEP wins by +0.9% test mean_rmse (56.35 -> 56.87), median_corr essentially tied (+0.9237 -> +0.9234, delta -0.0003). Per-group mean bias tells the more interesting story: bias_keep_% / bias_drop_% is +0.78 / -2.19 (moon), -1.70 / -0.54 (continuum), +0.49 / +0.41 (mesospheric), -1.87 / -0.98 (ionospheric), +0.68 / +0.87 (atomic). Under DROP the moon head gets worse (more magnitude of bias), but the continuum and ionospheric biases actually shrink. That is diagnostic: the moon head is using vanrhijn_* as a smooth altitude proxy alongside alt/airmass; the continuum head is using vanrhijn_87km to fake the van Rhijn correction that airglow_geometry_scale should have been applying to HO2/FeO/O2Ac, and hiding that column pushes it closer to unbiased. This is the direct empirical signature that motivates the continuum reclassification above. `drop_vanrhijn_from_context` stays at False; the moon-head benefit is real and larger than the accidental continuum benefit, and once the continuum fix is applied the accidental benefit disappears anyway.
rank, compressor, training) must be re-run against the new `filtered_triplet`
**2026-07-30 (asinh inverse clip).** `fit_group_compressor` now caches per-column min/max of the training data in forward-transformed (asinh / linear / sqrt / log) space as `y_train_min` / `y_train_max`, and `inverse_group_compressor` clips the reconstructed pre-inverse-transform `z` to `[y_train_min - 0.5, y_train_max + 0.5]` when the transform is `asinh` or `log`. Motivation: on the compressed default model, the diagnostic in cell 24 (mean bias per group) showed the mesospheric mean predicted amplitude sitting at ~7000% of the true mean (median matched), while the linear-inverse groups showed sub-1% bias. Cause: forward `asinh` is $\log$-like and bounded on physical inputs, but the inverse `sinh` is exponential — a small handful of predicted scores landing at $z \gtrsim 15$ per coefficient produces `sinh(z) * per_col_scale` values 3–6 orders of magnitude above the physical amplitude range, dominating the mean without moving the median. The clip is a no-op inside the training envelope and only fires on tail-of-tail extrapolation. Backward-compatible: `inverse_group_compressor` skips the clip if `y_train_min` is not present in the stored compressor. §5.5 and §12 unchanged for `linear` / `sqrt` groups, which are quadratic-inverse or identity and don't need the clip.

**2026-07-30 (continuum_group_weight restored, full loss-weight reporting).** `train_compressed_group_mlp` gained a `continuum_group_weight` kwarg (default 1.0) that mirrors the existing `moon_group_weight`: both are per-group multipliers $m_g$ applied on top of the base $1/\sqrt{n_{g,\text{score}}}$ weight. Internally the two scalars are collapsed into a `_group_multipliers` dict inside the training driver, so future per-group lifts add a single dict entry. `default_dual_group_config` records the new key and forwards it, and it is echoed into `artifacts['config']` so the value that trained a given checkpoint is auditable. The per-epoch printout was expanded from a one-line summary of $w_g$ into a per-group table showing `n_score`, the multiplier `m_g`, the effective `w_g`, and `w_g / n_g` (the per-coefficient weight), so under- and over-weighting are visible at a glance. Default behaviour is unchanged (`m_moon = 3`, `m_continuum = 1`); the knob is only meant to be used if the training-time bias diagnostic shows a persistent continuum under-fit. §5.5 (loss balance bullet) and §7 (retirement paragraph) were updated. The sweep cells still do not vary these multipliers.

**2026-07-30 (retired path removed).** `_group_loss`, `_group_loss_breakdown`, `train_coeff_prediction_group_mlp`, and `predict_sci_coefficients_group_mlp` were deleted from the network-architecture cell. Those are the loss, per-group breakdown, training driver, and predictor for the uncompressed baseline (§6), which the notebook has not trained as its default since the 2026-07-28 adoption of per-group compression. The parent class `DualEncoderGroupHeadMLP` is retained because `DualEncoderGroupHeadMLPCompressed` still inherits from it (shared encoder / trunk / head construction and the `_select_ctx` context-column filter). The `continuum_group_weight` and `moon_smooth_lambda` knobs were only ever consumed by the removed loss and driver; they are now gone from executable code as well as from the retirement notes. The `predict_sci_coefficients_group_mlp` reference in §4.2 was updated to `predict_sci_coefficients_default`.

**2026-07-30 (schema fix + new default config) — COEF_SCHEMA regex assignment, sweep re-run, `encoder_dims=(320, 160)` / `lr=1.5e-3` adopted.** A schema audit found that three broadband continuum coefficients (`HO2`, `FeO`, `O2Ac`) were being routed by the old substring-based grouper into the `mesospheric` group with the 402 OH coefficients, and that `ATOM_Or` / `ATOM_Orc_OI0777` / `ATOM_Orc_OI0845` (OI 6300/6364, 7774, 8446) were routed into `atomic` instead of `ionospheric` — the ionospheric group was in fact empty, since the old matcher looked for `"6300"` / `"redline"` tokens that no coefficient name carries. Both mistakes were silent because the grouper folded any unmatched name into `other` and the schema mapper never re-checked names it had claimed. `_build_group_indices()` was replaced with a hard-wired `COEF_SCHEMA` list of `(regex, group, wavelength, description)` covering every design_name emitted by `SkyDecomp._build_static_basis()`, and now raises `RuntimeError` on any unmatched name; the wavelength resolver was also stripped of its digit-token and species-substring fallbacks so the schema is the single source of truth. A cell-level self-test on a canonical name list runs at import time. New per-group sizes: `moon=29, continuum=3, mesospheric=403, atomic=4, ionospheric=3` (was `moon=29, continuum=0, mesospheric=406, atomic=7, ionospheric=0`). Loss-weight consequence: `w_continuum` 0→0.577 and `w_ionospheric` 0→0.577 (per-coef weight for the diffuse continuum block rose ~40×), so a sweep re-run at the fixed pipeline state was needed. Both `RUN_SWEEP` and `RUN_DENSE_SWEEP` were flipped True; the two-stage sweep was informative but the dense sweep is the trustworthy leaderboard (exhaustive grid at the default budget, same night-held-out split as `mlp_artifacts`, seed=42). Dense winner: `encoder_dims=(320, 160)`, `head_dim=192`, `trunk_dims=(320, 160)`, `lr=1.5e-3`, `weight_decay=1e-4` — improved test mean_rmse 59.06 → 56.64 (−2.42) and test median_corr 0.9212 → 0.9233 (+0.0021) versus the previous default (`encoder_dims=(384, 192)`, `lr=1e-3`, everything else identical). Adopted as the new `default_dual_group_config`; `moon_group_weight=3.0` is preserved. Both sweep gates flipped back to False after adoption. The 2026-07-30 (post-fold audit) entry below was written before the schema fix and its `mean_rmse ≈ 131` / winner geometry no longer apply — the schema fix changes the training signal, not just the leaderboard.

**2026-07-30 (post-fold audit) — wider sweep, no bugs found, moon_phase fold retained.** Followed up the moon_phase sin/cos adoption with a widened 36-config dense sweep (added encoder tier `(448, 224)` and lr `1.5e-3` to the previous 18-config grid) and an end-to-end code audit of the four sin/cos touchpoints (loader resolver, plot decoder, moon-phase reconstruction helper, and the compressor / driver split call sites). The audit found no functional bugs. One incidental side-effect worth noting: because `_moon_phase_deg_from_ctx` reconstructs degrees via `np.rad2deg(np.arctan2(sin, cos)) % 360` at float32 precision, the reconstructed phase differs from the raw META column by ~0.1° per row, which can flip a night very close to a phase-quantile boundary into an adjacent bin. So the phase-stratified split at seed=42 is not bit-identical to the pre-fold split, and the `pre-fold sweep mean_rmse ≈ 119` vs `post-fold sweep mean_rmse ≈ 131` gap is not a strict apples-to-apples comparison — the test rows differ slightly. The widened sweep confirms the current default (`encoder_dims=(384, 192)`, `head_dim=192`, `lr=1e-3`) is the winner across all 36 configs; runner-up (`(448, 224)/h=160/lr=1.5e-3`) is 8% worse on mean_rmse. The 10% gap versus the pre-fold pipeline is single-seed variance amplified by the phase-quantile-boundary sensitivity above; the sweep variance across configs spans 131 → 14 220, which is the natural noise floor of a single-seed comparison at this budget. No config change adopted; the folded moon_phase representation is retained as it removes a discontinuity the model cannot interpolate through.

**2026-07-30 — moon_phase sin/cos encoding, plot decoders, new default config.** The META `moon_phase` column (angle in degrees, 0–360) wraps discontinuously at the same 0°/360° boundary that motivated the azimuth folding. It is now added to `context_cols` as the pair `(moon_phase_sin, moon_phase_cos)` instead of raw degrees, driven by a new module-level constant `CYCLIC_META_DEGREE_FEATURES = {'moon_phase'}` in the loader and a corresponding branch in `_resolve_context_feature` that recognises any `<stem>_sin` / `<stem>_cos` name whose stem is in that set and folds it from the raw column. `context_cols` grew from 22 → 23 features. The moon-phase-stratified splitter still works: a new helper `_moon_phase_deg_from_ctx(filtered, arm='sci')` reads whichever encoding is present (raw degrees or sin/cos via `arctan2`) and returns degrees in [0, 360), and it is now wired into every call site of `split_indices_by_moon_phase` (uncompressed driver, compressed driver, compressor cell, effective-rank helper). For plot legibility a new `_decode_cyclic_context(ctx_names, ctx_matrix)` helper folds `<stem>_sin` / `<stem>_cos` pairs back into a single `<stem>` axis in degrees, and both the filter-cell distribution plot and the relationship-plots cell now render one 0–360° axis per cyclic feature (az, moon_az, sun_az, moon_phase) instead of two orthogonal sin/cos axes. Dense sweep re-run at the new pipeline state adopted `encoder_dims=(384, 192)`, `head_dim=192`, `lr=1e-3` (trunk and weight_decay unchanged) as the new `default_dual_group_config`. Compared with the previous default (encoder_dims=(320, 160), head_dim=192, lr=1e-3), test mean_rmse improves 187 → 131 (−30%), test median_corr 0.849 → 0.869. The sweep-best mean_rmse at the sweep-best geometry moved 119 → 131 relative to the pre-fold pipeline (median_corr unchanged at 0.869), i.e. the folding cost roughly 10% on mean_rmse at the sweep optimum in exchange for a continuous phase representation; the winning geometry class also grew back from (320, 160) to (384, 192), consistent with one extra input dimension benefiting from more first-layer capacity.

**2026-07-29 (blend head) — blend + residual head, and a moon loss multiplier.** Two structural changes to `DualEncoderGroupHeadMLPCompressed` motivated by a continuum-overprediction failure mode on bright-moon / near-adjacent rows: rows where the science pointing sat within a few degrees of the near arm and the model still over-predicted the near-arm continuum by roughly a factor of two, unchanged by adding `sci_sep` to the context earlier the same day. First, the forward pass now predicts a signed residual $\Delta_g$ on top of an explicit per-group learnable convex blend of the two sky-arm scores, $\hat{\mathbf{s}}_g = \alpha_g\,\mathbf{s}_{near,g} + (1-\alpha_g)\,\mathbf{s}_{far,g} + \Delta_g(\mathbf{h})$, with $\alpha_g = \sigma(\ell_g)$ initialised at 0.7 (near-arm biased). The previous "predict $\hat{\mathbf{s}}_g$ directly" head had no residual / skip path and its loss-minimising point was the training mean; because the encoder fusion $[\mathbf{e}_{mean};\mathbf{e}_{diff};|\mathbf{e}_{diff}|;\mathbf{e}_{ctx}]$ is symmetric-with-sign under near/far exchange and the PCA-kept subspace sits closer to the mean than to either arm, the model had no way to write "output the near arm verbatim" even when the near arm was a few degrees from the science pointing. Making the blend explicit fixes it, and the learned per-group $\alpha_g$ is printed at the best epoch. Second, `train_compressed_group_mlp` gained a `moon_group_weight` kwarg (default 3.0) which multiplies the moon head's $1/\sqrt{n_{g,\mathrm{score}}}$ loss weight; `default_dual_group_config` records it and forwards it through the sweep and verify paths, and the artifacts config records the value that was used. Effective per-group loss weights become moon=0.60, mesospheric=0.174, atomic=0.378 (from 0.200/0.174/0.378). Diagnostic mean coefficient bias on the training distribution went moon +4.2%, others flat → moon +1%, others -0.5%. `sci_sep` (per-pointing angular separation from the science pointing) was added to `context_cols` earlier the same day and is retained: on its own it did not shift the continuum bias, but the blend head can now condition its residual on how close the near arm actually is to the science target via the shared encoder. The relationship-plots cell and the filter-cell histogram both skip the `sci_sep` panel for the science field, where the value is 0 by construction.
wide random sweep and the dense sweep both remain in the notebook as
**2026-07-29 — azimuth sin/cos encoding and new default config.** Pointing,
moon, and sun azimuth (previously `az`, `moon_az`, `sun_az` in degrees) are
now folded to `(az_sin, az_cos, moon_az_sin, moon_az_cos, sun_az_sin,
sun_az_cos)` in the astropy geometry computation, matching the convention
already used for the folded obstime features. Raw azimuth in degrees jumps
at the 0°/360° boundary, which the model cannot interpolate through: on the
same dense hyperparameter grid this alone cut mean per-coefficient RMSE by
35% at the old default config (356 → 231) and 11% at the sweep-best geometry
(134 → 119). `context_cols` grew from 19 to 22 features (three raw azimuths
removed, six sin/cos pairs added). Dense sweep re-run at the new pipeline
state adopted `encoder_dims=(320, 160)`, `head_dim=192`, `lr=1e-3` (trunk
and weight_decay unchanged) as the new `default_dual_group_config`. Compared
with the previous default (encoder_dims=(384, 192), head_dim=224, lr=5e-4),
mean_rmse improves 231 → 119 (−49%), median_corr 0.833 → 0.868. Sky encoder
shrinks by ~30% (first-layer params, since the input width of the encoder is
the same: 77 compressed scores + 22 context). `default_dual_group_config`,
the geometry helper `_compute_astropy_geometry`, and the `context_cols` list
in the starter data-load cell are the touched code. The dense-sweep cell
also gained a missing `import ast` at cell scope; without it the
leaderboard printed but the final `dense_best_config` assembly raised
`NameError` when the wide-random sweep had not been run in the same kernel.

**2026-07-29 — geometry computed from RA / DEC + obstime via astropy.**
**2026-07-29 — geometry computed from RA / DEC + obstime via astropy.** The
pointing altitude, azimuth and airmass, together with Sun and Moon positions
(alt / az) and pointing-to-Sun / pointing-to-Moon angular separations, are now
computed from `SCI_RA`, `SCI_DEC`, `SKY_NEAR_RA/DEC`, `SKY_FAR_RA/DEC` and
`obstime_mjd` via `astropy.coordinates` at the LCO EarthLocation, rather than
read directly from META (see `_compute_astropy_geometry` and
`ASTROPY_GEOMETRY_FEATURES`). Centralising the geometry means a per-column
error in the input META (which happens for online-derived quantities) does
not silently propagate through the pipeline. Verification against META on the
17260-row meta_only input: astropy sun_alt, moon_alt and pointing alt agree
with the stored META values to a median 0.009° and 95th-percentile 0.014°
(consistent with a small LCO site-coordinate offset); airmass agrees to a
median 2×10⁻⁴ (the large max is from near-horizon rows where sec(z) diverges).
`context_cols` gained the four features the old META schema did not expose:
`az` (per-pointing azimuth, one of the requested additions), `moon_az`,
`sun_az`, and `sun_sep` (angular separation between the pointing and the Sun).
`moon_phase` is not in the astropy set: illumination fraction is not a pure
geometry feature and is still read from META. Rebuild the triplet and re-run
the geometry / structure-function / compressor cells for downstream state to
reflect the expanded context (n_ctx: 15 → 19).

**2026-07-29 — moon-phase-stratified split.** The night-level train/val/test
split was previously a plain random shuffle of nights (`split_indices_by_night`).
It is now stratified by moon phase (§8.1). Nights are sorted by their median
`moon_phase` and cut into ten equal-count phase quantiles; within each bin the
nights are shuffled and assigned 80/10/10 to train/val/test with a guarantee
of at least one night per split per bin. Val and test therefore cover the full
dark → bright range roughly uniformly, so error estimates on the small held-out
sub-samples are no longer at the mercy of the shuffle placing all dark (or all
bright) nights in one bucket. The compressor cell, the compressed training
driver, and the effective-rank helper all call `split_indices_by_moon_phase`.

**2026-07-28 — spectrum-space evaluation gated to the training filter set.**
The random-subset batch RMSE cell now applies chi2 (max reduced-$\chi^2$
over near/far/sci, 90th percentile capped at 10) and LMC/SMC field
exclusion (10$^\circ$ each) to the every10 triplet BEFORE drawing the
random rows. That keeps the evaluation set out of tiles the training set
excludes, so `sci_pred_vs_true` no longer picks up contamination from the
Magellanic Clouds or from decompositions the training set had already
thrown out. Hard coefficient bounds and kappa-sigma clipping are
deliberately NOT applied to the evaluation set: those are training-time
filters on the target, and applying them would remove the model's
hardest true cases from the reconstruction diagnostic.

**2026-07-28 — filter order reorganised.** Inside `apply_triplet_filters`,
the LMC/SMC field exclusion (§3, step 1) now runs before any other filter, and
thinning (§3, step 6) runs last. The intermediate per-filter counts are
reported against the full pre-thinning dataset (rather than a downsampled
subset), and "final kept rows" is announced at the very end. Boolean AND is
commutative, so the final `filtered_triplet` and every downstream number are
identical to the previous ordering; only the printed diagnostics change.

**2026-07-28 — Magellanic Cloud field exclusion.** `apply_triplet_filters`
now excludes tiles whose science pointing lies within 10$^\circ$ of the LMC
(SIMBAD J2000 centre $\alpha=80.894^\circ$, $\delta=-69.756^\circ$) or the SMC
($\alpha=13.187^\circ$, $\delta=-72.829^\circ$). Rationale: §11.2 — both

Clouds host bright stellar populations, dense H II regions and diffuse ionised

**2026-08-11 (4-seed ensemble as deployed default).** `predict_sci_coefficients_default` gained an `is_ensemble` dispatch at the top: when the artifact carries `members` and `is_ensemble=True` it recursively predicts with each member and returns the arithmetic mean in physical space. The trainer cell now fits N=4 members at seeds {42, 43, 44, 45} with identical config in a loop, assembles them into an ensemble `mlp_artifacts` dict (shared fields -- compressors, geom_kwargs, group_indices, score_slices, splits, coef/ctx names, config -- lifted from members[0] to the top level for backward-compat), and prints per-seed + ensemble test metrics plus the stderr analysis. Downstream cells (relationship plots, batch RMSE, single-row reconstruction, pipeline-state check, naive baselines) get the ensemble prediction transparently via the same `predict_sci_coefficients_default(mlp_artifacts, ...)` call they used for the single-seed model; no per-cell edits were needed. The multi-seed sanity cell no longer retrains; it reads `mlp_artifacts['members']` and prints the per-group bias table that shows whether individual seeds carry biases the ensemble averages out. -- Is 4 enough? Single-seed test `mean_rmse` std on this dataset is $\sigma \approx 0.72$ (8-seed confidence check). Ensemble-mean stderr = $\sigma/\sqrt{N}$: 0.72 at N=1, 0.36 at N=4, 0.25 at N=8, 0.18 at N=16. At N=4 the ensemble max per-group |bias| lands below 1% (from ~1.5-2% single-seed), which is the deployment-relevant threshold from §11.1; the residual ~1.8% mean_rmse uncertainty is well under the target-contamination floor of §11.2. The 30% mean_rmse stderr improvement at N=8 (0.36 -> 0.25) costs 2x training time and mostly buys tighter PIT calibration for §11.7, which is not blocking today. N is exposed as `default_dual_group_config['ensemble_seeds']` for one-line adjustment; a training-time "is N enough" printout (seed std, ensemble stderr, stderr as % of ensemble mean_rmse) makes the tradeoff visible on every run. §5.5 gained a new "Predictor is an N-seed ensemble" bullet; §11.7 (predictive uncertainty) is partially addressed by the ensemble spread and remains open only for calibrated per-row $\sigma$ from a Gaussian-NLL head.
**2026-08-11 (batch-RMSE cell reconstruction hoisted).** The RMSE-vs-truth diagnostic loop (25 rows x 3 arms) previously rebuilt `SkyDecompLSFSurfaceIterative` -- and its OH-stick basis, solar-reference read, moon B-spline, atomic/ORC/O2 blocks -- inside every one of the 75 reconstruction calls, and re-opened each decomp FITS twice per row (`load_lsf_state_if_available` opens once for the extension check and once via `load_lsf_surface_state`), plus re-reading the entire `VECTOR_O2` cube on every `load_o2_vector_if_available` call. Restructured cell 24 to (a) build ONE `SkyDecompLSFSurfaceIterative` before the loop (after asserting the sampled wave grid is shared -- true for every10 today; if that ever breaks a runtime error steers the user back to the per-row `reconstruct_with_lsf` fallback), and (b) precache `has_lsf` + the full `VECTOR_O2` cube once per decomp file, converting per-row lookups to array indexing. The loop now calls a local `_fast_reconstruct` that reuses the hoisted model via `_set_lsf_state` + `_assemble_refined_matrices` + `_components_from_coef`, injecting the cached O2 row into `mats['o2']` before evaluation. Numerically identical to the pre-optimization output (same basis, same coefficients, same LSF states); purely a runtime restructure. Timing prints (`recon setup` and `recon loop`) surface the payoff on the user's own dataset.**2026-08-11 (batch-RMSE cell reconstruction hoisted).** The RMSE-vs-truth diagnostic loop (25 rows x 3 arms) previously rebuilt `SkyDecompLSFSurfaceIterative` -- and its OH-stick basis, solar-reference read, moon B-spline, atomic/ORC/O2 blocks -- inside every one of the 75 reconstruction calls, and re-opened each decomp FITS twice per row (`load_lsf_state_if_available` opens once for the extension check and once via `load_lsf_surface_state`), plus re-reading the entire `VECTOR_O2` cube on every `load_o2_vector_if_available` call. Restructured cell 24 to (a) build ONE `SkyDecompLSFSurfaceIterative` before the loop (after asserting the sampled wave grid is shared -- true for every10 today; if that ever breaks a runtime error steers the user back to the per-row `reconstruct_with_lsf` fallback), and (b) precache `has_lsf` + the full `VECTOR_O2` cube once per decomp file, converting per-row lookups to array indexing. The loop now calls a local `_fast_reconstruct` that reuses the hoisted model via `_set_lsf_state` + `_assemble_refined_matrices` + `_components_from_coef`, injecting the cached O2 row into `mats['o2']` before evaluation. Numerically identical to the pre-optimization output (same basis, same coefficients, same LSF states); purely a runtime restructure. Timing prints (`recon setup` and `recon loop`) surface the payoff on the user's own dataset.

gas at velocities close enough to airglow to blend with the sky-component fit
**2026-08-11 (batch-RMSE cell reconstruction hoisted).** The RMSE-vs-truth diagnostic loop (25 rows x 3 arms) previously rebuilt `SkyDecompLSFSurfaceIterative` -- and its OH-stick basis, solar-reference read, moon B-spline, atomic/ORC/O2 blocks -- inside every one of the 75 reconstruction calls, and re-opened each decomp FITS twice per row (`load_lsf_state_if_available` opens once for the extension check and once via `load_lsf_surface_state`), plus re-reading the entire `VECTOR_O2` cube on every `load_o2_vector_if_available` call. Restructured cell 24 to (a) build ONE `SkyDecompLSFSurfaceIterative` before the loop (after asserting the sampled wave grid is shared -- true for every10 today; if that ever breaks a runtime error steers the user back to the per-row `reconstruct_with_lsf` fallback), and (b) precache `has_lsf` + the full `VECTOR_O2` cube once per decomp file, converting per-row lookups to array indexing. The loop now calls a local `_fast_reconstruct` that reuses the hoisted model via `_set_lsf_state` + `_assemble_refined_matrices` + `_components_from_coef`, injecting the cached O2 row into `mats['o2']` before evaluation. Numerically identical to the pre-optimization output (same basis, same coefficients, same LSF states); purely a runtime restructure. Timing prints (`recon setup` and `recon loop`) surface the payoff on the user's own dataset.


of `coef_sci`, and no per-tile source-mask exists that would reject them
**2026-08-11 (batch-RMSE cell reconstruction hoisted).** The RMSE-vs-truth diagnostic loop (25 rows x 3 arms) previously rebuilt `SkyDecompLSFSurfaceIterative` -- and its OH-stick basis, solar-reference read, moon B-spline, atomic/ORC/O2 blocks -- inside every one of the 75 reconstruction calls, and re-opened each decomp FITS twice per row (`load_lsf_state_if_available` opens once for the extension check and once via `load_lsf_surface_state`), plus re-reading the entire `VECTOR_O2` cube on every `load_o2_vector_if_available` call. Restructured cell 24 to (a) build ONE `SkyDecompLSFSurfaceIterative` before the loop (after asserting the sampled wave grid is shared -- true for every10 today; if that ever breaks a runtime error steers the user back to the per-row `reconstruct_with_lsf` fallback), and (b) precache `has_lsf` + the full `VECTOR_O2` cube once per decomp file, converting per-row lookups to array indexing. The loop now calls a local `_fast_reconstruct` that reuses the hoisted model via `_set_lsf_state` + `_assemble_refined_matrices` + `_components_from_coef`, injecting the cached O2 row into `mats['o2']` before evaluation. Numerically identical to the pre-optimization output (same basis, same coefficients, same LSF states); purely a runtime restructure. Timing prints (`recon setup` and `recon loop`) surface the payoff on the user's own dataset.**2026-08-11 (4-seed ensemble as deployed default).** `predict_sci_coefficients_default` gained an `is_ensemble` dispatch at the top: when the artifact carries `members` and `is_ensemble=True` it recursively predicts with each member and returns the arithmetic mean in physical space. The trainer cell now fits N=4 members at seeds {42, 43, 44, 45} with identical config in a loop, assembles them into an ensemble `mlp_artifacts` dict (shared fields -- compressors, geom_kwargs, group_indices, score_slices, splits, coef/ctx names, config -- lifted from members[0] to the top level for backward-compat), and prints per-seed + ensemble test metrics plus the stderr analysis. Downstream cells (relationship plots, batch RMSE, single-row reconstruction, pipeline-state check, naive baselines) get the ensemble prediction transparently via the same `predict_sci_coefficients_default(mlp_artifacts, ...)` call they used for the single-seed model; no per-cell edits were needed. The multi-seed sanity cell no longer retrains; it reads `mlp_artifacts['members']` and prints the per-group bias table that shows whether individual seeds carry biases the ensemble averages out. -- Is 4 enough? Single-seed test `mean_rmse` std on this dataset is $\sigma \approx 0.72$ (8-seed confidence check). Ensemble-mean stderr = $\sigma/\sqrt{N}$: 0.72 at N=1, 0.36 at N=4, 0.25 at N=8, 0.18 at N=16. At N=4 the ensemble max per-group |bias| lands below 1% (from ~1.5-2% single-seed), which is the deployment-relevant threshold from §11.1; the residual ~1.8% mean_rmse uncertainty is well under the target-contamination floor of §11.2. The 30% mean_rmse stderr improvement at N=8 (0.36 -> 0.25) costs 2x training time and mostly buys tighter PIT calibration for §11.7, which is not blocking today. N is exposed as `default_dual_group_config['ensemble_seeds']` for one-line adjustment; a training-time "is N enough" printout (seed std, ensemble stderr, stderr as % of ensemble mean_rmse) makes the tradeoff visible on every run. §5.5 gained a new "Predictor is an N-seed ensemble" bullet; §11.7 (predictive uncertainty) is partially addressed by the ensemble spread and remains open only for calibrated per-row $\sigma$ from a Gaussian-NLL head.

cleanly. Behaviour is controlled by `apply_triplet_filters`'s new

`exclude_field_regions` parameter (defaults to `[LMC_EXCLUSION, SMC_EXCLUSION]`;**2026-08-11 (empirical mean-bias calibration replaces analytic Jensen).** Follow-up to the same-day mesospheric no-PCA + multi-seed geometry adoption. That change removed the primary source of asinh-space residual variance (the 403 -> 28 xarm truncation), but the analytic Jensen factor $\exp(\sigma_j^2/2)$ with $\sigma$ estimated by robust MAD and capped at $\sigma_{cap}=0.4$ was retained. The cap had been tuned when the mesospheric truncation loss dominated the residual and a mean group lift of $\sim +2.3\%$ was exactly what was needed to null the pre-correction $-1.45\%$ bias. Post-no-PCA the analytic sigmas concentrate below the cap, but the group-mean lift still lands at $\sim +2\%$ -- flipping the mesospheric mean coefficient bias from near zero to $+2.4\%$ over-prediction that does NOT collapse under 4-seed ensembling (the analytic factor is deterministic in val residuals, not seed-varying). Replaced the block with an empirical per-group calibration: $\text{lift}_g = \overline{y_{true,g}}/\overline{y_{pred,g}^{naive}}$ measured in physical space on val, clipped to $[0.5, 2.0]$ as a broken-group guard, stored as a uniform per-coefficient scalar in `jensen_corrections[g]` so `inverse_group_compressor` accepts the same array shape. Only applied to asinh/log groups (sqrt/linear inverses are not multiplicatively biased in the same way and were never corrected). This directly nulls the group-mean-bias metric that §11.1 and the multi-seed check score, and absorbs any SmoothL1-induced mean bias too. No change to the network, split, compressor, or the `direct + init 0.7` blend parametrization; just the post-hoc calibration step.

pass `[]` to disable). `build_triplet_coef_dataset` now attaches `sci_ra` /

`sci_dec` from META so the filter has the coordinates without a second METARemoved nine settled experiment / A/B cells that trained models never used downstream: the ionospheric-head experiment, the two-stage `RUN_SWEEP` sweep, the `verify_artifacts` cell, the alpha-movement diagnostic, the `blend_optim` 4-way A/B, the multi-seed rerun of top-K sweep configs, the `drop_vanrhijn_from_context` A/B, the weight (moon/continuum) A/B, and the training-curve diagnostic. The only surviving training paths are (i) the default `mlp_artifacts` trainer at cell 18 (`DualEncoderGroupHeadMLPCompressed`, `blend_optim='direct'`, `blend_init_alpha=0.7`, `patience=16`, moon PCA at xarm>0.55, mesospheric identity, weights 3.5 / 1.5 / 3.0 / 1.0) and (ii) the 4-seed multi-seed ensemble at cell 26 that produces `_ensemble_test` for the naive-baseline comparison. All guarded confidence-check templates (dense sweep, alpha grid, 8-seed alpha, long-budget, pre-fit A/B, moon compression A/B) were kept so re-verification is one flag flip away.

load. Downstream cells (structure function, wavelength / extinction fit, PCA

rank, compressor, training) must be re-run against the new `filtered_triplet`### 2026-08-11 — notebook cleanup: single deployed path

for the model numbers to be consistent with the new data cut.



**2026-07-28 (latest)** — see the hyperparameter adoption entry above; bothshared row y-axes.

per-group compression and the sweep-adopted config now belong to the same date.`moon_phase` retained. Relationship plots fixed to four panels per row with

corrector and moon spline prior stage removed. `moon_illum` dropped from context;

**2026-07-28 — hyperparameter adoption for the compressed model.** A densescaled, passed through the API and dropped in `forward()`. Post-hoc Ridge

exhaustive sweep centred on the previous default (18 configs varyingthe default. Both sky contexts now reach the model; previously they were built,

encoder_dims, head_dim, lr, trunk_dims fixed at (320,160), weight_decay**2026-07-26 — splitting and plumbing.** Night-grouped splitting added and made

fixed at 1e-4) at the default budget (n_epochs=50, patience=4, seed=42)

identified a small improvement: encoder_dims=(384,192), head_dim=192→224,(was undefined on a clean kernel).

lr=7e-4→5e-4, converging at epoch 30 with mean RMSE 99.75 vs 102.4,grouping consolidated to a single definition. `y_te` in the driver cell fixed

MAE 71.3 vs 75.0, median correlation 0.890 vs 0.889 on the night-held-outcurve adopted. `assert_context_is_physical` added as a regression guard. Coefficient

test split. `default_dual_group_config` has been updated; the priorOH population model, replacing one wavelength per group. Real LCO extinction

wide random sweep and the dense sweep both remain in the notebook asheads. Per-coefficient effective wavelengths added, from the basis and from the

templates.`_apply_airglow_geometry` removed; `forward()` is now encode → fuse → trunk →

correct one. `_van_rhijn_factor_torch`, `_geometry_scale` and

**2026-07-28 — per-group compression is the default.** The rankand the combination produced a net transfer factor *anti-correlated* with the

diagnostic of §9.1 identified 79 transferable degrees of freedom in the 442robust-scaled context and applied in scaled square-root space; both were wrong,

coefficient columns. §5.5 documents the compression pipeline that(`airglow_geometry_scale`). Previously the correction was evaluated on

implements this: asinh for `mesospheric`, linear for `moon`, sqrt-identityof the network and into the data pipeline, in physical units

for the rest, column standardisation, PCA rotation, and cross-arm-**2026-07-27 — geometry restructuring.** Airglow geometry normalisation moved out

correlation selection at $r_{xarm} > 0.5$ on held-out nights.

`DualEncoderGroupHeadMLPCompressed` consumes the resulting per-arm scoreswith the basis primary, and the two are cross-checked.

and emits signed linear outputs; non-negativity is restored by clipping theBoth wavelength routes now read the same input product, precedence is explicit

reconstructed physical emissivity. The loss is SmoothL1 per group in scaledof 402 OH coefficients could otherwise sit near zero and hit the logarithm floor.

score space; `continuum_group_weight` and `moon_smooth_lambda` are retired.Its group amplitude now defaults to a sum rather than a median, which for a group

On the night-held-out test split (1386 rows) the compressed model beatsthan an order of magnitude and left it almost uncorrelated with the true field.

the uncompressed baseline on mean RMSE by ~33%, median RMSE by ~26%, MAEwhose spurious $-\log(X_{near}/X_{far})$ term inflated $D(\Delta\theta)$ by more

by ~34%, median per-coefficient correlation by 0.09; spectrum-spacehelper: it previously divided by van Rhijn *and* by airmass, a double correction

per-row RMSE tracks the coefficient-space result. The uncompressedseparately. The structure-function diagnostic now calls the shared geometry

training driver and its stale sweep have been removed from the notebook;`airglow_van_rhijn_matrix` and `airglow_extinction_matrix`, which the fit needs

`DualEncoderGroupHeadMLP` remains as the parent architecture.the training artifacts. `airglow_geometry_scale` was factored into

clipping; the result is threaded through `airglow_geometry_scale` and stored in

**2026-07-27 — dimensionality diagnostic.** Added §9.1: per-group effective rank on geometry-normalised amplitudes, with Horn parallel analysis on the correlation matrix, cross-arm score correlation on held-out nights, a transform sensitivity scan, a three-panel plot per group and a printed recommendation. No change to the model; this is decision support for whether to introduce a projection layer.intercept diagnostic; `resolve_coef_extinction_k()` applies it with fallback and

cluster-robust errors, column-level selection, split-half stability and an

**2026-07-27 — performance and diagnostics.** `fit_effective_extinction`$k_{\rm eff}$ per wavelength bin from the simultaneous sky pairs, with

vectorised: the per-cluster Python loop in the robust covariance was replaced bycurve and over-corrects an extended source. `fit_effective_extinction()` now fits

the closed form in `_masked_ols_rowconst`, the $(N \times P)$ van Rhijn matrices**2026-07-27 (later) — effective extinction.** The generic LCO curve is a stellar

by per-height $(N,)$ vectors, and the flattened design arrays by per-row

sufficient statistics. 8–10x faster, ~9x less memory, identical results. Basisnon-airglow heads keep those features.

wavelengths are now derived only for coefficients the population model could notepoch is the last. `drop_vanrhijn_from_context` added, defaulting to False so the

resolve plus a cross-check sample, instead of all of them, and the cache keylosses are recorded and printed at the best epoch, and training warns if the best

includes the wavelength grid so it cannot silently return values computed on adifferent grid. Cell 7 prints per-stage timings. Per-group unweighted validation

## Notes to self

- Instability with weights and including van rhijn: following test recommendation not consistent


In [ ]:
import os
os.environ.setdefault('LVMCORE_DIR', '/Users/droryn/prog/lvm/lvmcore')
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from astropy.io import fits
from astropy.table import Table
from IPython.display import HTML, display
from sky_decomp.fit import reconstruct_component_spectra
from sky_decomp.lsf_surface_iterative import (
    LSFSurfaceState,
    SkyDecompLSFSurfaceIterative,
    load_lsf_surface_state,
)
import re

# Journal-style axes: no gridlines, black box (mirrored) axes, outside ticks.
# Applied by mutating plotly_white in place so every explicit template='plotly_white'
# call in this notebook (and the default template) picks it up automatically.
for _ax in (pio.templates["plotly_white"].layout.xaxis,
            pio.templates["plotly_white"].layout.yaxis):
    _ax.showgrid = False
    _ax.showline = True
    _ax.mirror = True
    _ax.linecolor = "black"
    _ax.linewidth = 1
    _ax.ticks = "outside"
    _ax.zeroline = False
pio.templates.default = "plotly_white"

# Reused constants
FACTOR = 1e14
PALACE_DIR = '../'

In [ ]:
# Reused decomposition/context loading helpers
LAYER_HEIGHTS_KM = {
    'mesospheric_oh': 87.0,
    'mesospheric_atomic': 95.0,
    'ionospheric_red': 285.0,
}

TIME_FEATURE_NAMES = {
    'obstime_year_sin',
    'obstime_year_cos',
    'obstime_day_sin',
    'obstime_day_cos',
    'obstime_lunation_sin',
    'obstime_lunation_cos',
}

VAN_RHIJN_FEATURES = {
    'vanrhijn_87km': LAYER_HEIGHTS_KM['mesospheric_oh'],
    'vanrhijn_95km': LAYER_HEIGHTS_KM['mesospheric_atomic'],
    'vanrhijn_285km': LAYER_HEIGHTS_KM['ionospheric_red'],
}


def _as_array(x):
    arr = np.asarray(x)
    if arr.dtype.kind in ('U', 'S', 'O'):
        return None
    return arr.astype(np.float32)


def _coerce_coef_hdu_to_table(coef_hdu):
    data = coef_hdu.data
    if isinstance(coef_hdu, (fits.BinTableHDU, fits.TableHDU)):
        return Table(data)

    arr = np.asarray(data, dtype=np.float32)
    if arr.ndim != 2:
        raise ValueError(f'Expected 2D COEF image, got shape={arr.shape}')

    n_coef = arr.shape[1]
    names = []
    for i in range(n_coef):
        key = f'COEF{i:04d}'
        names.append(str(coef_hdu.header.get(key, f'coef_{i:04d}')))
    return Table({name: arr[:, i] for i, name in enumerate(names)})


def _select_context_from_labels(meta, meta_upper, labels, base_name):
    e_key = f'SKYE_{base_name.upper()}'
    w_key = f'SKYW_{base_name.upper()}'
    if e_key not in meta_upper or w_key not in meta_upper:
        return None

    arr_e = _as_array(meta[meta_upper[e_key]])
    arr_w = _as_array(meta[meta_upper[w_key]])
    if arr_e is None or arr_w is None:
        raise ValueError(f'Labeled context columns for {base_name} are non-numeric.')

    is_e = labels == 'SKYE'
    is_w = labels == 'SKYW'
    if not np.all(is_e | is_w):
        bad = np.unique(labels[~(is_e | is_w)])
        raise ValueError(f'Unexpected label values: {bad}')

    return np.where(is_e, arr_e, arr_w).astype(np.float32)


def _table_to_float32_matrix(tbl, value_name):
    names = list(tbl.colnames)
    cols = []
    numeric_names = []
    for name in names:
        arr = _as_array(tbl[name])
        if arr is not None:
            cols.append(arr)
            numeric_names.append(name)

    if len(cols) == 0:
        raise ValueError(f'No numeric {value_name} columns found.')

    return np.column_stack(cols).astype(np.float32), numeric_names


def _altitude_to_zenith_deg(alt_deg):
    alt = np.asarray(alt_deg, dtype=np.float64)
    return np.clip(90.0 - alt, 0.0, 89.9)


def _van_rhijn_factor(alt_deg, layer_height_km, earth_radius_km=6371.0):
    z_rad = np.deg2rad(_altitude_to_zenith_deg(alt_deg))
    shell_ratio = float(earth_radius_km) / (float(earth_radius_km) + float(layer_height_km))
    denom = 1.0 - (shell_ratio * np.sin(z_rad)) ** 2
    denom = np.clip(denom, 1e-6, None)
    return (1.0 / np.sqrt(denom)).astype(np.float32)


def _extract_obstime_mjd(meta, meta_upper):
    from astropy.time import Time

    candidates = ('OBSTIME', 'MJD', 'MJD_OBS', 'MJD-OBS', 'DATE_OBS', 'DATE-OBS')
    for key in candidates:
        if key not in meta_upper:
            continue

        raw = np.asarray(meta[meta_upper[key]])
        if raw.dtype.kind in ('i', 'u', 'f'):
            return raw.astype(np.float64)

        s = np.asarray(raw).astype(str)
        try:
            return Time(s, format='isot', scale='utc').mjd.astype(np.float64)
        except ValueError:
            return Time(s).mjd.astype(np.float64)

    raise KeyError('Could not find an OBSTIME/MJD/DATE-OBS-like META column for time features')


def _build_obstime_feature(meta, meta_upper, feature_name):
    mjd = _extract_obstime_mjd(meta, meta_upper)
    two_pi = 2.0 * np.pi

    if feature_name == 'obstime_year_sin':
        return np.sin(two_pi * mjd / 365.2422).astype(np.float32)

    if feature_name == 'obstime_year_cos':
        return np.cos(two_pi * mjd / 365.2422).astype(np.float32)

    frac_day = mjd - np.floor(mjd)
    if feature_name == 'obstime_day_sin':
        return np.sin(two_pi * frac_day).astype(np.float32)

    if feature_name == 'obstime_day_cos':
        return np.cos(two_pi * frac_day).astype(np.float32)

    if feature_name == 'obstime_lunation_sin':
        return np.sin(two_pi * mjd / 29.53058867).astype(np.float32)

    if feature_name == 'obstime_lunation_cos':
        return np.cos(two_pi * mjd / 29.53058867).astype(np.float32)

    raise KeyError(f'Unsupported obstime feature name: {feature_name}')


# --- Astropy-computed pointing geometry ------------------------------------
# alt / az / airmass and sun / moon positions are computed from RA, DEC and
# obstime via astropy rather than read directly from META. Centralising the
# geometry here means a per-column error in the input META (which happens for
# online-derived quantities) does not silently propagate through the pipeline.
# `moon_phase` is intentionally not in this set: illumination fraction is not
# a pure geometry feature and is still read from META.
ASTROPY_GEOMETRY_FEATURES = {
    'alt', 'az_sin', 'az_cos', 'airmass',
    'moon_alt', 'moon_az_sin', 'moon_az_cos', 'moon_sep',
    'sun_alt', 'sun_az_sin', 'sun_az_cos', 'sun_sep',
    'sci_sep',
}

# META columns stored as an angle in degrees that wraps at 0/360.
# `<stem>_sin` / `<stem>_cos` in context_cols pull the raw column and fold it.
CYCLIC_META_DEGREE_FEATURES = {'moon_phase'}

_LCO_EARTHLOCATION = None


def _lco_earth_location():
    """LCO EarthLocation, cached at first use."""
    global _LCO_EARTHLOCATION
    if _LCO_EARTHLOCATION is not None:
        return _LCO_EARTHLOCATION
    from astropy.coordinates import EarthLocation
    import astropy.units as u
    try:
        _LCO_EARTHLOCATION = EarthLocation.of_site('Las Campanas Observatory')
    except Exception:
        # Fallback to published LCO coordinates (Baade / du Pont site).
        _LCO_EARTHLOCATION = EarthLocation(
            lat=-29.00889 * u.deg,
            lon=-70.68992 * u.deg,
            height=2380.0 * u.m,
        )
    return _LCO_EARTHLOCATION


def _pointing_ra_dec_columns(meta_upper, kind):
    """Return the META column names for the pointing RA / DEC of one kind.

    Uses SKY_NEAR_RA / SKY_NEAR_DEC and SKY_FAR_RA / SKY_FAR_DEC for the two
    sky arms, so the near/far ordering (by separation from science) and the
    east/west assignment are picked up from the same columns that already
    drive the rest of the pipeline; no consultation of SKY_NEAR_LABEL /
    SKY_FAR_LABEL is needed for the pointing geometry.
    """
    if kind == 'sci':
        ra_key = next((meta_upper[k] for k in ('SCI_RA', 'RA') if k in meta_upper), None)
        dec_key = next((meta_upper[k] for k in ('SCI_DEC', 'DEC') if k in meta_upper), None)
    elif kind == 'sky1':
        ra_key = meta_upper.get('SKY_NEAR_RA')
        dec_key = meta_upper.get('SKY_NEAR_DEC')
    elif kind == 'sky2':
        ra_key = meta_upper.get('SKY_FAR_RA')
        dec_key = meta_upper.get('SKY_FAR_DEC')
    else:
        raise ValueError(f'unexpected kind {kind!r}')
    if ra_key is None or dec_key is None:
        raise KeyError(f'RA / DEC columns for kind={kind!r} not found in META')
    return ra_key, dec_key


def _compute_astropy_geometry(meta, meta_upper, kind):
    """Compute all astropy-derived geometry features for one pointing kind.

    Returns a dict keyed by feature name (subset of ASTROPY_GEOMETRY_FEATURES)
    with float32 arrays of length n_rows. Pointing RA / DEC come from the
    kind-specific columns; obstime comes from `_extract_obstime_mjd`. Sun and
    Moon topocentric positions are computed at LCO for every obstime. Airmass
    is sec(z) = 1 / sin(alt), NaN below the horizon.
    """
    from astropy.coordinates import AltAz, SkyCoord, get_body, get_sun
    from astropy.time import Time
    import astropy.units as u

    ra_key, dec_key = _pointing_ra_dec_columns(meta_upper, kind)
    ra = np.asarray(meta[ra_key], dtype=np.float64)
    dec = np.asarray(meta[dec_key], dtype=np.float64)

    # Science pointing coords are needed for the sky-to-sci angular separation feature.
    sci_ra_key, sci_dec_key = _pointing_ra_dec_columns(meta_upper, 'sci')
    sci_ra_col = np.asarray(meta[sci_ra_key], dtype=np.float64)
    sci_dec_col = np.asarray(meta[sci_dec_key], dtype=np.float64)

    mjd = _extract_obstime_mjd(meta, meta_upper)
    time = Time(mjd, format='mjd', scale='utc')

    lco = _lco_earth_location()
    frame = AltAz(obstime=time, location=lco)

    pointing = SkyCoord(ra=ra * u.deg, dec=dec * u.deg, frame='icrs')
    pointing_altaz = pointing.transform_to(frame)
    sci_pointing = SkyCoord(ra=sci_ra_col * u.deg, dec=sci_dec_col * u.deg, frame='icrs')
    sci_sep = pointing.separation(sci_pointing).to_value(u.deg)

    sun = get_sun(time)
    moon = get_body('moon', time, location=lco)
    sun_altaz = sun.transform_to(frame)
    moon_altaz = moon.transform_to(frame)

    sun_sep = pointing.separation(sun).to_value(u.deg)
    moon_sep = pointing.separation(moon).to_value(u.deg)

    alt_deg = pointing_altaz.alt.to_value(u.deg)
    sinalt = np.sin(np.deg2rad(alt_deg))
    airmass = np.where(sinalt > 1e-6, 1.0 / np.clip(sinalt, 1e-6, None), np.nan)

    az_rad = np.deg2rad(pointing_altaz.az.to_value(u.deg))
    moon_az_rad = np.deg2rad(moon_altaz.az.to_value(u.deg))
    sun_az_rad = np.deg2rad(sun_altaz.az.to_value(u.deg))

    return {
        'alt': alt_deg.astype(np.float32),
        'az_sin': np.sin(az_rad).astype(np.float32),
        'az_cos': np.cos(az_rad).astype(np.float32),
        'airmass': airmass.astype(np.float32),
        'moon_alt': moon_altaz.alt.to_value(u.deg).astype(np.float32),
        'moon_az_sin': np.sin(moon_az_rad).astype(np.float32),
        'moon_az_cos': np.cos(moon_az_rad).astype(np.float32),
        'moon_sep': moon_sep.astype(np.float32),
        'sun_alt': sun_altaz.alt.to_value(u.deg).astype(np.float32),
        'sun_az_sin': np.sin(sun_az_rad).astype(np.float32),
        'sun_az_cos': np.cos(sun_az_rad).astype(np.float32),
        'sun_sep': sun_sep.astype(np.float32),
        'sci_sep': sci_sep.astype(np.float32),
    }


# ---------------------------------------------------------------------------
# Solar activity proxies (F10.7 daily flux and Kp geomagnetic index).
#
# F10.7 is the standard EUV proxy driving thermospheric density and OI 6300 /
# OI 6364 recombination amplitude; Kp captures short-timescale auroral /
# geomagnetic driving that also modulates the 285 km group's brightness.
# Both are fetched from GFZ Potsdam's authoritative archive
#     https://kp.gfz.de/app/files/Kp_ap_Ap_SN_F107_since_1932.txt
# (definitive + quicklook, back to 1932, updated daily) and cached under
# solar_activity_cache/.  The raw file is re-downloaded if older than
# SOLAR_ACTIVITY_MAX_AGE_DAYS or missing.
# ---------------------------------------------------------------------------
SOLAR_ACTIVITY_FEATURES = {'f107', 'f107_81d', 'kp'}
SOLAR_ACTIVITY_SOURCE_URL = 'https://kp.gfz.de/app/files/Kp_ap_Ap_SN_F107_since_1932.txt'
SOLAR_ACTIVITY_CACHE_DIR = Path('solar_activity_cache')
SOLAR_ACTIVITY_RAW_PATH = SOLAR_ACTIVITY_CACHE_DIR / 'Kp_ap_Ap_SN_F107_since_1932.txt'
SOLAR_ACTIVITY_NPZ_PATH = SOLAR_ACTIVITY_CACHE_DIR / 'kp_f107.npz'
SOLAR_ACTIVITY_MAX_AGE_DAYS = 7

_SOLAR_ACTIVITY_TABLE = None  # module-level cache of parsed arrays


def _download_gfz_solar_activity(url, dest, timeout=60, verbose=True):
    """Fetch the GFZ Kp / ap / F10.7 archive to `dest` via a plain HTTP GET."""
    from urllib.request import Request, urlopen
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + '.part')
    req = Request(url, headers={'User-Agent': 'lvmsky-notebook/1.0'})
    with urlopen(req, timeout=timeout) as resp, open(tmp, 'wb') as fh:
        fh.write(resp.read())
    tmp.replace(dest)
    if verbose:
        print(f'  solar-activity archive downloaded: {dest} '
              f'({dest.stat().st_size / 1e6:.2f} MB)')


def _parse_gfz_solar_activity_raw(path):
    """Parse the GFZ text file. Returns dict of arrays: kp (3-hourly) + F10.7 (daily)."""
    df = pd.read_csv(path, comment='#', sep=r'\s+', header=None, engine='python')
    if df.shape[1] < 26:
        raise RuntimeError(
            f'Unexpected GFZ Kp/F10.7 file layout: got {df.shape[1]} columns, '
            f'expected at least 26. First data row: {df.iloc[0].tolist()!r}.'
        )
    year = df.iloc[:, 0].astype(int).to_numpy()
    month = df.iloc[:, 1].astype(int).to_numpy()
    day = df.iloc[:, 2].astype(int).to_numpy()
    # columns 3-6 = bookkeeping (days since 1932-01-01, Bartels rotation number, day of rotation)
    kp_cols = df.iloc[:, 7:15].astype(float).to_numpy()   # 8 three-hourly Kp values (Kp1..Kp8)
    # columns 15-22 = 8 ap values; 23 = Ap daily; 24 = SN; 25 = F10.7 obs; 26 = F10.7 adj
    f107_obs = df.iloc[:, 25].astype(float).to_numpy()

    from astropy.time import Time
    iso = [f'{y:04d}-{m:02d}-{d:02d}' for y, m, d in zip(year, month, day)]
    mjd_day0 = Time(iso, format='iso', scale='utc').mjd.astype(np.float64)
    # 3-hour Kp windows start at 0h, 3h, 6h, ..., 21h UT.
    hour_offsets = np.arange(8, dtype=np.float64) * (3.0 / 24.0)
    kp_mjd_start = (mjd_day0[:, None] + hour_offsets[None, :]).reshape(-1)
    kp_value = kp_cols.reshape(-1)
    # Daily F10.7 attached to noon UT so np.interp reads it as a midday sample.
    f107_mjd = mjd_day0 + 0.5
    good_kp = np.isfinite(kp_value) & (kp_value >= 0.0)
    good_f107 = np.isfinite(f107_obs) & (f107_obs > 0.0)
    return {
        'kp_mjd_start': kp_mjd_start[good_kp],
        'kp_value': kp_value[good_kp],
        'f107_mjd': f107_mjd[good_f107],
        'f107_obs': f107_obs[good_f107],
    }


def _f107_running_mean(mjd, f107, window_days=81.0):
    """Symmetric running mean of daily F10.7 over `window_days` (~3 solar rotations)."""
    order = np.argsort(mjd)
    mjd_s = mjd[order]
    f107_s = f107[order].astype(np.float64)
    n = mjd_s.size
    half = float(window_days) / 2.0
    lo = np.searchsorted(mjd_s, mjd_s - half, side='left')
    hi = np.searchsorted(mjd_s, mjd_s + half, side='right')
    csum = np.concatenate([[0.0], np.cumsum(f107_s)])
    sums = csum[hi] - csum[lo]
    counts = (hi - lo).astype(np.float64)
    with np.errstate(invalid='ignore', divide='ignore'):
        out_s = np.where(counts > 0, sums / counts, np.nan)
    inv = np.empty(n, dtype=int)
    inv[order] = np.arange(n)
    return out_s[inv]


def load_solar_activity_table(
    url=SOLAR_ACTIVITY_SOURCE_URL,
    raw_path=SOLAR_ACTIVITY_RAW_PATH,
    npz_path=SOLAR_ACTIVITY_NPZ_PATH,
    max_age_days=SOLAR_ACTIVITY_MAX_AGE_DAYS,
    force_refresh=False,
    verbose=True,
):
    """Return sorted numpy arrays for 3-hourly Kp and daily F10.7 (obs + 81-day mean).

    Downloads the GFZ archive if the raw cache is missing or older than
    `max_age_days` days.  Reuses a stale cache silently if the download fails.
    """
    global _SOLAR_ACTIVITY_TABLE
    if _SOLAR_ACTIVITY_TABLE is not None and not force_refresh:
        return _SOLAR_ACTIVITY_TABLE

    import time as _time_mod
    need_download = force_refresh or (not raw_path.exists())
    if not need_download:
        age_days = (_time_mod.time() - raw_path.stat().st_mtime) / 86400.0
        if age_days > float(max_age_days):
            if verbose:
                print(f'Solar-activity cache is {age_days:.1f} days old '
                      f'(> {max_age_days} d); refreshing.')
            need_download = True
    if need_download:
        try:
            _download_gfz_solar_activity(url, raw_path, verbose=verbose)
        except Exception as exc:
            if raw_path.exists():
                print(f'Solar-activity download failed '
                      f'({type(exc).__name__}: {exc}); reusing existing cache at {raw_path}.')
            else:
                raise RuntimeError(
                    f'Could not download solar-activity archive from {url} '
                    f'({type(exc).__name__}: {exc}); no local cache. '
                    f'Provide the file manually at {raw_path} to proceed.') from exc

    parsed = _parse_gfz_solar_activity_raw(raw_path)
    kp_order = np.argsort(parsed['kp_mjd_start'])
    kp_mjd_start = parsed['kp_mjd_start'][kp_order].astype(np.float64)
    kp_value = parsed['kp_value'][kp_order].astype(np.float32)
    f107_order = np.argsort(parsed['f107_mjd'])
    f107_mjd = parsed['f107_mjd'][f107_order].astype(np.float64)
    f107_obs = parsed['f107_obs'][f107_order].astype(np.float32)
    f107_81d = _f107_running_mean(f107_mjd, f107_obs, window_days=81.0).astype(np.float32)

    table = {
        'kp_mjd_start': kp_mjd_start,
        'kp_value': kp_value,
        'f107_mjd': f107_mjd,
        'f107_obs': f107_obs,
        'f107_81d': f107_81d,
    }
    npz_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez(npz_path, **table)
    _SOLAR_ACTIVITY_TABLE = table
    if verbose:
        print(f'Solar-activity table ready: '
              f'{kp_mjd_start.size} Kp windows '
              f'(MJD {kp_mjd_start.min():.1f} .. {kp_mjd_start.max():.1f}), '
              f'{f107_mjd.size} daily F10.7 samples; cached to {npz_path}.')
    return table


def _solar_activity_lookup(mjd_query, feature):
    """Evaluate one solar-activity feature at each observation MJD."""
    table = load_solar_activity_table()
    mjd_query = np.asarray(mjd_query, dtype=np.float64)
    if feature == 'kp':
        # nearest 3-hour window that starts at or before the observation
        idx = np.searchsorted(table['kp_mjd_start'], mjd_query, side='right') - 1
        n_kp = table['kp_value'].size
        oob = (idx < 0) | (idx >= n_kp)
        idx = np.clip(idx, 0, n_kp - 1)
        val = table['kp_value'][idx].astype(np.float32).copy()
        val[oob] = np.nan
        return val
    if feature in ('f107', 'f107_81d'):
        src = table['f107_obs'] if feature == 'f107' else table['f107_81d']
        val = np.interp(mjd_query, table['f107_mjd'], src.astype(np.float64),
                        left=np.nan, right=np.nan)
        return val.astype(np.float32)
    raise KeyError(f'unknown solar-activity feature: {feature!r}')


def _decode_cyclic_context(ctx_names, ctx_matrix):
    """Fold `<name>_sin`/`<name>_cos` pairs back into a single `<name>` in degrees [0, 360).

    Purely for display: the model still consumes the sin/cos pair, but plots
    (histograms, coefficient-vs-context relationship panels) are much more
    legible with a single 0-360 axis than with a pair of [-1, 1] projections.
    """
    names = list(ctx_names)
    mat = np.asarray(ctx_matrix, dtype=np.float64)
    out_names = []
    out_cols = []
    handled = set()
    for i, name in enumerate(names):
        if name in handled:
            continue
        if name.endswith('_sin'):
            stem = name[:-4]
            cos_name = stem + '_cos'
            if cos_name in names:
                cos_idx = names.index(cos_name)
                rad = np.arctan2(mat[:, i], mat[:, cos_idx])
                out_names.append(stem)
                out_cols.append((np.rad2deg(rad) % 360.0).astype(np.float32))
                handled.add(name)
                handled.add(cos_name)
                continue
        if name.endswith('_cos') and name in handled:
            continue
        out_names.append(name)
        out_cols.append(mat[:, i].astype(np.float32))
    return out_names, (np.column_stack(out_cols).astype(np.float32)
                       if out_cols else np.empty((mat.shape[0], 0), dtype=np.float32))


def _resolve_base_context_array(meta, meta_upper, labels, kind, base_name):
    key = base_name.upper()

    if kind == 'sci':
        sci_key = f'SCI_{key}'
        if sci_key in meta_upper:
            arr = _as_array(meta[meta_upper[sci_key]])
            if arr is None:
                raise ValueError(f'Context column {sci_key} is non-numeric.')
            return arr

    if key in meta_upper:
        arr = _as_array(meta[meta_upper[key]])
        if arr is None:
            raise ValueError(f'Context column {base_name} is non-numeric.')
        return arr

    if labels is not None:
        arr = _select_context_from_labels(meta, meta_upper, labels, base_name)
        if arr is not None:
            return arr

    return None


def _resolve_context_feature(meta, meta_upper, labels, kind, feature_name, cache):
    if feature_name in cache:
        return cache[feature_name]

    if feature_name in ('obstime_mjd', 'obstime_mjd_z'):
        raise ValueError('Direct time features are excluded from model context; use only periodic year/day/lunation sine and cosine terms.')

    if feature_name in ASTROPY_GEOMETRY_FEATURES:
        # Compute all astropy features in one pass the first time any is asked for.
        if '_astropy_computed' not in cache:
            for _name, _arr in _compute_astropy_geometry(meta, meta_upper, kind).items():
                cache[_name] = np.asarray(_arr, dtype=np.float32)
            cache['_astropy_computed'] = True
        return cache[feature_name]

    if feature_name in SOLAR_ACTIVITY_FEATURES:
        if '_solar_activity_computed' not in cache:
            _sa_mjd = _extract_obstime_mjd(meta, meta_upper)
            for _sa_name in SOLAR_ACTIVITY_FEATURES:
                cache[_sa_name] = np.asarray(
                    _solar_activity_lookup(_sa_mjd, _sa_name), dtype=np.float32)
            cache['_solar_activity_computed'] = True
        return cache[feature_name]

    if feature_name == 'ew':
        if kind == 'sci':
            arr = np.zeros(len(meta), dtype=np.float32)
        else:
            arr = np.where(labels == 'SKYE', 1.0,
                           np.where(labels == 'SKYW', -1.0, 0.0)).astype(np.float32)
        cache[feature_name] = arr
        return arr

    if feature_name in TIME_FEATURE_NAMES:
        arr = _build_obstime_feature(meta, meta_upper, feature_name)
    elif feature_name == 'zenith_deg':
        alt = _resolve_context_feature(meta, meta_upper, labels, kind, 'alt', cache)
        arr = _altitude_to_zenith_deg(alt).astype(np.float32)
    elif (feature_name.endswith('_sin') or feature_name.endswith('_cos')) \
            and feature_name[:-4] in CYCLIC_META_DEGREE_FEATURES:
        raw = _resolve_base_context_array(meta, meta_upper, labels, kind, feature_name[:-4])
        if raw is None:
            raise KeyError(feature_name)
        rad = np.deg2rad(np.asarray(raw, dtype=np.float64))
        arr = (np.sin(rad) if feature_name.endswith('_sin') else np.cos(rad)).astype(np.float32)
    elif feature_name in VAN_RHIJN_FEATURES:
        alt = _resolve_context_feature(meta, meta_upper, labels, kind, 'alt', cache)
        arr = _van_rhijn_factor(alt, VAN_RHIJN_FEATURES[feature_name])
    else:
        arr = _resolve_base_context_array(meta, meta_upper, labels, kind, feature_name)
        if arr is None:
            raise KeyError(feature_name)

    cache[feature_name] = np.asarray(arr, dtype=np.float32)
    return cache[feature_name]


def _build_context_matrix(meta, context_columns, kind):
    meta_upper = {c.upper(): c for c in meta.colnames}
    labels = None

    if kind in ('sky1', 'sky2'):
        label_col = 'SKY_NEAR_LABEL' if kind == 'sky1' else 'SKY_FAR_LABEL'
        if label_col not in meta_upper:
            raise KeyError(f'Missing required META label column: {label_col}')
        labels = np.char.upper(np.char.strip(np.asarray(meta[meta_upper[label_col]]).astype(str)))

    ctx_names = []
    ctx_cols = []
    missing_cols = []
    cache = {}

    for cname in context_columns:
        try:
            arr = _resolve_context_feature(meta, meta_upper, labels, kind, cname, cache)
        except KeyError:
            missing_cols.append(cname)
            continue
        ctx_names.append(cname)
        ctx_cols.append(arr)

    if missing_cols:
        raise KeyError(f'Missing requested context columns: {missing_cols}')
    if len(ctx_cols) == 0:
        raise ValueError('No usable context columns were assembled.')

    return np.column_stack(ctx_cols).astype(np.float32), ctx_names


def _find_chi2_column(meta_tbl):
    names = {c.upper(): c for c in meta_tbl.colnames}
    for cand in ('REDUCED_CHI2', 'CHI2_REDUCED', 'CHI2', 'RCHI2'):
        if cand in names:
            return names[cand]
    raise KeyError('No chi2-like column found in decomposition META table')


def read_decomp_dataset(decomp_fits_path, input_fits_path, context_columns,
                        decomp_kind='sky1', return_chi2=False, return_err=False):
    if context_columns is None or len(context_columns) == 0:
        raise ValueError('context_columns must be a non-empty list.')

    kind = decomp_kind.lower()
    if kind not in ('sky1', 'sky2', 'sci'):
        raise ValueError("decomp_kind must be one of: 'sky1', 'sky2', 'sci'")

    with fits.open(decomp_fits_path) as hdul_dec, fits.open(input_fits_path) as hdul_in:
        coef_tbl = _coerce_coef_hdu_to_table(hdul_dec['COEF'])
        coef_mat, coef_names = _table_to_float32_matrix(coef_tbl, 'coefficient')

        # COEF_ERR is the active-set 1-sigma uncertainty on each coefficient
        # (see fit.SkyDecomp._coef_err_active_set); older FITS files omit it,
        # so we fall back to NaN and callers can decide whether to weight.
        coef_err_mat = None
        if return_err:
            if 'COEF_ERR' in hdul_dec:
                coef_err_tbl = _coerce_coef_hdu_to_table(hdul_dec['COEF_ERR'])
                coef_err_mat, coef_err_names = _table_to_float32_matrix(coef_err_tbl, 'coefficient error')
                if coef_err_names != coef_names:
                    raise ValueError('COEF_ERR column ordering does not match COEF')
            else:
                coef_err_mat = np.full_like(coef_mat, np.nan, dtype=np.float32)

        meta = Table(hdul_in['META'].data)
        ctx_mat, ctx_names = _build_context_matrix(meta, context_columns, kind)

        if coef_mat.shape[0] != ctx_mat.shape[0]:
            raise ValueError(
                f'Row count mismatch: COEF has {coef_mat.shape[0]} rows, META has {ctx_mat.shape[0]} rows'
            )

        good = np.isfinite(coef_mat).all(axis=1) & np.isfinite(ctx_mat).all(axis=1)
        coef_mat = coef_mat[good]
        ctx_mat = ctx_mat[good]
        if coef_err_mat is not None:
            coef_err_mat = coef_err_mat[good]

        if not return_chi2 and not return_err:
            return coef_mat, ctx_mat, coef_names, ctx_names

        chi2_used = None
        if return_chi2:
            dec_meta = Table(hdul_dec['META'].data)
            chi2_col = _find_chi2_column(dec_meta)
            chi2_full = np.asarray(dec_meta[chi2_col], dtype=np.float64)
            chi2_used = chi2_full[good]
            if chi2_used.shape[0] != coef_mat.shape[0]:
                raise ValueError(
                    f'Aligned chi2 rows ({chi2_used.shape[0]}) do not match coef rows ({coef_mat.shape[0]})'
                )

        result = [coef_mat, ctx_mat, coef_names, ctx_names]
        if return_chi2:
            result.append(chi2_used)
        if return_err:
            result.append(coef_err_mat)
        return tuple(result)


# ---------------------------------------------------------------------------
# Coefficient grouping by physical origin / emission layer.
# Single definition, used by the wavelength resolver, the structure-function
# diagnostics and the model.  Coefficients are consumed in order so each one
# lands in exactly one group.
# ---------------------------------------------------------------------------

def _contains_any(name, needles):
    return any(token in name for token in needles)


# Coefficient-name schema. One-to-one with the design_names produced by
# SkyDecomp._build_static_basis() in skysub/sky_decomp/fit.py. Every entry
# is (regex, group, wavelength_a, description). Any coefficient that does
# not match a pattern raises in _build_group_indices() -- silently
# absorbing unknown names into an 'other' group previously hid that
# HO2/FeO/O2Ac were routed into 'mesospheric' and that OI 6300/6364 +
# OI 7774/8446 (ATOM_Or, ATOM_Orc_*) were routed into 'atomic' (~95 km)
# instead of 'ionospheric' (~285 km).
#
# Group layers (see GROUP_HEIGHT_FEATURE):
#   moon         no thin-shell height (scattered moonlight + spline)
#   continuum    ~87 km  HO2 / FeO / O2Ac mesopause chemiluminescence
#   mesospheric  ~87 km  OH Meinel bands + O2 atmospheric band
#   atomic       ~95 km  K, Na, N I, OI 5577 (mesopause metal / metastable)
#   ionospheric  ~285 km OI 6300/6364 red + OI 7774/8446 F-region recomb
COEF_SCHEMA = [
    (re.compile(r'^OH_\d{3}$'),         'mesospheric', None,   'OH Meinel band group (wavelength from basis)'),
    (re.compile(r'^O2_b\d+$'),          'mesospheric', 8645.0, 'O2 (b1Sigma) atmospheric band'),
    (re.compile(r'^Moon_bs\d+$'),       'moon',        None,   'Moon scattered-light spline knot'),
    (re.compile(r'^HO2$'),              'continuum',   None,   'HO2 diffuse continuum'),
    (re.compile(r'^FeO$'),              'continuum',   None,   'FeO orange-arc diffuse continuum'),
    (re.compile(r'^O2Ac$'),             'continuum',   None,   'O2 Chamberlain diffuse continuum'),
    (re.compile(r'^ATOM_K$'),           'atomic',      7698.96, 'K I 7699'),
    (re.compile(r'^ATOM_N$'),           'ionospheric', 5199.0,  'N I 5199 [NI] 2D->4S F-region metastable'),
    (re.compile(r'^ATOM_Na$'),          'atomic',      5892.9,  'Na D'),
    (re.compile(r'^ATOM_Og$'),          'atomic',      5577.34, 'OI 5577 green line'),
    (re.compile(r'^ATOM_Or$'),          'ionospheric', 6300.3,  'OI 6300/6364 red doublet'),
    (re.compile(r'^ATOM_Orc_OI0777$'),  'ionospheric', 7774.2,  'OI 7774 F-region recombination'),
    (re.compile(r'^ATOM_Orc_OI0845$'),  'ionospheric', 8446.4,  'OI 8446 F-region recombination'),
]

# Canonical output order for group dicts.
_COEF_GROUP_ORDER = ('moon', 'continuum', 'mesospheric', 'ionospheric', 'atomic')


def _lookup_coef_schema(name):
    """Return (group, wavelength_a, description) for one coefficient, else None."""
    for pat, group, lam, desc in COEF_SCHEMA:
        if pat.match(str(name)):
            return group, lam, desc
    return None


def _build_group_indices(coef_names):
    """Route each coefficient name to exactly one group via COEF_SCHEMA.

    Raises RuntimeError on any unmatched name so a new coefficient family
    added to fit.py becomes a loud error here rather than a silent 'other'.
    """
    groups = {g: [] for g in _COEF_GROUP_ORDER}
    unmatched = []
    for j, name in enumerate(coef_names):
        hit = _lookup_coef_schema(name)
        if hit is None:
            unmatched.append((j, str(name)))
        else:
            groups[hit[0]].append(j)
    if unmatched:
        raise RuntimeError(
            'Unrecognised coefficient names (no COEF_SCHEMA match): '
            f'{unmatched[:8]}{"..." if len(unmatched) > 8 else ""}. '
            'Extend COEF_SCHEMA in the loaders cell and cross-check '
            'against SkyDecomp._build_static_basis() in '
            'skysub/sky_decomp/fit.py.'
        )
    return {g: np.asarray(idx, dtype=int) for g, idx in groups.items() if idx}


# Self-test: verify the schema on a canonical name list. Runs when the
# cell is executed, so a regression in COEF_SCHEMA fails immediately.
_SCHEMA_TEST_NAMES = (
    [f'OH_{i:03d}' for i in (0, 42, 401)]
    + [f'Moon_bs{i:02d}' for i in (0, 15, 28)]
    + ['HO2', 'FeO', 'O2Ac']
    + ['ATOM_K', 'ATOM_N', 'ATOM_Na', 'ATOM_Og', 'ATOM_Or',
       'ATOM_Orc_OI0777', 'ATOM_Orc_OI0845']
    + ['O2_b01']
)
_SCHEMA_TEST_EXPECTED = {
    'moon': 3, 'continuum': 3,
    'mesospheric': 4,  # 3x OH_### + O2_b01
    'atomic': 3,       # K, Na, Og
    'ionospheric': 4,  # N, Or, Orc_OI0777, Orc_OI0845
}
_schema_test = _build_group_indices(_SCHEMA_TEST_NAMES)
_schema_test_sizes = {g: int(idx.size) for g, idx in _schema_test.items()}
if _schema_test_sizes != _SCHEMA_TEST_EXPECTED:
    raise RuntimeError(
        f'COEF_SCHEMA self-test failed: got {_schema_test_sizes}, '
        f'expected {_SCHEMA_TEST_EXPECTED}'
    )
try:
    _build_group_indices(['this_is_not_a_valid_coef'])
except RuntimeError:
    pass
else:
    raise RuntimeError('COEF_SCHEMA self-test: unknown name did not raise')

# ---------------------------------------------------------------------------
# Geometric normalisation of airglow coefficients  (PHYSICAL units)
#
# Airglow amplitudes scale as
#       A_obs = A_intrinsic * V(z; h) * 10**(-0.4 * k(lambda) * (X - 1))
# where V is the van Rhijn slant-path enhancement through a thin emitting
# shell at height h, and the power of ten is the extinction suffered by that
# emission on the way down.  Dividing an observed amplitude by this factor
# recovers the intrinsic layer emissivity, which is the only quantity
# comparable between two different lines of sight.
#
# The normalisation therefore has to happen HERE, on physical coefficients,
# before the sqrt transform and before any robust scaling.  Doing it on
# scaled or sqrt-transformed values is not equivalent: the scaler subtracts a
# median, so (sqrt(c) - m) / V is not sqrt(c / V) - m, and the sqrt means the
# correct divisor would be sqrt(V) rather than V.
# ---------------------------------------------------------------------------

# HO2 (H+O2+M chemiluminescence, ~87 km), FeO (Fe+O3 orange arc, ~85-90 km)
# and O2Ac (O+O+M -> O2 Chamberlain bands, ~90-95 km) are mesopause
# thin-shell emitters, not aerosol/Rayleigh continuum -- their sky-to-sci
# transfer needs the same van Rhijn factor as the OH bands. They are kept
# in a separate coefficient GROUP (broadband basis vs OH line groups, so
# their compression pipeline stays sqrt-identity with no PCA) but the
# geometric normalisation is now shared with mesospheric at 87 km.
AIRGLOW_GROUPS = {'mesospheric', 'atomic', 'ionospheric', 'continuum'}

GROUP_HEIGHT_FEATURE = {
    'mesospheric': 'vanrhijn_87km',
    'atomic': 'vanrhijn_95km',
    'ionospheric': 'vanrhijn_285km',
    'continuum': 'vanrhijn_87km',
}

# Only used as a last-resort fallback when a per-coefficient wavelength cannot
# be determined; see resolve_coef_wavelengths_a().
GROUP_EFFECTIVE_WAVELENGTH_A = {
    'mesospheric': 8500.0,
    'continuum': 7500.0,   # HO2/FeO/O2Ac mid-band; basis route usually resolves per-coefficient
    'atomic': 5750.0,
    'ionospheric': 6300.0,
}


def _group_height_feature_name(group_name):
    """van Rhijn context feature (hence effective layer height) for a group.

    mesospheric -> 87 km   (OH Meinel bands + O2 atmospheric band)
    continuum   -> 87 km   (HO2 / FeO / O2Ac mesopause chemiluminescence)
    atomic      -> 95 km   (K / Na / N I / OI 5577 mesopause metal / metastable)
    ionospheric -> 285 km  (OI 6300/6364 red + OI 7774/8446 F-region recomb)
    moon / other -> None (not thin-shell emission)
    """
    return GROUP_HEIGHT_FEATURE.get(group_name)


def _load_lco_extinction_curve():
    lvmcore_dir = os.environ.get('LVMCORE_DIR')
    if not lvmcore_dir:
        raise EnvironmentError('LVMCORE_DIR is not set; cannot load lco_extinction.txt')
    curve_path = Path(lvmcore_dir) / 'etc' / 'lco_extinction.txt'
    curve = np.loadtxt(curve_path, dtype=np.float64)
    if curve.ndim != 2 or curve.shape[1] < 2:
        raise ValueError(f'Unexpected extinction-curve shape: {curve.shape}')
    wave_a = np.asarray(curve[:, 0], dtype=np.float64)
    ext_k = np.asarray(curve[:, 1], dtype=np.float64)
    order = np.argsort(wave_a)
    return wave_a[order], ext_k[order]


LCO_EXTINCTION_WAVE_A, LCO_EXTINCTION_K = _load_lco_extinction_curve()


def _lco_extinction_k(wavelength_a):
    """Extinction coefficient (mag/airmass) at one or many wavelengths (Angstrom)."""
    lam = np.asarray(wavelength_a, dtype=np.float64)
    k = np.interp(
        lam,
        LCO_EXTINCTION_WAVE_A,
        LCO_EXTINCTION_K,
        left=float(LCO_EXTINCTION_K[0]),
        right=float(LCO_EXTINCTION_K[-1]),
    )
    return float(k) if np.ndim(wavelength_a) == 0 else np.asarray(k, dtype=np.float64)


# --- per-coefficient effective wavelength -----------------------------------
#
# Extinction varies strongly across the LVM range (k ~ 0.11 at 5577 A vs
# ~0.02 in the z band), so a single wavelength per group is not good enough:
# the 'mesospheric' group alone spans OI 5577, Na D and the OH/O2 bands.
#
# The primary wavelength source is coef_wavelengths_from_basis (reads the
# reconstructed basis directly). The name-based fallback below consults
# COEF_SCHEMA only, so new coefficient names cannot silently pick up a
# default guessed from digit tokens or species substrings.


def _wavelength_from_name(name):
    """Wavelength (Angstrom) from COEF_SCHEMA, else NaN."""
    hit = _lookup_coef_schema(name)
    if hit is None:
        return float('nan')
    _group, lam, _desc = hit
    return float(lam) if lam is not None and np.isfinite(lam) else float('nan')


def coef_wavelengths_from_basis(
    coef_names,
    wave,
    lsf_sigma,
    base_dir=None,
    n_spline_knots=25,
    cache_path=None,
    only_indices=None,
    verbose=True,
    return_k_eff=False,
):
    """Per-coefficient wavelength centroid and B^2-weighted effective extinction.

    Exploits linearity of the decomposition: reconstructing with coef = e_j
    returns basis component j on its own, so we read two scalars per
    coefficient in one reconstruction pass:

      1. INTENSITY-WEIGHTED CENTROID (reporting / binning):
             lambda_eff(j) = sum(lambda |f_j|) / sum(|f_j|)

      2. B^2-WEIGHTED EFFECTIVE EXTINCTION under the LCO stellar curve:
             k_eff(j) = sum(|f_j|^2 k(lambda)) / sum(|f_j|^2)
         For a broadband basis component the least-squares decomposition
         amplitude picks up the B^2-weighted average of any wavelength-dependent
         factor multiplying the true emissivity, so <k>_{B^2} is the correct
         effective extinction, not k(centroid). For narrow-line components
         |f_j| is sharply peaked and both weightings agree; the correction only
         matters for HO2 / FeO / O2Ac (continuum group) and any other broadband
         basis functions.

    Costs one reconstruction call per coefficient. Both scalars are cached to
    `cache_path` (npz keys `wavelengths_a`, `k_eff_a`); an older cache without
    `k_eff_a` is auto-invalidated and recomputed. Components that reconstruct
    to zero (outside the wavelength range) leave NaN in both arrays and fall
    back to name parsing / group defaults downstream.

    Returns `wavelengths_a` by default, or `(wavelengths_a, k_eff_a)` when
    `return_k_eff=True`.
    """
    coef_names = [str(n) for n in coef_names]
    n_coef = len(coef_names)
    wave = np.asarray(wave, dtype=np.float64)

    # The cache key must include the wavelength grid: the same coefficient
    # names evaluated on a different grid give different centroids, so keying
    # on names alone would silently return stale values after a change of input
    # product.
    grid_key = float(np.nansum(wave.astype(np.float64) * np.arange(1, wave.size + 1)))
    if cache_path is not None:
        cache_path = Path(cache_path)
        if cache_path.exists():
            cached = np.load(cache_path, allow_pickle=True)
            same_names = [str(x) for x in cached['coef_names']] == coef_names
            same_grid = ('grid_key' in cached.files
                         and np.isclose(float(cached['grid_key']), grid_key, rtol=1e-12))
            has_k_eff = 'k_eff_a' in cached.files
            if same_names and same_grid and has_k_eff:
                if verbose:
                    print(f'Basis wavelengths + k_eff loaded from cache: {cache_path}')
                lam_cached = np.asarray(cached['wavelengths_a'], dtype=np.float64)
                k_cached = np.asarray(cached['k_eff_a'], dtype=np.float64)
                return (lam_cached, k_cached) if return_k_eff else lam_cached
            if verbose:
                if not same_names:
                    reason = 'coefficient names'
                elif not same_grid:
                    reason = 'wavelength grid'
                else:
                    reason = 'missing k_eff_a (older cache format)'
                print(f'Cache {cache_path} does not match on {reason}; recomputing.')

    if base_dir is None:
        if '_infer_base_dir_for_reconstruction' not in globals():
            raise RuntimeError(
                'base_dir not given and _infer_base_dir_for_reconstruction() is '
                'not defined yet; run the reconstruction-helpers cell first or '
                'pass base_dir= explicitly.'
            )
        base_dir = _infer_base_dir_for_reconstruction()

    if only_indices is None:
        targets = np.arange(n_coef)
    else:
        targets = np.unique(np.asarray(only_indices, dtype=int))
        if verbose:
            print(f'Basis wavelengths: evaluating {targets.size} of {n_coef} '
                  f'coefficients (one reconstruction call each).')

    k_wave = np.asarray(_lco_extinction_k(wave), dtype=np.float64)
    lam_eff = np.full(n_coef, np.nan, dtype=np.float64)
    k_eff = np.full(n_coef, np.nan, dtype=np.float64)
    for j in targets:
        unit = np.zeros(n_coef, dtype=np.float64)
        unit[j] = 1.0
        comps = reconstruct_component_spectra(
            wave=wave,
            coef=unit,
            lsf_sigma=lsf_sigma,
            n_spline_knots=n_spline_knots,
            base_dir=base_dir,
        )
        f = np.abs(np.asarray(comps['total'], dtype=np.float64))
        tot = float(np.nansum(f))
        if np.isfinite(tot) and tot > 0.0:
            lam_eff[j] = float(np.nansum(wave * f) / tot)
            f2 = f * f
            f2_sum = float(np.nansum(f2))
            if np.isfinite(f2_sum) and f2_sum > 0.0:
                k_eff[j] = float(np.nansum(f2 * k_wave) / f2_sum)

    if cache_path is not None:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        np.savez(cache_path, coef_names=np.asarray(coef_names),
                 wavelengths_a=lam_eff, k_eff_a=k_eff,
                 grid_key=np.float64(grid_key))
        if verbose:
            print(f'Basis wavelengths + k_eff cached to: {cache_path}')

    if verbose:
        n_ok = int(np.isfinite(lam_eff).sum())
        n_ok_k = int(np.isfinite(k_eff).sum())
        print(f'Basis wavelengths derived for {n_ok}/{n_coef} coefficients; '
              f'basis B^2-weighted extinction k for {n_ok_k}/{n_coef}.')
    return (lam_eff, k_eff) if return_k_eff else lam_eff


def resolve_coef_wavelengths_a(
    coef_names,
    group_indices=None,
    basis_wavelengths_a=None,
    verbose=True,
):
    """Per-coefficient effective wavelength (Angstrom) plus its provenance.

    Priority, per coefficient:
      1. basis centroid from coef_wavelengths_from_basis(), where finite --
         reads the reconstructed basis directly, so it already carries the
         PMD-driven populations (including the mesopause rotational Boltzmann
         factor for OH), the LSF, and the blend structure across overlapping
         bands. This is the definitionally-correct weighting.
      2. explicit wavelength token or species lookup baked into COEF_SCHEMA.
      3. group default from GROUP_EFFECTIVE_WAVELENGTH_A.

    Returns (wavelengths_a, sources).  The printed table is deliberately
    verbose for the airglow groups: a silently wrong wavelength turns into a
    silently wrong extinction correction.
    """
    coef_names = [str(n) for n in coef_names]
    n_coef = len(coef_names)
    lam = np.full(n_coef, np.nan, dtype=np.float64)
    source = ['unset'] * n_coef

    if basis_wavelengths_a is not None:
        basis = np.asarray(basis_wavelengths_a, dtype=np.float64)
        if basis.size != n_coef:
            raise ValueError(
                f'basis_wavelengths_a has {basis.size} entries, expected {n_coef}'
            )
        ok = np.isfinite(basis) & (basis > 0.0)
        lam[ok] = basis[ok]
        for j in np.flatnonzero(ok):
            source[j] = 'basis'

    for j in range(n_coef):
        if np.isfinite(lam[j]):
            continue
        cand = _wavelength_from_name(coef_names[j])
        if np.isfinite(cand):
            lam[j] = cand
            source[j] = 'name'

    group_of = {}
    if group_indices is not None:
        for g, idx in group_indices.items():
            for j in np.asarray(idx, dtype=int):
                group_of[int(j)] = g
        for g, idx in group_indices.items():
            default = GROUP_EFFECTIVE_WAVELENGTH_A.get(g)
            if default is None:
                continue
            for j in np.asarray(idx, dtype=int):
                if not np.isfinite(lam[int(j)]):
                    lam[int(j)] = float(default)
                    source[int(j)] = f'group:{g}'

    if verbose:
        airglow_idx = [j for j in range(n_coef)
                       if group_of.get(j) in AIRGLOW_GROUPS]
        print(f'Per-coefficient wavelengths resolved for {n_coef} coefficients '
              f'({len(airglow_idx)} in airglow groups, which are the ones that '
              f'receive a geometry correction).')
        if airglow_idx:
            print(f"  {'coefficient':<28s} {'group':<13s} {'lambda[A]':>10s} "
                  f"{'k[mag/X]':>9s}  source")
            for j in airglow_idx:
                kj = _lco_extinction_k(lam[j]) if np.isfinite(lam[j]) else np.nan
                print(f'  {coef_names[j]:<28s} {group_of.get(j, "-"):<13s} '
                      f'{lam[j]:>10.1f} {kj:>9.4f}  {source[j]}')
            from collections import Counter
            print('  wavelength provenance:',
                  dict(Counter(source[j] for j in airglow_idx)))

    return lam, source


def assert_context_is_physical(ctx, ctx_names, rtol=1e-4, atol=1e-4):
    """Fail loudly if scaler-normalised context reaches the geometry code.

    Two independent checks:
      1. physical ranges (alt within +/-90 deg, airmass >= 1)
      2. any vanrhijn_* column must agree with recomputing van Rhijn from the
         'alt' column.  Those two are computed independently upstream, so
         disagreement means the context has been transformed on the way in.

    This is the regression test for the class of bug where geometry is
    evaluated on RobustScaler output: a scaled altitude of ~0.5 becomes a
    zenith angle of ~89.5 deg and every row gets a near-horizon van Rhijn
    factor of ~6.
    """
    ctx = np.asarray(ctx, dtype=np.float64)
    names = [str(n).strip().lower() for n in ctx_names]

    if ctx.ndim != 2:
        raise ValueError(f'context must be 2-D, got shape {ctx.shape}')
    if ctx.shape[1] != len(names):
        raise ValueError(
            f'context has {ctx.shape[1]} columns but {len(names)} names given'
        )
    for required in ('alt', 'airmass'):
        if required not in names:
            raise KeyError(
                f"geometry normalisation needs '{required}' among the context columns"
            )

    alt = ctx[:, names.index('alt')]
    airmass = ctx[:, names.index('airmass')]

    if np.nanmin(alt) < -90.0 or np.nanmax(alt) > 90.0:
        raise ValueError(
            f'alt outside [-90, 90] deg (min={np.nanmin(alt):.4g}, '
            f'max={np.nanmax(alt):.4g}); context looks scaled, not physical'
        )
    if np.nanmin(airmass) < 0.99:
        raise ValueError(
            f'airmass below 1 (min={np.nanmin(airmass):.4g}); '
            f'context looks scaled, not physical'
        )

    for feat, height_km in VAN_RHIJN_FEATURES.items():
        if feat not in names:
            continue
        stored = ctx[:, names.index(feat)]
        recomputed = np.asarray(_van_rhijn_factor(alt, float(height_km)), dtype=np.float64)
        if not np.allclose(stored, recomputed, rtol=rtol, atol=atol, equal_nan=True):
            worst = float(np.nanmax(np.abs(stored - recomputed)))
            raise ValueError(
                f"context column '{feat}' disagrees with van Rhijn recomputed "
                f"from 'alt' (max abs diff {worst:.4g}); context is not physical"
            )

    return True


def _airglow_ctx_columns(ctx_phys, ctx_names, check_physical=True):
    """Validate context and return (ctx, names, alt, airmass) in physical units."""
    ctx_phys = np.asarray(ctx_phys, dtype=np.float64)
    if ctx_phys.ndim == 1:
        ctx_phys = ctx_phys[None, :]
    names = [str(n).strip().lower() for n in ctx_names]
    if check_physical:
        assert_context_is_physical(ctx_phys, names)
    return (ctx_phys, names,
            ctx_phys[:, names.index('alt')],
            ctx_phys[:, names.index('airmass')])


def _coef_wavelengths_with_fallback(coef_wavelengths_a, group_indices, n_coef):
    """Per-coefficient wavelength array with group defaults filled in."""
    n_coef = int(n_coef)
    if coef_wavelengths_a is None:
        lam = np.full(n_coef, np.nan, dtype=np.float64)
    else:
        lam = np.asarray(coef_wavelengths_a, dtype=np.float64).astype(np.float64).copy()
        if lam.size != n_coef:
            raise ValueError(
                f'coef_wavelengths_a has {lam.size} entries, expected {n_coef}'
            )
    for group_name, idx in group_indices.items():
        fallback = GROUP_EFFECTIVE_WAVELENGTH_A.get(group_name)
        if fallback is None:
            continue
        idx = np.asarray(idx, dtype=int)
        if idx.size == 0:
            continue
        bad = ~(np.isfinite(lam[idx]) & (lam[idx] > 0.0))
        lam[idx[bad]] = float(fallback)
    return lam


def airglow_coef_extinction_k(coef_wavelengths_a, group_indices, n_coef,
                              coef_extinction_k=None):
    """Per-coefficient extinction coefficient (mag/airmass).

    If `coef_extinction_k` is supplied it is used verbatim -- that is the hook
    for the empirically fitted effective extinction (see
    fit_effective_extinction).  Otherwise the generic LCO stellar curve is
    interpolated at each coefficient's effective wavelength.
    """
    n_coef = int(n_coef)
    if coef_extinction_k is not None:
        k = np.asarray(coef_extinction_k, dtype=np.float64)
        if k.size != n_coef:
            raise ValueError(
                f'coef_extinction_k has {k.size} entries, expected {n_coef}'
            )
        return np.where(np.isfinite(k), k, 0.0)

    lam = _coef_wavelengths_with_fallback(coef_wavelengths_a, group_indices, n_coef)
    k = np.zeros(n_coef, dtype=np.float64)
    ok = np.isfinite(lam) & (lam > 0.0)
    if ok.any():
        k[ok] = np.atleast_1d(_lco_extinction_k(lam[ok])).astype(np.float64)
    return k


def airglow_van_rhijn_matrix(ctx_phys, ctx_names, group_indices, n_coef,
                             check_physical=True):
    """V(z; h) per row and coefficient; exactly 1.0 for non-airglow coefficients."""
    ctx_phys, _names, alt, _airmass = _airglow_ctx_columns(
        ctx_phys, ctx_names, check_physical)
    n_coef = int(n_coef)
    van_rhijn = np.ones((ctx_phys.shape[0], n_coef), dtype=np.float64)

    for group_name, idx in group_indices.items():
        if group_name not in AIRGLOW_GROUPS:
            continue
        feature_name = _group_height_feature_name(group_name)
        if feature_name is None or feature_name not in VAN_RHIJN_FEATURES:
            continue
        idx = np.asarray(idx, dtype=int)
        if idx.size == 0:
            continue
        height_km = float(VAN_RHIJN_FEATURES[feature_name])
        van_rhijn[:, idx] = np.asarray(
            _van_rhijn_factor(alt, height_km), dtype=np.float64)[:, None]

    return van_rhijn


def airglow_extinction_matrix(ctx_phys, ctx_names, group_indices, n_coef,
                              coef_wavelengths_a=None, coef_extinction_k=None,
                              check_physical=True):
    """10**(-0.4 k (X - 1)) per row and coefficient; 1.0 for non-airglow.

    Normalised to X = 1 rather than X = 0, so what this removes is the
    extinction *relative to zenith*.  The recovered quantity is therefore a
    zenith-equivalent amplitude, not an absolute layer emissivity.  That is
    self-consistent and cancels in the sky-to-science transfer.
    """
    ctx_phys, _names, _alt, airmass = _airglow_ctx_columns(
        ctx_phys, ctx_names, check_physical)
    n_coef = int(n_coef)
    k_all = airglow_coef_extinction_k(
        coef_wavelengths_a, group_indices, n_coef, coef_extinction_k)

    extinction = np.ones((ctx_phys.shape[0], n_coef), dtype=np.float64)
    for group_name, idx in group_indices.items():
        if group_name not in AIRGLOW_GROUPS:
            continue
        idx = np.asarray(idx, dtype=int)
        if idx.size == 0:
            continue
        extinction[:, idx] = 10.0 ** (
            -0.4 * k_all[idx][None, :] * (airmass[:, None] - 1.0))

    return extinction


def airglow_geometry_scale(
    ctx_phys,
    ctx_names,
    group_indices,
    n_coef,
    coef_wavelengths_a=None,
    coef_extinction_k=None,
    check_physical=True,
):
    """Geometry factor V(z; h) * 10**(-0.4 k (X - 1)) per row and coefficient.

    Returns an (n_rows, n_coef) array that is exactly 1.0 for every
    non-airglow coefficient (moon, other), since scattered moonlight needs
    scattering geometry rather than van Rhijn. The continuum group (HO2,
    FeO, O2Ac) IS airglow -- it's mesopause chemiluminescence, not aerosol
    or Rayleigh scattering -- and is treated at 87 km, same as OH.

    ctx_phys must be in physical units.  Divide physical coefficients by this
    to get zenith-equivalent amplitudes; multiply a prediction by it to get
    back to observed amplitudes at the target line of sight.

    Pass `coef_extinction_k` to use an empirically fitted effective extinction
    instead of the generic stellar curve.
    """
    van_rhijn = airglow_van_rhijn_matrix(
        ctx_phys, ctx_names, group_indices, n_coef, check_physical=check_physical)
    extinction = airglow_extinction_matrix(
        ctx_phys, ctx_names, group_indices, n_coef,
        coef_wavelengths_a=coef_wavelengths_a,
        coef_extinction_k=coef_extinction_k,
        check_physical=False)
    return np.clip(van_rhijn * extinction, 1e-6, None)


# ---------------------------------------------------------------------------
# Empirical effective airglow extinction
#
# The generic LCO curve is a *stellar* extinction curve.  Applying it to
# airglow over-corrects, because airglow is a quasi-uniform extended source:
# photons scattered out of the beam are largely replaced by photons scattered
# in from adjacent lines of sight, and scattering (Rayleigh + aerosol) is
# 70-100% of the total k everywhere airglow matters.  The effective airglow
# extinction is therefore well below the stellar value.
#
# Rather than model multiple scattering, fit the effective coefficient from
# the data.  The two sky pointings are simultaneous, so for coefficient j
#
#   ln(A_near/A_far) - ln(V_near/V_far) = -0.4 ln10 k_eff (X_near - X_far) + eps
#
# with eps the gravity-wave fluctuation between the two lines of sight (zero
# mean in log, uncorrelated with the airmass difference).  This is a Bouguer
# fit that uses airglow as its own source, and the recovered k_eff absorbs the
# multiple-scattering correction, the airglow-versus-stellar difference, the
# site aerosol level and any residual error in the assumed layer height.
#
# Two cautions, both documented in the methods section:
#   * h and k are nearly degenerate over the observed zenith range, so the
#     layer height is HELD FIXED here and only k_eff is fitted.  The product
#     V * 10^(-0.4 k (X-1)) is what the data constrain.
#   * a systematic horizontal gradient in layer brightness that correlates
#     with elevation at a fixed site would bias k_eff.  The returned table
#     includes split-half values so that can be checked.
# ---------------------------------------------------------------------------

def _airglow_height_per_coef(group_indices, n_coef):
    """Effective layer height per coefficient; NaN for non-airglow."""
    heights = np.full(int(n_coef), np.nan, dtype=np.float64)
    for group_name, idx in group_indices.items():
        if group_name not in AIRGLOW_GROUPS:
            continue
        feature_name = _group_height_feature_name(group_name)
        if feature_name is None or feature_name not in VAN_RHIJN_FEATURES:
            continue
        heights[np.asarray(idx, dtype=int)] = float(VAN_RHIJN_FEATURES[feature_name])
    return heights


def _masked_ols_rowconst(y, mask, x_row):
    """OLS of y_ij on [1, x_i] over masked entries, cluster-robust on rows.

    The airmass difference x is constant within a row, which collapses every
    sufficient statistic -- and the entire cluster-robust meat matrix -- to
    per-row counts and per-row sums:

        A_c' u_c = [sum_j u_ij, x_i sum_j u_ij] = g_i * [1, x_i]

    so no flattening and no per-cluster Python loop is needed.  The previous
    implementation looped over clusters (one iteration per row, via np.split)
    and was called about six times per wavelength bin, which dominated the
    runtime on large row counts.
    """
    y_masked = np.where(mask, y, 0.0)
    n_row = mask.sum(axis=1).astype(np.float64)
    s_row = y_masked.sum(axis=1)

    s1 = float(n_row.sum())
    sx = float((x_row * n_row).sum())
    sxx = float((x_row * x_row * n_row).sum())
    sy = float(s_row.sum())
    sxy = float((x_row * s_row).sum())

    xtx = np.array([[s1, sx], [sx, sxx]], dtype=np.float64)
    xtx_inv = np.linalg.pinv(xtx)
    beta = xtx_inv @ np.array([sy, sxy], dtype=np.float64)

    g = s_row - beta[0] * n_row - beta[1] * x_row * n_row
    g2 = g * g
    m01 = float((x_row * g2).sum())
    meat = np.array([[float(g2.sum()), m01],
                     [m01, float((x_row * x_row * g2).sum())]], dtype=np.float64)
    cov = xtx_inv @ meat @ xtx_inv
    return beta, cov, n_row


def fit_effective_extinction(
    coef_near,
    coef_far,
    ctx_near,
    ctx_far,
    ctx_names,
    group_indices,
    coef_wavelengths_a,
    n_wavelength_bins=8,
    wavelength_bin_edges=None,
    min_positive_fraction=0.80,
    clip_sigma=3.0,
    clip_iters=3,
    min_rows=100,
    min_pairs=500,
    clip_sample_rows=20000,
    seed=0,
    verbose=True,
):
    """Fit effective airglow extinction per wavelength bin from the sky pairs.

    Returns a DataFrame with one row per wavelength bin:
      lam_lo, lam_hi, lam_mid, n_coef, n_rows, n_pairs, retained_frac
      k_generic   -- generic LCO stellar curve at lam_mid, for comparison
      k_eff, k_eff_err  -- fitted effective extinction (cluster-robust error)
      intercept, intercept_err -- should be ~0; a significant value indicates a
                       relative throughput offset between the two sky channels
      k_eff_half1, k_eff_half2 -- split-half stability check
      ok          -- whether the bin is well enough constrained to be used

    Performance notes: van Rhijn is evaluated once per distinct layer height as
    an (n_rows,) vector rather than as an (n_rows, n_coef) matrix, the design
    statistics are accumulated from per-row sums instead of flattened
    (n_rows * n_coef) arrays, and the cluster-robust covariance is closed-form
    (see _masked_ols_rowconst).  Only the robust scale used for sigma clipping
    touches individual elements, and it is estimated on a row subsample.
    """
    rng = np.random.default_rng(seed)
    coef_near = np.asarray(coef_near, dtype=np.float64)
    coef_far = np.asarray(coef_far, dtype=np.float64)
    n_rows_all, n_coef = coef_near.shape

    ctx_near_arr, names, alt_near, airmass_near = _airglow_ctx_columns(
        ctx_near, ctx_names, check_physical=True)
    ctx_far_arr, _, alt_far, airmass_far = _airglow_ctx_columns(
        ctx_far, ctx_names, check_physical=False)
    delta_airmass = np.ascontiguousarray(airmass_near - airmass_far)

    height_per_coef = _airglow_height_per_coef(group_indices, n_coef)
    distinct_heights = np.unique(height_per_coef[np.isfinite(height_per_coef)])
    log_vr_ratio = {
        float(h): (np.log(_van_rhijn_factor(alt_near, float(h)))
                   - np.log(_van_rhijn_factor(alt_far, float(h))))
        for h in distinct_heights
    }

    lam = _coef_wavelengths_with_fallback(coef_wavelengths_a, group_indices, n_coef)
    airglow_cols = np.flatnonzero(np.isfinite(height_per_coef))
    if airglow_cols.size == 0:
        if verbose:
            print('fit_effective_extinction: no airglow coefficients; skipping.')
        return pd.DataFrame()

    if verbose:
        print(f'Effective-extinction fit: {airglow_cols.size} airglow coefficients, '
              f'{n_rows_all} rows, {len(distinct_heights)} distinct layer heights.')
        print(f'  airmass difference (near - far): '
              f'16-84% = {np.nanpercentile(delta_airmass, 16):+.3f}..'
              f'{np.nanpercentile(delta_airmass, 84):+.3f}, '
              f'|dX| median = {np.nanmedian(np.abs(delta_airmass)):.3f}')

    lam_ag = lam[airglow_cols]
    if wavelength_bin_edges is None:
        finite = np.isfinite(lam_ag)
        if finite.sum() < 2:
            return pd.DataFrame()
        wavelength_bin_edges = np.unique(np.nanpercentile(
            lam_ag[finite], np.linspace(0.0, 100.0, int(n_wavelength_bins) + 1)))
    wavelength_bin_edges = np.asarray(wavelength_bin_edges, dtype=np.float64)
    if wavelength_bin_edges.size < 2:
        return pd.DataFrame()

    half_assignment = rng.integers(0, 2, n_rows_all).astype(bool)
    slope_to_k = -1.0 / (0.4 * np.log(10.0))
    rows_out = []

    for b in range(wavelength_bin_edges.size - 1):
        lo, hi = wavelength_bin_edges[b], wavelength_bin_edges[b + 1]
        last = (b == wavelength_bin_edges.size - 2)
        in_bin = (lam_ag >= lo) & ((lam_ag <= hi) if last else (lam_ag < hi))
        cols = airglow_cols[in_bin]
        if cols.size == 0:
            continue

        near_block = coef_near[:, cols]
        far_block = coef_far[:, cols]

        # column-level selection only -- a per-row amplitude cut would be
        # selection on the outcome and biases the slope, because whichever arm
        # sits at lower airmass has the smaller van Rhijn factor and fails the
        # cut unless it carries a positive gravity-wave fluctuation
        positive = (near_block > 0.0) & (far_block > 0.0)
        keep_col = positive.mean(axis=0) >= float(min_positive_fraction)
        if not keep_col.any():
            continue
        cols = cols[keep_col]
        near_block = near_block[:, keep_col]
        far_block = far_block[:, keep_col]
        mask = positive[:, keep_col]

        # log amplitude ratio with van Rhijn removed, extinction left in.
        # van Rhijn is constant within a layer height, so it is subtracted as a
        # column vector per height rather than as a full matrix.
        with np.errstate(divide='ignore', invalid='ignore'):
            y_mat = np.log(np.where(mask, near_block, 1.0)) - np.log(
                np.where(mask, far_block, 1.0))
        for h in distinct_heights:
            sel = height_per_coef[cols] == h
            if sel.any():
                y_mat[:, sel] -= log_vr_ratio[float(h)][:, None]
        mask &= np.isfinite(y_mat)
        n_possible = int(mask.size)

        finite_x = np.isfinite(delta_airmass)
        if not finite_x.all():
            mask &= finite_x[:, None]

        for _ in range(int(clip_iters)):
            if mask.sum() < 10:
                break
            beta, _cov, _n_row = _masked_ols_rowconst(y_mat, mask, delta_airmass)
            resid = y_mat - (beta[0] + beta[1] * delta_airmass[:, None])
            if n_rows_all > int(clip_sample_rows):
                sample_rows = rng.choice(n_rows_all, int(clip_sample_rows), replace=False)
            else:
                sample_rows = np.arange(n_rows_all)
            vals = resid[sample_rows][mask[sample_rows]]
            if vals.size < 10:
                break
            med = float(np.median(vals))
            mad = 1.4826 * float(np.median(np.abs(vals - med)))
            if not np.isfinite(mad) or mad <= 0.0:
                break
            new_mask = mask & (np.abs(resid - med) < clip_sigma * mad)
            if new_mask.sum() == mask.sum():
                mask = new_mask
                break
            mask = new_mask

        n_pairs = int(mask.sum())
        lam_mid = float(np.nanmedian(lam[cols]))
        k_generic = float(_lco_extinction_k(lam_mid))
        if n_pairs < 10:
            continue

        beta, cov, n_row = _masked_ols_rowconst(y_mat, mask, delta_airmass)
        n_rows = int((n_row > 0).sum())
        retained = n_pairs / max(n_possible, 1)
        k_eff = float(beta[1] * slope_to_k)
        k_err = float(np.sqrt(max(cov[1, 1], 0.0)) * abs(slope_to_k))

        halves = []
        for which in (False, True):
            half_mask = mask & (half_assignment == which)[:, None]
            if half_mask.sum() >= 10:
                bh, _, _ = _masked_ols_rowconst(y_mat, half_mask, delta_airmass)
                halves.append(float(bh[1] * slope_to_k))
            else:
                halves.append(np.nan)

        rel_err = k_err / abs(k_eff) if k_eff != 0 else np.inf
        ok = bool(n_rows >= min_rows and n_pairs >= min_pairs
                  and np.isfinite(k_eff) and np.isfinite(k_err) and rel_err < 0.5)

        rows_out.append({
            'lam_lo': float(lo), 'lam_hi': float(hi), 'lam_mid': lam_mid,
            'n_coef': int(cols.size), 'n_rows': n_rows, 'n_pairs': n_pairs,
            'retained_frac': float(retained),
            'k_generic': k_generic, 'k_eff': k_eff, 'k_eff_err': k_err,
            'k_ratio': k_eff / k_generic if k_generic > 0 else np.nan,
            'intercept': float(beta[0]),
            'intercept_err': float(np.sqrt(max(cov[0, 0], 0.0))),
            'k_eff_half1': halves[0], 'k_eff_half2': halves[1],
            'ok': ok,
        })

    table = pd.DataFrame(rows_out)
    if verbose and not table.empty:
        print('  fitted effective extinction by wavelength bin:')
        cols_show = ['lam_mid', 'n_coef', 'n_rows', 'n_pairs', 'retained_frac',
                     'k_generic', 'k_eff', 'k_eff_err', 'k_ratio', 'intercept',
                     'k_eff_half1', 'k_eff_half2', 'ok']
        print(table[cols_show].to_string(
            index=False, float_format=lambda v: f'{v:.4g}'))
        low_ret = table[table['retained_frac'] < 0.5]
        if len(low_ret):
            print(f'  NOTE: {len(low_ret)} bin(s) retained under 50% of pairs. Heavy '
                  f'row-level loss reintroduces selection on the outcome and biases '
                  f'k_eff upward; consider raising min_positive_fraction so whole '
                  f'columns are dropped instead.')
        bad_int = table[np.abs(table['intercept'])
                        > 3.0 * table['intercept_err'].clip(lower=1e-12)]
        if len(bad_int):
            print(f'  NOTE: {len(bad_int)} bin(s) have an intercept >3 sigma from '
                  f'zero, which points to a relative throughput offset between '
                  f'the two sky channels rather than an extinction effect.')
    elif verbose:
        print('  effective-extinction fit produced no usable bins.')

    return table


def resolve_coef_extinction_k(
    coef_names,
    coef_wavelengths_a,
    group_indices,
    fit_table=None,
    clip_to_generic=True,
    verbose=True,
    coef_basis_k_generic=None,
):
    """Per-coefficient extinction, preferring the fitted k_eff over the generic curve.

    Well-constrained bins (`ok` in the fit table) are interpolated in
    wavelength; everything else falls back to the generic LCO curve.  With
    `clip_to_generic`, fitted values are restricted to [0, k_generic]: the
    multiple-scattering argument makes the effective airglow extinction a lower
    bound problem, so a fitted value above the stellar curve indicates noise or
    an unmodelled gradient rather than real physics.

    `coef_basis_k_generic`, when provided, is the per-coefficient B^2-weighted
    effective k from coef_wavelengths_from_basis(); where finite it overrides
    the point-sample `_lco_extinction_k(lambda_centroid)` as both the generic
    reference and the [0, k_generic] clip. For narrow-line coefficients this is
    a no-op (|f_j|^2 is peaked at the centroid); for broadband basis components
    (HO2, FeO, O2Ac) it correctly reflects the integrated behaviour of the LCO
    stellar curve across the basis function's support.
    """
    coef_names = [str(n) for n in coef_names]
    n_coef = len(coef_names)
    lam = _coef_wavelengths_with_fallback(coef_wavelengths_a, group_indices, n_coef)
    k_generic = airglow_coef_extinction_k(lam, group_indices, n_coef)
    if coef_basis_k_generic is not None:
        basis_k = np.asarray(coef_basis_k_generic, dtype=np.float64)
        if basis_k.size != n_coef:
            raise ValueError(
                f'coef_basis_k_generic has {basis_k.size} entries, expected {n_coef}'
            )
        replace = np.isfinite(basis_k) & (basis_k >= 0.0)
        if replace.any():
            k_generic = np.where(replace, basis_k, k_generic)
    k_out = k_generic.copy()
    source = ['generic'] * n_coef

    airglow_cols = np.concatenate(
        [np.asarray(idx, dtype=int) for g, idx in group_indices.items()
         if g in AIRGLOW_GROUPS and len(idx) > 0]
    ) if any(g in AIRGLOW_GROUPS and len(idx) > 0
             for g, idx in group_indices.items()) else np.zeros(0, dtype=int)

    usable = None
    if fit_table is not None and len(fit_table):
        usable = fit_table[fit_table['ok'].astype(bool)].sort_values('lam_mid')

    n_clipped = 0
    if usable is not None and len(usable) >= 1:
        lam_nodes = np.asarray(usable['lam_mid'], dtype=np.float64)
        k_nodes = np.asarray(usable['k_eff'], dtype=np.float64)
        for j in airglow_cols:
            if not np.isfinite(lam[j]):
                continue
            k_fit = float(np.interp(lam[j], lam_nodes, k_nodes,
                                    left=k_nodes[0], right=k_nodes[-1]))
            if clip_to_generic:
                k_clipped = float(np.clip(k_fit, 0.0, k_generic[j]))
                if k_clipped != k_fit:
                    n_clipped += 1
                k_fit = k_clipped
            k_out[j] = k_fit
            source[j] = 'fitted'

    if verbose:
        n_fit = sum(1 for s in source if s == 'fitted')
        print(f'Extinction resolved for {airglow_cols.size} airglow coefficients: '
              f'{n_fit} fitted, {airglow_cols.size - n_fit} generic.')
        if n_clipped:
            print(f'  {n_clipped} fitted value(s) clipped into [0, k_generic].')
        if airglow_cols.size:
            ratio = np.divide(k_out[airglow_cols], k_generic[airglow_cols],
                              out=np.full(airglow_cols.size, np.nan),
                              where=k_generic[airglow_cols] > 0)
            with np.errstate(invalid='ignore'):
                print(f'  k_eff / k_generic: median {np.nanmedian(ratio):.3f}, '
                      f'range {np.nanmin(ratio):.3f}..{np.nanmax(ratio):.3f}')

    return k_out, source




In [ ]:
# New helper: build simultaneous triplets (near, far -> sci) from decomposition files
def _load_decomp_with_row_index(
    decomp_fits_path,
    input_fits_path,
    context_columns,
    decomp_kind,
    return_chi2=False,
    return_err=False,
):
    """Load one decomp product and keep original source-row indices after finite filtering."""
    out = read_decomp_dataset(
        decomp_fits_path=decomp_fits_path,
        input_fits_path=input_fits_path,
        context_columns=context_columns,
        decomp_kind=decomp_kind,
        return_chi2=return_chi2,
        return_err=return_err,
    )

    # Unpack in the order used by read_decomp_dataset: base, then chi2, then err.
    _it = iter(out)
    coef_mat = next(_it); ctx_mat = next(_it); coef_names = next(_it); ctx_names = next(_it)
    chi2_used = next(_it) if return_chi2 else None
    coef_err_mat = next(_it) if return_err else None

    with fits.open(decomp_fits_path) as hdul_dec, fits.open(input_fits_path) as hdul_in:
        coef_tbl_full = _coerce_coef_hdu_to_table(hdul_dec['COEF'])
        coef_full, _ = _table_to_float32_matrix(coef_tbl_full, 'coefficient')
        meta_full = Table(hdul_in['META'].data)
        ctx_full, _ = _build_context_matrix(meta_full, context_columns, decomp_kind)
        meta_upper_full = {c.upper(): c for c in meta_full.colnames}
        obstime_mjd_full = _extract_obstime_mjd(meta_full, meta_upper_full)

    if coef_full.shape[0] != ctx_full.shape[0]:
        raise ValueError(
            f'Row count mismatch while building row_index: COEF has {coef_full.shape[0]} rows, META has {ctx_full.shape[0]} rows'
        )

    good = np.isfinite(coef_full).all(axis=1) & np.isfinite(ctx_full).all(axis=1)
    row_index = np.flatnonzero(good).astype(np.int64)
    obstime_mjd = np.asarray(obstime_mjd_full[good], dtype=np.float64)

    if row_index.size != coef_mat.shape[0]:
        raise RuntimeError(
            f'Internal row-index mismatch: row_index has {row_index.size}, data has {coef_mat.shape[0]}'
        )

    result = [coef_mat, ctx_mat, coef_names, ctx_names]
    if return_chi2:
        result.append(chi2_used)
    result.extend([row_index, obstime_mjd])
    if return_err:
        result.append(coef_err_mat)
    return tuple(result)


def build_triplet_coef_dataset(
    input_fits_path,
    sky_near_decomp_fits_path,
    sky_far_decomp_fits_path,
    sci_decomp_fits_path,
    context_columns,
    return_chi2=False,
    return_err=True,
):
    """Build aligned triplet arrays for coefficient-transfer experiments.

    ``return_err=True`` (default) also pulls the per-coefficient 1-sigma
    uncertainties from the ``COEF_ERR`` HDU (see fit.SkyDecomp._coef_err_active_set)
    and stores them as ``coef_err_near``, ``coef_err_far`` and ``coef_err_sci``.
    Older decomposition files without ``COEF_ERR`` yield all-NaN arrays.
    """

    def _load(kind, path):
        return _load_decomp_with_row_index(
            decomp_fits_path=path,
            input_fits_path=input_fits_path,
            context_columns=context_columns,
            decomp_kind=kind,
            return_chi2=return_chi2,
            return_err=return_err,
        )

    near = _load('sky1', sky_near_decomp_fits_path)
    far = _load('sky2', sky_far_decomp_fits_path)
    sci = _load('sci', sci_decomp_fits_path)

    def _unpack(out):
        it = iter(out)
        coef = next(it); ctx = next(it); cnames = next(it); xnames = next(it)
        chi2 = next(it) if return_chi2 else None
        row = next(it); mjd = next(it)
        err = next(it) if return_err else None
        return coef, ctx, cnames, xnames, chi2, row, mjd, err

    coef_near, ctx_near, coef_names_n, ctx_names_n, chi2_near, row_near, mjd_near, coef_err_near = _unpack(near)
    coef_far, ctx_far, coef_names_f, ctx_names_f, chi2_far, row_far, mjd_far, coef_err_far = _unpack(far)
    coef_sci, ctx_sci, coef_names_s, ctx_names_s, chi2_sci, row_sci, mjd_sci, coef_err_sci = _unpack(sci)

    if coef_names_n != coef_names_f or coef_names_n != coef_names_s:
        raise ValueError('Coefficient name mismatch across near/far/sci decomposition products.')
    if ctx_names_n != ctx_names_f or ctx_names_n != ctx_names_s:
        raise ValueError('Context name mismatch across near/far/sci products.')

    row_common = np.intersect1d(np.intersect1d(row_near, row_far), row_sci)
    if row_common.size == 0:
        raise ValueError('No shared source rows across near/far/sci after finite filtering.')

    idx_near = np.searchsorted(row_near, row_common)
    idx_far = np.searchsorted(row_far, row_common)
    idx_sci = np.searchsorted(row_sci, row_common)

    if (not np.array_equal(row_near[idx_near], row_common)
            or not np.array_equal(row_far[idx_far], row_common)
            or not np.array_equal(row_sci[idx_sci], row_common)):
        raise RuntimeError('Row-index alignment failure while building triplet dataset.')

    mjd_stack = np.column_stack([mjd_near[idx_near], mjd_far[idx_far], mjd_sci[idx_sci]])
    obstime_mjd = np.nanmedian(mjd_stack, axis=1).astype(np.float64)

    out = {
        'coef_near': coef_near[idx_near],
        'coef_far': coef_far[idx_far],
        'coef_sci': coef_sci[idx_sci],
        'ctx_near': ctx_near[idx_near],
        'ctx_far': ctx_far[idx_far],
        'ctx_sci': ctx_sci[idx_sci],
        'coef_names': coef_names_n,
        'ctx_names': ctx_names_n,
        'row_index': row_common.astype(np.int64),
        'obstime_mjd': obstime_mjd,
        'n_rows': int(row_common.size),
    }

    if return_chi2:
        out['chi2_near'] = chi2_near[idx_near]
        out['chi2_far'] = chi2_far[idx_far]
        out['chi2_sci'] = chi2_sci[idx_sci]

    if return_err:
        out['coef_err_near'] = coef_err_near[idx_near]
        out['coef_err_far'] = coef_err_far[idx_far]
        out['coef_err_sci'] = coef_err_sci[idx_sci]

    # Attach the science-pointing RA/Dec from META so downstream filters
    # can exclude tiles by sky region (see LMC/SMC exclusion in
    # apply_triplet_filters).  Try prefixed columns first, then unprefixed.
    try:
        with fits.open(input_fits_path) as _hdul_meta:
            _meta_tbl = Table(_hdul_meta['META'].data)
        _meta_upper = {c.upper(): c for c in _meta_tbl.colnames}
        _ra_col = next((_meta_upper[k] for k in ('SCI_RA', 'RA') if k in _meta_upper), None)
        _dec_col = next((_meta_upper[k] for k in ('SCI_DEC', 'DEC') if k in _meta_upper), None)
        if _ra_col is not None and _dec_col is not None:
            out['sci_ra'] = np.asarray(_meta_tbl[_ra_col], dtype=np.float64)[row_common]
            out['sci_dec'] = np.asarray(_meta_tbl[_dec_col], dtype=np.float64)[row_common]
            out['sci_radec_source'] = (_ra_col, _dec_col)
        else:
            print('  WARNING: sci RA/Dec columns not found in META '
                  '(looked for SCI_RA/SCI_DEC then RA/DEC); '
                  'field-level exclusion in apply_triplet_filters will be skipped.')
    except Exception as _exc:
        print(f'  WARNING: could not attach sci_ra/sci_dec '
              f'({type(_exc).__name__}: {_exc}).')

    print(
        f"Triplet dataset built: n_rows={out['n_rows']}, n_coef={out['coef_near'].shape[1]}, n_ctx={out['ctx_near'].shape[1]}"
        + (f" | sci pointing from META columns ({out['sci_radec_source'][0]}, {out['sci_radec_source'][1]})"
           if 'sci_radec_source' in out else "")
    )
    return out

In [ ]:
# Reused reconstruction/prediction helpers for quick visual checks
def _meta_row_to_dict_upper(meta_row):
    names = list(meta_row.colnames) if hasattr(meta_row, 'colnames') else list(meta_row.dtype.names)
    return {str(k).upper(): k for k in names}

def _safe_float(x):
    arr = np.asarray(x)
    if arr.size == 0:
        raise ValueError('Empty value cannot be converted to float')
    if arr.shape != ():
        arr = arr.ravel()[0]
    return float(arr)


def _infer_base_dir_for_reconstruction():
    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
    if 'PALACE_DIR' in globals():
        try:
            p = Path(PALACE_DIR).resolve()
            candidates.extend([p, p.parent])
        except Exception:
            pass

    for cand in candidates:
        if (cand / 'palace' / 'PMD').exists() and (cand / 'Spectre_HR_LATMOS_Meftah_V1_350_1000nm.txt').exists():
            return cand

    raise FileNotFoundError('Could not infer reconstruction base_dir containing palace/PMD and solar reference file')

def load_lsf_state_if_available(decomp_fits_path, spectrum_index):
    """Return an LSFSurfaceState from a decomposition FITS row, or None if absent.

    The new sky_decomp.lsf_surface_iterative module writes LSF_COEF / LSF_KNOTS /
    LSF_META extensions alongside COEF and META.  Older decomposition outputs do
    not have those extensions; this returns None so callers can fall back to a
    Gaussian LSF from the input FITS.
    """
    path = Path(decomp_fits_path)
    if not path.exists():
        return None
    try:
        with fits.open(str(path)) as hdul:
            ext_names = {h.name for h in hdul}
            if 'LSF_COEF' not in ext_names:
                return None
        return load_lsf_surface_state(str(path), int(spectrum_index))
    except (KeyError, IndexError, ValueError) as exc:
        print(f'  LSF surface state unavailable in {path.name} row {spectrum_index}: '
              f'{type(exc).__name__}: {exc}')
        return None


def load_o2_vector_if_available(decomp_fits_path, spectrum_index):
    """Return the unit-integrated O2 template for one row, or None if absent.

    The reworked decomposition pipeline (fit.py + decompose_parallel.py) writes a
    VECTOR_O2 ImageHDU (n_rows x n_wave) alongside COEF / META. Older
    decomposition products predate the split of the O2 amplitude into the
    coefficient, so their VECTOR_O2 is missing and the O2 basis reconstructs to
    zero (matching the historical behaviour of reconstruct_component_spectra).
    """
    path = Path(decomp_fits_path)
    if not path.exists():
        return None
    try:
        with fits.open(str(path)) as hdul:
            if 'VECTOR_O2' not in {h.name for h in hdul}:
                return None
            data = np.asarray(hdul['VECTOR_O2'].data, dtype=np.float64)
    except (KeyError, IndexError, ValueError) as exc:
        print(f'  VECTOR_O2 unavailable in {path.name} row {spectrum_index}: '
              f'{type(exc).__name__}: {exc}')
        return None
    if data.ndim != 2 or int(spectrum_index) >= data.shape[0]:
        return None
    row = data[int(spectrum_index)]
    if not np.isfinite(row).any() or float(np.nansum(np.abs(row))) == 0.0:
        return None
    return row


def reconstruct_with_lsf(wave, coef, lsf, *, n_spline_knots=25, base_dir=None,
                         o2_vector=None, coef_err=None):
    """Reconstruct component spectra, dispatching on the LSF representation.

    ``lsf`` is either an ``LSFSurfaceState`` (uses the wavelength-dependent
    B-spline kernel via ``SkyDecompLSFSurfaceIterative._assemble_refined_matrices``)
    or a per-pixel Gaussian sigma vector / scalar (uses the parent-class
    ``reconstruct_component_spectra``).  Returns the same components dict as
    ``reconstruct_component_spectra``.

    If ``coef_err`` (per-coefficient 1σ, same shape as ``coef``) is
    provided, the returned dict additionally contains ``sigma`` (dict of
    per-component 1σ flux uncertainty per pixel) and ``sigma_total``
    (quadrature sum across independent components).  See
    ``SkyDecompBase._components_sigma_from_coef_err`` for the propagation.
    """
    if isinstance(lsf, LSFSurfaceState):
        model = SkyDecompLSFSurfaceIterative(
            wave,
            lsf_sigma=1.0,  # dummy; only stick matrices are used with the surface path
            n_spline_knots=n_spline_knots,
            base_dir=base_dir,
        )
        coef_arr = np.asarray(coef, float).ravel()
        model._set_lsf_state(lsf)
        mats = model._assemble_refined_matrices()
        # VECTOR_O2 on disk is already convolved by the fitted LSF surface at fit
        # time; injecting it into matrix_o2_stick and letting _assemble_refined_matrices
        # re-convolve would double-broaden the O2 shape. Override the assembled O2
        # block verbatim instead.
        if o2_vector is not None:
            o2_vec = np.asarray(o2_vector, float).ravel()
            if o2_vec.shape != model.wave.shape:
                raise ValueError(
                    f'o2_vector shape mismatch: expected {model.wave.shape}, got {o2_vec.shape}'
                )
            mats['o2'] = o2_vec[None, :]
        n_expected = sum(m.shape[0] for m in mats.values())
        if coef_arr.size != n_expected:
            raise ValueError(
                f'Coefficient length mismatch: expected {n_expected}, got {coef_arr.size}'
            )
        comps = model._components_from_coef(coef_arr, mats)
        comps['total'] = (comps['oh'] + comps['moon'] + comps['diffuse']
                          + comps['atom'] + comps['orc'] + comps['o2'])
        if coef_err is not None:
            err_arr = np.asarray(coef_err, float).ravel()
            if err_arr.size != coef_arr.size:
                raise ValueError(
                    f'coef_err length mismatch: expected {coef_arr.size}, '
                    f'got {err_arr.size}'
                )
            sigma_comps = model._components_sigma_from_coef_err(err_arr, mats)
            comps['sigma'] = sigma_comps
            comps['sigma_total'] = np.sqrt(
                sigma_comps['oh'] ** 2
                + sigma_comps['moon'] ** 2
                + sigma_comps['diffuse'] ** 2
                + sigma_comps['atom'] ** 2
                + sigma_comps['orc'] ** 2
                + sigma_comps['o2'] ** 2
            )
        return comps
    return reconstruct_component_spectra(
        wave=wave, coef=coef, lsf_sigma=lsf,
        n_spline_knots=n_spline_knots, base_dir=base_dir, o2_vector=o2_vector,
        coef_err=coef_err,
    )

In [ ]:
# Starter data load for coefficient prediction experiments
context_cols = [
    'alt',
    'az_sin',
    'az_cos',
    'airmass',
    'moon_sep',
    'moon_alt',
    'moon_az_sin',
    'moon_az_cos',
    'moon_phase_sin',
    'moon_phase_cos',
    'sun_sep',
    'sun_alt',
    'sun_az_sin',
    'sun_az_cos',
    'sci_sep',
    'vanrhijn_87km',
    'vanrhijn_95km',
    'vanrhijn_285km',
    'obstime_day_sin',
    'obstime_day_cos',
    'obstime_lunation_sin',
    'obstime_lunation_cos',
    'obstime_year_sin',
    'obstime_year_cos',
    'f107',
    'f107_81d',
    'kp',
    'ew',
]

# Toggle the decomposition source: LSF-surface-iterative (wavelength-dependent LSF) or the baseline (Gaussian LSF).
USE_LSF_SURFACE_ITERATIVE = True
_DECOMP_SUFFIX = '_lsf_surface_iterative' if USE_LSF_SURFACE_ITERATIVE else ''
_DECOMP_STEM = 'spline_moon/lvmsframe_median_stack_1.2.1_p40_p70'
print(f'Using decomposition products: suffix={_DECOMP_SUFFIX!r}')

triplet = build_triplet_coef_dataset(
    input_fits_path=f'{_DECOMP_STEM}_meta_only.fits',
    sky_near_decomp_fits_path=f'{_DECOMP_STEM}_sky1_meta_coef{_DECOMP_SUFFIX}.fits',
    sky_far_decomp_fits_path=f'{_DECOMP_STEM}_sky2_meta_coef{_DECOMP_SUFFIX}.fits',
    sci_decomp_fits_path=f'{_DECOMP_STEM}_sci_meta_coef{_DECOMP_SUFFIX}.fits',
    context_columns=context_cols,
    return_chi2=True,
)

print('Shapes:')
print('  coef_near', triplet['coef_near'].shape)
print('  coef_far ', triplet['coef_far'].shape)
print('  coef_sci ', triplet['coef_sci'].shape)
print('  ctx_near ', triplet['ctx_near'].shape)
print('  ctx_far  ', triplet['ctx_far'].shape)
print('  ctx_sci  ', triplet['ctx_sci'].shape)
print('  obstime  ', triplet['obstime_mjd'].shape)


In [ ]:
# Filtering borrowed from the original notebook workflow, adapted to triplets
import plotly.express as px


def _angular_separation_deg_vec(ra_deg, dec_deg, ra_c_deg, dec_c_deg):
    """Great-circle angular separation (degrees), vectorised over (ra, dec)."""
    ra = np.deg2rad(np.asarray(ra_deg, dtype=np.float64))
    dec = np.deg2rad(np.asarray(dec_deg, dtype=np.float64))
    ra_c = np.deg2rad(float(ra_c_deg))
    dec_c = np.deg2rad(float(dec_c_deg))
    cos_sep = (np.sin(dec) * np.sin(dec_c)
               + np.cos(dec) * np.cos(dec_c) * np.cos(ra - ra_c))
    return np.rad2deg(np.arccos(np.clip(cos_sep, -1.0, 1.0)))

LMC_EXCLUSION = {'name': 'LMC', 'ra_deg': 81, 'dec_deg': -69.7, 'radius_deg': 10.0}
SMC_EXCLUSION = {'name': 'SMC', 'ra_deg': 14, 'dec_deg': -73, 'radius_deg': 10}

def _kappa_sigma_row_mask(x, kappa=5.0, n_iter=3):
    x = np.asarray(x, dtype=np.float64)
    keep = np.isfinite(x).all(axis=1)
    if not np.any(keep):
        return keep

    for _ in range(n_iter):
        mu = np.nanmean(x[keep], axis=0)
        sig = np.nanstd(x[keep], axis=0)
        sig = np.where(np.isfinite(sig) & (sig > 0), sig, 1.0)
        within = np.all(np.abs(x - mu) <= (kappa * sig), axis=1)
        within &= np.isfinite(x).all(axis=1)
        new_keep = keep & within
        if new_keep.sum() == keep.sum() or new_keep.sum() == 0:
            break
        keep = new_keep

    return keep


def apply_triplet_filters(
    triplet_data,
    thin_every_n=1,
    chi2_qmax=90.0,
    chi2_min=0.0,
    chi2_max=10.0,
    hard_coef_bounds=None,
    kappa=6.0,
    kappa_iter=3,
    exclude_field_regions=None,
    airmass_max=3.0,
):
    if hard_coef_bounds is None:
        hard_coef_bounds = {'feo': (0.0, 1.0), 'atom_k': (0.0, 1.0)}

    coef_names_local = [str(n) for n in triplet_data['coef_names']]
    coef_name_l = [n.lower() for n in coef_names_local]

    coef_near = np.asarray(triplet_data['coef_near'], dtype=np.float32)
    coef_far = np.asarray(triplet_data['coef_far'], dtype=np.float32)
    coef_sci = np.asarray(triplet_data['coef_sci'], dtype=np.float32)
    ctx_near = np.asarray(triplet_data['ctx_near'], dtype=np.float32)
    ctx_far = np.asarray(triplet_data['ctx_far'], dtype=np.float32)
    ctx_sci = np.asarray(triplet_data['ctx_sci'], dtype=np.float32)

    n0 = coef_near.shape[0]
    keep = np.ones(n0, dtype=bool)

    # Exclude tiles whose science pointing falls within a specified angular
    # radius of a listed sky region.  Defaults to LMC and SMC at 10 deg each --
    # both Clouds combine bright stellar populations, dense H II regions and
    # diffuse ionised gas at velocities close enough to airglow to blend with
    # the sky-component fit of coef_sci (methods sect 11.2).  Pass an empty
    # list to disable.
    if exclude_field_regions is None:
        exclude_field_regions = [LMC_EXCLUSION, SMC_EXCLUSION]
    if exclude_field_regions:
        if 'sci_ra' not in triplet_data or 'sci_dec' not in triplet_data:
            print("Science-field exclusion requested but triplet has no "
                  "sci_ra/sci_dec; skipping.  Re-run build_triplet_coef_dataset "
                  "so it attaches the science pointing coordinates.")
        else:
            sci_ra = np.asarray(triplet_data['sci_ra'], dtype=np.float64)
            sci_dec = np.asarray(triplet_data['sci_dec'], dtype=np.float64)
            field_mask = np.ones(n0, dtype=bool)
            for region in exclude_field_regions:
                reg_name = str(region.get('name', 'unnamed'))
                ra_c = float(region['ra_deg'])
                dec_c = float(region['dec_deg'])
                r_deg = float(region['radius_deg'])
                sep = _angular_separation_deg_vec(sci_ra, sci_dec, ra_c, dec_c)
                inside = np.isfinite(sep) & (sep <= r_deg)
                print(
                    f"Science-field exclusion around {reg_name} "
                    f"(ra={ra_c:.3f} deg, dec={dec_c:.3f} deg, radius={r_deg:.1f} deg): "
                    f"excluded {int(inside.sum())}/{inside.size} "
                    f"({100.0 * inside.mean():.1f}%)"
                )
                field_mask &= ~inside
            keep &= field_mask
            print(
                f"Combined science-field exclusion: kept {int(field_mask.sum())}/"
                f"{len(field_mask)} ({100.0 * field_mask.mean():.1f}%)"
            )

    ctx_names_l = [str(n).strip().lower() for n in triplet_data['ctx_names']]
    alt_idx = ctx_names_l.index('alt') if 'alt' in ctx_names_l else None
    airmass_idx = ctx_names_l.index('airmass') if 'airmass' in ctx_names_l else None
    vr87_idx = ctx_names_l.index('vanrhijn_87km') if 'vanrhijn_87km' in ctx_names_l else None
    vr95_idx = ctx_names_l.index('vanrhijn_95km') if 'vanrhijn_95km' in ctx_names_l else None
    vr285_idx = ctx_names_l.index('vanrhijn_285km') if 'vanrhijn_285km' in ctx_names_l else None

    if alt_idx is None:
        print("Altitude sanity filter: context column 'alt' not found; skipping altitude >= 0 check.")

    physical_mask = np.ones(n0, dtype=bool)
    if alt_idx is not None:
        alt_ok = (
            np.isfinite(ctx_near[:, alt_idx]) & (ctx_near[:, alt_idx] >= 0.0)
            & np.isfinite(ctx_far[:, alt_idx]) & (ctx_far[:, alt_idx] >= 0.0)
            & np.isfinite(ctx_sci[:, alt_idx]) & (ctx_sci[:, alt_idx] >= 0.0)
        )
        physical_mask &= alt_ok
        print(
            f"Altitude sanity filter (alt >= 0 in near/far/sci): kept {alt_ok.sum()}/{alt_ok.size} "
            f"({100.0 * alt_ok.mean():.1f}%)"
        )

    # airmass = sec(z) diverges near the horizon; anything above ~3 is not
    # science-usable and dominates the loss through the (X-1) extinction term.
    if airmass_idx is not None and airmass_max is not None:
        am_max = float(airmass_max)
        am_ok = (
            np.isfinite(ctx_near[:, airmass_idx]) & (ctx_near[:, airmass_idx] <= am_max)
            & np.isfinite(ctx_far[:, airmass_idx]) & (ctx_far[:, airmass_idx] <= am_max)
            & np.isfinite(ctx_sci[:, airmass_idx]) & (ctx_sci[:, airmass_idx] <= am_max)
        )
        physical_mask &= am_ok
        print(
            f"Airmass sanity filter (airmass <= {am_max:g} in near/far/sci): "
            f"kept {am_ok.sum()}/{am_ok.size} ({100.0 * am_ok.mean():.1f}%)"
        )
    elif airmass_idx is None and airmass_max is not None:
        print("Airmass sanity filter: context column 'airmass' not found; skipping.")

    for label, idx in (('vanrhijn_87km', vr87_idx), ('vanrhijn_95km', vr95_idx), ('vanrhijn_285km', vr285_idx)):
        if idx is None:
            continue
        vr_ok = (
            np.isfinite(ctx_near[:, idx]) & (ctx_near[:, idx] >= 1.0)
            & np.isfinite(ctx_far[:, idx]) & (ctx_far[:, idx] >= 1.0)
            & np.isfinite(ctx_sci[:, idx]) & (ctx_sci[:, idx] >= 1.0)
        )
        physical_mask &= vr_ok
        print(
            f"{label} sanity filter (finite and >= 1): kept {vr_ok.sum()}/{vr_ok.size} "
            f"({100.0 * vr_ok.mean():.1f}%)"
        )

    keep &= physical_mask
    print(
        f"Combined physical context filter: kept {physical_mask.sum()}/{len(physical_mask)} "
        f"({100.0 * physical_mask.mean():.1f}%)"
    )

    if all(k in triplet_data for k in ('chi2_near', 'chi2_far', 'chi2_sci')):
        chi2_stack = np.column_stack(
            [
                np.asarray(triplet_data['chi2_near'], dtype=np.float64),
                np.asarray(triplet_data['chi2_far'], dtype=np.float64),
                np.asarray(triplet_data['chi2_sci'], dtype=np.float64),
            ]
        )
        # Use max (not nanmax) so any-arm-NaN chi2 propagates NaN and
        # the row gets dropped by the isfinite gate below.  This
        # matches the intent that ANY per-arm decomposition failure
        # disqualifies the whole observation.
        chi2_combined = np.max(chi2_stack, axis=1)
        chi2_finite = chi2_combined[np.isfinite(chi2_combined)]
        chi2_hi = np.nanpercentile(chi2_finite, chi2_qmax)
        chi2_upper = min(float(chi2_max), float(chi2_hi)) if chi2_max is not None else float(chi2_hi)
        chi2_mask = np.isfinite(chi2_combined) & (chi2_combined >= chi2_min) & (chi2_combined <= chi2_upper)
        keep &= chi2_mask
        print(
            f"Triplet chi2 filter: min={chi2_min:.3g}, qmax={chi2_qmax:.1f}%=>{chi2_hi:.3g}, "
            f"upper={chi2_upper:.3g} | keep={chi2_mask.sum()}/{len(chi2_mask)} ({100.0*chi2_mask.mean():.1f}%)"
        )
        fig_chi2_triplet = px.histogram(
            x=chi2_combined[keep],
            nbins=80,
            title='Triplet combined reduced chi2 distribution (rows used for training)',
            labels={'x': 'max(reduced chi2 near/far/sci)', 'y': 'count'},
        )
        fig_chi2_triplet.update_layout(template='plotly_white', bargap=0.03)
        fig_chi2_triplet.show()
    else:
        print('Triplet chi2 columns not present; chi2 filtering skipped.')

    for cname, (lo, hi) in hard_coef_bounds.items():
        idxs = np.where(np.array(coef_name_l) == str(cname).lower())[0]
        if idxs.size == 0:
            print(f"Manual hard clip: coefficient {cname} not found; skipping.")
            continue

        j = int(idxs[0])
        within = (
            np.isfinite(coef_near[:, j]) & (coef_near[:, j] >= lo) & (coef_near[:, j] <= hi)
            & np.isfinite(coef_far[:, j]) & (coef_far[:, j] >= lo) & (coef_far[:, j] <= hi)
            & np.isfinite(coef_sci[:, j]) & (coef_sci[:, j] >= lo) & (coef_sci[:, j] <= hi)
        )
        keep &= within
        print(
            f"Manual hard clip {cname}: [{lo:.3g}, {hi:.3g}] | kept {within.sum()}/{within.size} ({100.0 * within.mean():.1f}%)"
        )

    coef_concat = np.hstack([coef_near, coef_far, coef_sci]).astype(np.float32)
    kappa_mask = _kappa_sigma_row_mask(coef_concat, kappa=float(kappa), n_iter=int(kappa_iter))
    keep &= kappa_mask
    print(
        f"Kappa-sigma filter (kappa={kappa:.1f}): kept {kappa_mask.sum()}/{len(kappa_mask)} ({100.0 * kappa_mask.mean():.1f}%)"
    )

    # Thinning is applied LAST so the filter-fraction prints above reflect
    # counts against the full pre-thinning dataset.  It selects every N-th row
    # in the ORIGINAL row indexing; combined with the accumulated filter mask
    # via boolean AND, the result is the intersection.
    if int(thin_every_n) > 1:
        thin_mask = np.zeros(n0, dtype=bool)
        thin_mask[:: int(thin_every_n)] = True
        keep &= thin_mask
        print(f"Final thinning: every {int(thin_every_n)}-th of {n0} original rows; "
              f"combined with earlier filters, kept {int(keep.sum())} rows")
    else:
        print(f"Final thinning disabled: kept {int(keep.sum())}/{n0} rows after filters")

    if keep.sum() == 0:
        raise RuntimeError('Filtering removed all rows; relax thresholds.')

    out = {
        'coef_near': coef_near[keep],
        'coef_far': coef_far[keep],
        'coef_sci': coef_sci[keep],
        'ctx_near': ctx_near[keep],
        'ctx_far': ctx_far[keep],
        'ctx_sci': ctx_sci[keep],
        'coef_names': coef_names_local,
        'ctx_names': list(triplet_data['ctx_names']),
        'mask': keep,
        'row_index': np.asarray(triplet_data['row_index'])[keep],
    }
    if 'obstime_mjd' in triplet_data:
        out['obstime_mjd'] = np.asarray(triplet_data['obstime_mjd'], dtype=np.float64)[keep]
    for k in ('chi2_near', 'chi2_far', 'chi2_sci',
              'coef_err_near', 'coef_err_far', 'coef_err_sci',
              'sci_ra', 'sci_dec'):
        if k in triplet_data:
            out[k] = np.asarray(triplet_data[k])[keep]
    if 'sci_radec_source' in triplet_data:
        out['sci_radec_source'] = triplet_data['sci_radec_source']

    print(
        f"Filtered triplet shapes: near={out['coef_near'].shape}, far={out['coef_far'].shape}, "
        f"sci={out['coef_sci'].shape}"
    )

    # Fold cyclic sin/cos pairs (az, moon_az, sun_az, moon_phase) back to
    # a single 0-360 degree axis per feature so histograms are readable.
    _display_names, _display_near = _decode_cyclic_context(out['ctx_names'], out['ctx_near'])
    _, _display_far = _decode_cyclic_context(out['ctx_names'], out['ctx_far'])
    _, _display_sci = _decode_cyclic_context(out['ctx_names'], out['ctx_sci'])

    # Distribution plot: one panel per context feature, three fields shown as
    # step lines (no fill).  Bin edges are computed per panel from that
    # feature's own min/max, so each panel has an independent x-range and
    # bin width -- which matters because the features span very different
    # ranges (degrees, airmass, sin/cos, van Rhijn factors).
    _hist_field_arrays = {
        'sky_near': _display_near,
        'sky_far': _display_far,
        'science': _display_sci,
    }
    _hist_field_colors = {
        'sky_near': '#1f77b4',
        'sky_far': '#9467bd',
        'science': '#2ca02c',
    }
    _hist_field_order = ['sky_near', 'sky_far', 'science']
    _hist_n_bins = 60

    context_order = list(_display_names)
    n_context = len(context_order)
    n_facet_cols = min(5, max(1, n_context))
    n_facet_rows = int(np.ceil(n_context / n_facet_cols))
    _n_panels_total = n_facet_rows * n_facet_cols

    fig_ctx_hist = make_subplots(
        rows=n_facet_rows,
        cols=n_facet_cols,
        subplot_titles=[c.replace('_', ' ') for c in context_order]
            + [''] * (_n_panels_total - n_context),
        horizontal_spacing=0.05,
        vertical_spacing=0.09,
    )

    for k, cname in enumerate(context_order):
        row = k // n_facet_cols + 1
        col = k % n_facet_cols + 1
        # sci_sep is 0 by construction for the science pointing; skip its delta at 0.
        panel_field_order = [f for f in _hist_field_order
                             if not (cname == 'sci_sep' and f == 'science')]
        per_field_vals = {}
        for f in panel_field_order:
            v = np.asarray(_hist_field_arrays[f][:, k], dtype=np.float64)
            per_field_vals[f] = v[np.isfinite(v)]
        combined = np.concatenate([per_field_vals[f] for f in panel_field_order])
        if combined.size < 2:
            continue
        lo = float(np.nanmin(combined))
        hi = float(np.nanmax(combined))
        if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            continue
        edges = np.linspace(lo, hi, _hist_n_bins + 1)
        step_x = np.repeat(edges, 2)[1:-1]
        for f in panel_field_order:
            v = per_field_vals[f]
            if v.size == 0:
                continue
            counts, _ = np.histogram(v, bins=edges)
            step_y = np.repeat(counts, 2)
            fig_ctx_hist.add_trace(
                go.Scattergl(
                    x=step_x,
                    y=step_y,
                    mode='lines',
                    line=dict(color=_hist_field_colors[f], width=1.4),
                    name=f,
                    legendgroup=f,
                    showlegend=(k == 0),
                    hovertemplate=f + ': %{y}<extra></extra>',
                ),
                row=row,
                col=col,
            )

    fig_ctx_hist.for_each_annotation(lambda a: a.update(font=dict(size=11)))
    fig_ctx_hist.update_xaxes(showline=True, mirror=True, ticks='outside',
                              ticklen=4, showticklabels=True)
    fig_ctx_hist.update_yaxes(showline=True, mirror=True, ticks='outside',
                              ticklen=4, showticklabels=True)
    fig_ctx_hist.update_layout(
        template='plotly_white',
        title='Context parameter distributions by field (rows used after all filters)',
        height=n_facet_rows * 220 + 200,
        width=min(2000, n_facet_cols * 340 + 140),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0.0),
        margin=dict(l=60, r=20, t=90, b=50),
    )
    fig_ctx_hist.show()

    return out


filtered_triplet = apply_triplet_filters(
    triplet,
    thin_every_n=1,
    chi2_qmax=90.0,
    chi2_min=0.0,
    chi2_max=10.0,
    hard_coef_bounds={'feo': (0.0, 10.01), 'atom_k': (0.0, 10.01)},
    kappa=6.0,
    kappa_iter=3,
    exclude_field_regions=[LMC_EXCLUSION, SMC_EXCLUSION],
)

In [ ]:
# ECLIPTIC-CTX-V1: augment triplet ctx with PER-ARM ecliptic coordinates.
# Adds three features to each arm's ctx (following the same convention as
# `alt`, `az_sin`, `az_cos`: shared feature names, arm-specific values):
#   ecl_beta_deg -- raw ecliptic latitude in degrees (-90..+90). Zodi
#     amplitude peaks at beta=0 (ecliptic plane), so cos(beta) drives the
#     scaling; the MLP learns that from the raw latitude directly.
#   ecl_lon_sin/cos -- cyclic embedding of ecliptic longitude.
# `build_triplet_coef_dataset` in this notebook only attaches sci_ra/sci_dec;
# `_fill_arm_radec_from_meta_fits` reads SKY_NEAR/SKY_FAR RA/Dec directly from
# a meta FITS so per-arm ecliptic works even without changing the loader.
from astropy.coordinates import SkyCoord, BarycentricMeanEcliptic
import astropy.units as u

ECLIPTIC_FEATURE_NAMES = ['ecl_beta_deg', 'ecl_lon_sin', 'ecl_lon_cos']
# Older names we strip on sight so re-running is idempotent.
_LEGACY_ECLIPTIC_NAMES = [
    'sci_ecl_lat_sin', 'sci_ecl_lat_cos',
    'sci_ecl_lon_sin', 'sci_ecl_lon_cos',
    'ecl_lat_sin', 'ecl_lat_cos',
]


def _ecliptic_features(ra_deg, dec_deg):
    _coords = SkyCoord(ra=np.asarray(ra_deg) * u.deg,
                       dec=np.asarray(dec_deg) * u.deg, frame='icrs')
    _ecl = _coords.transform_to(BarycentricMeanEcliptic())
    _lat_deg = np.asarray(_ecl.lat.degree, dtype=np.float32)
    _lon_rad = np.asarray(_ecl.lon.radian, dtype=np.float64)
    return np.stack([_lat_deg,
                     np.sin(_lon_rad).astype(np.float32),
                     np.cos(_lon_rad).astype(np.float32)],
                    axis=1).astype(np.float32)


def _fill_arm_radec_from_meta_fits(triplet, meta_fits_path=None):
    """Attach near/far RA/Dec to triplet by reading a meta FITS directly.

    Tries an explicit ``meta_fits_path`` first; otherwise falls back to
    ``{_DECOMP_STEM}_meta_only.fits`` then ``{_DECOMP_STEM}.fits``.
    """
    _row_idx = np.asarray(triplet['row_index'], dtype=int)
    _candidates = []
    if meta_fits_path is not None:
        _candidates.append(str(meta_fits_path))
    _stem = globals().get('_DECOMP_STEM',
                          'spline_moon/lvmsframe_median_stack_1.2.1_p40_p70')
    _candidates.extend([f'{_stem}_meta_only.fits', f'{_stem}.fits'])
    _meta = None
    _used_path = None
    for _p in _candidates:
        if not Path(_p).exists():
            continue
        try:
            with fits.open(_p) as _hdul:
                _meta = Table(_hdul['META'].data)
            _used_path = _p
            break
        except (KeyError, OSError) as _exc:
            print(f'  meta FITS candidate {_p} unusable: '
                  f'{type(_exc).__name__}: {_exc}')
    if _meta is None:
        print('  WARNING: no meta FITS available for per-arm RA/Dec fill; '
              'ecliptic will fall back to sci pointing for near/far arms')
        return
    _meta_up = {c.upper(): c for c in _meta.colnames}
    for _pref, _rakey, _deckey in (
        ('near', 'SKY_NEAR_RA', 'SKY_NEAR_DEC'),
        ('far', 'SKY_FAR_RA', 'SKY_FAR_DEC'),
    ):
        if f'{_pref}_ra' in triplet and f'{_pref}_dec' in triplet:
            continue
        if _rakey in _meta_up and _deckey in _meta_up:
            triplet[f'{_pref}_ra'] = np.asarray(
                _meta[_meta_up[_rakey]], dtype=np.float64)[_row_idx]
            triplet[f'{_pref}_dec'] = np.asarray(
                _meta[_meta_up[_deckey]], dtype=np.float64)[_row_idx]
    print(f'  filled per-arm RA/Dec from meta FITS ({_used_path})')


def _augment_triplet_with_ecliptic(triplet, force=True, meta_fits_path=None):
    """Append per-arm ecliptic features to ctx_near/ctx_far/ctx_sci in place.

    ``force=True`` (default): strip pre-existing ecliptic features and
    recompute from RA/Dec so re-running never leaves stale values in place.
    ``meta_fits_path``: optional override for the meta FITS used when the
    triplet is missing near/far RA/Dec (see ``_fill_arm_radec_from_meta_fits``).
    """
    _names = list(triplet.get('ctx_names', []))
    _to_strip = set(_LEGACY_ECLIPTIC_NAMES)
    if force:
        _to_strip.update(ECLIPTIC_FEATURE_NAMES)
    _strip_idx = [i for i, _n in enumerate(_names) if _n in _to_strip]
    if _strip_idx:
        _keep = [i for i in range(len(_names)) if i not in _strip_idx]
        for _arm in ('ctx_near', 'ctx_far', 'ctx_sci'):
            triplet[_arm] = np.asarray(triplet[_arm],
                                       dtype=np.float32)[:, _keep]
        _names = [_names[i] for i in _keep]
        triplet['ctx_names'] = _names
        print(f'  stripped {len(_strip_idx)} pre-existing ecliptic feature(s) '
              'so per-arm augment recomputes cleanly')
    if 'sci_ra' not in triplet or 'sci_dec' not in triplet:
        raise RuntimeError('triplet missing sci_ra/sci_dec; cannot compute '
                           'ecliptic features')
    if ('near_ra' not in triplet or 'far_ra' not in triplet
            or 'near_dec' not in triplet or 'far_dec' not in triplet):
        try:
            _fill_arm_radec_from_meta_fits(triplet, meta_fits_path=meta_fits_path)
        except Exception as _e:
            print(f'  meta FITS fallback failed: {type(_e).__name__}: {_e}')
    _sci_feats = _ecliptic_features(triplet['sci_ra'], triplet['sci_dec'])
    for _arm_prefix, _ctx_key in (('near', 'ctx_near'), ('far', 'ctx_far'),
                                  ('sci', 'ctx_sci')):
        if _arm_prefix == 'sci':
            _feats = _sci_feats
        elif f'{_arm_prefix}_ra' in triplet and f'{_arm_prefix}_dec' in triplet:
            _feats = _ecliptic_features(triplet[f'{_arm_prefix}_ra'],
                                        triplet[f'{_arm_prefix}_dec'])
        else:
            print(f'  WARNING: {_arm_prefix}_ra/{_arm_prefix}_dec unavailable; '
                  f'falling back to sci ecliptic for ctx_{_arm_prefix}')
            _feats = _sci_feats
        _prev = np.asarray(triplet[_ctx_key], dtype=np.float32)
        triplet[_ctx_key] = np.concatenate([_prev, _feats],
                                           axis=1).astype(np.float32)
    triplet['ctx_names'] = _names + ECLIPTIC_FEATURE_NAMES


_augment_triplet_with_ecliptic(filtered_triplet, force=True)
print(f'filtered_triplet ctx augmented: n_ctx={len(filtered_triplet["ctx_names"])} '
      f'(added {ECLIPTIC_FEATURE_NAMES}, per-arm ecliptic)')


In [ ]:
# Per-coefficient effective wavelength and effective extinction.
#
# Both are needed because extinction varies strongly across the LVM range: the
# 'mesospheric' group alone spans OI 5577, Na D and the OH/O2 bands, and even
# within the OH Meinel system k varies by nearly a factor of two.
#
INPUT_FITS_FOR_BASIS = 'spline_moon/lvmsframe_median_stack_1.2.1_p40_p70_every10.fits'
WAVELENGTH_CACHE = Path('spline_moon/coef_wavelengths_basis_v3.npz')  # p40_p70 decomposition (2026-08-15); uses v3 cache based on p70 improved continuumnuum basis
USE_FITTED_EXTINCTION = True   # False -> generic LCO stellar curve only

import time as _time
_t_start = _time.perf_counter()


def _lap(label):
    global _t_start
    now = _time.perf_counter()
    print(f'    [{label}: {now - _t_start:.1f} s]')
    _t_start = now


# --- 1. basis-derived wavelengths + B^2-weighted effective extinction -----
# One reconstruction call per coefficient. Reads the actual basis (built from
# the same PMD populations, LSF, and wavelength grid the decomposition uses at
# fit time), so the returned centroid already carries the mesopause rotational
# Boltzmann factor for OH and the correct blend structure across overlapping
# bands. There is no auxiliary population-model route to cross-check against;
# it was retired 2026-08-04 (closes methods sect 11.5).
_group_indices_preview = _build_group_indices(filtered_triplet['coef_names'])
coef_wavelengths_basis = None
coef_k_eff_basis = None
try:
    with fits.open(INPUT_FITS_FOR_BASIS) as _hdul:
        _ext_names = [h.name for h in _hdul]
        if 'WAVE' not in _ext_names:
            raise KeyError(f'{INPUT_FITS_FOR_BASIS} has no WAVE extension')
        _wave_ref = np.asarray(_hdul['WAVE'].data, dtype=np.float64)
        _lsf_name = 'LSF_SCI' if 'LSF_SCI' in _ext_names else (
            'LSF' if 'LSF' in _ext_names else None)
        if _lsf_name is None:
            raise KeyError(f'{INPUT_FITS_FOR_BASIS} has no LSF_SCI or LSF extension')
        _lsf_ref = np.asarray(_hdul[_lsf_name].data, dtype=np.float64)
    if _wave_ref.ndim > 1:
        _wave_ref = _wave_ref[0]
    if _lsf_ref.ndim > 1:
        _lsf_ref = _lsf_ref[0]

    coef_wavelengths_basis, coef_k_eff_basis = coef_wavelengths_from_basis(
        coef_names=filtered_triplet['coef_names'],
        wave=_wave_ref,
        lsf_sigma=_lsf_ref / 2.35,
        n_spline_knots=25,
        cache_path=WAVELENGTH_CACHE,
        only_indices=None,   # every coefficient; disk cache absorbs the one-shot cost
        return_k_eff=True,
        verbose=True,
    )
except Exception as exc:
    print(f'Basis wavelengths unavailable ({type(exc).__name__}: {exc}); '
          f'falling back to name-token and group defaults.')

_lap('basis wavelengths + B^2 k_eff')

# --- 2. resolve one wavelength per coefficient ---------------------------
_grp_sizes = {g: int(idx.size) for g, idx in _group_indices_preview.items()}
_expected_sizes = {'moon': 29, 'continuum': 3, 'mesospheric': 403,
                   'atomic': 3, 'ionospheric': 4}
print(f'Coefficient group sizes: {_grp_sizes} (total {sum(_grp_sizes.values())})')
if _grp_sizes != _expected_sizes:
    print(f'  NOTE: expected {_expected_sizes} for the lvmsframe_median_stack_1.2.1_p70 '
          f'product; a different input product can legitimately shift the '
          f'OH_### count (basis coverage of the OH bands) or the Moon_bs## '
          f'count (n_spline_knots). If the diff is elsewhere, extend '
          f'COEF_SCHEMA to cover the new coefficient family.')
coef_wavelengths_a, coef_wavelength_source = resolve_coef_wavelengths_a(
    filtered_triplet['coef_names'],
    group_indices=_group_indices_preview,
    basis_wavelengths_a=coef_wavelengths_basis,
    verbose=True,
)
filtered_triplet['coef_wavelengths_a'] = coef_wavelengths_a
_lap('wavelength resolution')

# --- 5. contexts must be physical before any geometry is evaluated -------
for _label, _ctx in (('near', filtered_triplet['ctx_near']),
                     ('far', filtered_triplet['ctx_far']),
                     ('sci', filtered_triplet['ctx_sci'])):
    assert_context_is_physical(_ctx, filtered_triplet['ctx_names'])
    print(f'context[{_label}] verified physical (van Rhijn columns consistent with alt)')

# --- 6. fit the EFFECTIVE extinction from the simultaneous sky pairs ------
# The tabulated LCO curve is a stellar curve. Airglow is a quasi-uniform
# extended source, so photons scattered out of the beam are largely replaced by
# photons scattered in from adjacent lines of sight, and the effective
# attenuation is well below the stellar value. Fitting it from the near/far
# pairs sidesteps having to model that, and simultaneously absorbs the
# airglow-versus-stellar difference, the site aerosol level and any residual
# error in the assumed layer height.
extinction_fit_table = pd.DataFrame()
if USE_FITTED_EXTINCTION:
    extinction_fit_table = fit_effective_extinction(
        coef_near=filtered_triplet['coef_near'],
        coef_far=filtered_triplet['coef_far'],
        ctx_near=filtered_triplet['ctx_near'],
        ctx_far=filtered_triplet['ctx_far'],
        ctx_names=filtered_triplet['ctx_names'],
        group_indices=_group_indices_preview,
        coef_wavelengths_a=coef_wavelengths_a,
        n_wavelength_bins=8,
        verbose=True,
    )

coef_extinction_k, coef_extinction_source = resolve_coef_extinction_k(
    filtered_triplet['coef_names'],
    coef_wavelengths_a,
    _group_indices_preview,
    fit_table=extinction_fit_table if USE_FITTED_EXTINCTION else None,
    clip_to_generic=True,
    coef_basis_k_generic=coef_k_eff_basis,
    verbose=True,
)
filtered_triplet['coef_extinction_k'] = coef_extinction_k
_lap('effective-extinction fit')


## Layer-Normalized Structure Function Diagnostics

This section measures the near/far pair structure function for grouped coefficients after optional van Rhijn normalization. It is intended as a first empirical check on the spatial variability scale available to the model.

In [ ]:
# Structure-function diagnostics for grouped coefficients after van Rhijn + airmass normalization

def _angular_separation_deg(ra1_deg, dec1_deg, ra2_deg, dec2_deg):
    ra1 = np.deg2rad(np.asarray(ra1_deg, dtype=np.float64))
    dec1 = np.deg2rad(np.asarray(dec1_deg, dtype=np.float64))
    ra2 = np.deg2rad(np.asarray(ra2_deg, dtype=np.float64))
    dec2 = np.deg2rad(np.asarray(dec2_deg, dtype=np.float64))
    cos_sep = (
        np.sin(dec1) * np.sin(dec2)
        + np.cos(dec1) * np.cos(dec2) * np.cos(ra1 - ra2)
    )
    cos_sep = np.clip(cos_sep, -1.0, 1.0)
    return np.rad2deg(np.arccos(cos_sep))


def _group_amplitude(coef_block, reducer='sum'):
    """Collapse a group's coefficients to one amplitude per row.

    'sum' is the default because it is the total zenith-equivalent emissivity
    of the group, which is what the structure function is meant to track. A
    median over a group is dominated by whichever coefficients happen to be
    mid-magnitude, and for a group like the 402 OH line coefficients it can
    land near zero and be pushed into the logarithm floor.
    """
    coef_block = np.asarray(coef_block, dtype=np.float64)
    coef_block = np.clip(coef_block, 0.0, None)
    if reducer == 'median':
        return np.nanmedian(coef_block, axis=1)
    if reducer == 'sum':
        return np.nansum(coef_block, axis=1)
    raise ValueError(f'unknown reducer {reducer!r}')


# Grouping is defined once, in the loaders cell.  These aliases keep the
# older _sf call sites working without a second copy that can drift.
_build_group_indices_sf = _build_group_indices


required = ['filtered_triplet']
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError('Run the filtering cell first. Missing: ' + ', '.join(missing))

with fits.open('spline_moon/lvmsframe_median_stack_1.2.1_p40_p70_meta_only.fits') as hdul:
    meta_tbl = Table(hdul['META'].data)

row_idx = np.asarray(filtered_triplet['row_index'], dtype=np.int64)
meta_used = meta_tbl[row_idx]
sep_near_far = _angular_separation_deg(
    meta_used['sky_near_ra'],
    meta_used['sky_near_dec'],
    meta_used['sky_far_ra'],
    meta_used['sky_far_dec'],
)

ctx_names_sf = [str(n) for n in filtered_triplet['ctx_names']]
airmass_idx = ctx_names_sf.index('airmass') if 'airmass' in ctx_names_sf else None
if airmass_idx is None:
    raise RuntimeError("airmass context feature is required for residual normalization; rerun Cell 6 + Cell 7 after adding 'airmass' to context_cols.")
group_indices_sf = _build_group_indices_sf(filtered_triplet['coef_names'])

coef_names_sf = [str(n) for n in filtered_triplet['coef_names']]
n_total_coef = len(coef_names_sf)

def _family_tag(name):
    sl = str(name).lower()
    if sl.startswith('moon_bs'):
        return 'moon_bs'
    if sl.startswith('oh'):
        return 'oh'
    return None

group_summary_rows = []
for gname, gidx in group_indices_sf.items():
    idx = np.asarray(gidx, dtype=int)
    raw_names = [coef_names_sf[i] for i in idx]

    moon_names = [n for n in raw_names if _family_tag(n) == 'moon_bs']
    oh_names = [n for n in raw_names if _family_tag(n) == 'oh']
    other_coeff = [n for n in raw_names if _family_tag(n) is None]

    family_tokens = []
    if moon_names:
        family_tokens.append(f"moon_bs[{len(moon_names)}]")
    if oh_names:
        family_tokens.append(f"oh[{len(oh_names)}]")

    group_summary_rows.append({
        'group': gname,
        'vanrhijn_feature': _group_height_feature_name(gname) or 'none',
        'n_coeff': int(idx.size),
        'frac_coeff': float(idx.size / max(n_total_coef, 1)),
        'collapsed_families': ', '.join(family_tokens),
        'moon_bs_example': moon_names[0] if moon_names else '',
        'oh_example': oh_names[0] if oh_names else '',
        'other_coeff': ', '.join(other_coeff),
    })

grouping_summary_df = pd.DataFrame(group_summary_rows).sort_values('n_coeff', ascending=False).reset_index(drop=True)
print('Van Rhijn assignment rules:')
print(_group_height_feature_name.__doc__.strip())
print('\nCoefficient grouping + van Rhijn assignment summary:')
print(
    grouping_summary_df[
        ['group', 'vanrhijn_feature', 'n_coeff', 'frac_coeff', 'collapsed_families', 'moon_bs_example', 'oh_example']
    ].to_string(index=False, float_format=lambda v: f'{v:.4f}')
)
print('Other coefficients by group (verbatim; wavelength-like 4-digit names remain visible):')
for _, r in grouping_summary_df.iterrows():
    print(f"[{r['group']}] {r['other_coeff']}")

# Normalise with EXACTLY the transform the model applies, via the shared
# helper. The previous version divided by van Rhijn and then again by airmass
# itself; dividing by X is not an extinction correction, and on top of the van
# Rhijn division it is a double correction. It injected a spurious
# -log(X_near / X_far) term whose scatter is an order of magnitude larger than
# the real extinction term, which inflated D(dtheta) severalfold and
# contaminated its shape -- making the measured information ceiling far too
# pessimistic. Sharing one code path with the model removes that whole class of
# discrepancy.
_sf_geom_kwargs = dict(
    ctx_names=filtered_triplet['ctx_names'],
    group_indices=group_indices_sf,
    n_coef=filtered_triplet['coef_near'].shape[1],
    coef_wavelengths_a=filtered_triplet.get('coef_wavelengths_a'),
    coef_extinction_k=filtered_triplet.get('coef_extinction_k'),
)
_sf_scale_near = airglow_geometry_scale(filtered_triplet['ctx_near'], **_sf_geom_kwargs)
_sf_scale_far = airglow_geometry_scale(filtered_triplet['ctx_far'], **_sf_geom_kwargs)
_sf_coef_near_em = np.asarray(filtered_triplet['coef_near'], dtype=np.float64) / _sf_scale_near
_sf_coef_far_em = np.asarray(filtered_triplet['coef_far'], dtype=np.float64) / _sf_scale_far
_sf_normalization = 'vanrhijn+extinction (airglow_geometry_scale)'
print(f'Structure function normalised by: {_sf_normalization}')

rows = []

for group_name, idx in group_indices_sf.items():
    if len(idx) == 0 or group_name in ('moon', 'continuum', 'other'):
        continue

    # amplitudes are already in zenith-equivalent (geometry-removed) units,
    # so the only remaining step is the logarithm
    near_amp = _group_amplitude(_sf_coef_near_em[:, idx])
    far_amp = _group_amplitude(_sf_coef_far_em[:, idx])

    near_resid_log_amp = np.log(np.clip(near_amp, 1e-8, None))
    far_resid_log_amp = np.log(np.clip(far_amp, 1e-8, None))

    normalization = _sf_normalization

    delta_resid_log = near_resid_log_amp - far_resid_log_amp
    d_theta = np.asarray(sep_near_far, dtype=np.float64)
    sf_value = delta_resid_log ** 2
    finite = np.isfinite(d_theta) & np.isfinite(sf_value)
    if finite.sum() < 8:
        continue

    d_theta = d_theta[finite]
    sf_value = sf_value[finite]
    edges = np.linspace(d_theta.min(), d_theta.max(), 9)
    edges = np.unique(edges)
    if edges.size < 3:
        continue

    bin_id = np.digitize(d_theta, edges[1:-1], right=False)
    for b in range(edges.size - 1):
        use = bin_id == b
        if use.sum() == 0:
            continue
        rows.append({
            'group': group_name,
            'normalization': normalization,
            'sep_deg_mid': float(np.nanmedian(d_theta[use])),
            'sep_deg_lo': float(edges[b]),
            'sep_deg_hi': float(edges[b + 1]),
            'count': int(use.sum()),
            'structure_function': float(np.nanmean(sf_value[use])),
            'rms_delta_log_amp_resid': float(np.sqrt(np.nanmean(sf_value[use]))),
        })

structure_function_df = pd.DataFrame(rows)
if structure_function_df.empty:
    raise RuntimeError('Structure-function table is empty; check filtered rows and group assignments.')

print('Structure-function summary by group:')
print(
    structure_function_df.groupby(['group', 'normalization'], as_index=False)
    .agg(
        n_bins=('count', 'size'),
        mean_count=('count', 'mean'),
        mean_structure_function=('structure_function', 'mean'),
        mean_rms_delta_log_amp_resid=('rms_delta_log_amp_resid', 'mean'),
    )
    .to_string(index=False, float_format=lambda v: f'{v:.6g}')
)

fig_structure = px.line(
    structure_function_df,
    x='sep_deg_mid',
    y='structure_function',
    color='group',
    markers=True,
    hover_data=['count', 'normalization', 'rms_delta_log_amp_resid'],
    title='Near/far grouped-coefficient residual structure function vs sky separation',
    labels={
        'sep_deg_mid': 'near-far angular separation [deg]',
        'structure_function': 'D(Δθ) = <(Δ log A_resid)^2>',
    },
)
fig_structure.update_layout(template='plotly_white', height=520)
fig_structure.show()

In [ ]:
# Shared ML utilities for coefficient prediction models
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

class RobustScaler:
    def fit(self, x):
        x = np.asarray(x, dtype=np.float32)
        self.med_ = np.nanmedian(x, axis=0)
        q25 = np.nanpercentile(x, 25, axis=0)
        q75 = np.nanpercentile(x, 75, axis=0)
        iqr = q75 - q25
        self.scale_ = np.where(iqr > 1e-8, iqr, 1.0).astype(np.float32)
        return self

    def transform(self, x):
        x = np.asarray(x, dtype=np.float32)
        return (x - self.med_) / self.scale_

    def inverse_transform(self, x):
        x = np.asarray(x, dtype=np.float32)
        return x * self.scale_ + self.med_

def _set_reproducibility(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def split_indices(n, train_frac=0.8, val_frac=0.1, seed=42):
    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    rng.shuffle(idx)
    n_train = int(train_frac * n)
    n_val = int(val_frac * n)
    train_idx = idx[:n_train]
    val_idx = idx[n_train:n_train + n_val]
    test_idx = idx[n_train + n_val:]
    return train_idx, val_idx, test_idx

def split_indices_by_night(obstime_mjd, train_frac=0.8, val_frac=0.1, seed=42):
    t = np.asarray(obstime_mjd, dtype=np.float64).reshape(-1)
    if t.size == 0:
        raise ValueError('obstime_mjd is empty')
    if not np.isfinite(t).all():
        raise ValueError('obstime_mjd contains non-finite values')

    night_id = np.floor(t - 0.5).astype(int)
    unique_nights = np.unique(night_id)
    rng = np.random.default_rng(seed)
    shuffled_nights = unique_nights.copy()
    rng.shuffle(shuffled_nights)

    n_nights = shuffled_nights.size
    n_train = int(train_frac * n_nights)
    n_val = int(val_frac * n_nights)
    train_nights = shuffled_nights[:n_train]
    val_nights = shuffled_nights[n_train:n_train + n_val]
    test_nights = shuffled_nights[n_train + n_val:]

    train_idx = np.flatnonzero(np.isin(night_id, train_nights))
    val_idx = np.flatnonzero(np.isin(night_id, val_nights))
    test_idx = np.flatnonzero(np.isin(night_id, test_nights))
    return train_idx, val_idx, test_idx

def _moon_phase_deg_from_ctx(filtered, arm='sci'):
    """Per-row moon phase in degrees [0, 360), reading whichever encoding is present."""
    names = [str(n) for n in filtered['ctx_names']]
    ctx = np.asarray(filtered[f'ctx_{arm}'], dtype=np.float64)
    if 'moon_phase' in names:
        return ctx[:, names.index('moon_phase')]
    s = names.index('moon_phase_sin')
    c = names.index('moon_phase_cos')
    return np.rad2deg(np.arctan2(ctx[:, s], ctx[:, c])) % 360.0


def split_indices_by_moon_phase(
    obstime_mjd,
    moon_phase,
    train_frac=0.8,
    val_frac=0.1,
    seed=42,
    n_bins=10,
):
    """Night-level split stratified by moon phase.

    Rows are grouped by night (floor(mjd - 0.5)). Each night is assigned a
    representative moon phase (median across its exposures). Nights are sorted
    by that phase and cut into ``n_bins`` equal-count quantiles. Within each
    quantile the nights are shuffled and split into train / val / test with
    the requested fractions rounded per-bin, so val and test each cover the
    full moon-phase range roughly uniformly. Whole nights stay together, so
    within-night airglow autocorrelation is not leaked across splits.
    """
    t = np.asarray(obstime_mjd, dtype=np.float64).reshape(-1)
    phase = np.asarray(moon_phase, dtype=np.float64).reshape(-1)
    if t.size == 0:
        raise ValueError('obstime_mjd is empty')
    if t.shape != phase.shape:
        raise ValueError('obstime_mjd and moon_phase must have the same shape')
    if not np.isfinite(t).all():
        raise ValueError('obstime_mjd contains non-finite values')
    if not np.isfinite(phase).all():
        raise ValueError('moon_phase contains non-finite values')

    night_id = np.floor(t - 0.5).astype(int)
    unique_nights = np.unique(night_id)
    n_nights = unique_nights.size

    # Per-night representative moon phase (median across the night's exposures).
    night_phase = np.empty(n_nights, dtype=np.float64)
    for k, nid in enumerate(unique_nights):
        night_phase[k] = np.median(phase[night_id == nid])

    rng = np.random.default_rng(seed)
    # Random tie-break so nights sharing a phase aren't ordered by their id.
    tie_break = rng.random(n_nights)
    order = np.lexsort((tie_break, night_phase))
    sorted_nights = unique_nights[order]

    n_bins_eff = max(1, min(int(n_bins), n_nights))
    bin_edges = np.linspace(0, n_nights, n_bins_eff + 1, dtype=int)
    test_frac = max(0.0, 1.0 - float(train_frac) - float(val_frac))

    train_list, val_list, test_list = [], [], []
    for k in range(n_bins_eff):
        lo, hi = int(bin_edges[k]), int(bin_edges[k + 1])
        block = sorted_nights[lo:hi].copy()
        b = block.size
        if b == 0:
            continue
        # Random assignment within the phase bin.
        perm = rng.permutation(b)
        block = block[perm]
        if b >= 3:
            n_val = max(1, int(round(float(val_frac) * b)))
            n_test = max(1, int(round(test_frac * b)))
            if n_val + n_test >= b:
                # Leave at least one train per bin.
                n_val = max(1, min(n_val, b - 2))
                n_test = max(1, min(n_test, b - 1 - n_val))
        elif b == 2:
            n_val, n_test = 1, 0
        else:
            n_val, n_test = 0, 0
        val_list.extend(block[:n_val].tolist())
        test_list.extend(block[n_val:n_val + n_test].tolist())
        train_list.extend(block[n_val + n_test:].tolist())

    train_nights = np.asarray(train_list, dtype=int)
    val_nights = np.asarray(val_list, dtype=int)
    test_nights = np.asarray(test_list, dtype=int)

    train_idx = np.flatnonzero(np.isin(night_id, train_nights))
    val_idx = np.flatnonzero(np.isin(night_id, val_nights))
    test_idx = np.flatnonzero(np.isin(night_id, test_nights))
    return train_idx, val_idx, test_idx


def _moon_bs_indices_from_names(coef_names_local):
    """Column indices of moon spline (moon_bs*) coefficients within a coef-name list."""
    return np.asarray([i for i, n in enumerate(coef_names_local)
                       if str(n).lower().startswith('moon_bs')], dtype=int)


def _row_spline_roughness(vals):
    """Root-mean-second-difference of a coefficient vector; a spline-noise diagnostic."""
    vals = np.asarray(vals, dtype=np.float64)
    if vals.size < 3:
        return np.nan
    d2 = vals[2:] - 2.0 * vals[1:-1] + vals[:-2]
    return float(np.sqrt(np.nanmean(d2 * d2)))


def _metric_row(y_true, y_pred, model_name, *,
                sigma=None, group_indices=None, floor_by_group=None):
    """Distributional summary of per-coefficient error (eRMSE), MAE, and Pearson r.

    Terminology (also used in §9 of the top notebook doc):
      * eRMSE  -- ensemble RMSE, i.e. per-coefficient RMSE aggregated
        over the ensemble of rows (axis=0).  ``mean_eRMSE`` /
        ``median_eRMSE`` are the mean / median across the per-coefficient
        RMSE vector.  Literature name: per-target RMSE.
      * sRMSE  -- spectral RMSE, per-row RMSE aggregated over the
        coefficient axis (axis=1).  Emitted per row by
        ``weighted_rmse_per_row``.  Literature name: per-sample RMSE.
      * pRMSE  -- pixel RMSE, per-row RMSE aggregated over the pixel
        axis of the reconstructed spectrum.  Emitted per row by
        ``pixel_wrmse_per_row``.  Distinct from sRMSE because it lives
        in flux space rather than coefficient space.
      * eWRMSE / sWRMSE / pWRMSE -- weighted counterparts (mirror the
        trainer's coef_err-weighted loss so the coefficient-space numbers
        stay comparable to the training objective; pWRMSE uses per-pixel
        sigma from the FLUX_SIGMA_TOTAL HDU or the on-the-fly propagator).

    When ``sigma`` (per-element decomposition COEF_ERR, same shape as
    ``y_true``) is provided along with ``group_indices`` and
    ``floor_by_group``, weighted counterparts are also reported:
    ``mean_eWRMSE`` / ``median_eWRMSE`` (per-column WRMSE with per-group
    sigma floor, aggregated), and ``total_eWRMSE`` (row + column pooled).
    """
    rmse = np.sqrt(np.mean((y_pred - y_true) ** 2, axis=0))
    mae = np.mean(np.abs(y_pred - y_true), axis=0)
    corr = []
    for j in range(y_true.shape[1]):
        x = y_true[:, j]
        y = y_pred[:, j]
        if np.std(x) < 1e-12 or np.std(y) < 1e-12:
            corr.append(np.nan)
        else:
            corr.append(float(np.corrcoef(x, y)[0, 1]))
    corr = np.asarray(corr)
    out = {
        'model': model_name,
        'mean_eRMSE': float(np.nanmean(rmse)),
        'median_eRMSE': float(np.nanmedian(rmse)),
        'mean_eMAE': float(np.nanmean(mae)),
        'mean_corr': float(np.nanmean(corr)),
        'median_corr': float(np.nanmedian(corr)),
    }
    if sigma is not None and group_indices is not None and floor_by_group is not None:
        wrmse_col, total_wrmse = _per_column_wrmse(
            y_true, y_pred, sigma, group_indices, floor_by_group,
        )
        out['mean_eWRMSE'] = float(np.nanmean(wrmse_col))
        out['median_eWRMSE'] = float(np.nanmedian(wrmse_col))
        out['total_eWRMSE'] = float(total_wrmse)
    return out


def _per_column_wrmse(y_true, y_pred, sigma, group_indices, floor_by_group):
    """Per-column WRMSE with per-group sigma floor + total WRMSE.

    Weights mirror the trainer: sigma is floored per group at
    ``floor_by_group[g] * median_col(sigma)``, per-column normalised to
    E[w]=1 so the mean/median WRMSE stay on the same scale as RMSE.
    Returns (per_col_wrmse, total_wrmse).
    """
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    sigma = np.asarray(sigma, dtype=np.float64)
    resid = y_pred - y_true
    finite = np.where(np.isfinite(sigma) & (sigma > 0.0), sigma, np.nan)
    median_col = np.nanmedian(finite, axis=0)
    median_col = np.where(np.isfinite(median_col) & (median_col > 0.0),
                          median_col, 1.0)
    floor_rel_col = np.full(sigma.shape[1], 0.05, dtype=np.float64)
    for g, idx in group_indices.items():
        idx = np.asarray(idx, dtype=int)
        if idx.size:
            floor_rel_col[idx] = float(floor_by_group.get(g, 0.05))
    floor_col = floor_rel_col * median_col
    sigma_eff = np.where(np.isfinite(sigma) & (sigma > 0.0),
                         sigma, floor_col[None, :])
    sigma_eff = np.maximum(sigma_eff, floor_col[None, :])
    w = 1.0 / (sigma_eff ** 2)
    wcm = np.mean(w, axis=0)
    wcm = np.where(wcm > 0.0, wcm, 1.0)
    w = w / wcm[None, :]
    f = np.isfinite(w) & np.isfinite(resid)
    num_col = np.sum(w * resid ** 2 * f, axis=0)
    den_col = np.sum(w * f, axis=0)
    with np.errstate(divide='ignore', invalid='ignore'):
        per_col = np.sqrt(np.where(den_col > 0.0, num_col / den_col, np.nan))
    tot_num = float(np.sum(w[f] * resid[f] ** 2))
    tot_den = float(np.sum(w[f]))
    total = float(np.sqrt(tot_num / max(tot_den, 1e-30)))
    return per_col, total


In [ ]:
# --- WRMSE per-row helper (used by R^2 -> WRMSE map/plot cells) --------
# Companion to cell 12's _per_column_wrmse: computes per-row weighted RMSE
# in coefficient space, using the trainer's per-group sigma floor and
# per-column weight normalisation.  Small stand-alone cell so downstream
# reporting/plotting cells never depend on the compressor/trainer being
# loaded first.
import numpy as np


def weighted_rmse_per_row(y_true, y_pred, sigma, group_indices, floor_by_group):
    """Return per-row WRMSE across all coefficients.

    ``sigma`` is native COEF_ERR (same shape as y_true/y_pred).
    Weights = 1 / max(sigma, floor_g * median_col(sigma))**2 with per-column
    mean normalisation over all rows (matches the aggregate reporting).
    Rows with no finite weighted residuals return NaN.
    """
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    sigma = np.asarray(sigma, dtype=np.float64)
    resid = y_pred - y_true
    finite = np.where(np.isfinite(sigma) & (sigma > 0.0), sigma, np.nan)
    median_col = np.nanmedian(finite, axis=0)
    median_col = np.where(np.isfinite(median_col) & (median_col > 0.0),
                          median_col, 1.0)
    floor_rel_col = np.full(sigma.shape[1], 0.05, dtype=np.float64)
    for g, idx in group_indices.items():
        idx = np.asarray(idx, dtype=int)
        if idx.size:
            floor_rel_col[idx] = float(floor_by_group.get(g, 0.05))
    floor_col = floor_rel_col * median_col
    sigma_eff = np.where(np.isfinite(sigma) & (sigma > 0.0),
                         sigma, floor_col[None, :])
    sigma_eff = np.maximum(sigma_eff, floor_col[None, :])
    w = 1.0 / (sigma_eff ** 2)
    wcm = np.mean(w, axis=0)
    wcm = np.where(wcm > 0.0, wcm, 1.0)
    w = w / wcm[None, :]
    f = np.isfinite(w) & np.isfinite(resid)
    num = np.sum(w * resid ** 2 * f, axis=1)
    den = np.sum(w * f, axis=1)
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.sqrt(np.where(den > 0.0, num / den, np.nan))


In [ ]:
# --- Pixel-space per-row WRMSE helpers ---------------------------------
# Companion to `weighted_rmse_per_row` (cell 13) but in flux/pixel space,
# used by every "spectrum-space RMSE" reporting site downstream (§9).
#
# Sigma sourcing (documented in §9 for future decompositions):
#   1. If the decomposition FITS carries a `FLUX_SIGMA_TOTAL` HDU (planned
#      output of the LSF-aware sigma propagator in `sky_decomp/fit.py`
#      -- see `SkyDecompBase._components_sigma_from_coef_err`), the loader
#      picks it up directly.  Shape must match FLUX_* (n_row, n_pix).
#   2. If only `COEF_ERR` is present (current schema), callers derive
#      sigma on-the-fly via `reconstruct_component_spectra(coef=..., coef_err=...)`
#      which returns the same LSF-propagated per-pixel sigma; both routes
#      produce identical arrays.
#   3. If neither is present, pass sigma=None and the WRMSE reduces to
#      plain per-pixel RMSE with a per-row median floor (no error).
import numpy as np


PIXEL_SIGMA_FLOOR_REL = 0.05
"""Relative floor on the per-pixel sigma used by `pixel_wrmse_per_row`.

Floor per row = `PIXEL_SIGMA_FLOOR_REL * median(|flux_true|)`; guards low-noise
regions where the propagator can return near-zero sigma from making WRMSE
diverge.  0.05 matches the moon/continuum/ionospheric floor used in
coefficient-space WRMSE (§7)."""


PIXEL_SIGMA_HDU_NAMES = ("FLUX_SIGMA_TOTAL", "FLUX_SIGMA")
"""HDU names the loader accepts, in preference order.

`FLUX_SIGMA_TOTAL` is the canonical name for the future LSF-aware sigma
propagator output; `FLUX_SIGMA` is a shorter alias for external files."""


def load_pixel_sigma_if_available(fits_path, row_indices=None):
    """Return per-pixel sigma array from *fits_path* or `None`.

    Returns None (does NOT raise) when neither HDU name in
    `PIXEL_SIGMA_HDU_NAMES` is present, so upstream reporting can degrade
    gracefully to plain RMSE.  When ``row_indices`` is given, only those rows
    are returned in that order.
    """
    from astropy.io import fits
    try:
        with fits.open(str(fits_path)) as hdul:
            _names = [str(getattr(h, "name", "")).upper() for h in hdul]
            for _wanted in PIXEL_SIGMA_HDU_NAMES:
                if _wanted in _names:
                    _data = np.asarray(hdul[_wanted].data, dtype=np.float64)
                    if row_indices is not None:
                        _data = _data[np.asarray(row_indices, dtype=int)]
                    return _data
    except FileNotFoundError:
        pass
    return None


def pixel_wrmse_per_row(flux_pred, flux_true, sigma_pix=None,
                        floor_rel=PIXEL_SIGMA_FLOOR_REL):
    """Per-row pixel-space WRMSE.

    Parameters
    ----------
    flux_pred, flux_true : (n_row, n_pix) or (n_pix,) arrays
        Predicted and observed spectra, same shape.  A 1-D input is
        broadcast to (1, n_pix).
    sigma_pix : (n_row, n_pix) array or None
        Per-pixel 1sigma.  None triggers floor-only weighting (uniform
        weights per row --> plain per-row RMSE with median-based floor).
    floor_rel : float
        Relative floor factor -- floor per row = ``floor_rel *
        median(|flux_true|)`` per row.

    Returns
    -------
    (n_row,) array of WRMSE values (NaN where a row has no usable pixels).
    """
    flux_pred = np.atleast_2d(np.asarray(flux_pred, dtype=np.float64))
    flux_true = np.atleast_2d(np.asarray(flux_true, dtype=np.float64))
    if flux_pred.shape != flux_true.shape:
        raise ValueError(f"shape mismatch: pred {flux_pred.shape} vs true {flux_true.shape}")
    resid = flux_pred - flux_true
    true_abs = np.abs(flux_true)
    median_row = np.nanmedian(true_abs, axis=1, keepdims=True)
    median_row = np.where(np.isfinite(median_row) & (median_row > 0.0),
                          median_row, 1.0)
    floor_row = float(floor_rel) * median_row
    if sigma_pix is None:
        sigma_eff = np.broadcast_to(floor_row, resid.shape).copy()
    else:
        s = np.atleast_2d(np.asarray(sigma_pix, dtype=np.float64))
        if s.shape != resid.shape:
            raise ValueError(f"sigma_pix shape {s.shape} != flux shape {resid.shape}")
        sigma_eff = np.where(np.isfinite(s) & (s > 0.0), s, floor_row)
        sigma_eff = np.maximum(sigma_eff, floor_row)
    w = 1.0 / (sigma_eff ** 2)
    # Row-wise normalisation: mean(w) = 1 per row so WRMSE stays on the
    # same scale as unweighted per-row RMSE.
    wrm = np.nanmean(w, axis=1, keepdims=True)
    wrm = np.where(wrm > 0.0, wrm, 1.0)
    w = w / wrm
    f = np.isfinite(w) & np.isfinite(resid)
    num = np.sum(w * resid ** 2 * f, axis=1)
    den = np.sum(w * f, axis=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.sqrt(np.where(den > 0.0, num / den, np.nan))


def pixel_wrmse_pointwise(flux_pred, flux_true, sigma_pix=None,
                          floor_rel=PIXEL_SIGMA_FLOOR_REL):
    """Per-pixel weighted squared-residual (before per-row reduction).

    Returns ``(w * resid**2)`` (n_row, n_pix), useful for the residual-band
    plots in the batch-RMSE cell.
    """
    flux_pred = np.atleast_2d(np.asarray(flux_pred, dtype=np.float64))
    flux_true = np.atleast_2d(np.asarray(flux_true, dtype=np.float64))
    resid = flux_pred - flux_true
    true_abs = np.abs(flux_true)
    median_row = np.nanmedian(true_abs, axis=1, keepdims=True)
    median_row = np.where(np.isfinite(median_row) & (median_row > 0.0),
                          median_row, 1.0)
    floor_row = float(floor_rel) * median_row
    if sigma_pix is None:
        sigma_eff = np.broadcast_to(floor_row, resid.shape).copy()
    else:
        s = np.atleast_2d(np.asarray(sigma_pix, dtype=np.float64))
        sigma_eff = np.where(np.isfinite(s) & (s > 0.0), s, floor_row)
        sigma_eff = np.maximum(sigma_eff, floor_row)
    w = 1.0 / (sigma_eff ** 2)
    wrm = np.nanmean(w, axis=1, keepdims=True)
    wrm = np.where(wrm > 0.0, wrm, 1.0)
    return w / wrm * resid ** 2


## Effective Dimensionality of the Coefficient Groups

Should the spline and line coefficients be projected onto a reduced basis before
the network sees them? This cell answers that empirically, per group, on
geometry-normalised amplitudes, and prints a recommendation.

Two ranks are reported because they answer different questions: how many
directions carry *structure* (Horn parallel analysis, the right rank for the
encoder input) and how many carry *transferable* structure (cross-arm score
correlation, the right rank for the target). Variance-fraction ranks are shown
only to demonstrate that they are misleading here.


In [ ]:
# Effective dimensionality of the coefficient groups.
#
# The question this answers: how many degrees of freedom do the coefficient
# groups actually carry, and how many of those are *transferable* from the sky
# arms to the science pointing? 402 OH line coefficients are not 402 degrees of
# freedom -- Meinel emission from one layer is set by the vibrational
# populations and a single rotational temperature, so the physical dimension is
# closer to ten.
#
# Three rank criteria are computed, because they answer different questions and
# disagree badly:
#
#   1. VARIANCE FRACTION (reported for reference only, and it is a trap). Past
#      the real structure, the leftover variance is noise spread evenly over
#      hundreds of directions, so "99.9% of variance" tries to capture the
#      noise and returns a rank in the hundreds.
#
#   2. HORN PARALLEL ANALYSIS. Shuffle each coefficient column independently to
#      destroy cross-coefficient correlation while preserving the marginals,
#      recompute the spectrum, and keep components above the shuffled null.
#      This is "how many directions carry structure rather than noise", and it
#      is the right rank for the ENCODER INPUT: denoise the sky arms without
#      discarding real signal.
#
#   3. CROSS-ARM SCORE CORRELATION. Project the near and far arms onto the same
#      basis and correlate their scores per component. A direction that is
#      uncorrelated between two simultaneous lines of sight cannot be predicted
#      at a third one, so including it in the target only adds unpredictable
#      variance. This is the right rank for the TARGET, and it is the number
#      that matters for this model.
#
# Everything runs on geometry-normalised coefficients (zenith-equivalent
# amplitudes), because PCA on raw coefficients would put airmass into PC1 and
# rediscover geometry that is already removed analytically. The basis is fit on
# TRAINING NIGHTS ONLY; cross-arm correlations are evaluated on held-out nights.

import matplotlib.pyplot as plt


def _pca_transform(coef, scale, kind='asinh'):
    """Map coefficients into the space the PCA is performed in.

    asinh is the default: it behaves like log for amplitudes well above the
    scale, which linearises the multiplicative structure (a change in rotational
    temperature is a single linear direction in log space but a curved manifold
    in linear space), while staying finite and monotonic through zero and
    negative values. sqrt is variance-stabilising and is the right space for the
    *loss*, but it is a worse space in which to look for structure.
    """
    coef = np.asarray(coef, dtype=np.float64)
    if kind == 'asinh':
        return np.arcsinh(coef / scale[None, :])
    if kind == 'log':
        return np.log(np.clip(coef / scale[None, :], 1e-3, None))
    if kind == 'sqrt':
        return np.sqrt(np.clip(coef, 0.0, None))
    if kind == 'linear':
        return coef.copy()
    raise ValueError(f'unknown transform {kind!r}')


def _spectrum(x_centered):
    """Eigenvalues of the sample covariance, descending."""
    n = max(x_centered.shape[0] - 1, 1)
    cov = (x_centered.T @ x_centered) / n
    eig = np.linalg.eigvalsh(cov)[::-1]
    return np.clip(eig, 0.0, None)


def _horn_threshold(x_centered, n_permutations, rng, percentile=95.0):
    """Null eigenvalue spectrum from independently shuffled columns."""
    null = []
    for _ in range(int(n_permutations)):
        shuffled = x_centered.copy()
        for j in range(shuffled.shape[1]):
            rng.shuffle(shuffled[:, j])
        null.append(_spectrum(shuffled))
    return np.percentile(np.vstack(null), percentile, axis=0)


def _leading_rank(values, threshold):
    """Number of leading components that stay above threshold."""
    above = np.asarray(values) > np.asarray(threshold)
    if not above.any():
        return 0
    if above.all():
        return int(above.size)
    return int(np.argmax(~above))


def pca_rank_diagnostic(
    filtered,
    group_indices,
    transform='asinh',
    min_group_size=3,
    max_rows_per_arm=4000,
    n_permutations=3,
    xarm_threshold=0.50,
    transform_sensitivity=True,
    sensitivity_rows=2000,
    seed=0,
    make_plot=True,
    verbose=True,
):
    """Per-group effective rank of the coefficient space, with a recommendation."""
    rng = np.random.default_rng(seed)
    coef_names = [str(n) for n in filtered['coef_names']]
    n_coef_total = len(coef_names)

    # --- geometry-normalised (zenith-equivalent) amplitudes -----------------
    geom_kwargs = dict(
        ctx_names=filtered['ctx_names'],
        group_indices=group_indices,
        n_coef=n_coef_total,
        coef_wavelengths_a=filtered.get('coef_wavelengths_a'),
        coef_extinction_k=filtered.get('coef_extinction_k'),
    )
    emissivity = {}
    for arm, ckey, xkey in (('near', 'coef_near', 'ctx_near'),
                            ('far', 'coef_far', 'ctx_far'),
                            ('sci', 'coef_sci', 'ctx_sci')):
        scale = airglow_geometry_scale(filtered[xkey], **geom_kwargs)
        emissivity[arm] = np.asarray(filtered[ckey], dtype=np.float64) / scale

    # --- split by night (phase-stratified when moon_phase is present);
    # basis fit on train, correlations on held-out --------------------
    n_rows = emissivity['near'].shape[0]
    if 'obstime_mjd' in filtered:
        _ctx_names_list = list(filtered.get('ctx_names', []))
        if 'moon_phase' in _ctx_names_list or 'moon_phase_sin' in _ctx_names_list:
            _moon_phase_rank = _moon_phase_deg_from_ctx(filtered)
            train_idx, val_idx, test_idx = split_indices_by_moon_phase(
                filtered['obstime_mjd'], _moon_phase_rank)
        else:
            train_idx, val_idx, test_idx = split_indices_by_night(filtered['obstime_mjd'])
        held_idx = np.concatenate([val_idx, test_idx])
    else:
        if verbose:
            print('WARNING: no obstime_mjd; falling back to a random split, which '
                  'leaks across nights and will overstate the transferable rank.')
        train_idx, val_idx, test_idx = split_indices(n_rows)
        held_idx = np.concatenate([val_idx, test_idx])
    if verbose:
        null_r = 1.0 / np.sqrt(max(min(held_idx.size, max_rows_per_arm), 2))
        print(f'Rows: {n_rows} total, {train_idx.size} train (basis), '
              f'{held_idx.size} held out (cross-arm correlations).')
        print(f'Correlation null level ~1/sqrt(n_held) = {null_r:.3f}; the '
              f'threshold below is {xarm_threshold / max(null_r, 1e-9):.1f} sigma.')
        print(f'Transform: {transform};  Horn permutations: {n_permutations};  '
              f'cross-arm threshold: {xarm_threshold:.2f}')

    def _subsample(idx, cap):
        if idx.size <= cap:
            return idx
        return rng.choice(idx, int(cap), replace=False)

    train_sub = _subsample(train_idx, max_rows_per_arm)
    held_sub = _subsample(held_idx, max_rows_per_arm)

    results = []
    for group_name, gidx in group_indices.items():
        gidx = np.asarray(gidx, dtype=int)
        n_g = int(gidx.size)
        if n_g == 0:
            continue
        if n_g < int(min_group_size):
            results.append({'group': group_name, 'n_coef': n_g, 'skipped': True})
            continue

        # per-coefficient robust scale from training rows
        block_tr = emissivity['near'][np.ix_(train_sub, gidx)]
        positive = np.where(block_tr > 0, block_tr, np.nan)
        scale = np.nanmedian(positive, axis=0)
        scale = np.where(np.isfinite(scale) & (scale > 0), scale, 1.0)

        z = {arm: _pca_transform(emissivity[arm][:, gidx], scale, transform)
             for arm in ('near', 'far', 'sci')}

        pooled_train = np.vstack([z[a][train_sub] for a in ('near', 'far', 'sci')])
        keep = np.isfinite(pooled_train).all(axis=1)
        pooled_train = pooled_train[keep]
        if pooled_train.shape[0] < max(20, n_g // 2):
            results.append({'group': group_name, 'n_coef': n_g, 'skipped': True})
            continue

        # Standardise columns (i.e. work from the correlation matrix). Without
        # this, Horn's null inherits the column variances, and those span a
        # huge range here -- a high-excitation OH line is far more sensitive to
        # rotational temperature than a low-excitation one -- so the shuffled
        # null acquires large leading eigenvalues and the method badly
        # under-detects. It is also the space the pipeline works in, since a
        # per-coefficient robust scaler is applied before the network.
        mean_vec = pooled_train.mean(axis=0)
        sd_vec = pooled_train.std(axis=0)
        sd_vec = np.where(sd_vec > 1e-12, sd_vec, 1.0)
        centered = (pooled_train - mean_vec) / sd_vec
        eig = _spectrum(centered)
        var_frac = eig / max(eig.sum(), 1e-300)
        cum = np.cumsum(var_frac)

        null_thresh = _horn_threshold(centered, n_permutations, rng)
        rank_horn = _leading_rank(eig, null_thresh)

        # basis vectors
        cov = (centered.T @ centered) / max(centered.shape[0] - 1, 1)
        evals, evecs = np.linalg.eigh(cov)
        order = np.argsort(evals)[::-1]
        basis = evecs[:, order]

        # cross-arm score correlation on held-out rows
        scores_near = ((z['near'][held_sub] - mean_vec) / sd_vec) @ basis
        scores_far = ((z['far'][held_sub] - mean_vec) / sd_vec) @ basis
        finite = np.isfinite(scores_near).all(axis=1) & np.isfinite(scores_far).all(axis=1)
        scores_near, scores_far = scores_near[finite], scores_far[finite]
        xarm = np.full(n_g, np.nan)
        if scores_near.shape[0] > 10:
            a = scores_near - scores_near.mean(axis=0)
            b = scores_far - scores_far.mean(axis=0)
            denom = np.sqrt((a * a).sum(axis=0) * (b * b).sum(axis=0))
            xarm = np.where(denom > 0, (a * b).sum(axis=0) / np.maximum(denom, 1e-300), 0.0)
        # COUNT, not a leading block. PCA orders components by variance, not by
        # transferability, so a shared mode can sit behind an arm-private one.
        # The transferable subspace is therefore selected by correlation, not by
        # variance rank -- which also means the target basis is a *chosen subset*
        # of components rather than the first k.
        xarm_keep = np.flatnonzero(xarm > xarm_threshold)
        rank_xarm = int(xarm_keep.size)
        rank_xarm50 = int(np.sum(xarm > 0.50))

        resid_frac = np.sqrt(np.clip(1.0 - np.concatenate([[0.0], cum]), 0.0, None))

        sensitivity = {}
        if transform_sensitivity:
            sens_rows = _subsample(train_sub, sensitivity_rows)
            for kind in ('asinh', 'log', 'sqrt', 'linear'):
                zz = np.vstack([_pca_transform(emissivity[a][np.ix_(sens_rows, gidx)],
                                               scale, kind)
                                for a in ('near', 'far', 'sci')])
                zz = zz[np.isfinite(zz).all(axis=1)]
                if zz.shape[0] < 20:
                    continue
                zsd = zz.std(axis=0)
                zc = (zz - zz.mean(axis=0)) / np.where(zsd > 1e-12, zsd, 1.0)
                sensitivity[kind] = _leading_rank(
                    _spectrum(zc), _horn_threshold(zc, 1, rng))

        results.append({
            'group': group_name, 'n_coef': n_g, 'skipped': False,
            'xarm_keep': xarm_keep,
            'eig': eig, 'var_frac': var_frac, 'cum': cum,
            'null_thresh': null_thresh, 'xarm': xarm, 'resid_frac': resid_frac,
            'rank_horn': rank_horn, 'rank_xarm': rank_xarm, 'rank_xarm50': rank_xarm50,
            'rank_var90': int(np.searchsorted(cum, 0.90)) + 1,
            'rank_var99': int(np.searchsorted(cum, 0.99)) + 1,
            'rank_var999': int(np.searchsorted(cum, 0.999)) + 1,
            'sensitivity': sensitivity,
        })

    def _fmt(v):
        return '-' if v is None or (isinstance(v, float) and not np.isfinite(v)) else int(v)

    table = pd.DataFrame([
        {'group': r['group'], 'n_coef': r['n_coef'],
         'var90': _fmt(r.get('rank_var90')), 'var99': _fmt(r.get('rank_var99')),
         'var99.9': _fmt(r.get('rank_var999')), 'horn': _fmt(r.get('rank_horn')),
         'transferable': _fmt(r.get('rank_xarm')), 'xarm>0.5': _fmt(r.get('rank_xarm50')),
         'input_rank': r['n_coef'] if r['skipped']
                       else max(r['rank_horn'], r['rank_xarm']),
         'target_rank': r['n_coef'] if r['skipped'] else r['rank_xarm'],
         'compressed': not r['skipped']}
        for r in results
    ])

    if verbose and len(table):
        print()
        print('Effective rank by group')
        print(table.to_string(index=False))
        for r in results:
            if r.get('sensitivity'):
                s = r['sensitivity']
                print(f"  {r['group']:<14s} Horn rank by transform: "
                      + '  '.join(f'{k}={v}' for k, v in s.items()))

    if make_plot:
        plotted = [r for r in results if not r['skipped']]
        if plotted:
            n = len(plotted)
            fig, axes = plt.subplots(n, 3, figsize=(13.5, 2.9 * n), squeeze=False)
            for row, r in enumerate(plotted):
                k = np.arange(1, r['n_coef'] + 1)
                show = min(r['n_coef'], max(3 * max(r['rank_horn'], 4), 30))

                ax = axes[row][0]
                ax.semilogy(k[:show], np.clip(r['eig'][:show], 1e-300, None),
                            'o-', ms=3, lw=1.2, color='#1f5fa8', label='data')
                ax.semilogy(k[:show], np.clip(r['null_thresh'][:show], 1e-300, None),
                            '--', lw=1.2, color='#c02c2c', label='shuffled null (95%)')
                ax.axvline(r['rank_horn'] + 0.5, color='#444', lw=1.0, ls=':')
                ax.set_title(f"{r['group']}  (n={r['n_coef']})  Horn rank = {r['rank_horn']}",
                             fontsize=10)
                ax.set_xlabel('component'); ax.set_ylabel('eigenvalue')
                ax.legend(fontsize=7, frameon=False)

                ax = axes[row][1]
                ax.plot(k[:show], r['xarm'][:show], '-', lw=1.0, color='#bbb', zorder=1)
                sel = r['xarm_keep'][r['xarm_keep'] < show]
                rej = np.setdiff1d(np.arange(show), sel)
                ax.scatter(rej + 1, r['xarm'][rej], s=14, color='#c9c9c9',
                           zorder=2, label='not transferable')
                ax.scatter(sel + 1, r['xarm'][sel], s=22, color='#0f8a63',
                           zorder=3, label='kept for target')
                ax.axhline(xarm_threshold, color='#c02c2c', ls='--', lw=1.0)
                ax.axhline(0.0, color='#999', lw=0.6)
                ax.set_ylim(-0.15, 1.05)
                ax.set_title(f"cross-arm correlation  ->  {r['rank_xarm']} transferable "
                             f"of {r['rank_horn']} real", fontsize=10)
                ax.legend(fontsize=7, frameon=False, loc='upper right')
                ax.set_xlabel('component'); ax.set_ylabel('corr(near, far)')

                ax = axes[row][2]
                rr = np.arange(r['resid_frac'].size)
                ax.semilogy(rr[:show + 1], np.clip(r['resid_frac'][:show + 1], 1e-6, None),
                            '-', lw=1.4, color='#4a3aa7')
                ax.axvline(r['rank_horn'], color='#c02c2c', lw=1.0, ls='--',
                           label=f"Horn = {r['rank_horn']}")
                ax.axvline(r['rank_xarm'], color='#0f8a63', lw=1.0, ls=':',
                           label=f"transferable = {r['rank_xarm']}")
                ax.set_title('reconstruction residual vs rank', fontsize=10)
                ax.set_xlabel('components kept')
                ax.set_ylabel('residual / total scatter')
                ax.legend(fontsize=7, frameon=False)
            fig.suptitle('Effective dimensionality of geometry-normalised coefficient groups',
                         fontsize=12, y=1.002)
            fig.tight_layout()
            plt.show()

    # ---------------------------- recommendation ---------------------------
    if verbose and len(table):
        input_dim = int(table['input_rank'].sum())
        target_dim = int(table['target_rank'].sum())
        n_ctx = len(filtered['ctx_names'])
        enc0 = 384
        head_dim = 192
        enc_before = (n_coef_total + n_ctx) * enc0
        enc_after = (input_dim + n_ctx) * enc0
        head_before = head_dim * n_coef_total
        head_after = head_dim * target_dim

        def _percoef_weight(sizes):
            return {g: (1.0 / np.sqrt(max(n, 1))) / max(n, 1) for g, n in sizes.items()}

        sizes_before = dict(zip(table['group'], table['n_coef']))
        sizes_after = dict(zip(table['group'], table['target_rank']))
        wb, wa = _percoef_weight(sizes_before), _percoef_weight(sizes_after)
        ratio_before = ratio_after = np.nan
        if 'continuum' in wb and 'mesospheric' in wb:
            ratio_before = wb['continuum'] / wb['mesospheric']
            ratio_after = wa['continuum'] / wa['mesospheric']

        print()
        print('=' * 78)
        print('RECOMMENDATION')
        print('=' * 78)
        print(f'  coefficient dimension   {n_coef_total}  ->  {input_dim} (encoder input), '
              f'{target_dim} (target)')
        print(f'  encoder first layer     {enc_before/1e3:.0f}k  ->  {enc_after/1e3:.0f}k params '
              f'({enc_before/max(enc_after,1):.1f}x)')
        print(f'  all group heads         {head_before/1e3:.0f}k  ->  {head_after/1e3:.0f}k params '
              f'({head_before/max(head_after,1):.1f}x)')
        if np.isfinite(ratio_before):
            print(f'  per-coefficient weight ratio continuum:mesospheric  '
                  f'{ratio_before:.0f}x  ->  {ratio_after:.0f}x')
        print()

        compression = n_coef_total / max(target_dim, 1)
        verdict = []
        if compression >= 3.0:
            verdict.append(
                f'PROCEED. {compression:.1f}x compression of the target space. Fit the basis '
                f'on training nights only, on geometry-normalised amplitudes in {transform!r} '
                f'space, per group, pooled over near/far/sci so all three share one basis.')
        else:
            verdict.append(
                f'DO NOT BOTHER. Only {compression:.1f}x compression; the coefficients are '
                f'already close to independent and PCA would buy little while costing '
                f'interpretability and the non-negativity constraint.')

        for r in results:
            if r['skipped']:
                continue
            if r['rank_xarm'] > r['rank_horn']:
                verdict.append(
                    f"{r['group']!r} has more transferable ({r['rank_xarm']}) than "
                    f"single-arm-detectable ({r['rank_horn']}) directions. That is expected "
                    f"where a mode is weak but shared: correlating two arms beats the noise "
                    f"floor of either one alone. Size the encoder input by the larger number.")
            if r['rank_horn'] >= 0.5 * r['n_coef']:
                verdict.append(
                    f"Leave {r['group']!r} uncompressed: Horn rank {r['rank_horn']} is over half "
                    f"of its {r['n_coef']} coefficients, so there is little redundancy.")
            elif r['rank_xarm'] <= 0.35 * r['rank_horn']:
                verdict.append(
                    f"{r['group']!r} carries {r['rank_horn']} real directions but only "
                    f"{r['rank_xarm']} are correlated between the two sky arms. The rest is "
                    f"structure the sky telescopes cannot transfer to the science pointing -- "
                    f"that gap is an information ceiling, not a modelling failure, and it "
                    f"should agree with the structure function for this group.")
            if r['sensitivity'] and 'asinh' in r['sensitivity'] and 'linear' in r['sensitivity']:
                if r['sensitivity']['linear'] > 1.5 * r['sensitivity']['asinh']:
                    verdict.append(
                        f"{r['group']!r} is markedly more compact under asinh/log than linear "
                        f"({r['sensitivity']['asinh']} vs {r['sensitivity']['linear']} "
                        f"directions), confirming the variability is multiplicative. Do the "
                        f"projection in that space even though the loss stays in sqrt space.")

        verdict.append(
            'Use the Horn rank for the encoder input and the cross-arm count for the target: '
            'a direction uncorrelated between two simultaneous lines of sight cannot be '
            'predicted at a third, so keeping it in the target only adds unpredictable '
            'variance. Select the target components BY CORRELATION, not by variance order -- '
            'PCA sorts by variance and a transferable mode can sit behind an arm-private one. '
            'The retained indices are in the returned results under "xarm_keep".')
        for r in results:
            if r['skipped'] or r['rank_xarm'] == 0:
                continue
            keep = r['xarm_keep']
            contiguous = np.array_equal(keep, np.arange(keep.size))
            if not contiguous:
                verdict.append(
                    f"{r['group']!r}: the transferable components are not the leading block "
                    f"(indices {[int(v) for v in keep[:12]]}"
                    f"{' ...' if keep.size > 12 else ''}), so "
                    f"truncating at the first k would keep unpredictable directions and "
                    f"discard predictable ones.")
        verdict.append(
            'Two consequences to decide deliberately: Softplus non-negativity is lost in score '
            'space (predict scores linearly and clip after reconstruction, or accept the small '
            'negatives the decomposition already contains), and PCA truncation on the moon '
            'spline is itself a smoothness prior, so moon_smooth_lambda becomes redundant.')
        verdict.append(
            'Reconstruction residual in this basis doubles as an out-of-distribution flag on '
            'nights whose conditions fall outside the training span.')

        for i, line in enumerate(verdict, 1):
            print(f'  {i}. ' + line.replace('\n', '\n     '))
        print('=' * 78)

    return table, results


pca_rank_table, pca_rank_results = pca_rank_diagnostic(
    filtered_triplet,
    _build_group_indices(filtered_triplet['coef_names']),
    transform='asinh',
    verbose=True,
    make_plot=True,
)


## Symmetric Dual-Encoder Group-Head Model

Key design choices in this implementation:
1. Shared near/far encoder weights for symmetry and sample efficiency.
2. Fusion vector uses:
   - $e_{mean} = 0.5(e_{near} + e_{far})$
   - $e_{diff} = e_{near} - e_{far}$
   - $|e_{diff}|$
3. Context branch encodes science-context only, now including folded time features and van Rhijn shell factors.
4. Group-specific heads map fused representation into layer-aware coefficient groups rather than a single flat atomic bucket.

**Note.** `DualEncoderGroupHeadMLP` below is the parent architecture. What
the notebook actually trains is `DualEncoderGroupHeadMLPCompressed` (defined
two cells down, together with the compressor fit), which reuses this
architecture verbatim on a per-group compressed coefficient space and emits
signed linear head outputs. See §5.5 in the methods cell above for the
compression pipeline and the empirical argument for switching to it.


In [ ]:
# Symmetric dual-encoder group-head MLP for same-time SCI coefficient prediction
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


class DualEncoderGroupHeadMLP(nn.Module):
    def __init__(self, n_coef, n_ctx, group_indices, ctx_names=None, encoder_dims=(384, 192), ctx_dims=(64,), trunk_dims=(256, 128), head_dim=128,
                 drop_vanrhijn_from_context=False):
        super().__init__()
        self.group_indices = group_indices
        self.ctx_names = [str(n).strip().lower() for n in ctx_names] if ctx_names is not None else None

        # Airglow geometry is divided out analytically before anything reaches
        # this model, so the vanrhijn_* columns are redundant FOR THE AIRGLOW
        # GROUPS.  They are not redundant for the moon and continuum groups,
        # which receive no analytic correction and must learn their airmass
        # dependence from context -- and the encoders are shared across groups,
        # so dropping the columns takes those smooth non-linear functions of
        # altitude away from every head.  Default is therefore to keep them.
        if self.ctx_names is not None and bool(drop_vanrhijn_from_context):
            self.ctx_keep_idx = [i for i, name in enumerate(self.ctx_names) if name not in VAN_RHIJN_FEATURES]
        elif self.ctx_names is not None:
            self.ctx_keep_idx = list(range(len(self.ctx_names)))
        else:
            self.ctx_keep_idx = list(range(int(n_ctx)))

        self.ctx_dim = len(self.ctx_keep_idx)
        self.sky_encoder = self._make_mlp(n_coef + self.ctx_dim, encoder_dims)
        sky_embed_dim = int(encoder_dims[-1])

        self.ctx_encoder = self._make_mlp(self.ctx_dim, ctx_dims)
        ctx_embed_dim = int(ctx_dims[-1]) if len(ctx_dims) > 0 else self.ctx_dim

        fusion_in = 3 * sky_embed_dim + ctx_embed_dim
        self.trunk = self._make_mlp(fusion_in, trunk_dims)
        trunk_out = int(trunk_dims[-1]) if len(trunk_dims) > 0 else fusion_in

        self.heads = nn.ModuleDict({
            g: nn.Sequential(
                nn.Linear(trunk_out, int(head_dim)),
                nn.GELU(),
                nn.Linear(int(head_dim), len(idx)),
            )
            for g, idx in group_indices.items()
        })

    @staticmethod
    def _make_mlp(in_dim, dims):
        layers = []
        d0 = int(in_dim)
        for d1 in dims:
            d1 = int(d1)
            layers += [nn.Linear(d0, d1), nn.LayerNorm(d1), nn.GELU()]
            d0 = d1
        return nn.Sequential(*layers) if layers else nn.Identity()

    def _select_ctx(self, ctx):
        if self.ctx_names is None or len(self.ctx_keep_idx) == ctx.shape[1]:
            return ctx
        if len(self.ctx_keep_idx) == 0:
            return ctx[:, :0]
        return ctx[:, self.ctx_keep_idx]

    def forward(self, near_coef, far_coef, near_ctx, far_ctx, sci_ctx):
        near_ctx_model = self._select_ctx(near_ctx)
        far_ctx_model = self._select_ctx(far_ctx)
        sci_ctx_model = self._select_ctx(sci_ctx)

        e_near = self.sky_encoder(torch.cat([near_coef, near_ctx_model], dim=1))
        e_far = self.sky_encoder(torch.cat([far_coef, far_ctx_model], dim=1))
        e_mean = 0.5 * (e_near + e_far)
        e_diff = e_near - e_far
        e_absdiff = torch.abs(e_diff)
        e_ctx = self.ctx_encoder(sci_ctx_model)

        z = torch.cat([e_mean, e_diff, e_absdiff, e_ctx], dim=1)
        h = self.trunk(z)
        # Outputs are intrinsic emissivities in model space; the caller
        # multiplies by airglow_geometry_scale(ctx_sci) to get observed
        # amplitudes at the science line of sight.
        return {g: F.softplus(head(h)) for g, head in self.heads.items()}


In [ ]:
# ============================================================================
# Per-group coefficient compression -- points 2, 3, 4 of the diagnostic.
#
#   2. Compress OH ("mesospheric") in ASINH space, moon in LINEAR space.
#      Atomic / continuum / ionospheric / other pass through as sqrt "identity
#      compressors" so the score-space pipeline is uniform even where nothing
#      is actually compressed.
#   3. Select KEPT COMPONENTS BY CROSS-ARM CORRELATION > 0.5 on held-out
#      nights.  This is NOT a variance-order truncation; the retained indices
#      are the target subspace flagged by Cell 13's `xarm_keep` at
#      xarm_threshold = 0.5.
#   4. The encoder input dimension shrinks by the same factor as the target,
#      because near/far/sci all pass through the same per-group basis.
#
# The compressor for each group is:
#     em (n_g)  -->  z = transform(em)          (asinh / linear / sqrt)
#     z         -->  z_std = (z - mean) / sd    (standardise columns)
#     z_std     -->  scores = z_std @ basis     (basis = PCA of correlation)
#     scores    -->  scores[:, kept]            (r_xarm > threshold on held-out)
# and the inverse puts zeros in dropped-component slots.  Non-negativity is
# recovered by clipping the emissivity after the inverse transform (also, the
# sqrt "identity" branch inverts via square which is non-negative by design).
# ============================================================================

COMPRESSION_XARM_THRESHOLD = 0.10  # 2026-08-19b: was 0.55.  The two PCs the 0.55
                                    # threshold dropped are PC4 (|xarm|=0.20, centroid 9644 A) and PC6
                                    # (|xarm|=0.53, centroid 8842 A) -- both red-heavy.  Dropping them
                                    # caused a +0.14 bias/0.43 rms wiggle in 9500-9800 A.  Their low
                                    # xarm reflects moon being SNR-limited in the extreme red, not
                                    # non-transferability.  Threshold 0.10 keeps all 29 moon PCs;
                                    # only the moon group has use_pca=True, so no other group changes.

COMPRESSION_TRANSFORM_BY_GROUP = {
    'mesospheric': 'asinh',
    # 2026-08-19b: moon 'asinh' -> 'sqrt'.  asinh compressed moon coefs so aggressively that the
    # round-trip lost precision at extreme-blue knots (where moon flux is large).  sqrt is a
    # milder tail-taming transform that still bounds the bright-moon-row loss share (was ~70%
    # of moon SS under 'linear') but preserves round-trip precision at the extreme blue and
    # red edges.  Encoder-free compressor round-trip rms in 3600-3800 A: linear 0.009, sqrt
    # 0.005, asinh 8.25 (diagnostic run 2026-08-19b).
    'moon': 'sqrt',
    'atomic': 'sqrt',
    'continuum': 'sqrt',
    'ionospheric': 'sqrt',
    'other': 'sqrt',
}

COMPRESSION_USE_PCA_BY_GROUP = {
    'mesospheric': False,  # 2026-08-11: asinh-identity beats PCA + xarm truncation on OH
    'moon': True,
    'atomic': False,
    'continuum': False,
    'ionospheric': False,
    'other': False,
}


def _score_forward(x, kind, per_col_scale):
    x = np.asarray(x, dtype=np.float64)
    if kind == 'linear':
        return x / per_col_scale[None, :]
    if kind == 'sqrt':
        return np.sqrt(np.clip(x, 0.0, None))
    if kind == 'asinh':
        return np.arcsinh(x / per_col_scale[None, :])
    if kind == 'log':
        return np.log(np.clip(x / per_col_scale[None, :], 1e-6, None))
    raise ValueError(f'unknown score transform {kind!r}')


def _score_inverse(y, kind, per_col_scale):
    y = np.asarray(y, dtype=np.float64)
    if kind == 'linear':
        return y * per_col_scale[None, :]
    if kind == 'sqrt':
        # square is non-negative by construction; sign is intentionally lost
        return np.square(np.clip(y, 0.0, None))
    if kind == 'asinh':
        return np.sinh(y) * per_col_scale[None, :]
    if kind == 'log':
        return np.exp(y) * per_col_scale[None, :]
    raise ValueError(f'unknown score transform {kind!r}')


def fit_group_compressor(em_near_g, em_far_g, em_sci_g,
                         train_idx, held_idx,
                         kind, use_pca, xarm_threshold):
    """Fit one group's compressor and return the matrices needed to apply it."""
    em_near_g = np.asarray(em_near_g, dtype=np.float64)
    em_far_g = np.asarray(em_far_g, dtype=np.float64)
    em_sci_g = np.asarray(em_sci_g, dtype=np.float64)
    n_g = em_near_g.shape[1]

    # per-column robust scale set from training rows on the near arm.
    pos = em_near_g[train_idx].copy()
    pos[pos <= 0] = np.nan
    per_col_scale = np.nanmedian(pos, axis=0)
    per_col_scale = np.where(np.isfinite(per_col_scale) & (per_col_scale > 0),
                             per_col_scale, 1.0)

    z_near = _score_forward(em_near_g, kind, per_col_scale)
    z_far = _score_forward(em_far_g, kind, per_col_scale)
    z_sci = _score_forward(em_sci_g, kind, per_col_scale)

    pooled_tr = np.vstack([z_near[train_idx], z_far[train_idx], z_sci[train_idx]])
    finite = np.isfinite(pooled_tr).all(axis=1)
    pooled_tr = pooled_tr[finite]
    if pooled_tr.shape[0] < max(20, n_g // 2):
        raise RuntimeError(
            'not enough finite training rows to standardise this group')
    mean_vec = pooled_tr.mean(axis=0)
    sd_vec = pooled_tr.std(axis=0)
    sd_vec = np.where(sd_vec > 1e-12, sd_vec, 1.0)
    # Per-column bounds of training data in forward-transformed space;
    # inverse_group_compressor clips to these for asinh/log to cap sinh/exp blowup.
    # Robust percentiles (not raw min/max) so a handful of anomalous decomposition
    # rows -- e.g. a bright OH band spiking to 1e12 on a bad fit -- do not push
    # the clip range so wide that sinh() blows up on typical predictions. Two
    # rows out of ~30k determined the previous max; the tail contraction here is
    # deliberate.
    _CLIP_LO_PCT, _CLIP_HI_PCT = 0.1, 99.9
    y_train_min = np.nanpercentile(pooled_tr, _CLIP_LO_PCT, axis=0)
    y_train_max = np.nanpercentile(pooled_tr, _CLIP_HI_PCT, axis=0)

    if not use_pca:
        # identity: keep every column, no rotation, no selection.
        return {
            'kind': kind,
            'use_pca': False,
            'per_col_scale': per_col_scale,
            'mean_vec': mean_vec,
            'sd_vec': sd_vec,
            'basis': np.eye(n_g),
            'kept': np.arange(n_g, dtype=int),
            'xarm_kept': np.full(n_g, np.nan),
            'n_coef_group': n_g,
            'y_train_min': y_train_min,
            'y_train_max': y_train_max,
        }

    centered = (pooled_tr - mean_vec) / sd_vec
    cov = (centered.T @ centered) / max(centered.shape[0] - 1, 1)
    evals, evecs = np.linalg.eigh(cov)
    order = np.argsort(evals)[::-1]
    basis = evecs[:, order]  # (n_g, n_g); columns are components, ordered by variance

    # score cross-arm correlation on held-out rows -- selects the KEPT set.
    scores_near_h = ((z_near[held_idx] - mean_vec) / sd_vec) @ basis
    scores_far_h = ((z_far[held_idx] - mean_vec) / sd_vec) @ basis
    finite_h = (np.isfinite(scores_near_h).all(axis=1)
                & np.isfinite(scores_far_h).all(axis=1))
    a = scores_near_h[finite_h] - scores_near_h[finite_h].mean(axis=0)
    b = scores_far_h[finite_h] - scores_far_h[finite_h].mean(axis=0)
    denom = np.sqrt((a * a).sum(axis=0) * (b * b).sum(axis=0))
    xarm = np.where(denom > 0, (a * b).sum(axis=0) / np.maximum(denom, 1e-300),
                    0.0)
    kept = np.flatnonzero(xarm > float(xarm_threshold))

    return {
        'kind': kind,
        'use_pca': True,
        'per_col_scale': per_col_scale,
        'mean_vec': mean_vec,
        'sd_vec': sd_vec,
        'basis': basis,
        'kept': kept,
        'xarm_kept': xarm[kept],
        'xarm_full': xarm,
        'n_coef_group': n_g,
        'y_train_min': y_train_min,
        'y_train_max': y_train_max,
    }


def apply_group_compressor(comp, em_g):
    """Transform emissivity to compressed scores (n_rows, len(kept))."""
    em_g = np.asarray(em_g, dtype=np.float64)
    z = _score_forward(em_g, comp['kind'], comp['per_col_scale'])
    centered = (z - comp['mean_vec']) / comp['sd_vec']
    scores_full = centered @ comp['basis']
    return scores_full[:, comp['kept']]


def inverse_group_compressor(comp, scores, jensen_correction=None):
    """Compressed scores -> emissivity (n_rows, n_g).

    `jensen_correction`, when supplied, is a per-coefficient array of
    multiplicative factors applied after the naive inverse.  Originally
    introduced to cancel the sinh-mean shrinkage of asinh compressors
    (E[sinh(y_true) | y_pred] ~= sinh(y_pred) * E[cosh(delta)]), it is now
    used more broadly as an empirical mean-bias null for any group whose
    predictions land on the SmoothL1 median rather than the mean.  Estimated
    on train+val rows inside `train_compressed_group_mlp` and stored in the
    training artifacts; clipped there to [0.5, 2.0].
    """
    scores = np.asarray(scores, dtype=np.float64)
    n_pc = comp['basis'].shape[1]
    full = np.zeros((scores.shape[0], n_pc), dtype=np.float64)
    full[:, comp['kept']] = scores
    centered = full @ comp['basis'].T
    z = centered * comp['sd_vec'] + comp['mean_vec']
    # asinh/log inverses (sinh/exp) are exponential; cap extrapolation
    # to the training y-range plus a small margin to avoid tail blowup.
    if comp['kind'] in ('asinh', 'log') and 'y_train_min' in comp:
        _margin = 0.5
        z = np.clip(z,
                    comp['y_train_min'] - _margin,
                    comp['y_train_max'] + _margin)
    x = _score_inverse(z, comp['kind'], comp['per_col_scale'])
    if jensen_correction is not None:
        jc = np.asarray(jensen_correction, dtype=np.float64)
        if jc.shape != (x.shape[1],):
            raise ValueError(
                f'jensen_correction shape {jc.shape} must match n_g={x.shape[1]}')
        x = x * jc[None, :]
    return x


def fit_all_group_compressors(filtered, group_indices, train_idx, held_idx,
                              transforms_by_group=None, use_pca_by_group=None,
                              xarm_threshold=0.5, verbose=True):
    if transforms_by_group is None:
        transforms_by_group = COMPRESSION_TRANSFORM_BY_GROUP
    if use_pca_by_group is None:
        use_pca_by_group = COMPRESSION_USE_PCA_BY_GROUP

    n_coef = filtered['coef_near'].shape[1]
    geom_kwargs = dict(
        ctx_names=filtered['ctx_names'],
        group_indices=group_indices,
        n_coef=n_coef,
        coef_wavelengths_a=filtered.get('coef_wavelengths_a'),
        coef_extinction_k=filtered.get('coef_extinction_k'),
    )
    sc_near = airglow_geometry_scale(filtered['ctx_near'], **geom_kwargs)
    sc_far = airglow_geometry_scale(filtered['ctx_far'], **geom_kwargs)
    sc_sci = airglow_geometry_scale(filtered['ctx_sci'], **geom_kwargs)
    em_near = np.asarray(filtered['coef_near'], dtype=np.float64) / sc_near
    em_far = np.asarray(filtered['coef_far'], dtype=np.float64) / sc_far
    em_sci = np.asarray(filtered['coef_sci'], dtype=np.float64) / sc_sci

    compressors = {}
    if verbose:
        print(f'Fitting per-group compressors on {train_idx.size} training rows, '
              f'{held_idx.size} held-out rows (xarm_threshold={xarm_threshold:.2f}).')
        print(f"  {'group':<14s} {'n_coef':>6s} {'transform':>9s} {'PCA':>4s} "
              f"{'n_score':>7s} {'median r':>8s} {'compression':>11s}")

    total_in, total_out = 0, 0
    for gname, gidx in group_indices.items():
        gidx = np.asarray(gidx, dtype=int)
        n_in = int(gidx.size)
        if n_in == 0:
            continue
        kind = transforms_by_group.get(gname, 'sqrt')
        use_pca = bool(use_pca_by_group.get(gname, False)) and n_in >= 3
        comp = fit_group_compressor(
            em_near[:, gidx], em_far[:, gidx], em_sci[:, gidx],
            train_idx=train_idx, held_idx=held_idx,
            kind=kind, use_pca=use_pca, xarm_threshold=xarm_threshold,
        )
        comp['coef_indices'] = gidx
        compressors[gname] = comp
        n_out = int(comp['kept'].size)
        total_in += n_in
        total_out += n_out
        if verbose:
            if comp['use_pca'] and n_out:
                r_med = f"{float(np.median(comp['xarm_kept'])):.3f}"
            else:
                r_med = ' n/a  '
            comp_ratio = f'{n_in/max(n_out,1):.1f}x'
            print(f"  {gname:<14s} {n_in:>6d} {kind:>9s} "
                  f"{('yes' if comp['use_pca'] else 'no'):>4s} "
                  f"{n_out:>7d} {r_med:>8s} {comp_ratio:>11s}")

    if verbose:
        print(f"  {'TOTAL':<14s} {total_in:>6d} {'':>9s} {'':>4s} "
              f"{total_out:>7d} {'':>8s} "
              f"{total_in/max(total_out,1):>10.1f}x")
    return compressors, geom_kwargs


def compress_coef_err_to_score_sigma(coef_phys, coef_err, ctx_phys, compressors,
                                     geom_kwargs, group_indices):
    """Propagate per-coefficient 1-sigma uncertainties into compressed score space.

    First-order (Jacobian) propagation through the same fixed pipeline that
    ``compress_coefs_to_scores`` implements:

        em       = coef / geom_scale(ctx)
        z        = f(em)                                 element-wise
        centered = (z - mean_vec) / sd_vec
        scores   = centered @ basis[:, kept]

    For independent per-coefficient variances only the DIAGONAL of the score
    covariance is needed, so the variance in score space reduces to

        Var(score_j) = sum_k (basis[k, j_kept] / sd_k)**2
                            * (df/dem_k)**2 * (1 / scale_k)**2
                            * sigma_coef_k**2

    NaN entries in ``coef_err`` (coefficients pinned at the c>=0 bound in the
    decomposition) contribute zero to the variance -- the boundary carries no
    symmetric error.  For ``sqrt``/``log`` transforms the derivative is
    undefined at em<=0 and is treated as zero for the same reason.
    """
    coef = np.asarray(coef_phys, dtype=np.float64)
    sig = np.nan_to_num(np.asarray(coef_err, dtype=np.float64), nan=0.0)
    sig = np.clip(sig, 0.0, np.inf)

    scale = airglow_geometry_scale(np.asarray(ctx_phys, dtype=np.float64),
                                   **geom_kwargs)
    em = coef / scale
    sig_em2 = (sig / scale) ** 2

    parts = []
    slices = {}
    offset = 0
    for gname, gidx in group_indices.items():
        gidx = np.asarray(gidx, dtype=int)
        if gname not in compressors or gidx.size == 0:
            continue
        comp = compressors[gname]
        pcs = np.asarray(comp['per_col_scale'], dtype=np.float64)
        sd_vec = np.asarray(comp['sd_vec'], dtype=np.float64)
        basis = np.asarray(comp['basis'], dtype=np.float64)
        kept = np.asarray(comp['kept'], dtype=int)

        em_g = em[:, gidx]
        sig_em2_g = sig_em2[:, gidx]

        kind = comp['kind']
        if kind == 'linear':
            dzdem = np.broadcast_to(1.0 / pcs[None, :], em_g.shape)
        elif kind == 'sqrt':
            dzdem = np.where(em_g > 0.0,
                             0.5 / np.sqrt(np.clip(em_g, 1e-30, None)),
                             0.0)
        elif kind == 'asinh':
            dzdem = 1.0 / (pcs[None, :]
                           * np.sqrt(1.0 + (em_g / pcs[None, :]) ** 2))
        elif kind == 'log':
            dzdem = np.where(em_g > 1e-6,
                             1.0 / (np.clip(em_g, 1e-6, None) * pcs[None, :]),
                             0.0)
        else:
            raise ValueError(f'unknown score transform {kind!r}')

        sig_z2 = (dzdem ** 2) * sig_em2_g
        sig_centered2 = sig_z2 / (sd_vec[None, :] ** 2)

        basis_kept = basis[:, kept]
        sig_scores2 = sig_centered2 @ (basis_kept ** 2)

        parts.append(np.sqrt(np.clip(sig_scores2, 0.0, np.inf)))
        slices[gname] = (offset, offset + parts[-1].shape[1])
        offset += parts[-1].shape[1]

    if not parts:
        return np.empty((coef.shape[0], 0), dtype=np.float64), slices
    return np.concatenate(parts, axis=1).astype(np.float64), slices


def compress_coefs_to_scores(coef_phys, ctx_phys, compressors, geom_kwargs,
                             group_indices):
    """Full pipeline: physical coefs -> concatenated per-group scores."""
    scale = airglow_geometry_scale(np.asarray(ctx_phys, dtype=np.float64),
                                   **geom_kwargs)
    em = np.asarray(coef_phys, dtype=np.float64) / scale
    parts = []
    slices = {}
    offset = 0
    for gname, gidx in group_indices.items():
        gidx = np.asarray(gidx, dtype=int)
        if gname not in compressors or gidx.size == 0:
            continue
        comp = compressors[gname]
        s = apply_group_compressor(comp, em[:, gidx])
        parts.append(s)
        slices[gname] = (offset, offset + s.shape[1])
        offset += s.shape[1]
    if not parts:
        return np.empty((coef_phys.shape[0], 0), dtype=np.float64), slices
    return np.concatenate(parts, axis=1), slices


def expand_scores_to_coefs(scores, ctx_phys, compressors, group_indices,
                           geom_kwargs, n_coef, score_slices,
                           jensen_corrections=None, coef_upper_bound=None):
    """Full pipeline: predicted scores -> physical coefs at ctx_phys geometry.

    `jensen_corrections`, when provided, is a {group_name: per-coefficient
    factor array} dict passed through to `inverse_group_compressor` for the
    asinh/log groups.

    `coef_upper_bound`, when provided, is a {group_name: per-coefficient
    upper-bound array} dict. Predictions above the bound are clipped to it
    (§11 item 11): defensive guard against asinh-inverse blowups when the
    trunk output lands outside the training envelope; does not touch training.
    Default trainer choice is 3 x max(coef_sci_train, axis=0) per group.
    """
    scale = airglow_geometry_scale(np.asarray(ctx_phys, dtype=np.float64),
                                   **geom_kwargs)
    em = np.zeros((scores.shape[0], n_coef), dtype=np.float64)
    for gname, gidx in group_indices.items():
        gidx = np.asarray(gidx, dtype=int)
        if gname not in compressors or gidx.size == 0:
            continue
        lo, hi = score_slices[gname]
        _jc = None if jensen_corrections is None else jensen_corrections.get(gname)
        em[:, gidx] = inverse_group_compressor(
            compressors[gname], scores[:, lo:hi], jensen_correction=_jc)
    coef = em * scale
    coef = np.clip(coef, 0.0, None)
    if isinstance(coef_upper_bound, dict):
        for gname, gidx in group_indices.items():
            _ub = coef_upper_bound.get(gname)
            if _ub is None:
                continue
            gidx = np.asarray(gidx, dtype=int)
            if gidx.size == 0:
                continue
            coef[:, gidx] = np.minimum(
                coef[:, gidx], np.asarray(_ub, dtype=np.float64)[None, :])
    return coef


# --- Fit compressors -------------------------------------------------------
_group_indices_compress = _build_group_indices(filtered_triplet['coef_names'])
_moon_phase_compress = _moon_phase_deg_from_ctx(filtered_triplet)
_split_tr, _split_va, _split_te = split_indices_by_moon_phase(
    filtered_triplet['obstime_mjd'], _moon_phase_compress, seed=42)
# Xarm component selection uses val only; val+test would let test-set
# structure shape which components survive the r_xarm > threshold cut.
_split_held = np.asarray(_split_va, dtype=int)

group_compressors, compress_geom_kwargs = fit_all_group_compressors(
    filtered_triplet, _group_indices_compress,
    train_idx=_split_tr, held_idx=_split_held,
    xarm_threshold=COMPRESSION_XARM_THRESHOLD,
    verbose=True,
)

filtered_triplet['compress_train_idx'] = _split_tr
filtered_triplet['compress_val_idx'] = _split_va
filtered_triplet['compress_test_idx'] = _split_te


In [ ]:
# ============================================================================
# Compressed dual-encoder group-head MLP.  THIS is the notebook's default model.
#
# See §5.5 of the methods cell for the compression rationale and the empirical
# argument for switching from the uncompressed baseline of §6 to this one.
# The uncompressed model is kept only as the parent class of
# `DualEncoderGroupHeadMLPCompressed`; its architecture, geometry handling,
# splitting and optimiser are inherited unchanged.
#
# The compressed model differs from the parent baseline in three ways:
#   * input to the encoder is per-arm compressed scores (n_input_score) rather
#     than sqrt-scaled coefficients (n_coef).
#   * group heads emit SIGNED score-space predictions (linear, no Softplus);
#     non-negativity is recovered by clipping the reconstructed physical
#     emissivity after inverse projection through the per-group compressor.
#   * loss is SmoothL1 on scaled score-space vectors, per group, with the
#     1/sqrt(n_g_score) balancing described in §7.  Because compressed groups
#     have comparable n_g_score, the per-coefficient weight imbalance between
#     continuum and OH of the uncompressed loss collapses by an order of
#     magnitude. Both `moon_group_weight` (default 3.0) and
#     `continuum_group_weight` (default 1.0) are exposed as per-group
#     multipliers $m_g$ on top of $1/\sqrt{n_g}$. `moon_smooth_lambda`
#     is retired (subsumed by the PCA truncation on the moon spline).
# ============================================================================


class DualEncoderGroupHeadMLPCompressed(DualEncoderGroupHeadMLP):
    """Subclass of the baseline arch with per-group heads emitting signed scores."""

    def __init__(self, *, n_score, n_ctx, group_score_dims, ctx_names=None,
                 encoder_dims=(384, 192), ctx_dims=(64,),
                 trunk_dims=(320, 160), head_dim=192,
                 drop_vanrhijn_from_context=False,
                 blend_init_alpha=0.7,
                 blend_use_direct=False,
                 moon_alt_conditional_alpha=False):
        fake_group_indices = {g: np.arange(int(n))
                              for g, n in group_score_dims.items()}
        super().__init__(
            n_coef=int(n_score),
            n_ctx=int(n_ctx),
            group_indices=fake_group_indices,
            ctx_names=ctx_names,
            encoder_dims=encoder_dims,
            ctx_dims=ctx_dims,
            trunk_dims=trunk_dims,
            head_dim=head_dim,
            drop_vanrhijn_from_context=drop_vanrhijn_from_context,
        )
        self.group_score_dims = dict(group_score_dims)
        # Cumulative offsets track how compress_coefs_to_scores concatenates the per-group score blocks.
        _offsets = np.cumsum([0] + [int(group_score_dims[g]) for g in group_score_dims])
        self._score_offsets = {g: (int(_offsets[i]), int(_offsets[i + 1]))
                               for i, g in enumerate(group_score_dims)}
        # Per-group learnable blend weight. Two parametrizations:
        #   blend_use_direct=False (default) -> alpha_g = sigmoid(logit_g).
        #   blend_use_direct=True            -> alpha_g stored directly, clamped in [eps, 1-eps] after each opt step.
        self.blend_use_direct = bool(blend_use_direct)
        self.blend_alpha_eps = 1e-3
        # Moon-alt-conditional alpha: 'moon' blend uses a per-row 2-value lookup
        # selected by the sign of moon_alt at the science pointing (2026-08-12).
        self.moon_alt_conditional_alpha = bool(moon_alt_conditional_alpha)
        _init_alpha = float(np.clip(blend_init_alpha, self.blend_alpha_eps, 1 - self.blend_alpha_eps))
        _init_logit = float(np.log(_init_alpha / (1.0 - _init_alpha)))
        _scalar_alpha_groups = [str(g) for g in group_score_dims
                                if not (self.moon_alt_conditional_alpha and g == 'moon')]
        if self.blend_use_direct:
            self.blend_alpha_direct = nn.ParameterDict({
                str(g): nn.Parameter(torch.tensor(_init_alpha))
                for g in _scalar_alpha_groups
            })
            if self.moon_alt_conditional_alpha and 'moon' in group_score_dims:
                # Index 0 = moon_alt <= 0 (moon down); 1 = moon_alt > 0 (moon up).
                self.blend_alpha_moon_by_alt = nn.Parameter(
                    torch.tensor([_init_alpha, _init_alpha]))
        else:
            self.blend_logit = nn.ParameterDict({
                str(g): nn.Parameter(torch.tensor(_init_logit))
                for g in _scalar_alpha_groups
            })
            if self.moon_alt_conditional_alpha and 'moon' in group_score_dims:
                self.blend_moon_by_alt_logit = nn.Parameter(
                    torch.tensor([_init_logit, _init_logit]))

    def forward(self, near_score, far_score, near_ctx, far_ctx, sci_ctx,
                sci_moon_alt_sign=None):
        near_ctx_m = self._select_ctx(near_ctx)
        far_ctx_m = self._select_ctx(far_ctx)
        sci_ctx_m = self._select_ctx(sci_ctx)
        e_near = self.sky_encoder(torch.cat([near_score, near_ctx_m], dim=1))
        e_far = self.sky_encoder(torch.cat([far_score, far_ctx_m], dim=1))
        e_mean = 0.5 * (e_near + e_far)
        e_diff = e_near - e_far
        e_absdiff = torch.abs(e_diff)
        e_ctx = self.ctx_encoder(sci_ctx_m)
        z = torch.cat([e_mean, e_diff, e_absdiff, e_ctx], dim=1)
        h = self.trunk(z)
        # Predict a residual on top of an explicit per-group blend of the two sky arms
        # so delta=0 is the interpolation baseline rather than the training mean.
        out = {}
        for g, head in self.heads.items():
            lo, hi = self._score_offsets[g]
            if (self.moon_alt_conditional_alpha and g == 'moon'
                    and sci_moon_alt_sign is not None):
                if self.blend_use_direct:
                    _alpha_row = self.blend_alpha_moon_by_alt[sci_moon_alt_sign]
                else:
                    _alpha_row = torch.sigmoid(
                        self.blend_moon_by_alt_logit[sci_moon_alt_sign])
                blend = (_alpha_row.unsqueeze(1) * near_score[:, lo:hi]
                         + (1.0 - _alpha_row.unsqueeze(1)) * far_score[:, lo:hi])
            else:
                if self.blend_use_direct:
                    alpha = self.blend_alpha_direct[str(g)]
                else:
                    alpha = torch.sigmoid(self.blend_logit[str(g)])
                blend = (alpha * near_score[:, lo:hi]
                         + (1.0 - alpha) * far_score[:, lo:hi])
            out[g] = blend + head(h)
        return out


DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP = {
    # Per-group floors on the relative sigma (fraction of the per-column
    # median finite sigma).  These were set on 2026-08-16 from the sigma /
    # weight diagnostic (§7): moon / continuum / ionospheric have
    # well-behaved tails and take the historical 5% floor; mesospheric
    # (403 lines) and atomic have p99 / p50 ratios of 10^3-10^6 in
    # compressed-score space, so they need a 20% floor to keep
    # w_p99 / w_median below ~30 per column.
    'moon': 0.05,
    'continuum': 0.05,
    'mesospheric': 0.20,
    'ionospheric': 0.05,
    'atomic': 0.20,
}


def train_compressed_group_mlp(
    filtered, compressors, group_indices, geom_kwargs,
    split_indices=None,
    n_epochs=50, batch_size=256, lr=7e-4,
    encoder_dims=(384, 192), ctx_dims=(64,),
    trunk_dims=(320, 160), head_dim=192,
    weight_decay=1e-4, grad_clip=1.0, patience=4,
    seed=42, drop_vanrhijn_from_context=False,
    moon_group_weight=3.5,
    continuum_group_weight=1.5,
    mesospheric_group_weight=3.0,
    ionospheric_group_weight=1.0,
    blend_init_alpha=0.7,
    blend_optim='direct',
    moon_alt_conditional_alpha=False,
    high_airmass_boost=1.0,
    # New (2026-08-16): heteroscedastic-Gaussian loss weighting from the
    # decomposition COEF_ERR (§7 of the top methods cell).  The floor is
    # now per group; the mesospheric and atomic groups need a larger floor
    # because their p99/p50 sigma ratio is 10^3-10^6 (§7 diagnostic).
    use_coef_err_weights=True,
    coef_err_sigma_floor_rel=None,
):
    _set_reproducibility(seed)

    ctx_near = np.asarray(filtered['ctx_near'], dtype=np.float32)
    ctx_far = np.asarray(filtered['ctx_far'], dtype=np.float32)
    ctx_sci = np.asarray(filtered['ctx_sci'], dtype=np.float32)

    if split_indices is not None:
        train_idx, val_idx, test_idx = split_indices
    else:
        _moon_phase_col = _moon_phase_deg_from_ctx(filtered)
        train_idx, val_idx, test_idx = split_indices_by_moon_phase(
            filtered['obstime_mjd'], _moon_phase_col, seed=seed)
    train_idx = np.asarray(train_idx, dtype=int)
    val_idx = np.asarray(val_idx, dtype=int)
    test_idx = np.asarray(test_idx, dtype=int)

    coef_near = np.asarray(filtered['coef_near'], dtype=np.float64)
    coef_far = np.asarray(filtered['coef_far'], dtype=np.float64)
    coef_sci = np.asarray(filtered['coef_sci'], dtype=np.float64)

    scores_near, slices_near = compress_coefs_to_scores(
        coef_near, ctx_near, compressors, geom_kwargs, group_indices)
    scores_far, slices_far = compress_coefs_to_scores(
        coef_far, ctx_far, compressors, geom_kwargs, group_indices)
    scores_sci, slices_sci = compress_coefs_to_scores(
        coef_sci, ctx_sci, compressors, geom_kwargs, group_indices)
    assert slices_near == slices_far == slices_sci, 'score slices disagree'
    score_slices = slices_near
    n_input_score = scores_near.shape[1]
    group_score_dims = {g: (hi - lo) for g, (lo, hi) in score_slices.items()}

    print(f'Compressed input dimension: {n_input_score} '
          f'(uncompressed was {coef_near.shape[1]}); per-group scores: '
          + ', '.join(f'{g}={n}' for g, n in group_score_dims.items()))

    # Per-group multiplier m_g on top of 1/sqrt(n_g_score); defaults to 1.0.
    _group_multipliers = {'moon': float(moon_group_weight),
                          'continuum': float(continuum_group_weight),
                          'mesospheric': float(mesospheric_group_weight),
                          'ionospheric': float(ionospheric_group_weight)}
    group_loss_weight = {
        g: (1.0 / max(float(np.sqrt(max(int(n), 1))), 1.0))
           * _group_multipliers.get(g, 1.0)
        for g, n in group_score_dims.items()
    }
    print(f'Loss weights (moon_group_weight={float(moon_group_weight):.2f}, '
          f'continuum_group_weight={float(continuum_group_weight):.2f}, '
          f'mesospheric_group_weight={float(mesospheric_group_weight):.2f}, '
          f'ionospheric_group_weight={float(ionospheric_group_weight):.2f}):')
    print(f"  {'group':<14s} {'n_score':>7s} {'m_g':>6s} {'w_g':>8s} {'w_g/n':>10s}")
    for _g, _n in group_score_dims.items():
        _m = _group_multipliers.get(_g, 1.0)
        _w = group_loss_weight[_g]
        print(f'  {_g:<14s} {int(_n):>7d} {_m:>6.2f} {_w:>8.4f} {(_w / max(int(_n), 1)):>10.5f}')

    score_scaler = RobustScaler().fit(np.vstack([
        scores_near[train_idx], scores_far[train_idx], scores_sci[train_idx],
    ]).astype(np.float32))
    ctx_scaler = RobustScaler().fit(np.vstack([
        ctx_near[train_idx], ctx_far[train_idx], ctx_sci[train_idx],
    ]))

    near_s = np.clip(score_scaler.transform(scores_near.astype(np.float32)),
                     -25.0, 25.0).astype(np.float32)
    far_s = np.clip(score_scaler.transform(scores_far.astype(np.float32)),
                    -25.0, 25.0).astype(np.float32)
    sci_s = np.clip(score_scaler.transform(scores_sci.astype(np.float32)),
                    -25.0, 25.0).astype(np.float32)

    # -----------------------------------------------------------------
    # Per-element inverse-variance weights from decomposition COEF_ERR.
    # Propagate through the compressor with a first-order Jacobian, then
    # divide out the score_scaler scale so the weights live in the same
    # scaled-score space as `sci_s`; finally clip to a per-column floor
    # (frac of median finite sigma) and normalise so E_train[w_pe] = 1
    # per column.  Missing/boundary sigmas fall through to the floor.
    # See §7 of the top methods cell for the derivation.
    coef_err_sci_local = np.asarray(
        filtered.get('coef_err_sci', np.full_like(coef_sci, np.nan)),
        dtype=np.float64,
    )
    _n_finite_err = int(np.sum(np.isfinite(coef_err_sci_local)))
    _n_total_err = int(coef_err_sci_local.size)
    _weights_enabled = bool(use_coef_err_weights) and _n_finite_err > 0
    if _weights_enabled:
        _sigma_scores_sci, _sigma_slices_sci = compress_coef_err_to_score_sigma(
            coef_sci, coef_err_sci_local, ctx_sci,
            compressors, geom_kwargs, group_indices,
        )
        assert _sigma_slices_sci == score_slices, 'sigma slices disagree with score slices'
        _scale_ = np.asarray(getattr(score_scaler, 'scale_',
                                    np.ones(n_input_score)),
                             dtype=np.float64)
        _scale_ = np.where(_scale_ > 0.0, _scale_, 1.0)
        _sigma_scaled = _sigma_scores_sci / _scale_[None, :]
        _finite = np.where(np.isfinite(_sigma_scaled) & (_sigma_scaled > 0.0),
                           _sigma_scaled, np.nan)
        _floor_col = np.nanmedian(_finite, axis=0)
        _floor_col = np.where(np.isfinite(_floor_col) & (_floor_col > 0.0),
                              _floor_col, 1.0)
        # Resolve per-group floors: accept scalar (broadcast), dict
        # (per-group), or None (built-in DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP).
        _floor_arg = coef_err_sigma_floor_rel
        if _floor_arg is None:
            _floor_arg = DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP
        if isinstance(_floor_arg, dict):
            _floor_by_group = {str(k): float(v) for k, v in _floor_arg.items()}
        else:
            _floor_by_group = {g: float(_floor_arg) for g in score_slices}
        _floor_rel_col = np.ones(_floor_col.shape[0], dtype=np.float64)
        _missing_groups = []
        for _g, (_lo, _hi) in score_slices.items():
            if _g not in _floor_by_group:
                _missing_groups.append(_g)
                _fr = 0.05
            else:
                _fr = _floor_by_group[_g]
            _floor_rel_col[_lo:_hi] = _fr
        if _missing_groups:
            print(f'  warning: coef_err_sigma_floor_rel missing groups '
                  f'{_missing_groups}; using fallback 0.05 for each.')
        _floor_col = _floor_rel_col * _floor_col
        _sigma_scaled = np.where(_sigma_scaled > 0.0, _sigma_scaled,
                                 _floor_col[None, :])
        _sigma_scaled = np.maximum(_sigma_scaled, _floor_col[None, :])
        w_pe_np = 1.0 / (_sigma_scaled ** 2)
        _wcm = np.mean(w_pe_np[train_idx], axis=0)
        _wcm = np.where(_wcm > 0.0, _wcm, 1.0)
        w_pe_np = (w_pe_np / _wcm[None, :]).astype(np.float32)
        _floor_report = ', '.join(f'{g}={_floor_by_group.get(g, 0.05):.3g}'
                                  for g in score_slices)
        print(
            f'Per-element loss weights from coef_err_sci: '
            f'finite fraction={_n_finite_err / max(_n_total_err, 1):.3f} in native space, '
            f'compressed w median={float(np.median(w_pe_np[train_idx])):.3f}, '
            f'1-99% = [{float(np.percentile(w_pe_np[train_idx], 1)):.2f}, '
            f'{float(np.percentile(w_pe_np[train_idx], 99)):.2f}]'
        )
        print(f'  per-group floor_rel: {_floor_report}')
        # Save the resolved dict so the artifact/config can round-trip it.
        _resolved_floor_by_group = {g: float(_floor_by_group.get(g, 0.05))
                                    for g in score_slices}
    else:
        # Uniform per-element weights: reduces to the previous unweighted loss.
        w_pe_np = np.ones_like(sci_s, dtype=np.float32)
        print('Per-element loss weights: disabled '
              '(use_coef_err_weights=False or no finite COEF_ERR); '
              'compressed_loss falls back to the unweighted SmoothL1.')

    ctx_near_n = np.clip(ctx_scaler.transform(ctx_near), -25.0, 25.0).astype(np.float32)
    ctx_far_n = np.clip(ctx_scaler.transform(ctx_far), -25.0, 25.0).astype(np.float32)
    ctx_sci_n = np.clip(ctx_scaler.transform(ctx_sci), -25.0, 25.0).astype(np.float32)

    # Row weights: up-weight high-airmass rows so training sees them proportional
    # to deployment relevance rather than raw frequency. Normalized so the mean
    # training weight = 1. See §12 (2026-08-12) for the ablation that rejected
    # bright-moon and high-F10.7 boosts.
    _ctx_names_local = list(filtered['ctx_names'])
    _row_weights_np = np.ones(ctx_sci.shape[0], dtype=np.float32)
    if 'airmass' in _ctx_names_local and float(high_airmass_boost) > 1.0:
        _ai = _ctx_names_local.index('airmass')
        _mask = ctx_sci[:, _ai] > 1.5
        _b = float(high_airmass_boost)
        _row_weights_np = np.where(_mask, _b, 1.0).astype(np.float32)
        _train_mean_w = float(np.mean(_row_weights_np[train_idx]))
        if _train_mean_w > 1e-9:
            _row_weights_np = (_row_weights_np / _train_mean_w).astype(np.float32)
        print(f'Row weights: airmass>1.5 x{_b:.2f}: n_all={int(_mask.sum())} '
              f'| train mean pre-norm = {_train_mean_w:.3f}')

    # Moon_alt sign per row (physical ctx_sci). 0 = moon down (alt<=0), 1 = up.
    _moon_alt_idx_local = _ctx_names_local.index('moon_alt')
    _moon_alt_sign_all = (ctx_sci[:, _moon_alt_idx_local] > 0.0).astype(np.int64)

    tr_loader = DataLoader(TensorDataset(
        torch.from_numpy(near_s[train_idx]),
        torch.from_numpy(far_s[train_idx]),
        torch.from_numpy(ctx_near_n[train_idx]),
        torch.from_numpy(ctx_far_n[train_idx]),
        torch.from_numpy(ctx_sci_n[train_idx]),
        torch.from_numpy(_moon_alt_sign_all[train_idx]),
        torch.from_numpy(_row_weights_np[train_idx]),
        torch.from_numpy(w_pe_np[train_idx]),
        torch.from_numpy(sci_s[train_idx]),
    ), batch_size=int(batch_size), shuffle=True)
    va_loader = DataLoader(TensorDataset(
        torch.from_numpy(near_s[val_idx]),
        torch.from_numpy(far_s[val_idx]),
        torch.from_numpy(ctx_near_n[val_idx]),
        torch.from_numpy(ctx_far_n[val_idx]),
        torch.from_numpy(ctx_sci_n[val_idx]),
        torch.from_numpy(_moon_alt_sign_all[val_idx]),
        torch.from_numpy(_row_weights_np[val_idx]),
        torch.from_numpy(w_pe_np[val_idx]),
        torch.from_numpy(sci_s[val_idx]),
    ), batch_size=512, shuffle=False)

    if torch.cuda.is_available():
        device = 'cuda'
    elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
        device = 'mps'
    else:
        device = 'cpu'

    model = DualEncoderGroupHeadMLPCompressed(
        n_score=n_input_score, n_ctx=ctx_near_n.shape[1],
        group_score_dims=group_score_dims,
        ctx_names=[str(x) for x in filtered['ctx_names']],
        encoder_dims=tuple(int(v) for v in encoder_dims),
        ctx_dims=tuple(int(v) for v in ctx_dims),
        trunk_dims=tuple(int(v) for v in trunk_dims),
        head_dim=int(head_dim),
        drop_vanrhijn_from_context=bool(drop_vanrhijn_from_context),
        blend_init_alpha=float(blend_init_alpha),
        blend_use_direct=(str(blend_optim).lower()
                          in ('direct', 'prefit_freeze', 'prefit_warmstart')),
        moon_alt_conditional_alpha=bool(moon_alt_conditional_alpha),
    ).to(device)

    # Optimizer construction depends on blend_optim mode:
    #   'default' -> single AdamW param group.
    #   'no_wd'   -> blend params in a group with weight_decay=0.
    #   'lr10x'   -> blend params in a group with lr=10*base_lr.
    #   'direct'  -> alpha_g stored directly (no sigmoid); blend group has weight_decay=0
    #                and is clamped to [eps, 1-eps] after each step.
    _blend_mode = str(blend_optim).lower()
    _ALLOWED_BLEND_MODES = ('default', 'no_wd', 'lr10x', 'direct',
                            'prefit_freeze', 'prefit_warmstart')
    if _blend_mode not in _ALLOWED_BLEND_MODES:
        raise ValueError(f"Unknown blend_optim={blend_optim!r}; "
                         f"expected one of {_ALLOWED_BLEND_MODES}.")
    # Pre-fit alpha_g in scaled score space on training rows. Closed-form OLS:
    #   alpha_g = <s_near - s_far, s_sci - s_far> / ||s_near - s_far||^2
    # summed across all coefficients in group g. 'prefit_freeze' locks the
    # value; 'prefit_warmstart' uses it as an init and lets AdamW adjust.
    _prefit_alphas = None
    if _blend_mode in ('prefit_freeze', 'prefit_warmstart'):
        _prefit_alphas = {}
        for _g_pf, (_lo_pf, _hi_pf) in score_slices.items():
            if model.moon_alt_conditional_alpha and _g_pf == 'moon':
                continue  # conditional moon alpha does not use the closed-form prefit.
            _sn = near_s[train_idx, _lo_pf:_hi_pf].astype(np.float64)
            _sf = far_s[train_idx, _lo_pf:_hi_pf].astype(np.float64)
            _ss = sci_s[train_idx, _lo_pf:_hi_pf].astype(np.float64)
            _diff = _sn - _sf
            _target = _ss - _sf
            _num = float(np.sum(_diff * _target))
            _den = float(np.sum(_diff * _diff))
            if _den <= 1e-12:
                # near ≈ far for this group; blend is ill-defined -> midpoint.
                _alpha_star = 0.5
            else:
                _alpha_star = float(np.clip(_num / _den,
                                            float(model.blend_alpha_eps),
                                            1.0 - float(model.blend_alpha_eps)))
            _prefit_alphas[_g_pf] = _alpha_star
            with torch.no_grad():
                model.blend_alpha_direct[str(_g_pf)].data.fill_(float(_alpha_star))
        if _blend_mode == 'prefit_freeze':
            for _g_pf in group_score_dims:
                model.blend_alpha_direct[str(_g_pf)].requires_grad_(False)
        print(f'Pre-fit alpha per group ({_blend_mode}): '
              + '  '.join(f'{_g_pf}={_prefit_alphas[_g_pf]:.3f}'
                          for _g_pf in group_score_dims))

    blend_pnames = set()
    blend_params = []
    if model.blend_use_direct:
        for _k, _p in model.blend_alpha_direct.items():
            blend_params.append(_p)
            blend_pnames.add(f'blend_alpha_direct.{_k}')
        if model.moon_alt_conditional_alpha and hasattr(model, 'blend_alpha_moon_by_alt'):
            blend_params.append(model.blend_alpha_moon_by_alt)
            blend_pnames.add('blend_alpha_moon_by_alt')
    else:
        for _k, _p in model.blend_logit.items():
            blend_params.append(_p)
            blend_pnames.add(f'blend_logit.{_k}')
        if model.moon_alt_conditional_alpha and hasattr(model, 'blend_moon_by_alt_logit'):
            blend_params.append(model.blend_moon_by_alt_logit)
            blend_pnames.add('blend_moon_by_alt_logit')
    other_params = [p for n, p in model.named_parameters() if n not in blend_pnames]
    if _blend_mode == 'default':
        opt = torch.optim.AdamW(model.parameters(), lr=float(lr),
                                weight_decay=float(weight_decay))
    elif _blend_mode == 'no_wd':
        opt = torch.optim.AdamW(
            [{'params': other_params, 'weight_decay': float(weight_decay)},
             {'params': blend_params, 'weight_decay': 0.0}],
            lr=float(lr))
    elif _blend_mode == 'lr10x':
        opt = torch.optim.AdamW(
            [{'params': other_params, 'lr': float(lr)},
             {'params': blend_params, 'lr': float(lr) * 10.0}],
            weight_decay=float(weight_decay))
    elif _blend_mode == 'prefit_freeze':
        # blend params carry the OLS alpha and are frozen (requires_grad=False).
        opt = torch.optim.AdamW(other_params, lr=float(lr),
                                weight_decay=float(weight_decay))
    else:  # 'direct' or 'prefit_warmstart'
        opt = torch.optim.AdamW(
            [{'params': other_params, 'weight_decay': float(weight_decay)},
             {'params': blend_params, 'weight_decay': 0.0}],
            lr=float(lr))
    print(f"Blend optim mode: {_blend_mode} "
          f"(use_direct={bool(model.blend_use_direct)}, "
          f"n_blend_params={sum(p.numel() for p in blend_params)})")

    def compressed_loss(pred_dict, yb, w_row, w_pe):
        loss = torch.tensor(0.0, device=yb.device)
        for g, y_head in pred_dict.items():
            lo, hi = score_slices[g]
            target = yb[:, lo:hi].contiguous()
            w_pe_g = w_pe[:, lo:hi].contiguous()
            _per_elem = F.smooth_l1_loss(y_head, target, reduction='none') * w_pe_g
            _per_row = _per_elem.mean(dim=1)
            _weighted = (w_row * _per_row).mean()
            _w_g = group_loss_weight[g]
            loss = loss + float(_w_g) * _weighted
        return loss / max(len(pred_dict), 1)

    def _snapshot_alpha():
        result = {}
        for g in group_score_dims:
            if model.moon_alt_conditional_alpha and g == 'moon':
                if model.blend_use_direct:
                    _a = model.blend_alpha_moon_by_alt.detach().cpu().numpy()
                else:
                    _a = torch.sigmoid(model.blend_moon_by_alt_logit).detach().cpu().numpy()
                result['moon_dn'] = float(_a[0])
                result['moon_up'] = float(_a[1])
            elif model.blend_use_direct:
                result[g] = float(model.blend_alpha_direct[str(g)].item())
            else:
                result[g] = float(torch.sigmoid(model.blend_logit[str(g)]).item())
        return result
    history = []
    blend_history = []  # per-epoch snapshot of learned alpha_g
    _init_blend = _snapshot_alpha()
    blend_history.append({'epoch': 0, **_init_blend})
    best_val = np.inf
    best_epoch = -1
    best_state = None
    stale = 0
    for ep in range(1, int(n_epochs) + 1):
        model.train()
        tr_loss = 0.0
        tr_n = 0
        for near_b, far_b, near_ctx_b, far_ctx_b, ctx_b, ma_sign_b, w_row_b, w_pe_b, yb in tr_loader:
            near_b = near_b.to(device); far_b = far_b.to(device)
            near_ctx_b = near_ctx_b.to(device); far_ctx_b = far_ctx_b.to(device)
            ctx_b = ctx_b.to(device); ma_sign_b = ma_sign_b.to(device)
            w_row_b = w_row_b.to(device); w_pe_b = w_pe_b.to(device); yb = yb.to(device)
            pred = model(near_b, far_b, near_ctx_b, far_ctx_b, ctx_b,
                         sci_moon_alt_sign=(ma_sign_b
                                            if model.moon_alt_conditional_alpha else None))
            loss = compressed_loss(pred, yb, w_row_b, w_pe_b)
            if not torch.isfinite(loss):
                continue
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), float(grad_clip))
            opt.step()
            if model.blend_use_direct:
                with torch.no_grad():
                    _eps = float(model.blend_alpha_eps)
                    for _g in group_score_dims:
                        if model.moon_alt_conditional_alpha and _g == 'moon':
                            continue
                        model.blend_alpha_direct[str(_g)].clamp_(_eps, 1.0 - _eps)
                    if model.moon_alt_conditional_alpha and hasattr(model, 'blend_alpha_moon_by_alt'):
                        model.blend_alpha_moon_by_alt.clamp_(_eps, 1.0 - _eps)
            tr_loss += float(loss.item())
            tr_n += 1

        model.eval()
        va_loss = 0.0
        va_n = 0
        with torch.no_grad():
            for near_b, far_b, near_ctx_b, far_ctx_b, ctx_b, ma_sign_b, w_row_b, w_pe_b, yb in va_loader:
                near_b = near_b.to(device); far_b = far_b.to(device)
                near_ctx_b = near_ctx_b.to(device); far_ctx_b = far_ctx_b.to(device)
                ctx_b = ctx_b.to(device); ma_sign_b = ma_sign_b.to(device)
                w_row_b = w_row_b.to(device); w_pe_b = w_pe_b.to(device); yb = yb.to(device)
                pred = model(near_b, far_b, near_ctx_b, far_ctx_b, ctx_b,
                             sci_moon_alt_sign=(ma_sign_b
                                                if model.moon_alt_conditional_alpha else None))
                loss = compressed_loss(pred, yb, w_row_b, w_pe_b)
                if torch.isfinite(loss):
                    va_loss += float(loss.item())
                    va_n += 1

        tr_mean = tr_loss / max(tr_n, 1)
        va_mean = va_loss / max(va_n, 1)
        history.append({'epoch': ep, 'train_loss': tr_mean, 'val_loss': va_mean})
        blend_history.append({'epoch': ep, **_snapshot_alpha()})
        if va_mean < best_val:
            best_val = va_mean
            best_epoch = ep
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
        if ep == 1 or ep % 5 == 0 or ep == int(n_epochs):
            print(f'[compressed] epoch={ep:03d} train={tr_mean:.6f} val={va_mean:.6f}')
        if stale >= int(patience):
            print(f'Early stopping at epoch {ep} (patience={patience})')
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    _final_blend = _snapshot_alpha()
    _parts = []
    for _g in group_score_dims:
        if model.moon_alt_conditional_alpha and _g == 'moon':
            _parts.append(f'moon_dn={_final_blend["moon_dn"]:.3f}')
            _parts.append(f'moon_up={_final_blend["moon_up"]:.3f}')
        else:
            _parts.append(f'{_g}={_final_blend[_g]:.3f}')
    print(f"Learned per-group near-arm blend alpha at best epoch [{_blend_mode}]: "
          + '  '.join(_parts))
    if _init_blend is not None:
        init_s = float(blend_init_alpha)
        _deltas = []
        for _g in group_score_dims:
            if model.moon_alt_conditional_alpha and _g == 'moon':
                _deltas.append(f'moon_dn={_final_blend["moon_dn"] - _init_blend["moon_dn"]:+.3f}')
                _deltas.append(f'moon_up={_final_blend["moon_up"] - _init_blend["moon_up"]:+.3f}')
            else:
                _deltas.append(f'{_g}={_final_blend[_g] - _init_blend[_g]:+.3f}')
        print(f'  init alpha={init_s:.3f} (uniform)  |  delta alpha at best epoch: '
              + '  '.join(_deltas))

    # --- Fit per-group empirical mean-bias calibration on train + val rows ---
    # Since 2026-08-16: fit on train+val (test excluded), applied to ALL groups.
    # Previously fit on val only and gated to asinh/log groups -- that left
    # linear-compressor groups (continuum, ionospheric, atomic) with the raw
    # SmoothL1 median-bias, producing a -1.77% systematic mean bias on continuum
    # in the group-bias diagnostic.  The lift is a scalar mean_true/mean_pred
    # ratio: valid for any group with a coherent non-zero mean, so the asinh/log
    # gate was unnecessarily restrictive.  Train+val gives ~85% of rows for
    # groups as small as n=3 (continuum, atomic) where val-only sampling noise
    # dominated the estimator.  Stored as a uniform per-coefficient array so
    # `inverse_group_compressor` continues to accept the same shape.
    jensen_corrections = {}
    _calib_idx = np.concatenate([train_idx, val_idx]).astype(int)
    model.eval()
    with torch.no_grad():
        _calib_pred_dict = model(
            torch.from_numpy(near_s[_calib_idx]).to(device),
            torch.from_numpy(far_s[_calib_idx]).to(device),
            torch.from_numpy(ctx_near_n[_calib_idx]).to(device),
            torch.from_numpy(ctx_far_n[_calib_idx]).to(device),
            torch.from_numpy(ctx_sci_n[_calib_idx]).to(device),
            sci_moon_alt_sign=(torch.from_numpy(_moon_alt_sign_all[_calib_idx]).to(device)
                               if model.moon_alt_conditional_alpha else None),
        )
    _calib_scores_scaled = np.zeros((_calib_idx.size, n_input_score), dtype=np.float64)
    for _g, (_lo, _hi) in score_slices.items():
        _calib_scores_scaled[:, _lo:_hi] = _calib_pred_dict[_g].detach().cpu().numpy().astype(np.float64)
    _calib_pred_scores = score_scaler.inverse_transform(
        _calib_scores_scaled.astype(np.float32)).astype(np.float64)

    _calib_ctx_sci_phys = np.asarray(ctx_sci, dtype=np.float64)[_calib_idx]
    _calib_pred_phys_naive = expand_scores_to_coefs(
        _calib_pred_scores, _calib_ctx_sci_phys,
        compressors, group_indices, geom_kwargs,
        int(coef_sci.shape[1]), score_slices,
        jensen_corrections=None,
    )
    _calib_true_phys = np.asarray(coef_sci, dtype=np.float64)[_calib_idx]

    _CALIB_LIFT_CLIP = (0.5, 2.0)  # sanity bounds; outside indicates a broken group
    # 2026-08-19: moon uses a per-coefficient lift; every other group uses the historical
    # scalar lift.  Rationale: cell 27 shows the moon residual has a spectral tilt bias
    # (+0.054 blue / -0.018 NIR) that a single per-group scalar cannot correct.  Per-coef
    # lift on the 29 Moon_bs knots nulls the tilt by construction.  Tight [0.7, 1.4] clip
    # so a broken knot doesn't cascade.
    _MOON_LIFT_CLIP = (0.7, 1.4)
    for _gname, _comp in compressors.items():
        _gidx = np.asarray(_comp['coef_indices'], dtype=int)
        if _gname == 'moon':
            _mean_true_pc = np.mean(_calib_true_phys[:, _gidx], axis=0).astype(np.float64)
            _mean_pred_pc = np.mean(_calib_pred_phys_naive[:, _gidx], axis=0).astype(np.float64)
            _rel_mag_pc = np.abs(_mean_pred_pc) / np.maximum(np.abs(_mean_true_pc), 1e-30)
            _lift_pc = np.where(
                (_mean_true_pc > 0) & (_mean_pred_pc > 0) & (_rel_mag_pc >= 0.05),
                _mean_true_pc / np.where(_mean_pred_pc != 0.0, _mean_pred_pc, 1.0),
                1.0,
            )
            _lift_pc_clipped = np.clip(_lift_pc, _MOON_LIFT_CLIP[0], _MOON_LIFT_CLIP[1])
            _n_clipped = int(np.sum(_lift_pc != _lift_pc_clipped))
            jensen_corrections[_gname] = _lift_pc_clipped.astype(np.float64)
            print(f'Calibration: moon per-coef lift range=[{_lift_pc_clipped.min():.3f}, '
                  f'{_lift_pc_clipped.max():.3f}] median={float(np.median(_lift_pc_clipped)):.3f} '
                  f'({_n_clipped}/{_lift_pc_clipped.size} knots clipped to {_MOON_LIFT_CLIP})')
            continue
        _mean_true = float(np.mean(_calib_true_phys[:, _gidx]))
        _mean_pred = float(np.mean(_calib_pred_phys_naive[:, _gidx]))
        _rel_mag = abs(_mean_pred) / max(abs(_mean_true), 1e-30)
        if (not np.isfinite(_mean_true) or not np.isfinite(_mean_pred)
                or _mean_true * _mean_pred <= 0.0
                or _rel_mag < 0.05):
            print(f'Calibration: {_gname} skipped (degenerate or near-zero mean; '
                  f'true={_mean_true:.4g}, pred_naive={_mean_pred:.4g}).')
            continue
        _raw_lift = _mean_true / _mean_pred
        _lift = float(np.clip(_raw_lift, _CALIB_LIFT_CLIP[0], _CALIB_LIFT_CLIP[1]))
        if _raw_lift != _lift:
            print(f'Calibration: {_gname} lift {_raw_lift:.4f} clipped to {_lift:.4f}.')
        jensen_corrections[_gname] = np.full(_gidx.size, _lift, dtype=np.float64)

    if jensen_corrections:
        print('Empirical per-group mean-bias calibration (train+val rows, uniform per-group scalar):')
        print(f"  {'group':<14s} {'n_g':>4s} {'mean_true':>10s} {'mean_pred_naive':>16s} "
              f"{'lift':>7s} {'delta_%':>8s}")
        for _gname, _corr in jensen_corrections.items():
            _gidx = np.asarray(compressors[_gname]['coef_indices'], dtype=int)
            _mean_true = float(np.mean(_calib_true_phys[:, _gidx]))
            _mean_pred = float(np.mean(_calib_pred_phys_naive[:, _gidx]))
            _lift = float(_corr[0])
            _delta_pct = 100.0 * (_lift - 1.0)
            print(f'  {_gname:<14s} {len(_corr):>4d} {_mean_true:>10.4g} '
                  f'{_mean_pred:>16.4g} {_lift:>7.4f} {_delta_pct:>+7.2f}%')

    # Per-group upper cap = 3.0 x max(coef_sci_train, axis=0) per coefficient.
    # Defensive guard applied at inference in expand_scores_to_coefs (§11 item 11).
    coef_upper_bound = {}
    _coef_sci_train_arr = np.asarray(coef_sci, dtype=np.float64)[train_idx]
    for _gname, _gidx_local in group_indices.items():
        _gidx_local = np.asarray(_gidx_local, dtype=int)
        if _gidx_local.size == 0:
            continue
        _max_train = np.max(_coef_sci_train_arr[:, _gidx_local], axis=0)
        coef_upper_bound[_gname] = (
            3.0 * np.maximum(_max_train, 0.0)).astype(np.float32)

    return {
        'model': model,
        'device': device,
        'score_scaler': score_scaler,
        'ctx_scaler': ctx_scaler,
        'compressors': compressors,
        'jensen_corrections': jensen_corrections,
        'coef_upper_bound': coef_upper_bound,
        'geom_kwargs': geom_kwargs,
        'group_indices': group_indices,
        'score_slices': score_slices,
        'group_score_dims': group_score_dims,
        'n_input_score': n_input_score,
        'history': history,
        'blend_history': blend_history,
        'blend_init_alpha': float(blend_init_alpha),
        'blend_optim': str(_blend_mode),
        'blend_use_direct': bool(model.blend_use_direct),
        'blend_prefit_alphas': (dict(_prefit_alphas)
                                if _prefit_alphas is not None else None),
        'best_val_loss': float(best_val),
        'best_epoch': int(best_epoch),
        'train_idx': train_idx, 'val_idx': val_idx, 'test_idx': test_idx,
        'coef_names': [str(x) for x in filtered['coef_names']],
        'ctx_names': [str(x) for x in filtered['ctx_names']],
        'config': {
            'n_epochs': int(n_epochs), 'batch_size': int(batch_size),
            'lr': float(lr), 'encoder_dims': tuple(int(v) for v in encoder_dims),
            'ctx_dims': tuple(int(v) for v in ctx_dims),
            'trunk_dims': tuple(int(v) for v in trunk_dims),
            'head_dim': int(head_dim), 'weight_decay': float(weight_decay),
            'patience': int(patience), 'seed': int(seed),
            'drop_vanrhijn_from_context': bool(drop_vanrhijn_from_context),
            'moon_group_weight': float(moon_group_weight),
            'continuum_group_weight': float(continuum_group_weight),
            'mesospheric_group_weight': float(mesospheric_group_weight),
            'ionospheric_group_weight': float(ionospheric_group_weight),
            'blend_init_alpha': float(blend_init_alpha),
            'blend_optim': str(_blend_mode),
            'moon_alt_conditional_alpha': bool(moon_alt_conditional_alpha),
            'high_airmass_boost': float(high_airmass_boost),
            'use_coef_err_weights': bool(_weights_enabled),
            'coef_err_sigma_floor_rel': (
                _resolved_floor_by_group
                if _weights_enabled
                else dict(DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP)
            ),
        },
    }


def predict_sci_coefficients_default(artifacts, coef_near_phys, coef_far_phys,
                                     ctx_near_phys, ctx_far_phys, ctx_sci_phys):
    """Default sky-to-science coefficient predictor (compressed model, see methods §5.5).

    Pipeline: physical coefficients -> divide by geometry factor (§4) ->
    per-group forward compressor (asinh / linear / sqrt + PCA rotation +
    xarm-selected retained subspace) -> encoder + fusion + trunk + heads ->
    inverse compressor -> multiply by science-pointing geometry factor ->
    non-negativity clip.

    When ``artifacts`` is an N-seed ensemble (``is_ensemble=True``), predicts
    with each member and returns the arithmetic mean in physical space
    (the deployed default since 2026-08-11; see §12).
    """
    if artifacts.get('is_ensemble', False):
        _preds = [predict_sci_coefficients_default(
                      m, coef_near_phys, coef_far_phys,
                      ctx_near_phys, ctx_far_phys, ctx_sci_phys)
                  for m in artifacts['members']]
        return np.mean(np.stack(_preds, axis=0), axis=0).astype(np.float32)
    model = artifacts['model']
    device = artifacts['device']
    score_scaler = artifacts['score_scaler']
    ctx_scaler = artifacts['ctx_scaler']
    compressors = artifacts['compressors']
    geom_kwargs = artifacts['geom_kwargs']
    group_indices = artifacts['group_indices']
    score_slices = artifacts['score_slices']
    n_coef = len(artifacts['coef_names'])

    scores_near, _ = compress_coefs_to_scores(
        np.asarray(coef_near_phys, dtype=np.float64),
        np.asarray(ctx_near_phys, dtype=np.float64),
        compressors, geom_kwargs, group_indices)
    scores_far, _ = compress_coefs_to_scores(
        np.asarray(coef_far_phys, dtype=np.float64),
        np.asarray(ctx_far_phys, dtype=np.float64),
        compressors, geom_kwargs, group_indices)

    near_s = np.clip(score_scaler.transform(scores_near.astype(np.float32)),
                     -25.0, 25.0).astype(np.float32)
    far_s = np.clip(score_scaler.transform(scores_far.astype(np.float32)),
                    -25.0, 25.0).astype(np.float32)
    ctx_near_n = np.clip(ctx_scaler.transform(np.asarray(ctx_near_phys, dtype=np.float32)),
                         -25.0, 25.0).astype(np.float32)
    ctx_far_n = np.clip(ctx_scaler.transform(np.asarray(ctx_far_phys, dtype=np.float32)),
                        -25.0, 25.0).astype(np.float32)
    ctx_sci_n = np.clip(ctx_scaler.transform(np.asarray(ctx_sci_phys, dtype=np.float32)),
                        -25.0, 25.0).astype(np.float32)

    _pred_ma_sign = None
    if getattr(model, 'moon_alt_conditional_alpha', False):
        _ctx_names_p = list(artifacts.get('ctx_names', []))
        if 'moon_alt' in _ctx_names_p:
            _ma_idx = _ctx_names_p.index('moon_alt')
            _ma_sign_np = (np.asarray(ctx_sci_phys, dtype=np.float32)[:, _ma_idx] > 0.0
                           ).astype(np.int64)
            _pred_ma_sign = torch.from_numpy(_ma_sign_np).to(device)

    with torch.no_grad():
        pred_dict = model(
            torch.from_numpy(near_s).to(device),
            torch.from_numpy(far_s).to(device),
            torch.from_numpy(ctx_near_n).to(device),
            torch.from_numpy(ctx_far_n).to(device),
            torch.from_numpy(ctx_sci_n).to(device),
            sci_moon_alt_sign=_pred_ma_sign,
        )

    n_rows = near_s.shape[0]
    n_score_total = artifacts['n_input_score']
    pred_scaled = np.zeros((n_rows, n_score_total), dtype=np.float64)
    for g, (lo, hi) in score_slices.items():
        pred_scaled[:, lo:hi] = pred_dict[g].detach().cpu().numpy().astype(np.float64)

    pred_scores = score_scaler.inverse_transform(
        pred_scaled.astype(np.float32)).astype(np.float64)

    coef_predicted = expand_scores_to_coefs(
        pred_scores, np.asarray(ctx_sci_phys, dtype=np.float64),
        compressors, group_indices, geom_kwargs, n_coef, score_slices,
        jensen_corrections=artifacts.get('jensen_corrections'),
        coef_upper_bound=artifacts.get('coef_upper_bound'))
    return coef_predicted.astype(np.float32)


# --- Train the compressed model as an N-seed ensemble, register as the default predictor --
required = ['filtered_triplet', 'group_compressors', 'compress_geom_kwargs',
            '_group_indices_compress', '_metric_row']
_missing = [k for k in required if k not in globals()]
if _missing:
    raise RuntimeError('Run prerequisite cells first (compressor fit + ML utils). '
                       'Missing: ' + ', '.join(_missing))

default_dual_group_config = {
    # 2026-08-11 (multi-seed winner): smaller encoder + narrow trunk; picked for robustness across seeds.
    # Single-seed (42) mean_rmse=20.40; 4-seed mean=21.07+/-0.75 vs previous default 21.43+/-1.15.
    'name': 'dual_group_mlp_compressed',
    'n_epochs': 50,
    'batch_size': 256,
    'lr': 7.0e-4,
    'encoder_dims': (768, 384),
    'ctx_dims': (64,),
    'trunk_dims': (320, 160),
    # 2026-08-16 arch retune (trunk/head sweep at the flat-weight defaults):
    # head_dim 384 -> 192 shrinks per-group heads by 2x and improves test RMSE
    # by ~3% (278.1 -> 270.2 single-seed) with cleaner convergence (best_ep 22/30
    # vs 19/27).  trunk_dims=(640,320) gave a marginally better 269.1 but did NOT
    # combine additively with head/2 and cost 21% more per-epoch, so head_dim=192
    # is the cheap win.  Do NOT combine with trunk*2 (they interact negatively).
    'head_dim': 192,
    # 2026-08-16 loss-surface retune: wd 1e-4 -> 3.3e-5, patience 16 -> 8,
    # all per-group multipliers and airmass boost -> 1.0.  Justified by the
    # LR/WD sweep at flat weights (§12 entry "2026-08-16 retune"): matched
    # val loss with lower wd and better test RMSE (-2.4% total).
    'weight_decay': 3.3e-5,
    'patience': 8,
    # 2026-08-19b: moon_group_weight 3.0 -> 4.0.  Under sqrt+all-29-PC compressor the
    # trainer no longer needs to compensate for compressor loss (per-coef lift ~1.02,
    # 0/29 knots clipped), so a stronger weight goes directly into shape fit; the goal is
    # to recover the ~0.35 blue-band moon rms that step-1a asinh reached before the sqrt
    # switch relaxed it to 0.40 (see 2026-08-19b memory entry).
    'moon_group_weight': 4.0,
    'continuum_group_weight': 1.0,
    'mesospheric_group_weight': 1.0,
    'ionospheric_group_weight': 1.0,
    'moon_alt_conditional_alpha': False,
    'high_airmass_boost': 1.0,
    'use_coef_err_weights': True,
    # 2026-08-16: per-group floor -- moon / continuum / ionospheric take
    # 0.05, mesospheric / atomic take 0.20 (see §7 diagnostic).
    'coef_err_sigma_floor_rel': dict(DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP),
    # 2026-08-12: N=10 (raised from 4 after the item-6 predictive-sigma diagnostic
    # of §11.7 showed the 4-seed spread under-estimates true error by ~2.5x on test).
    # Ensemble-mean stderr scales as sigma_seed / sqrt(N): with single-seed std ~0.72,
    # stderr drops from ~0.36 at N=4 to ~0.23 at N=10 (~1.2% of aggregate mean_rmse).
    # Larger N tightens the mean-bias floor of §11.1 and the sigma_scale estimate of
    # item 6, but does NOT fix the under-dispersion factor itself (a proper
    # Gaussian-NLL head does; §11.7 remains partially open). Cost is 10x training +
    # 10x inference, still sub-second per call on this dataset.
    'ensemble_seeds': (42, 43, 44, 45, 46, 47, 48, 49, 50, 51),
}
print(f'=== Training compressed dual-encoder group-head MLP ({len(default_dual_group_config["ensemble_seeds"])}-seed ensemble default) ===')
print(default_dual_group_config)

_shared_train_kwargs = dict(
    n_epochs=int(default_dual_group_config['n_epochs']),
    batch_size=int(default_dual_group_config['batch_size']),
    lr=float(default_dual_group_config['lr']),
    encoder_dims=tuple(int(v) for v in default_dual_group_config['encoder_dims']),
    ctx_dims=tuple(int(v) for v in default_dual_group_config['ctx_dims']),
    trunk_dims=tuple(int(v) for v in default_dual_group_config['trunk_dims']),
    head_dim=int(default_dual_group_config['head_dim']),
    weight_decay=float(default_dual_group_config['weight_decay']),
    patience=int(default_dual_group_config['patience']),
    moon_group_weight=float(default_dual_group_config['moon_group_weight']),
    continuum_group_weight=float(default_dual_group_config['continuum_group_weight']),
    mesospheric_group_weight=float(default_dual_group_config['mesospheric_group_weight']),
    ionospheric_group_weight=float(default_dual_group_config['ionospheric_group_weight']),
    moon_alt_conditional_alpha=bool(default_dual_group_config['moon_alt_conditional_alpha']),
    high_airmass_boost=float(default_dual_group_config['high_airmass_boost']),
    use_coef_err_weights=bool(default_dual_group_config['use_coef_err_weights']),
    coef_err_sigma_floor_rel=default_dual_group_config['coef_err_sigma_floor_rel'],
)
_ensemble_seeds = tuple(int(s) for s in default_dual_group_config['ensemble_seeds'])
_split_for_members = (
    filtered_triplet['compress_train_idx'],
    filtered_triplet['compress_val_idx'],
    filtered_triplet['compress_test_idx'],
)
_ensemble_members = []
for _seed in _ensemble_seeds:
    print(f'\n--- Ensemble member seed={_seed} ---')
    _member = train_compressed_group_mlp(
        filtered_triplet, group_compressors, _group_indices_compress,
        compress_geom_kwargs,
        split_indices=_split_for_members,
        seed=int(_seed),
        **_shared_train_kwargs,
    )
    _ensemble_members.append(_member)

# Shared fields lifted to top level so downstream cells (pipeline-state check,
# relationship plots, batch RMSE, naive baseline) continue to read the same keys.
_first_member = _ensemble_members[0]
mlp_artifacts = {
    'is_ensemble': True,
    'seeds': list(_ensemble_seeds),
    'members': _ensemble_members,
    'compressors': _first_member['compressors'],
    'coef_upper_bound': _first_member['coef_upper_bound'],
    'geom_kwargs': _first_member['geom_kwargs'],
    'group_indices': _first_member['group_indices'],
    'score_slices': _first_member['score_slices'],
    'group_score_dims': _first_member['group_score_dims'],
    'n_input_score': _first_member['n_input_score'],
    'coef_names': _first_member['coef_names'],
    'ctx_names': _first_member['ctx_names'],
    'train_idx': _first_member['train_idx'],
    'val_idx': _first_member['val_idx'],
    'test_idx': _first_member['test_idx'],
    'config': _first_member['config'],
    'best_epochs': [int(m['best_epoch']) for m in _ensemble_members],
    'best_val_losses': [float(m['best_val_loss']) for m in _ensemble_members],
}
print(f"\nEnsemble assembled: {len(_ensemble_members)} members, "
      f"best_epochs={mlp_artifacts['best_epochs']}, "
      f"best_val_losses={[f'{v:.5f}' for v in mlp_artifacts['best_val_losses']]}")

# Convenience globals used by the downstream evaluation cells.
train_idx = np.asarray(mlp_artifacts['train_idx'], dtype=int)
val_idx = np.asarray(mlp_artifacts['val_idx'], dtype=int)
test_idx = np.asarray(mlp_artifacts['test_idx'], dtype=int)

coef_near_all = np.asarray(filtered_triplet['coef_near'], dtype=np.float32)
coef_far_all = np.asarray(filtered_triplet['coef_far'], dtype=np.float32)
coef_sci_all = np.asarray(filtered_triplet['coef_sci'], dtype=np.float32)
ctx_near_all = np.asarray(filtered_triplet['ctx_near'], dtype=np.float32)
ctx_far_all = np.asarray(filtered_triplet['ctx_far'], dtype=np.float32)
ctx_sci_all = np.asarray(filtered_triplet['ctx_sci'], dtype=np.float32)

# 1-sigma per-coefficient uncertainties from the decomposition COEF_ERR HDU.
# NaN for coefficients pinned at the c>=0 boundary; the loss cell below
# replaces those with a floor before use.
coef_err_near_all = np.asarray(
    filtered_triplet.get('coef_err_near', np.full_like(coef_near_all, np.nan)),
    dtype=np.float32,
)
coef_err_far_all = np.asarray(
    filtered_triplet.get('coef_err_far', np.full_like(coef_far_all, np.nan)),
    dtype=np.float32,
)
coef_err_sci_all = np.asarray(
    filtered_triplet.get('coef_err_sci', np.full_like(coef_sci_all, np.nan)),
    dtype=np.float32,
)

y_te = coef_sci_all[test_idx]
coef_pred_det = predict_sci_coefficients_default(
    mlp_artifacts,
    coef_near_phys=coef_near_all[test_idx],
    coef_far_phys=coef_far_all[test_idx],
    ctx_near_phys=ctx_near_all[test_idx],
    ctx_far_phys=ctx_far_all[test_idx],
    ctx_sci_phys=ctx_sci_all[test_idx],
).astype(np.float32)

# Per-seed test metrics + ensemble, so seed-to-seed noise and the ensemble-mean gain are visible.
_per_seed_test_metrics = []
# 2026-08-16: pass sigma to _metric_row so it emits mean_wrmse / median_wrmse / total_wrmse.
_sig_te = coef_err_sci_all[test_idx] if 'coef_err_sci_all' in globals() else None
_floor_te = dict(DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP) if 'DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP' in globals() else None
_gidx_te = (_group_indices_compress if '_group_indices_compress' in globals()
             else group_indices_sf if 'group_indices_sf' in globals() else None)
for _seed, _member in zip(mlp_artifacts['seeds'], _ensemble_members):
    _pred_seed = predict_sci_coefficients_default(
        _member,
        coef_near_phys=coef_near_all[test_idx], coef_far_phys=coef_far_all[test_idx],
        ctx_near_phys=ctx_near_all[test_idx], ctx_far_phys=ctx_far_all[test_idx],
        ctx_sci_phys=ctx_sci_all[test_idx],
    ).astype(np.float32)
    _per_seed_test_metrics.append({'variant': f'seed={_seed}',
                                   **_metric_row(y_te, _pred_seed, f'seed={_seed}',
                                                 sigma=_sig_te, group_indices=_gidx_te,
                                                 floor_by_group=_floor_te)})
_per_seed_test_metrics.append({'variant': f'{len(_ensemble_members)}-seed ensemble (default)',
                               **_metric_row(y_te, coef_pred_det, 'ensemble',
                                             sigma=_sig_te, group_indices=_gidx_te,
                                             floor_by_group=_floor_te)})

cmp_df = pd.DataFrame(_per_seed_test_metrics)
print(f"\nPer-seed and ensemble test metrics on the night-held-out split "
      f"(n_test = {test_idx.size} rows):")
print(cmp_df.to_string(index=False, float_format=lambda v: f'{v:.6g}'))

# "Is N seeds enough?" quantitative answer from the seed-to-seed spread.
_per_seed_rmse_arr = np.array([r['mean_eRMSE'] for r in _per_seed_test_metrics[:len(_ensemble_seeds)]])
_seed_std_rmse = float(np.std(_per_seed_rmse_arr, ddof=1))
_ensemble_rmse = float(cmp_df.iloc[-1]['mean_eRMSE'])
_ensemble_stderr = _seed_std_rmse / (len(_ensemble_seeds) ** 0.5)
print(f'\nSeed-to-seed test mean_eRMSE std: {_seed_std_rmse:.3f}  '
      f'(single-seed noise scale on this dataset)')
print(f'Ensemble-mean stderr = std/sqrt(N={len(_ensemble_seeds)}) = '
      f'{_ensemble_stderr:.3f}  '
      f'(~{100.0 * _ensemble_stderr / max(_ensemble_rmse, 1e-30):.1f}% of ensemble '
      f'mean_eRMSE {_ensemble_rmse:.3f}).')
print(f'Doubling N reduces stderr by sqrt(2) ~ 1.41x. N={len(_ensemble_seeds)} is the deployment '
      'default: ensemble max |bias| stays below 1% (§11.1) and the ~1% mean_eRMSE stderr is '
      'well under the target-contamination floor of §11.2. N was raised from 4 to 10 on '
      '2026-08-12 after the item-6 predictive-sigma diagnostic (§11.7) tightened the '
      'sigma-scale bolt-on that N feeds. See §12 for the tradeoff.')


In [ ]:
# ============================================================================
# Denser sweep centred on the current default's neighbourhood.
#
# Motivation: the wide random sweep (previous cell) sampled 14 of ~1200
# candidates and missed the exact current-default geometry
# (encoder=(384,192), trunk=(320,160), head=192, lr=7e-4, wd=1e-4).
# Its "winner" (encoder=(256,128), head=224, lr=1e-3) turned out to be
# WORSE than the current default when retrained at the default budget.
#
# This cell does the opposite: a small EXHAUSTIVE grid around the current
# default, evaluated at the same budget as `mlp_artifacts`, so the
# comparison is direct and every candidate at seed=42 is deterministic.
#
#   encoder_dims  in {(320,160), (384,192)}                 -> 2
#   trunk_dims    fixed (320,160)                           -> 1
#   head_dim      in {160, 192, 224}                        -> 3
#   lr            in {5e-4, 7e-4, 1e-3}                     -> 3
#   weight_decay  fixed 1e-4                                -> 1
#   -> 18 candidates, run all
#
# Everything else matches the default training cell: seed=42,
# n_epochs=50, patience=4, batch_size=256, ctx_dims=(64,),
# grad_clip=1.0, same night-held-out split.
#
# Gate: RUN_DENSE_SWEEP.  Never overwrites `mlp_artifacts`.  When done,
# stores the leaderboard in `dense_sweep_df` and the winning config in
# `dense_best_config`.
# ============================================================================
required = ['filtered_triplet', 'group_compressors', 'compress_geom_kwargs',
            '_group_indices_compress', 'train_compressed_group_mlp',
            'predict_sci_coefficients_default', '_metric_row',
            'default_dual_group_config', 'mlp_artifacts', 'cmp_df',
            'coef_near_all', 'coef_far_all', 'coef_sci_all',
            'ctx_near_all', 'ctx_far_all', 'ctx_sci_all', 'test_idx']
_missing = [k for k in required if k not in globals()]
if _missing:
    raise RuntimeError('Run prerequisite cells first. Missing: ' + ', '.join(_missing))

import ast
import contextlib
import io
import itertools

RUN_DENSE_SWEEP = False
DENSE_SUPPRESS_STDOUT = True
DENSE_SEED = 42

# Grid expanded 2026-08-08: previous default sat at the (448,224)/h=224/lr=5e-4 corner,
# so this iteration drops smaller tiers and probes larger encoder/head + lower LR.
dense_grid = {
    # 2026-08-11: probing larger geometries after mesospheric no-PCA adoption
    # widened compressor input from ~67 scores to 441; the (512, 256) encoder
    # had ~1.7x fanout and the 160-wide trunk output capped mesospheric head rank.
    'encoder_dims': [(512, 256), (768, 384), (1024, 512)],
    'trunk_dims': [(320, 160), (512, 256)],
    'head_dim': [288, 384],
    'lr': [5e-4, 7e-4],
    'weight_decay': [1e-4],
}

dense_cfgs = [
    {'encoder_dims': e, 'trunk_dims': t, 'head_dim': h,
     'lr': lr, 'weight_decay': wd,
     'n_epochs': int(default_dual_group_config['n_epochs']),
     'patience': int(default_dual_group_config['patience'])}
    for e, t, h, lr, wd in itertools.product(
        dense_grid['encoder_dims'], dense_grid['trunk_dims'],
        dense_grid['head_dim'], dense_grid['lr'],
        dense_grid['weight_decay'],
    )
]


def _eval_dense_cfg(cfg, idx_all):
    ctx_mgr = (contextlib.redirect_stdout(io.StringIO())
               if DENSE_SUPPRESS_STDOUT else contextlib.nullcontext())
    with ctx_mgr:
        art = train_compressed_group_mlp(
            filtered_triplet, group_compressors, _group_indices_compress,
            compress_geom_kwargs,
            split_indices=(
                filtered_triplet['compress_train_idx'],
                filtered_triplet['compress_val_idx'],
                filtered_triplet['compress_test_idx'],
            ),
            n_epochs=int(cfg['n_epochs']),
            batch_size=int(default_dual_group_config['batch_size']),
            lr=float(cfg['lr']),
            encoder_dims=tuple(int(v) for v in cfg['encoder_dims']),
            ctx_dims=tuple(int(v) for v in default_dual_group_config['ctx_dims']),
            trunk_dims=tuple(int(v) for v in cfg['trunk_dims']),
            head_dim=int(cfg['head_dim']),
            weight_decay=float(cfg['weight_decay']),
            grad_clip=1.0,
            patience=int(cfg['patience']),
            seed=DENSE_SEED,
        )
        _te = np.asarray(art['test_idx'], dtype=int)
        assert np.array_equal(_te, idx_all), \
            'test split changed between default and dense-sweep runs'
        y_te = coef_sci_all[_te]
        y_pred = predict_sci_coefficients_default(
            art,
            coef_near_phys=coef_near_all[_te],
            coef_far_phys=coef_far_all[_te],
            ctx_near_phys=ctx_near_all[_te],
            ctx_far_phys=ctx_far_all[_te],
            ctx_sci_phys=ctx_sci_all[_te],
        ).astype(np.float32)
    m = _metric_row(y_te, y_pred, 'dense_sweep')
    is_current_default = (
        tuple(int(v) for v in cfg['encoder_dims'])
            == tuple(int(v) for v in default_dual_group_config['encoder_dims'])
        and tuple(int(v) for v in cfg['trunk_dims'])
            == tuple(int(v) for v in default_dual_group_config['trunk_dims'])
        and int(cfg['head_dim']) == int(default_dual_group_config['head_dim'])
        and float(cfg['lr']) == float(default_dual_group_config['lr'])
        and float(cfg['weight_decay']) == float(default_dual_group_config['weight_decay'])
    )
    return {
        'encoder_dims': str(tuple(int(v) for v in cfg['encoder_dims'])),
        'trunk_dims': str(tuple(int(v) for v in cfg['trunk_dims'])),
        'head_dim': int(cfg['head_dim']),
        'lr': float(cfg['lr']),
        'weight_decay': float(cfg['weight_decay']),
        'is_current_default': bool(is_current_default),
        'best_epoch': int(art['best_epoch']),
        'best_val_loss': float(art['best_val_loss']),
        'test_mean_eRMSE': float(m['mean_eRMSE']),
        'test_median_eRMSE': float(m['median_eRMSE']),
        'test_mean_eMAE': float(m['mean_eMAE']),
        'test_median_corr': float(m['median_corr']),
    }


dense_best_config = None
if not RUN_DENSE_SWEEP:
    print('Dense sweep skipped. Set RUN_DENSE_SWEEP = True to execute.')
else:
    print(f'Running dense sweep exhaustively: {len(dense_cfgs)} configs at '
          f"n_epochs={default_dual_group_config['n_epochs']}, "
          f"patience={default_dual_group_config['patience']}, "
          f'seed={DENSE_SEED}')
    _te_ref = np.asarray(test_idx, dtype=int)
    _rows = []
    for i, cfg in enumerate(dense_cfgs, start=1):
        row = _eval_dense_cfg(cfg, _te_ref)
        marker = '  <-- current default' if row['is_current_default'] else ''
        print(f'  [{i:02d}/{len(dense_cfgs)}] '
              f"enc={row['encoder_dims']} h={row['head_dim']} "
              f"lr={row['lr']:.1e} wd={row['weight_decay']:.1e} -> "
              f"mean_eRMSE={row['test_mean_eRMSE']:.4g} "
              f"median_corr={row['test_median_corr']:.3f} "
              f"@ep{row['best_epoch']}{marker}")
        _rows.append(row)

    dense_sweep_df = (pd.DataFrame(_rows)
                     .sort_values(['test_median_eRMSE', 'test_mean_eRMSE', 'test_mean_eMAE',
                                   'best_val_loss'])
                     .reset_index(drop=True))
    print('\nFull dense-sweep leaderboard (sorted by test_median_eRMSE; test_mean_eRMSE for context):')
    print(dense_sweep_df.to_string(
        index=False, float_format=lambda v: f'{v:.6g}'))

    best_row = dense_sweep_df.iloc[0]
    dense_best_config = {
        'name': 'dual_group_mlp_compressed_dense_best',
        'n_epochs': int(default_dual_group_config['n_epochs']),
        'batch_size': int(default_dual_group_config['batch_size']),
        'lr': float(best_row['lr']),
        'encoder_dims': tuple(int(v) for v in ast.literal_eval(best_row['encoder_dims'])),
        'ctx_dims': tuple(int(v) for v in default_dual_group_config['ctx_dims']),
        'trunk_dims': tuple(int(v) for v in ast.literal_eval(best_row['trunk_dims'])),
        'head_dim': int(best_row['head_dim']),
        'weight_decay': float(best_row['weight_decay']),
        'patience': int(default_dual_group_config['patience']),
    }

    _default_row = cmp_df.iloc[0]
    _default_mean_rmse = float(_default_row['mean_eRMSE'])
    _default_median_corr = float(_default_row['median_corr'])
    _best_mean_rmse = float(best_row['test_mean_eRMSE'])
    _best_median_corr = float(best_row['test_median_corr'])
    _delta_mean_rmse = _best_mean_rmse - _default_mean_rmse
    _delta_median_corr = _best_median_corr - _default_median_corr
    is_default_winner = bool(best_row['is_current_default'])

    print(f'\nCurrent `mlp_artifacts` (default) test metrics for reference:')
    print(f'  mean_eRMSE={_default_mean_rmse:.4g}, '
          f'median_corr={_default_median_corr:.4f}')
    print(f'\nDense-sweep winner:')
    print(f'  enc={best_row["encoder_dims"]} trk={best_row["trunk_dims"]} '
          f'h={best_row["head_dim"]} lr={best_row["lr"]:.1e} '
          f'wd={best_row["weight_decay"]:.1e}')
    print(f'  mean_eRMSE={_best_mean_rmse:.4g} '
          f'({_delta_mean_rmse:+.4g} vs default), '
          f'median_corr={_best_median_corr:.4f} '
          f'({_delta_median_corr:+.4f} vs default)')
    if is_default_winner:
        print('\nVERDICT: current default IS the sweep winner. No change needed.')
    else:
        adopt = (_delta_mean_rmse < 0.0) and (_delta_median_corr > -0.005)
        print(f'\nAdoption verdict (mean_eRMSE improves AND median_corr does not '
              f'regress by more than 0.005): '
              f"{'ADOPT' if adopt else 'DO NOT ADOPT'}")
        print('\nRecommended dense_best_config:')
        print(dense_best_config)


## Copied Relationship Visualizations
The following visualization cell is copied from the existing notebook and uses the default predictor interface.

In [ ]:
# Enhanced coefficient-vs-context scatter matrix (2026-08-19 replacement of the
# 2026-08-10 raw-coefficient version). Improvements (1-10 from the discussion of
# "hidden correlations in raw coefficients"):
#   (1) shape rows (log_amp, red/blue tilt, PC1, PC2) for the Moon_bs spline and
#       the OH block replace the amplitude-dominated `median` aggregates so
#       zodi- / shape-scale signals stop being drowned by the common brightness
#       scalar;
#   (2) Y_TRANSFORM='log' switches raw-coefficient rows to log10; shape rows are
#       already dimensionless;
#   (3) Y_TRANSFORM='geom' divides coefficients by airglow_geometry_scale per
#       arm (same path the loader feeds the encoder);
#   (4) Y_TRANSFORM='residual' regresses y on [moon_alt, moon_sep, airmass,
#       sun_alt, moon_phase] fit on train rows and plots the residual (partial-
#       correlation panel) -- this is the transform that exposed the ecl->
#       Moon_bs signal invisible on raw coefficients;
#   (5) three extra ecliptic display columns (|ecl_beta|, cos(ecl_beta), signed
#       ecl_beta) appended to the context axis so the zodi-relevant projections
#       are directly on the grid;
#   (6) cyclic sin/cos pairs still decoded to degrees for readability;
#   (7) per-panel Pearson r annotation for the STAT_SERIES ('sci_true' default);
#   (8) 95% Fisher CI on r and sample size n annotated inline;
#   (9) partial-r shown when Y_TRANSFORM='residual' (the residualisation happens
#       upstream of the correlation, so the annotated `r` IS the partial r);
#  (10) Spearman rho and non-linearity indicator delta = rho^2 - r^2 annotated
#       alongside r so monotone-but-nonlinear trends stand out. Bolded when
#       |r|>0.15 or |rho|>0.15.
#
# Toggle Y_TRANSFORM at the top of the cell; the four modes are:
#   'raw'      -- coefficients as-fitted (matches the 2026-08-10 version).
#   'log'      -- log10(clip(coef, floor, None)); makes multiplicative scaling
#                 visible.
#   'geom'     -- coef / airglow_geometry_scale(ctx_arm, **compress_geom_kwargs);
#                 removes van-Rhijn * extinction so residual scatter is
#                 intrinsic emissivity.
#   'residual' -- nuisance-regressed partial residual (train-fit LSQ).
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import pearsonr, spearmanr

required = ["mlp_artifacts", "filtered_triplet", "predict_sci_coefficients_default",
            "_decode_cyclic_context"]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError("Run training + prediction cells first. Missing: "
                       + ", ".join(missing))

# --- Configuration -------------------------------------------------------
Y_TRANSFORM = 'residual'  # one of: 'raw', 'log', 'geom', 'residual'
STAT_SERIES = 'sci_true'  # which series drives the per-panel r/rho annotation
_MAX_POINTS = 3000        # scatter markers per panel; stats always use all rows
_TREND_ONLY_STAT = True   # draw the black running-median only for STAT_SERIES

_Y_LABEL = {
    'raw': 'raw coef',
    'log': 'log10(coef)',
    'geom': 'coef / airglow_geometry_scale',
    'residual': 'partial residual (train-fit LSQ vs moon geom)',
}[Y_TRANSFORM]

# --- Load per-arm coefficient + context matrices --------------------------
coef_near_all = np.asarray(filtered_triplet["coef_near"], dtype=np.float64)
coef_far_all = np.asarray(filtered_triplet["coef_far"], dtype=np.float64)
coef_sci_all = np.asarray(filtered_triplet["coef_sci"], dtype=np.float64)
ctx_near_all = np.asarray(filtered_triplet["ctx_near"], dtype=np.float32)
ctx_far_all = np.asarray(filtered_triplet["ctx_far"], dtype=np.float32)
ctx_sci_all = np.asarray(filtered_triplet["ctx_sci"], dtype=np.float32)
coef_names_all = [str(n) for n in filtered_triplet["coef_names"]]
ctx_names_all = [str(n) for n in filtered_triplet["ctx_names"]]
n_all = coef_sci_all.shape[0]

coef_pred_all = predict_sci_coefficients_default(
    mlp_artifacts,
    coef_near_phys=coef_near_all.astype(np.float32),
    coef_far_phys=coef_far_all.astype(np.float32),
    ctx_near_phys=ctx_near_all,
    ctx_far_phys=ctx_far_all,
    ctx_sci_phys=ctx_sci_all,
).astype(np.float64)

# --- (3) Geometry-normalise if requested --------------------------------
def _geom_scale_or_ones(ctx):
    if 'airglow_geometry_scale' not in globals() or 'compress_geom_kwargs' not in globals():
        return None
    try:
        return airglow_geometry_scale(np.asarray(ctx, dtype=np.float64),
                                      **compress_geom_kwargs)
    except (KeyError, ValueError) as exc:
        print(f'  geometry-scale unavailable ({type(exc).__name__}: {exc}); '
              'geom transform will pass through unchanged.')
        return None

_apply_geom = (Y_TRANSFORM == 'geom')
if _apply_geom:
    _scale_near = _geom_scale_or_ones(ctx_near_all)
    _scale_far = _geom_scale_or_ones(ctx_far_all)
    _scale_sci = _geom_scale_or_ones(ctx_sci_all)
    _apply_geom = _scale_near is not None and _scale_far is not None and _scale_sci is not None

if _apply_geom:
    coef_near_use = coef_near_all / np.where(_scale_near > 0, _scale_near, 1.0)
    coef_far_use = coef_far_all / np.where(_scale_far > 0, _scale_far, 1.0)
    coef_sci_use = coef_sci_all / np.where(_scale_sci > 0, _scale_sci, 1.0)
    coef_pred_use = coef_pred_all / np.where(_scale_sci > 0, _scale_sci, 1.0)
elif Y_TRANSFORM == 'log':
    _floor = 1e-6
    coef_near_use = np.log10(np.clip(coef_near_all, _floor, None))
    coef_far_use = np.log10(np.clip(coef_far_all, _floor, None))
    coef_sci_use = np.log10(np.clip(coef_sci_all, _floor, None))
    coef_pred_use = np.log10(np.clip(coef_pred_all, _floor, None))
else:
    coef_near_use = coef_near_all
    coef_far_use = coef_far_all
    coef_sci_use = coef_sci_all
    coef_pred_use = coef_pred_all

# --- (6) Cyclic pairs -> degrees --------------------------------------
_ctx_names_display, _ctx_near_disp = _decode_cyclic_context(ctx_names_all, ctx_near_all)
_, _ctx_far_disp = _decode_cyclic_context(ctx_names_all, ctx_far_all)
_, _ctx_sci_disp = _decode_cyclic_context(ctx_names_all, ctx_sci_all)

# --- (5) Ecliptic display column enrichment ------------------------------
def _augment_ecl_columns(names_disp, ctx_disp):
    if 'ecl_beta_deg' not in names_disp:
        return names_disp, ctx_disp
    _idx = names_disp.index('ecl_beta_deg')
    _b = np.asarray(ctx_disp[:, _idx], dtype=np.float64)
    _extra = np.column_stack([np.abs(_b), np.cos(np.deg2rad(_b))]).astype(np.float32)
    return (list(names_disp) + ['|ecl_beta|', 'cos(ecl_beta)'],
            np.concatenate([ctx_disp, _extra], axis=1))

_orig_display_names = list(_ctx_names_display)  # ecl column-add is fragile on [:-2] slice.
_ctx_names_display, _ctx_near_disp = _augment_ecl_columns(_orig_display_names, _ctx_near_disp)
_, _ctx_far_disp = _augment_ecl_columns(_orig_display_names, _ctx_far_disp)
_, _ctx_sci_disp = _augment_ecl_columns(_orig_display_names, _ctx_sci_disp)

# --- Coefficient index bookkeeping -----------------------------------
_names_lower = [n.lower() for n in coef_names_all]
_moon_idx = np.array([i for i, n in enumerate(_names_lower)
                      if n.startswith("moon_bs")], dtype=int)
_oh_idx = np.array([i for i, n in enumerate(_names_lower)
                    if n.startswith("oh_")], dtype=int)

# --- Train mask for PCA fit + nuisance regression --------------------
_train_idx_local = np.asarray(mlp_artifacts.get('train_idx', np.arange(n_all)),
                              dtype=int)
_train_mask = np.zeros(n_all, dtype=bool)
_train_mask[_train_idx_local[(_train_idx_local >= 0) & (_train_idx_local < n_all)]] = True
if _train_mask.sum() < 10:
    print('  train mask empty; using all rows for PCA / nuisance regression.')
    _train_mask[:] = True

# --- (1) Shape features: log-amp, red/blue tilt, PC1/PC2 -----------------
def _fit_shape_pca(block_idx, k=2):
    """PCA on training-set L2-normalised block; returns (mean, components)."""
    idx = np.asarray(block_idx, dtype=int)
    _b = np.clip(coef_sci_all[_train_mask][:, idx], 0.0, None)
    _n = np.linalg.norm(_b, axis=1, keepdims=True)
    _n = np.where(_n > 0, _n, 1.0)
    _shape = _b / _n
    _mean = _shape.mean(axis=0)
    _centered = _shape - _mean
    _u, _s, _vt = np.linalg.svd(_centered, full_matrices=False)
    return _mean, _vt[:k]


def _make_shape_functions(block_idx, tilt_frac=0.35):
    """Return dict of (label -> function(coef_mat)) producing all-row series."""
    idx = np.asarray(block_idx, dtype=int)
    if idx.size < 4:
        return {}
    lo_end = max(1, int(round(idx.size * tilt_frac)))
    hi_start = max(lo_end, int(round(idx.size * (1 - tilt_frac))))
    _mean, _components = _fit_shape_pca(idx, k=2)

    def _log_amp(coef_mat):
        _b = np.clip(coef_mat[:, idx], 0.0, None)
        return np.log10(np.clip(_b.sum(axis=1), 1e-6, None))

    def _tilt(coef_mat):
        _b = np.clip(coef_mat[:, idx], 0.0, None)
        _lo = np.clip(_b[:, :lo_end].mean(axis=1), 1e-6, None)
        _hi = np.clip(_b[:, hi_start:].mean(axis=1), 1e-6, None)
        return np.log10(_hi / _lo)

    def _pc(coef_mat, k):
        _b = np.clip(coef_mat[:, idx], 0.0, None)
        _n = np.linalg.norm(_b, axis=1, keepdims=True)
        _n = np.where(_n > 0, _n, 1.0)
        return (_b / _n - _mean) @ _components[k]

    return {
        'log_amp': _log_amp,
        'red_blue_tilt': _tilt,
        'PC1': (lambda cm: _pc(cm, 0)),
        'PC2': (lambda cm: _pc(cm, 1)),
    }


_moon_shape = _make_shape_functions(_moon_idx)
_oh_shape = _make_shape_functions(_oh_idx)

# --- Individual coefficient rows (unchanged from 2026-08-10) ----------
_INDIVIDUAL = [
    "HO2", "FeO", "O2Ac",
    "ATOM_K", "ATOM_Na", "ATOM_Og",
    "ATOM_N", "ATOM_Or", "ATOM_Orc_OI0777", "ATOM_Orc_OI0845",
    "O2_b01",
]


def _find_col(name):
    lname = name.lower()
    matches = [i for i, n in enumerate(_names_lower) if n == lname]
    if not matches:
        raise KeyError(f"coefficient {name!r} not in coef_names_all")
    return matches[0]


_individual_idx = {name: _find_col(name) for name in _INDIVIDUAL}

# --- Row definitions -------------------------------------------------
_ROW_DEFS = []
if _moon_shape:
    _ROW_DEFS += [
        (f'moon log_amp (n={_moon_idx.size})', 'shape_moon_log_amp'),
        ('moon red/blue tilt', 'shape_moon_red_blue_tilt'),
        ('moon PC1', 'shape_moon_PC1'),
        ('moon PC2', 'shape_moon_PC2'),
    ]
if _oh_shape:
    _ROW_DEFS += [
        (f'OH log_amp (n={_oh_idx.size})', 'shape_oh_log_amp'),
        ('OH PC1', 'shape_oh_PC1'),
        ('OH PC2', 'shape_oh_PC2'),
    ]
for _n in _INDIVIDUAL:
    _ROW_DEFS.append((_n, _n))


# Raw coef matrices per series so shape features (PCA / L2-norm / red-blue
# tilt) always operate on physical coefficients, independently of Y_TRANSFORM.
_RAW_BY_SERIES = {
    'near': coef_near_all,
    'far': coef_far_all,
    'sci_true': coef_sci_all,
    'sci_pred_default': coef_pred_all,
}


def _series_full(series_name, coef_mat_use, kind):
    """Series value per row on the FULL dataset (all n_all rows)."""
    if kind.startswith('shape_moon_'):
        return _moon_shape[kind[len('shape_moon_'):]](_RAW_BY_SERIES[series_name])
    if kind.startswith('shape_oh_'):
        return _oh_shape[kind[len('shape_oh_'):]](_RAW_BY_SERIES[series_name])
    return coef_mat_use[:, _individual_idx[kind]]


# --- (4) Nuisance regression ------------------------------------------
_NUISANCE_COLS = ['moon_alt', 'moon_sep', 'airmass', 'sun_alt', 'moon_phase']


def _nuisance_matrix(ctx_names, ctx_disp):
    _idx = [ctx_names.index(c) for c in _NUISANCE_COLS if c in ctx_names]
    if not _idx:
        return None
    return ctx_disp[:, _idx].astype(np.float64)


_N_near = _nuisance_matrix(_ctx_names_display, _ctx_near_disp)
_N_far = _nuisance_matrix(_ctx_names_display, _ctx_far_disp)
_N_sci = _nuisance_matrix(_ctx_names_display, _ctx_sci_disp)


def _residualise(y_all, N_all):
    """Fit y ~ 1 + N on train rows; return residuals on ALL rows."""
    if N_all is None:
        return y_all
    _tr = _train_mask & np.isfinite(y_all) & np.all(np.isfinite(N_all), axis=1)
    if _tr.sum() < 10:
        return y_all
    X_tr = np.column_stack([np.ones(_tr.sum()), N_all[_tr]])
    beta, *_ = np.linalg.lstsq(X_tr, y_all[_tr], rcond=None)
    X_all = np.column_stack([np.ones(n_all), N_all])
    return y_all - X_all @ beta

# --- Downsample rows for markers only (stats always use full data) -----
if n_all > _MAX_POINTS:
    _rng = np.random.default_rng(42)
    _row_sel = np.sort(_rng.choice(n_all, size=_MAX_POINTS, replace=False))
else:
    _row_sel = np.arange(n_all)

_SERIES = [
    ('near',             coef_near_use, _ctx_near_disp, _N_near, "#1f77b4"),
    ('far',              coef_far_use,  _ctx_far_disp,  _N_far,  "#9467bd"),
    ('sci_true',         coef_sci_use,  _ctx_sci_disp,  _N_sci,  "#2ca02c"),
    ('sci_pred_default', coef_pred_use, _ctx_sci_disp,  _N_sci,  "#d62728"),
]

# --- Trend line: 15-bin running median between p2..p98 of x -----------
def _running_median(x, y, n_bins=15):
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 30:
        return np.array([]), np.array([])
    x_ = x[m]; y_ = y[m]
    lo, hi = np.percentile(x_, [2, 98])
    if not (hi > lo):
        return np.array([]), np.array([])
    edges = np.linspace(lo, hi, n_bins + 1)
    x_mid = 0.5 * (edges[:-1] + edges[1:])
    y_med = np.full(n_bins, np.nan)
    for b in range(n_bins):
        _in = (x_ >= edges[b]) & (x_ <= edges[b + 1])
        if _in.sum() >= 10:
            y_med[b] = np.median(y_[_in])
    return x_mid, y_med


def _fisher_ci(r, n, z_=1.96):
    if not np.isfinite(r) or n < 4 or abs(r) >= 1.0:
        return (np.nan, np.nan)
    _z = 0.5 * np.log((1 + r) / (1 - r))
    _se = 1.0 / np.sqrt(n - 3)
    return (float(np.tanh(_z - z_ * _se)), float(np.tanh(_z + z_ * _se)))


def _stat_pair(x, y):
    m = np.isfinite(x) & np.isfinite(y)
    n = int(m.sum())
    if n < 5:
        return dict(n=n, r=np.nan, r_lo=np.nan, r_hi=np.nan, rho=np.nan)
    r, _ = pearsonr(x[m], y[m])
    rho, _ = spearmanr(x[m], y[m])
    lo, hi = _fisher_ci(float(r), n)
    return dict(n=n, r=float(r), r_lo=lo, r_hi=hi, rho=float(rho))


# --- Build the figure -----------------------------------------------
n_rows = len(_ROW_DEFS)
n_cols = len(_ctx_names_display)

_subplot_titles = []
for r in range(n_rows):
    for c in range(n_cols):
        _subplot_titles.append(_ctx_names_display[c].replace('_', ' ') if r == 0 else '')

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=_subplot_titles,
    shared_xaxes='columns',
    shared_yaxes='rows',
    vertical_spacing=0.012,
    horizontal_spacing=0.003,
)

_marker_base = dict(size=3.0, opacity=0.25)

# Precompute y_full per (row_def, series) once so we can reuse for stats + trend.
_y_full_cache = {}
for row_label, kind in _ROW_DEFS:
    for series_name, coef_mat_use, _ctx, N_mat, _color in _SERIES:
        y_full = _series_full(series_name, coef_mat_use, kind)
        if Y_TRANSFORM == 'residual':
            y_full = _residualise(y_full, N_mat)
        _y_full_cache[(row_label, series_name)] = y_full

# Populate scatter + trend traces.
for r_i, (row_label, kind) in enumerate(_ROW_DEFS, start=1):
    for series_name, _cm, ctx_disp, _N, color in _SERIES:
        y_full = _y_full_cache[(row_label, series_name)]
        y_sel = y_full[_row_sel]
        for c_i in range(n_cols):
            if (_ctx_names_display[c_i] == 'sci_sep'
                    and series_name in ('sci_true', 'sci_pred_default')):
                continue
            x_vals = ctx_disp[_row_sel, c_i]
            fig.add_trace(
                go.Scattergl(
                    x=x_vals, y=y_sel,
                    mode='markers',
                    marker=dict(color=color, **_marker_base),
                    name=series_name,
                    legendgroup=series_name,
                    showlegend=(r_i == 1 and c_i == 0),
                    hoverinfo='skip',
                ),
                row=r_i, col=c_i + 1,
            )
            if _TREND_ONLY_STAT and series_name != STAT_SERIES:
                continue
            _xm, _ym = _running_median(ctx_disp[:, c_i], y_full, n_bins=15)
            if _xm.size:
                fig.add_trace(
                    go.Scattergl(
                        x=_xm, y=_ym,
                        mode='lines',
                        line=dict(color='#111111', width=1.2),
                        showlegend=(r_i == 1 and c_i == 0 and series_name == STAT_SERIES),
                        name=f'{series_name} running median',
                        legendgroup=f'{series_name}_median',
                        hoverinfo='skip',
                    ),
                    row=r_i, col=c_i + 1,
                )
    # Row axis label.
    fig.update_yaxes(title_text=row_label, row=r_i, col=1,
                     title_font=dict(size=9))

# --- Style original subplot title annotations (columns headers) BEFORE
#     appending our per-panel stat annotations so their smaller font survives.
fig.for_each_annotation(lambda a: a.update(font=dict(size=9)))

# --- Bottom-row column labels + tick styling ---
for c_i in range(n_cols):
    fig.update_xaxes(title_text=_ctx_names_display[c_i].replace('_', ' '),
                     row=n_rows, col=c_i + 1, title_font=dict(size=9))
fig.update_xaxes(showline=True, mirror=True, ticks='outside', ticklen=3,
                 tickfont=dict(size=7), tickangle=30)
fig.update_yaxes(showline=True, mirror=True, ticks='outside', ticklen=3,
                 tickfont=dict(size=7))

# --- (7-10) Per-panel Pearson r + Spearman rho + Fisher CI annotations ---
def _panel_ref(r_i, c_i):
    """Subplot axis names for annotation domain-relative refs."""
    idx = (r_i - 1) * n_cols + c_i
    xr = 'x' if idx == 1 else f'x{idx}'
    yr = 'y' if idx == 1 else f'y{idx}'
    return xr, yr


# Index STAT_SERIES entry for x-vector lookup.
_stat_ctx_disp = next(_ctx for _sn, _cm, _ctx, _N, _col in _SERIES
                       if _sn == STAT_SERIES)

_stat_annotations = []
for r_i, (row_label, kind) in enumerate(_ROW_DEFS, start=1):
    y_full = _y_full_cache.get((row_label, STAT_SERIES))
    if y_full is None:
        continue
    for c_i in range(n_cols):
        if (_ctx_names_display[c_i] == 'sci_sep'
                and STAT_SERIES in ('sci_true', 'sci_pred_default')):
            continue
        x_full = _stat_ctx_disp[:, c_i]
        st = _stat_pair(x_full, y_full)
        if not np.isfinite(st['r']):
            _r_str = 'r=n/a'
            _err = 0.0
        else:
            _err = max(st['r_hi'] - st['r'], st['r'] - st['r_lo']) if np.isfinite(st['r_lo']) else 0.0
            _r_str = f"r={st['r']:+.2f}±{_err:.2f}" if _err > 0 else f"r={st['r']:+.2f}"
        _rho_str = (', ρ=n/a' if not np.isfinite(st['rho'])
                    else f", ρ={st['rho']:+.2f}")
        _nl = (''
               if not (np.isfinite(st['rho']) and np.isfinite(st['r']))
               else f", Δ={st['rho']**2 - st['r']**2:+.02f}")
        _n_str = (f" (n={st['n']//1000}k)" if st['n'] >= 1000
                  else f" (n={st['n']})")
        _txt = f"{_r_str}{_rho_str}{_nl}{_n_str}"
        _strong = ((np.isfinite(st['r']) and abs(st['r']) > 0.15)
                   or (np.isfinite(st['rho']) and abs(st['rho']) > 0.15))
        _color = '#000000' if _strong else '#7a7a7a'
        _xr, _yr = _panel_ref(r_i, c_i + 1)
        _stat_annotations.append(dict(
            x=0.02, y=0.98,
            xref=f'{_xr} domain', yref=f'{_yr} domain',
            text=f'<b>{_txt}</b>' if _strong else _txt,
            showarrow=False, xanchor='left', yanchor='top',
            font=dict(size=7, color=_color),
        ))

# Merge our stat annotations onto the figure without touching subplot titles.
fig.update_layout(
    annotations=list(fig.layout.annotations) + _stat_annotations,
)

_panel_w = 190
_panel_h = 155
fig.update_layout(
    template='plotly_white',
    title=(f"Coefficient vs context (n_rows={n_rows}, n_cols={n_cols}, "
           f"scatter n={_row_sel.size}/full n={n_all}). "
           f"y-transform: <b>{Y_TRANSFORM}</b> [{_Y_LABEL}]. "
           f"black line = 15-bin running median of {STAT_SERIES}. "
           f"panel annotation: Pearson r ±95% Fisher CI, Spearman ρ, "
           f"Δ=ρ²−r² (non-linearity), n (rows used for r, on full data)."),
    height=_panel_h * n_rows + 200,
    width=_panel_w * n_cols + 220,
    legend=dict(orientation='h', yanchor='bottom', y=1.005, xanchor='left',
                x=0.0, itemsizing='constant'),
    margin=dict(l=110, r=30, t=120, b=100),
)
fig.show()


## Copied Full-Spectrum Verification
The following verification cell is copied from the existing notebook and reconstructs a selected science row from predicted coefficients.

In [ ]:
# Full-spectrum reconstruction test for a single requested row using default coefficients
import plotly.graph_objects as go

_e10_stem = "spline_moon/lvmsframe_median_stack_1.2.1_p40_p70_every10"
_e10_suffix = _DECOMP_SUFFIX  # inherited from cell 6
EVERY10_INPUT = f"{_e10_stem}.fits"
EVERY10_NEAR = f"{_e10_stem}_decomp_sky1{_e10_suffix}.fits"
EVERY10_FAR = f"{_e10_stem}_decomp_sky2{_e10_suffix}.fits"
EVERY10_SCI = f"{_e10_stem}_decomp_sci{_e10_suffix}.fits"

# Set the row to reconstruct and inspect.
# REQUESTED_ROW = 612
REQUESTED_ROW = 500
# REQUESTED_ROW = 978

required = [
    "mlp_artifacts",
    "predict_sci_coefficients_default",
    "context_cols",
    "build_triplet_coef_dataset",
    "reconstruct_component_spectra",
    "reconstruct_with_lsf",
    "load_lsf_state_if_available",
    "load_o2_vector_if_available",
    "_infer_base_dir_for_reconstruction",
    "_moon_bs_indices_from_names",
    "_row_spline_roughness",
]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError("Run the training + residual-correction cells first. Missing: " + ", ".join(missing))

# 1) Load coefficients/context from every10 decomposition products.
e10_triplet = build_triplet_coef_dataset(
    input_fits_path=EVERY10_INPUT,
    sky_near_decomp_fits_path=EVERY10_NEAR,
    sky_far_decomp_fits_path=EVERY10_FAR,
    sci_decomp_fits_path=EVERY10_SCI,
    context_columns=context_cols,
    return_chi2=False,
)
# ECLIPTIC-CTX-V1: match training-time ctx layout on the e10 triplet.
if '_augment_triplet_with_ecliptic' in globals():
    _augment_triplet_with_ecliptic(e10_triplet, meta_fits_path=EVERY10_INPUT)
n_e10 = int(e10_triplet["n_rows"])
row_index_e10 = np.asarray(e10_triplet["row_index"], dtype=np.int64)
coef_names_e10 = [str(n) for n in e10_triplet["coef_names"]]
moon_idx = _moon_bs_indices_from_names(coef_names_e10)

if moon_idx.size < 3:
    raise RuntimeError("Expected at least 3 Moon_bs coefficients for spline diagnostics")

# 2) Load observed spectra, wavelength grid, and LSF from every10 input.
with fits.open(EVERY10_INPUT) as hdul:
    for ext in ("FLUX_SKY_NEAR", "FLUX_SKY_FAR", "FLUX_SCI", "WAVE", "LSF_SCI"):
        if ext not in hdul:
            raise KeyError(f"Missing extension {ext} in {EVERY10_INPUT}")

    wave_arr = np.asarray(hdul["WAVE"].data, dtype=np.float64)
    flux_near_all = np.asarray(hdul["FLUX_SKY_NEAR"].data, dtype=np.float64)
    flux_far_all = np.asarray(hdul["FLUX_SKY_FAR"].data, dtype=np.float64)
    flux_sci_true_all = np.asarray(hdul["FLUX_SCI"].data, dtype=np.float64)
    lsf_sci_arr = np.asarray(hdul["LSF_SCI"].data, dtype=np.float64)

n_spec, n_wave = flux_sci_true_all.shape
if row_index_e10.size != n_e10:
    raise ValueError(f"Triplet row_index length mismatch: {row_index_e10.size} vs n_rows={n_e10}")
if np.any(row_index_e10 < 0) or np.any(row_index_e10 >= n_spec):
    raise ValueError(
        f"Triplet row_index contains values outside [0, {n_spec - 1}] for {EVERY10_INPUT}"
    )

idx_row = int(REQUESTED_ROW)
triplet_pos = np.flatnonzero(row_index_e10 == idx_row)
if triplet_pos.size == 0:
    raise IndexError(
        f"REQUESTED_ROW={idx_row} is not available in aligned triplet rows. "
        f"Choose one of e10_triplet['row_index'] (size={row_index_e10.size})."
    )
triplet_pos = int(triplet_pos[0])

# Normalize WAVE/LSF arrays to per-row vectors, then select requested row.
wave_row = wave_arr if wave_arr.ndim == 1 else wave_arr[idx_row]
lsf_row = lsf_sci_arr if lsf_sci_arr.ndim == 1 else lsf_sci_arr[idx_row]

flux_near_row = flux_near_all[idx_row]
flux_far_row = flux_far_all[idx_row]
flux_sci_true_row = flux_sci_true_all[idx_row]

# 3) Predict SCI coefficients for the requested row using global default path.
coef_pred_row_batch = predict_sci_coefficients_default(
    mlp_artifacts,
    coef_near_phys=e10_triplet["coef_near"][triplet_pos: triplet_pos + 1],
    coef_far_phys=e10_triplet["coef_far"][triplet_pos: triplet_pos + 1],
    ctx_near_phys=e10_triplet["ctx_near"][triplet_pos: triplet_pos + 1],
    ctx_far_phys=e10_triplet["ctx_far"][triplet_pos: triplet_pos + 1],
    ctx_sci_phys=e10_triplet["ctx_sci"][triplet_pos: triplet_pos + 1],
)
coef_pred_row = np.asarray(coef_pred_row_batch[0], dtype=np.float64)

# 3a) Context values for the near-sky, far-sky and science pointings at this row.
#     Cyclic sin/cos pairs are folded back to 0-360 degree axes for readability.
_ctx_names_e10 = list(e10_triplet["ctx_names"])
_ctx_row_stack = np.stack([
    e10_triplet["ctx_near"][triplet_pos],
    e10_triplet["ctx_far"][triplet_pos],
    e10_triplet["ctx_sci"][triplet_pos],
], axis=0)
_ctx_disp_names, _ctx_disp_stack = _decode_cyclic_context(_ctx_names_e10, _ctx_row_stack)
_ctx_row_df = pd.DataFrame({
    "feature": _ctx_disp_names,
    "sky_near": _ctx_disp_stack[0],
    "sky_far": _ctx_disp_stack[1],
    "science": _ctx_disp_stack[2],
})
print(f"Context values at row {idx_row} (sin/cos pairs decoded to degrees):")
print(_ctx_row_df.to_string(index=False, float_format=lambda v: f'{v:.4g}'))
print()

# 3b) Moon_bs coefficient diagnostics (global prior already applied).
coef_near_row = np.asarray(e10_triplet["coef_near"][triplet_pos], dtype=np.float64)
coef_far_row = np.asarray(e10_triplet["coef_far"][triplet_pos], dtype=np.float64)
coef_sci_row = np.asarray(e10_triplet["coef_sci"][triplet_pos], dtype=np.float64)

# Per-row coef_err arrays (LSF-propagated sigma feeds off these when the
# decomposition FITS lacks a FLUX_SIGMA_TOTAL HDU).  Missing HDU -> None,
# and the downstream WRMSE degrades to the median-floor path.
def _row_coef_err(triplet_key):
    _arr = e10_triplet.get(triplet_key)
    if _arr is None:
        return None
    _row = np.asarray(_arr[triplet_pos], dtype=np.float64)
    return _row if np.any(np.isfinite(_row)) else None

coef_err_near_row = _row_coef_err("coef_err_near")
coef_err_far_row  = _row_coef_err("coef_err_far")
coef_err_sci_row  = _row_coef_err("coef_err_sci")
moon_pred = coef_pred_row[moon_idx]
moon_near = coef_near_row[moon_idx]
moon_far = coef_far_row[moon_idx]
moon_true = coef_sci_row[moon_idx]

# 4) Reconstruct this row for SCI prediction and near/far self-consistency checks.
base_dir_guess = _infer_base_dir_for_reconstruction()

# Prefer the fitted wavelength-dependent LSF surface from each decomp file's
# LSF_COEF/LSF_KNOTS/LSF_META extensions (written by sky_decomp.lsf_surface_iterative);
# fall back to the input FITS's Gaussian LSF_SCI if that isn't present.
_lsf_state_near = load_lsf_state_if_available(EVERY10_NEAR, idx_row)
_lsf_state_far  = load_lsf_state_if_available(EVERY10_FAR,  idx_row)
_lsf_state_sci  = load_lsf_state_if_available(EVERY10_SCI,  idx_row)
_lsf_sigma_fallback = lsf_row / 2.35
print(f"  LSF source per arm (row {idx_row}): "
      f"near={'surface' if _lsf_state_near is not None else 'gaussian (LSF_SCI)'}, "
      f"far={'surface' if _lsf_state_far is not None else 'gaussian (LSF_SCI)'}, "
      f"sci={'surface' if _lsf_state_sci is not None else 'gaussian (LSF_SCI)'}")

# Per-arm unit-integrated O2 templates from the decomposition FITS
# (VECTOR_O2 extension). If absent (older decomp), the O2 basis stays at
# zero -- matching pre-2026-08-10 behaviour. For the predicted-sci
# reconstruction, use the sci-arm template so the shape is anchored on the
# same layer temperature the science pointing had; the amplitude comes
# from the predicted coef['O2_b01'].
_o2_vec_near = load_o2_vector_if_available(EVERY10_NEAR, idx_row)
_o2_vec_far  = load_o2_vector_if_available(EVERY10_FAR,  idx_row)
_o2_vec_sci  = load_o2_vector_if_available(EVERY10_SCI,  idx_row)
print(f"  O2 template per arm (row {idx_row}): "
      f"near={'VECTOR_O2' if _o2_vec_near is not None else 'zero'}, "
      f"far={'VECTOR_O2' if _o2_vec_far is not None else 'zero'}, "
      f"sci={'VECTOR_O2' if _o2_vec_sci is not None else 'zero'}")

comps_sci = reconstruct_with_lsf(
    wave=wave_row,
    coef=coef_pred_row,
    lsf=_lsf_state_sci if _lsf_state_sci is not None else _lsf_sigma_fallback,
    n_spline_knots=25,
    base_dir=base_dir_guess,
    o2_vector=_o2_vec_sci,
)
comps_near_from_near = reconstruct_with_lsf(
    wave=wave_row,
    coef=coef_near_row,
    lsf=_lsf_state_near if _lsf_state_near is not None else _lsf_sigma_fallback,
    n_spline_knots=25,
    base_dir=base_dir_guess,
    o2_vector=_o2_vec_near,
    coef_err=coef_err_near_row,
)
comps_far_from_far = reconstruct_with_lsf(
    wave=wave_row,
    coef=coef_far_row,
    lsf=_lsf_state_far if _lsf_state_far is not None else _lsf_sigma_fallback,
    n_spline_knots=25,
    base_dir=base_dir_guess,
    o2_vector=_o2_vec_far,
    coef_err=coef_err_far_row,
)
# Reconstruction of the observed sci spectrum from the fitted sci coefs, so panel 3
# can separate the sky-decomposition fit residual (obs vs recon-from-sci-coef)
# from our transfer-model error (recon-from-pred vs recon-from-sci-coef).
comps_sci_true = reconstruct_with_lsf(
    wave=wave_row,
    coef=coef_sci_row,
    lsf=_lsf_state_sci if _lsf_state_sci is not None else _lsf_sigma_fallback,
    n_spline_knots=25,
    base_dir=base_dir_guess,
    o2_vector=_o2_vec_sci,
    coef_err=coef_err_sci_row,
)

flux_sci_pred_row = np.asarray(comps_sci["total"], dtype=np.float64) / FACTOR
flux_sci_true_recon_row = np.asarray(comps_sci_true["total"], dtype=np.float64) / FACTOR
flux_near_recon_row = np.asarray(comps_near_from_near["total"], dtype=np.float64) / FACTOR
flux_far_recon_row = np.asarray(comps_far_from_far["total"], dtype=np.float64) / FACTOR

# 5) Single-row metrics.
resid_row = flux_sci_pred_row - flux_sci_true_row
rmse_row = float(np.sqrt(np.mean(resid_row ** 2)))
rmse_row_display = float(rmse_row * FACTOR)
mae_row = float(np.mean(np.abs(resid_row)))
rel_resid_row = resid_row / np.where(flux_sci_true_row != 0, flux_sci_true_row, np.nan)

rmse_near_recon = float(np.sqrt(np.mean((flux_near_recon_row - flux_near_row) ** 2)))
rmse_far_recon = float(np.sqrt(np.mean((flux_far_recon_row - flux_far_row) ** 2)))

# Pixel-space WRMSE for the same three arms.  Sigma source order:
#   1. FLUX_SIGMA_TOTAL HDU in the decomposition FITS (future LSF-aware
#      propagator output);
#   2. sigma_total returned by `reconstruct_component_spectra(coef_err=...)`
#      when the corresponding COEF_ERR array is available on this row;
#   3. None -> pixel_wrmse_per_row degrades to a per-row median-floor RMSE.
def _sigma_for_single_row(fits_path, hdu_row_idx, comps_dict, comps_true_dict=None):
    """Prefer FITS HDU sigma; else use comps sigma_total (from coef_err)."""
    _fits_sigma = load_pixel_sigma_if_available(fits_path,
                                                row_indices=[int(hdu_row_idx)])
    if _fits_sigma is not None:
        return np.asarray(_fits_sigma[0], dtype=np.float64) / FACTOR
    # Fallback: use the sigma_total attached to the reconstructed comps dict.
    _sig = comps_dict.get("sigma_total") if isinstance(comps_dict, dict) else None
    if _sig is not None:
        return np.asarray(_sig, dtype=np.float64) / FACTOR
    # As a last resort, try the "true" recomposition's sigma (only meaningful
    # for the sci arm where the true decomposition sigma is more directly
    # comparable to the observation noise floor).
    _sig_alt = (comps_true_dict.get("sigma_total")
                if isinstance(comps_true_dict, dict) else None)
    if _sig_alt is not None:
        return np.asarray(_sig_alt, dtype=np.float64) / FACTOR
    return None

_sig_near_pix = _sigma_for_single_row(EVERY10_NEAR, idx_row, comps_near_from_near)
_sig_far_pix  = _sigma_for_single_row(EVERY10_FAR,  idx_row, comps_far_from_far)
_sig_sci_pix  = _sigma_for_single_row(EVERY10_SCI,  idx_row, comps_sci, comps_sci_true)

wrmse_near_recon = float(pixel_wrmse_per_row(
    flux_near_recon_row, flux_near_row, _sig_near_pix)[0])
wrmse_far_recon  = float(pixel_wrmse_per_row(
    flux_far_recon_row,  flux_far_row,  _sig_far_pix)[0])
wrmse_row_pix    = float(pixel_wrmse_per_row(
    flux_sci_pred_row,   flux_sci_true_row, _sig_sci_pix)[0])

print("Single-row reconstruction summary (every10, default coefficients)")
print(f"  row index (input file) = {idx_row}")
print(f"  row index (triplet pos) = {triplet_pos}")
print(f"  n_wave     = {n_wave}")
print("  predictor  = deep group-head MLP")
print(
    "  Moon_bs roughness: "
    f"pred={_row_spline_roughness(moon_pred):.4g}, "
    f"near={_row_spline_roughness(moon_near):.4g}, "
    f"far={_row_spline_roughness(moon_far):.4g}, "
    f"sci_true={_row_spline_roughness(moon_true):.4g}"
)
print(f"  near self-recon pRMSE  = {rmse_near_recon:.6g}")
print(f"  near self-recon pWRMSE = {wrmse_near_recon:.6g}")
print(f"  far  self-recon pRMSE  = {rmse_far_recon:.6g}")
print(f"  far  self-recon pWRMSE = {wrmse_far_recon:.6g}")
print(f"  sci row pRMSE          = {rmse_row:.6g}")
print(f"  sci row pWRMSE         = {wrmse_row_pix:.6g}")
print(f"  sci row pRMSE (x{FACTOR:.3g} display units) = {rmse_row_display:.6g}")
print(f"  sci row MAE            = {mae_row:.6g}")

# 6) Four-panel diagnostic plot:
#    row1: near observed vs reconstruction from near coefficients
#    row2: far observed vs reconstruction from far coefficients
#    row3: science true vs science prediction
#    row4: science relative residual
fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    subplot_titles=(
        "Near: observed vs reconstructed from near coefficients",
        "Far: observed vs reconstructed from far coefficients",
        "Science: observed / recon(sci coef) / recon(pred)",
        "Science residual: (pred - true) / true",
    ),
    row_heights=[0.24, 0.24, 0.34, 0.18],
)

fig.add_trace(
    go.Scattergl(
        x=wave_row,
        y=flux_near_row * FACTOR,
        mode="lines",
        name="near true",
        line=dict(color="#7f7f7f", width=1.0),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scattergl(
        x=wave_row,
        y=flux_near_recon_row * FACTOR,
        mode="lines",
        name="near recon(from near coef)",
        line=dict(color="#e41a1c", width=1.4),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scattergl(
        x=wave_row,
        y=flux_far_row * FACTOR,
        mode="lines",
        name="far true",
        line=dict(color="#7f7f7f", width=1.0),
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scattergl(
        x=wave_row,
        y=flux_far_recon_row * FACTOR,
        mode="lines",
        name="far recon(from far coef)",
        line=dict(color="#ff7f00", width=1.4),
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scattergl(
        x=wave_row,
        y=flux_sci_true_row * FACTOR,
        mode="lines",
        name="science observed",
        line=dict(color="#7f7f7f", width=1.0),
    ),
    row=3,
    col=1,
)
fig.add_trace(
    go.Scattergl(
        x=wave_row,
        y=flux_sci_true_recon_row * FACTOR,
        mode="lines",
        name="science recon(from sci coef)",
        line=dict(color="#2ca02c", width=1.2, dash="dash"),
    ),
    row=3,
    col=1,
)
fig.add_trace(
    go.Scattergl(
        x=wave_row,
        y=flux_sci_pred_row * FACTOR,
        mode="lines",
        name="science recon(pred)",
        line=dict(color="#1f78b4", width=1.4),
    ),
    row=3,
    col=1,
)

fig.add_trace(
    go.Scattergl(
        x=wave_row,
        y=rel_resid_row,
        mode="lines",
        name="science residual",
        line=dict(color="#d62728", width=1.0),
    ),
    row=4,
    col=1,
)
fig.add_hline(y=0, line=dict(color="black", width=0.8, dash="dash"), row=4, col=1)

fig.update_yaxes(type="log", title_text="Near flux", row=1, col=1)
fig.update_yaxes(type="log", title_text="Far flux", row=2, col=1)
fig.update_yaxes(type="log", title_text="Science flux", row=3, col=1)
fig.update_yaxes(type="linear", title_text="(pred-true)/true", row=4, col=1)
fig.update_xaxes(title_text="Wavelength [A]", row=4, col=1)

fig.update_layout(
    template="plotly_white",
    title=(
        f"Every10 row {idx_row} | near pRMSE={rmse_near_recon:.3g} "
        f"(pWRMSE={wrmse_near_recon:.3g}), far pRMSE={rmse_far_recon:.3g} "
        f"(pWRMSE={wrmse_far_recon:.3g}), sci pRMSE={rmse_row:.3g} "
        f"(pWRMSE={wrmse_row_pix:.3g}, disp={rmse_row_display:.3g})"
    ),
    height=1160,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
)
fig.show()

# 7) Moon spline coefficient diagnostic figure (global-prior result).
moon_axis = np.arange(moon_idx.size)
fig_moon = go.Figure()
fig_moon.add_trace(
    go.Scatter(
        x=moon_axis,
        y=moon_near,
        mode="lines+markers",
        name="near",
        line=dict(color="#7f7f7f"),
    )
)
fig_moon.add_trace(
    go.Scatter(
        x=moon_axis,
        y=moon_far,
        mode="lines+markers",
        name="far",
        line=dict(color="#bdbdbd"),
    )
)
fig_moon.add_trace(
    go.Scatter(
        x=moon_axis,
        y=moon_true,
        mode="lines+markers",
        name="sci true",
        line=dict(color="#1f78b4"),
    )
)
fig_moon.add_trace(
    go.Scatter(
        x=moon_axis,
        y=moon_pred,
        mode="lines+markers",
        name="pred default",
        line=dict(color="#e41a1c"),
    )
)
fig_moon.update_layout(
    template="plotly_white",
    title="Moon spline coefficients (base prediction) for selected row",
    xaxis_title="Moon_bs coefficient index",
    yaxis_title="coefficient value",
    height=420,
)
fig_moon.show()

# 8) Per-component reconstructions for all four arms, mirroring the
#    four-trace layout of the Moon spline diagnostic above but as
#    spectra over wavelength rather than coefficients over index.
#    comps_sci_true was reconstructed in section 4 so row 3 can use it.
_comps_by_arm = {
    "near": comps_near_from_near,
    "far": comps_far_from_far,
    "sci true": comps_sci_true,
    "pred default": comps_sci,
}
_arm_colors = {
    "near": "#7f7f7f",
    "far": "#bdbdbd",
    "sci true": "#1f78b4",
    "pred default": "#e41a1c",
}

# comps["*"] is in the same display scale as comps["total"], so no *FACTOR here.
def _non_moon_continuum(comps):
    return np.asarray(comps["diffuse"], dtype=np.float64)

def _line_component(comps):
    return (np.asarray(comps["oh"], dtype=np.float64)
            + np.asarray(comps["atom"], dtype=np.float64)
            + np.asarray(comps["orc"], dtype=np.float64)
            + np.asarray(comps["o2"], dtype=np.float64))

def _moon_spectrum(comps):
    return np.asarray(comps["moon"], dtype=np.float64)

# Sanity: total is defined by reconstruct_component_spectra as
#   oh + moon + diffuse + atom + orc + o2
# so lines + moon_spectrum + non_moon_continuum must equal it.
_total_pred = np.asarray(comps_sci["total"], dtype=np.float64)
_sum_pred = (_line_component(comps_sci)
             + _moon_spectrum(comps_sci)
             + _non_moon_continuum(comps_sci))
_max_diff = float(np.nanmax(np.abs(_total_pred - _sum_pred)))
_max_rel = float(np.nanmax(np.abs(_total_pred - _sum_pred)
                          / np.clip(np.abs(_total_pred), 1e-30, None)))
print(f"Component-sum check (pred): max abs diff = {_max_diff:.3g}, "
      f"max rel diff = {_max_rel:.3g}")

fig_continuum = go.Figure()
for _arm, _comps in _comps_by_arm.items():
    fig_continuum.add_trace(
        go.Scattergl(
            x=wave_row,
            y=_non_moon_continuum(_comps),
            mode="lines",
            name=_arm,
            line=dict(color=_arm_colors[_arm], width=1.2),
        )
    )
fig_continuum.update_layout(
    template="plotly_white",
    title="Reconstructed non-moon continuum (diffuse = HO2 + FeO + O2ac) for selected row",
    xaxis_title="Wavelength [A]",
    yaxis_title=f"Flux (display units x{FACTOR:.3g})",
    height=420,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
)
fig_continuum.show()

fig_moon_spectrum = go.Figure()
for _arm, _comps in _comps_by_arm.items():
    fig_moon_spectrum.add_trace(
        go.Scattergl(
            x=wave_row,
            y=_moon_spectrum(_comps),
            mode="lines",
            name=_arm,
            line=dict(color=_arm_colors[_arm], width=1.2),
        )
    )
fig_moon_spectrum.update_layout(
    template="plotly_white",
    title="Reconstructed moon spline spectrum (comps['moon']) for selected row",
    xaxis_title="Wavelength [A]",
    yaxis_title=f"Flux (display units x{FACTOR:.3g})",
    height=420,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
)
fig_moon_spectrum.show()

fig_lines = go.Figure()
for _arm, _comps in _comps_by_arm.items():
    fig_lines.add_trace(
        go.Scattergl(
            x=wave_row,
            y=_line_component(_comps),
            mode="lines",
            name=_arm,
            line=dict(color=_arm_colors[_arm], width=1.2),
        )
    )
fig_lines.update_layout(
    template="plotly_white",
    title="Reconstructed line emission (OH + atom + ORC + O2) for selected row",
    xaxis_title="Wavelength [A]",
    yaxis_title=f"Flux (display units x{FACTOR:.3g})",
    height=460,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
)
fig_lines.show()

# 9) Per-component (pred - sci_true) residual spectrum. Row 3 shows only
#    the total residual; this decomposes it into moon / diffuse / lines
#    so a small broadband deficit in a component that spans the whole
#    wavelength range is visible even when the component panels above
#    make it look "close" on a linear-y axis. By construction the three
#    traces must sum to the blue-minus-green curve of row 3, and that
#    sum is drawn as a black dashed reference.
_delta_moon = _moon_spectrum(comps_sci) - _moon_spectrum(comps_sci_true)
_delta_diffuse = _non_moon_continuum(comps_sci) - _non_moon_continuum(comps_sci_true)
_delta_lines = _line_component(comps_sci) - _line_component(comps_sci_true)
_delta_total = _delta_moon + _delta_diffuse + _delta_lines

fig_deltas = go.Figure()
for _label, _y, _color in (
    ("moon (pred - sci recon)", _delta_moon, "#e41a1c"),
    ("diffuse (pred - sci recon)", _delta_diffuse, "#377eb8"),
    ("lines (pred - sci recon)", _delta_lines, "#4daf4a"),
    ("total (pred - sci recon)", _delta_total, "#000000"),
):
    fig_deltas.add_trace(
        go.Scattergl(
            x=wave_row,
            y=_y,
            mode="lines",
            name=_label,
            line=dict(color=_color, width=1.2,
                      dash="dash" if _label.startswith("total") else "solid"),
        )
    )
fig_deltas.add_hline(y=0, line=dict(color="rgba(0,0,0,0.4)", width=0.8, dash="dot"))
fig_deltas.update_layout(
    template="plotly_white",
    title="Per-component prediction minus sci-arm reconstruction (linear)",
    xaxis_title="Wavelength [A]",
    yaxis_title=f"Delta flux (display units x{FACTOR:.3g})",
    height=460,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
)
fig_deltas.show()

# 10) Numeric integrated deltas per component in three wavelength bands, so
#     the sign and magnitude of each contribution to the red deficit is
#     visible even where the plot traces are noisy line-by-line.
_bands = [
    ("blue  (< 5500 A)", wave_row < 5500.0),
    ("green (5500-7500)", (wave_row >= 5500.0) & (wave_row < 7500.0)),
    ("red   (>= 7500 A)", wave_row >= 7500.0),
]
print()
print("Integrated (pred - sci recon) per component and wavelength band:")
print(f"  {'band':<18s} {'moon':>12s} {'diffuse':>12s} {'lines':>12s} {'total':>12s}")
for _bname, _mask in _bands:
    if not _mask.any():
        continue
    _sm = float(np.nansum(_delta_moon[_mask]))
    _sd = float(np.nansum(_delta_diffuse[_mask]))
    _sl = float(np.nansum(_delta_lines[_mask]))
    _st = float(np.nansum(_delta_total[_mask]))
    print(f"  {_bname:<18s} {_sm:>+12.4g} {_sd:>+12.4g} {_sl:>+12.4g} {_st:>+12.4g}")



In [ ]:
# Batch RMSE stats on a random subset of every10 rows (same inputs as Cell 17)
import numpy as np
import pandas as pd
import plotly.graph_objects as go

RUN_RMSE_SUBSET_EVAL = True  # Set True to execute this slower evaluation cell.

required = [
    "mlp_artifacts",
    "predict_sci_coefficients_default",
    "context_cols",
    "build_triplet_coef_dataset",
    "reconstruct_component_spectra",
    "reconstruct_with_lsf",
    "load_lsf_state_if_available",
    "load_o2_vector_if_available",
    "_infer_base_dir_for_reconstruction",
    "FACTOR",
]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError("Run the training + residual-correction cells first. Missing: " + ", ".join(missing))

if not RUN_RMSE_SUBSET_EVAL:
    print("Cell 19 skipped. Set RUN_RMSE_SUBSET_EVAL = True to run the random-subset RMSE evaluation.")
else:
    # Use the same file inputs as the reconstruction diagnostic cell.
    _e10_stem = "spline_moon/lvmsframe_median_stack_1.2.1_p40_p70_every10"
    _e10_suffix = _DECOMP_SUFFIX  # inherited from cell 6
    EVERY10_INPUT = f"{_e10_stem}.fits"
    EVERY10_NEAR = f"{_e10_stem}_decomp_sky1{_e10_suffix}.fits"
    EVERY10_FAR = f"{_e10_stem}_decomp_sky2{_e10_suffix}.fits"
    EVERY10_SCI = f"{_e10_stem}_decomp_sci{_e10_suffix}.fits"

    n_sample = 100
    rng_seed = 42

    # 1) Build aligned triplet rows, keeping chi2 so we can apply the same
    #    quality gates the training set went through.
    e10_triplet = build_triplet_coef_dataset(
        input_fits_path=EVERY10_INPUT,
        sky_near_decomp_fits_path=EVERY10_NEAR,
        sky_far_decomp_fits_path=EVERY10_FAR,
        sci_decomp_fits_path=EVERY10_SCI,
        context_columns=context_cols,
        return_chi2=True,
    )
    # ECLIPTIC-CTX-V1: match training-time ctx layout on the e10 triplet.
    if '_augment_triplet_with_ecliptic' in globals():
        _augment_triplet_with_ecliptic(e10_triplet, meta_fits_path=EVERY10_INPUT)

    row_index_e10 = np.asarray(e10_triplet["row_index"], dtype=np.int64)
    _e10_n0 = int(e10_triplet["n_rows"])
    if _e10_n0 == 0:
        raise RuntimeError("No aligned rows available in e10_triplet")

    # 1a) Apply chi2 gating and LMC/SMC field exclusion BEFORE the random
    #     subsample, so the reconstructions we score are drawn from tiles
    #     that pass the same data-quality gates the training set went
    #     through.  Hard coefficient bounds and kappa-sigma clipping are
    #     intentionally NOT applied here -- those are training-time filters
    #     on the target that would remove the model's hardest true cases
    #     from the evaluation set.
    _e10_keep = np.ones(_e10_n0, dtype=bool)

    if "sci_ra" in e10_triplet and "sci_dec" in e10_triplet:
        _sci_ra = np.asarray(e10_triplet["sci_ra"], dtype=np.float64)
        _sci_dec = np.asarray(e10_triplet["sci_dec"], dtype=np.float64)
        _field_mask = np.ones(_e10_n0, dtype=bool)
        for _region in (LMC_EXCLUSION, SMC_EXCLUSION):
            _sep = _angular_separation_deg_vec(
                _sci_ra, _sci_dec,
                _region["ra_deg"], _region["dec_deg"])
            _inside = np.isfinite(_sep) & (_sep <= float(_region["radius_deg"]))
            print(
                f"  every10 field exclusion around {_region['name']}: "
                f"excluded {int(_inside.sum())}/{_e10_n0}"
            )
            _field_mask &= ~_inside
        _e10_keep &= _field_mask
    else:
        print("  every10 field exclusion skipped: no sci_ra/sci_dec in triplet.")

    if all(_k in e10_triplet for _k in ("chi2_near", "chi2_far", "chi2_sci")):
        # 2026-08-19: use max (not nanmax) so any-arm-NaN chi2 propagates
        # NaN and disqualifies the whole observation via the isfinite gate.
        _chi2_combined = np.max(
            np.column_stack([
                np.asarray(e10_triplet["chi2_near"], dtype=np.float64),
                np.asarray(e10_triplet["chi2_far"], dtype=np.float64),
                np.asarray(e10_triplet["chi2_sci"], dtype=np.float64),
            ]),
            axis=1,
        )
        _chi2_finite = _chi2_combined[np.isfinite(_chi2_combined)]
        _chi2_hi = (float(np.nanpercentile(_chi2_finite, 90.0))
                    if _chi2_finite.size else np.inf)
        _chi2_upper = min(10.0, _chi2_hi)
        _chi2_mask = (np.isfinite(_chi2_combined)
                      & (_chi2_combined >= 0.0)
                      & (_chi2_combined <= _chi2_upper))
        print(
            f"  every10 chi2 filter (qmax=90% -> {_chi2_hi:.3g}, "
            f"upper={_chi2_upper:.3g}): "
            f"kept {int(_chi2_mask.sum())}/{_e10_n0}"
        )
        _e10_keep &= _chi2_mask
    else:
        print("  every10 chi2 filter skipped: chi2 columns not in triplet.")

    _e10_valid_pos = np.flatnonzero(_e10_keep)
    n_rows = int(_e10_valid_pos.size)
    print(
        f"  every10 rows passing chi2 + field exclusion: "
        f"{n_rows}/{_e10_n0} ({100.0 * n_rows / max(_e10_n0, 1):.1f}%)"
    )
    if n_rows == 0:
        raise RuntimeError(
            "No aligned rows survive chi2/field filtering; relax thresholds.")

    n_use = int(min(n_sample, n_rows))
    rng = np.random.default_rng(rng_seed)

    # Stratify the draw by lunar phase so the diagnostic set spans dark -> bright
    # roughly uniformly (matches split_indices_by_moon_phase in §8.1). A plain
    # rng.choice over _e10_valid_pos would inherit the phase distribution of the
    # dataset -- weighted toward whichever quantiles happen to hold more filtered
    # rows -- and the sci_pred_vs_true RMSE would then be dominated by that
    # region rather than being representative of deployment conditions.
    _e10_moon_phase = _moon_phase_deg_from_ctx(e10_triplet)
    _valid_phase = _e10_moon_phase[_e10_valid_pos]
    if not np.isfinite(_valid_phase).all():
        raise RuntimeError('Non-finite moon_phase in the valid every10 rows; '
                           'cannot stratify by lunar phase.')

    _n_phase_bins = int(min(10, n_use))
    _phase_edges = np.quantile(_valid_phase,
                               np.linspace(0.0, 1.0, _n_phase_bins + 1))
    _phase_edges[0], _phase_edges[-1] = -np.inf, np.inf
    _bin_id = np.digitize(_valid_phase, _phase_edges[1:-1], right=False)

    # Round-robin quota with the +1s scattered randomly so no bin is systematically favored.
    _quota = np.full(_n_phase_bins, n_use // _n_phase_bins, dtype=int)
    _quota[:n_use - int(_quota.sum())] += 1
    rng.shuffle(_quota)

    _picked = []
    for _b in range(_n_phase_bins):
        _in_bin = _e10_valid_pos[_bin_id == _b]
        _take = int(min(_quota[_b], _in_bin.size))
        if _take > 0:
            _picked.append(rng.choice(_in_bin, size=_take, replace=False))
    _selected = (np.concatenate(_picked).astype(int)
                 if _picked else np.array([], dtype=int))

    # If any bin was smaller than its quota, backfill from the remaining pool.
    _shortfall = n_use - _selected.size
    if _shortfall > 0:
        _remaining = np.setdiff1d(_e10_valid_pos, _selected, assume_unique=False)
        if _remaining.size >= _shortfall:
            _selected = np.concatenate(
                [_selected, rng.choice(_remaining, size=_shortfall, replace=False)])

    sel_pos = np.sort(_selected)
    sel_rows = row_index_e10[sel_pos]

    _sel_phases = _e10_moon_phase[sel_pos]
    print(f"  phase-stratified sample: n_use={n_use} across {_n_phase_bins} "
          f"quantile bins; phase deg quartiles (min / 25 / 50 / 75 / max) = "
          f"{float(np.min(_sel_phases)):.1f} / "
          f"{float(np.percentile(_sel_phases, 25)):.1f} / "
          f"{float(np.percentile(_sel_phases, 50)):.1f} / "
          f"{float(np.percentile(_sel_phases, 75)):.1f} / "
          f"{float(np.max(_sel_phases)):.1f}")

    # 2) Load observed spectra, wavelength grid, and LSF from the same input file.
    with fits.open(EVERY10_INPUT) as hdul:
        flux_near_all = np.asarray(hdul["FLUX_SKY_NEAR"].data, dtype=np.float64)
        flux_far_all = np.asarray(hdul["FLUX_SKY_FAR"].data, dtype=np.float64)
        flux_sci_all = np.asarray(hdul["FLUX_SCI"].data, dtype=np.float64)
        wave_arr = np.asarray(hdul["WAVE"].data, dtype=np.float64)
        lsf_sci_arr = np.asarray(hdul["LSF_SCI"].data, dtype=np.float64)
        # expnum per every10 row for the per-line hover tooltip on the residual figure.
        _expnum_all = None
        if "META" in hdul:
            _meta_e10 = Table(hdul["META"].data)
            _meta_up_e10 = {c.upper(): c for c in _meta_e10.colnames}
            _expnum_col = next(
                (_meta_up_e10[k] for k in ("EXPNUM", "EXP_NUM", "EXPOSURE")
                 if k in _meta_up_e10), None)
            if _expnum_col is not None:
                _expnum_all = np.asarray(_meta_e10[_expnum_col])

    n_spec = int(flux_sci_all.shape[0])
    if np.any(sel_rows < 0) or np.any(sel_rows >= n_spec):
        raise ValueError("Selected row index is outside the valid range of EVERY10_INPUT")

    # 3) Predict SCI coefficients for sampled rows.
    coef_sci_pred = predict_sci_coefficients_default(
        mlp_artifacts,
        coef_near_phys=e10_triplet["coef_near"][sel_pos],
        coef_far_phys=e10_triplet["coef_far"][sel_pos],
        ctx_near_phys=e10_triplet["ctx_near"][sel_pos],
        ctx_far_phys=e10_triplet["ctx_far"][sel_pos],
        ctx_sci_phys=e10_triplet["ctx_sci"][sel_pos],
    ).astype(np.float64)

    coef_near_sel = np.asarray(e10_triplet["coef_near"][sel_pos], dtype=np.float64)
    coef_far_sel = np.asarray(e10_triplet["coef_far"][sel_pos], dtype=np.float64)
    coef_sci_sel = np.asarray(e10_triplet["coef_sci"][sel_pos], dtype=np.float64)

    # 4) Reconstruct and compute per-row RMSE + pixel-space WRMSE.
    base_dir_guess = _infer_base_dir_for_reconstruction()
    near_rmse = np.full(n_use, np.nan, dtype=np.float64)
    far_rmse = np.full(n_use, np.nan, dtype=np.float64)
    sci_rmse = np.full(n_use, np.nan, dtype=np.float64)
    near_wrmse = np.full(n_use, np.nan, dtype=np.float64)
    far_wrmse = np.full(n_use, np.nan, dtype=np.float64)
    sci_wrmse = np.full(n_use, np.nan, dtype=np.float64)

    # Try to load per-pixel sigma HDUs from the decomposition FITS files
    # up-front (fast path when new decompositions land).  Falls back to
    # on-the-fly propagation via coef_err inside the loop when absent.
    _pix_sigma_near_all = load_pixel_sigma_if_available(EVERY10_NEAR)
    _pix_sigma_far_all  = load_pixel_sigma_if_available(EVERY10_FAR)
    _pix_sigma_sci_all  = load_pixel_sigma_if_available(EVERY10_SCI)
    _pix_sigma_source = {
        arm: ("FITS HDU" if arr is not None else "coef_err propagation")
        for arm, arr in (("near", _pix_sigma_near_all),
                         ("far",  _pix_sigma_far_all),
                         ("sci",  _pix_sigma_sci_all))
    }
    print(f"  pixel sigma source: near={_pix_sigma_source['near']}, "
          f"far={_pix_sigma_source['far']}, sci={_pix_sigma_source['sci']}")

    # Grab coef_err arrays for the fallback path (may be all-NaN when the
    # decomposition FITS lacks a COEF_ERR HDU; the WRMSE helper falls back
    # to floor-only weighting so nothing breaks).
    _e10_coef_err_near = (np.asarray(e10_triplet.get("coef_err_near",
                                     np.full_like(coef_near_sel, np.nan)),
                                     dtype=np.float64)[sel_pos]
                          if _pix_sigma_near_all is None else None)
    _e10_coef_err_far  = (np.asarray(e10_triplet.get("coef_err_far",
                                     np.full_like(coef_far_sel, np.nan)),
                                     dtype=np.float64)[sel_pos]
                          if _pix_sigma_far_all is None else None)
    _e10_coef_err_sci  = (np.asarray(e10_triplet.get("coef_err_sci",
                                     np.full_like(coef_sci_pred, np.nan)),
                                     dtype=np.float64)[sel_pos]
                          if _pix_sigma_sci_all is None else None)
    sci_resid_rows = []
    sci_wave_rows = []
    # Per-component residuals: comps_sci[<comp>] - comps_sci_true[<comp>] per row,
    # stored in native units (like sci_resid_rows) and multiplied by FACTOR at plot time.
    sci_moon_resid_rows = []
    sci_diffuse_resid_rows = []
    sci_lines_resid_rows = []

    def _lines_sum(_c):
        return (np.asarray(_c["oh"], dtype=np.float64)
                + np.asarray(_c["atom"], dtype=np.float64)
                + np.asarray(_c["orc"], dtype=np.float64)
                + np.asarray(_c["o2"], dtype=np.float64))

    # 4a) Optimization (2026-08-11): build ONE reconstruction model outside the loop and
    #     precache per-file LSF-availability + full VECTOR_O2 cubes. Previously the loop
    #     rebuilt SkyDecompLSFSurfaceIterative (basis + solar-reference + moon spline)
    #     3 x n_use times and re-read VECTOR_O2 on every call, both of which dominated
    #     the runtime. See §12 (2026-08-11 batch-RMSE cell reconstruction hoisted).
    import time as _time
    _t_recon0 = _time.perf_counter()
    _wave_ref_recon = (wave_arr if wave_arr.ndim == 1
                       else np.asarray(wave_arr[int(sel_rows[0])], dtype=np.float64))
    if wave_arr.ndim > 1:
        _wave_probe = np.asarray(wave_arr[int(sel_rows[-1])], dtype=np.float64)
        if _wave_probe.shape != _wave_ref_recon.shape or not np.allclose(
                _wave_probe, _wave_ref_recon, rtol=0.0, atol=1e-8):
            raise RuntimeError(
                "wave_arr rows differ between sampled rows; model hoisting assumes a shared grid. "
                "Fall back to per-row reconstruct_with_lsf if this ever triggers on your dataset.")
    _lsf_model = SkyDecompLSFSurfaceIterative(
        _wave_ref_recon, lsf_sigma=1.0, n_spline_knots=25, base_dir=base_dir_guess,
    )

    def _precache_decomp_state(decomp_path):
        state = {"path": Path(decomp_path), "has_lsf": False, "o2_cube": None}
        if not state["path"].exists():
            return state
        try:
            with fits.open(str(state["path"]), memmap=False) as _hdul_dec:
                _ext_names = {h.name for h in _hdul_dec}
                state["has_lsf"] = all(_e in _ext_names
                                       for _e in ("LSF_COEF", "LSF_KNOTS", "LSF_META"))
                if state["has_lsf"]:
                    # Refuse to open an inconsistent LSF cube (observed on the
                    # _p25_every10 decomp files, where LSF_COEF was rewritten
                    # to every10 size but LSF_META was inherited from every1).
                    # Loading anyway would slice cube row k with an n_basis
                    # taken from META row k -- a different fiber -- silently
                    # applying the wrong LSF to ~99% of rows and hitting NaN
                    # padding on the rest.
                    _coef_rows = int(_hdul_dec["LSF_COEF"].data.shape[0])
                    _meta_rows = int(len(_hdul_dec["LSF_META"].data))
                    _expected_meta = 3 * _coef_rows
                    if _meta_rows != _expected_meta:
                        raise RuntimeError(
                            f"Inconsistent LSF HDUs in {state['path'].name}: "
                            f"LSF_COEF has {_coef_rows} rows but LSF_META has "
                            f"{_meta_rows} rows (expected 3 x {_coef_rows} = "
                            f"{_expected_meta}). Regenerate this decomposition "
                            f"file with matching LSF_META; do NOT fall back to "
                            f"LSF_SCI sigma because the per-row LSF would be "
                            f"silently wrong on all other rows."
                        )
                if "VECTOR_O2" in _ext_names:
                    _data = np.asarray(_hdul_dec["VECTOR_O2"].data, dtype=np.float64)
                    if _data.ndim == 2:
                        state["o2_cube"] = _data
        except (KeyError, IndexError, ValueError) as _exc:
            print(f"  precache failed for {state['path'].name}: "
                  f"{type(_exc).__name__}: {_exc}")
        return state

    _state_near = _precache_decomp_state(EVERY10_NEAR)
    _state_far  = _precache_decomp_state(EVERY10_FAR)
    _state_sci  = _precache_decomp_state(EVERY10_SCI)
    print(f"  precache: has_lsf near/far/sci = "
          f"{_state_near['has_lsf']}/{_state_far['has_lsf']}/{_state_sci['has_lsf']}, "
          f"VECTOR_O2 near/far/sci = "
          f"{_state_near['o2_cube'] is not None}/"
          f"{_state_far['o2_cube'] is not None}/"
          f"{_state_sci['o2_cube'] is not None}")

    def _lsf_state_from_cache(state_dict, row_idx):
        if not state_dict["has_lsf"]:
            return None
        try:
            return load_lsf_surface_state(str(state_dict["path"]), int(row_idx))
        except (KeyError, IndexError, ValueError) as _exc:
            print(f"  LSF surface state unavailable in {state_dict['path'].name} "
                  f"row {int(row_idx)}: {type(_exc).__name__}: {_exc}")
            return None

    def _o2_vec_from_cache(state_dict, row_idx):
        cube = state_dict["o2_cube"]
        if cube is None or int(row_idx) >= cube.shape[0]:
            return None
        row = cube[int(row_idx)]
        if not np.isfinite(row).any() or float(np.nansum(np.abs(row))) == 0.0:
            return None
        return row

    def _fast_reconstruct(coef, lsf_state, o2_vec, lsf_sigma_fallback, coef_err=None):
        if isinstance(lsf_state, LSFSurfaceState):
            _lsf_model._set_lsf_state(lsf_state)
            _mats = _lsf_model._assemble_refined_matrices()
            if o2_vec is not None:
                _o2_arr = np.asarray(o2_vec, float).ravel()
                if _o2_arr.shape != _lsf_model.wave.shape:
                    raise ValueError(
                        f"o2_vector shape mismatch: expected {_lsf_model.wave.shape}, "
                        f"got {_o2_arr.shape}")
                _mats["o2"] = _o2_arr[None, :]
            _coef_arr = np.asarray(coef, float).ravel()
            _comps = _lsf_model._components_from_coef(_coef_arr, _mats)
            _comps["total"] = (_comps["oh"] + _comps["moon"] + _comps["diffuse"]
                                + _comps["atom"] + _comps["orc"] + _comps["o2"])
            if coef_err is not None:
                _err_arr = np.asarray(coef_err, float).ravel()
                _sigmas = _lsf_model._components_sigma_from_coef_err(_err_arr, _mats)
                _comps["sigma"] = _sigmas
                _comps["sigma_total"] = np.sqrt(
                    _sigmas["oh"] ** 2 + _sigmas["moon"] ** 2
                    + _sigmas["diffuse"] ** 2 + _sigmas["atom"] ** 2
                    + _sigmas["orc"] ** 2 + _sigmas["o2"] ** 2)
            return _comps
        # Rare fallback path: no LSF surface state for this row -- take the slow route.
        return reconstruct_with_lsf(
            wave=_lsf_model.wave, coef=coef, lsf=lsf_sigma_fallback,
            n_spline_knots=25, base_dir=base_dir_guess, o2_vector=o2_vec,
            coef_err=coef_err)

    print(f"  recon setup: {_time.perf_counter() - _t_recon0:.2f} s "
          f"(one-time basis build + FITS precache)")

    _t_loop0 = _time.perf_counter()
    for i, r in enumerate(sel_rows):
        rr = int(r)
        wave_row = wave_arr if wave_arr.ndim == 1 else np.asarray(wave_arr[rr], dtype=np.float64)
        lsf_row = lsf_sci_arr if lsf_sci_arr.ndim == 1 else np.asarray(lsf_sci_arr[rr], dtype=np.float64)

        flux_near_true = np.asarray(flux_near_all[rr], dtype=np.float64)
        flux_far_true = np.asarray(flux_far_all[rr], dtype=np.float64)
        flux_sci_true = np.asarray(flux_sci_all[rr], dtype=np.float64)

        _lsf_state_near = _lsf_state_from_cache(_state_near, rr)
        _lsf_state_far  = _lsf_state_from_cache(_state_far,  rr)
        _lsf_state_sci  = _lsf_state_from_cache(_state_sci,  rr)
        _lsf_sigma_fallback = lsf_row / 2.35

        _o2_vec_near = _o2_vec_from_cache(_state_near, rr)
        _o2_vec_far  = _o2_vec_from_cache(_state_far,  rr)
        _o2_vec_sci  = _o2_vec_from_cache(_state_sci,  rr)

        # For each arm, only ask the reconstructor for sigma when the
        # loaded FITS sigma is absent; otherwise the propagator call would
        # duplicate work already done by the pipeline.
        _cerr_near_row = (_e10_coef_err_near[i]
                          if _e10_coef_err_near is not None else None)
        _cerr_far_row  = (_e10_coef_err_far[i]
                          if _e10_coef_err_far is not None else None)
        _cerr_sci_row  = (_e10_coef_err_sci[i]
                          if _e10_coef_err_sci is not None else None)

        comps_near = _fast_reconstruct(coef_near_sel[i], _lsf_state_near,
                                        _o2_vec_near, _lsf_sigma_fallback,
                                        coef_err=_cerr_near_row)
        comps_far  = _fast_reconstruct(coef_far_sel[i],  _lsf_state_far,
                                        _o2_vec_far,  _lsf_sigma_fallback,
                                        coef_err=_cerr_far_row)
        comps_sci  = _fast_reconstruct(coef_sci_pred[i], _lsf_state_sci,
                                        _o2_vec_sci,  _lsf_sigma_fallback,
                                        coef_err=_cerr_sci_row)

        flux_near_recon = np.asarray(comps_near["total"], dtype=np.float64) / FACTOR
        flux_far_recon = np.asarray(comps_far["total"], dtype=np.float64) / FACTOR
        flux_sci_pred = np.asarray(comps_sci["total"], dtype=np.float64) / FACTOR

        # nanmean so isolated NaN pixels (~1 pixel/row on ~40% of every10) don't
        # poison the pRMSE; a whole-row veto lives with the chi2/field filter above.
        near_rmse[i] = float(np.sqrt(np.nanmean((flux_near_recon - flux_near_true) ** 2)))
        far_rmse[i] = float(np.sqrt(np.nanmean((flux_far_recon - flux_far_true) ** 2)))

        sci_resid = flux_sci_pred - flux_sci_true
        sci_rmse[i] = float(np.sqrt(np.nanmean(sci_resid ** 2)))
        sci_resid_rows.append(np.asarray(sci_resid, dtype=np.float64))
        sci_wave_rows.append(np.asarray(wave_row, dtype=np.float64))

        # Reconstruct the sci-arm spectrum from the FITTED sci coefficients so
        # per-component residuals (pred - recon(sci_true)) can be separated in
        # the multi-panel residual plot below (matches cell 26 fig_deltas).
        comps_sci_true_batch = _fast_reconstruct(
            coef_sci_sel[i], _lsf_state_sci, _o2_vec_sci, _lsf_sigma_fallback,
            coef_err=None,
        )
        _dmoon    = (np.asarray(comps_sci["moon"], dtype=np.float64)
                     - np.asarray(comps_sci_true_batch["moon"], dtype=np.float64)) / FACTOR
        _ddiffuse = (np.asarray(comps_sci["diffuse"], dtype=np.float64)
                     - np.asarray(comps_sci_true_batch["diffuse"], dtype=np.float64)) / FACTOR
        _dlines   = (_lines_sum(comps_sci) - _lines_sum(comps_sci_true_batch)) / FACTOR
        sci_moon_resid_rows.append(_dmoon)
        sci_diffuse_resid_rows.append(_ddiffuse)
        sci_lines_resid_rows.append(_dlines)

        # Pixel-space WRMSE: prefer the FITS-side sigma if present, else use
        # the propagator output from _fast_reconstruct.  All three sources
        # deliver the same LSF-aware sigma; the fallback of last resort is
        # the median-floor path inside pixel_wrmse_per_row.
        _sig_near_row = (_pix_sigma_near_all[int(sel_pos[i])] * FACTOR
                         if _pix_sigma_near_all is not None
                         else comps_near.get("sigma_total"))
        _sig_far_row  = (_pix_sigma_far_all[int(sel_pos[i])] * FACTOR
                         if _pix_sigma_far_all is not None
                         else comps_far.get("sigma_total"))
        _sig_sci_row  = (_pix_sigma_sci_all[int(sel_pos[i])] * FACTOR
                         if _pix_sigma_sci_all is not None
                         else comps_sci.get("sigma_total"))
        # comps_*['sigma_total'] comes out in native units; the flux_* arrays
        # here are already divided by FACTOR, so the sigma from propagation
        # must be divided by FACTOR too for a scale match.
        if _pix_sigma_near_all is None and _sig_near_row is not None:
            _sig_near_row = np.asarray(_sig_near_row) / FACTOR
        if _pix_sigma_far_all is None and _sig_far_row is not None:
            _sig_far_row = np.asarray(_sig_far_row) / FACTOR
        if _pix_sigma_sci_all is None and _sig_sci_row is not None:
            _sig_sci_row = np.asarray(_sig_sci_row) / FACTOR

        near_wrmse[i] = float(pixel_wrmse_per_row(
            flux_near_recon, flux_near_true, _sig_near_row)[0])
        far_wrmse[i]  = float(pixel_wrmse_per_row(
            flux_far_recon,  flux_far_true,  _sig_far_row)[0])
        sci_wrmse[i]  = float(pixel_wrmse_per_row(
            flux_sci_pred,   flux_sci_true,  _sig_sci_row)[0])

    print(f"  recon loop:  {_time.perf_counter() - _t_loop0:.2f} s "
          f"({n_use} rows x 3 arms = {3 * n_use} reconstructions with hoisted basis)")

    def _rmse_stats(arr):
        x = np.asarray(arr, dtype=np.float64)
        x = x[np.isfinite(x)]
        if x.size == 0:
            return {
                "count": 0,
                "mean": np.nan,
                "median": np.nan,
                "std": np.nan,
                "min": np.nan,
                "p05": np.nan,
                "p95": np.nan,
                "max": np.nan,
            }
        return {
            "count": int(x.size),
            "mean": float(np.mean(x)),
            "median": float(np.median(x)),
            "std": float(np.std(x)),
            "min": float(np.min(x)),
            "p05": float(np.percentile(x, 5.0)),
            "p95": float(np.percentile(x, 95.0)),
            "max": float(np.max(x)),
        }

    summary_df = pd.DataFrame(
        [
            {"series": "near_self_recon_pRMSE",  **_rmse_stats(near_rmse)},
            {"series": "near_self_recon_pWRMSE", **_rmse_stats(near_wrmse)},
            {"series": "far_self_recon_pRMSE",   **_rmse_stats(far_rmse)},
            {"series": "far_self_recon_pWRMSE",  **_rmse_stats(far_wrmse)},
            {"series": "sci_pred_vs_true_pRMSE", **_rmse_stats(sci_rmse)},
            {"series": "sci_pred_vs_true_pWRMSE",**_rmse_stats(sci_wrmse)},
        ]
    )

    summary_disp_df = summary_df.copy()
    for c in ["mean", "median", "std", "min", "p05", "p95", "max"]:
        summary_disp_df[c] = summary_disp_df[c] * FACTOR

    print(f"Random per-row pRMSE / pWRMSE evaluation on {n_use} spectra from every10 inputs (seed={rng_seed})")
    print("Per-row pixel-space pRMSE / pWRMSE stats in physical flux units:")
    print(summary_df.to_string(index=False, float_format=lambda v: f"{v:.6g}"))
    print("")
    print(f"Per-row pRMSE / pWRMSE stats in display units (x{FACTOR:.3g}):")
    print(summary_disp_df.to_string(index=False, float_format=lambda v: f"{v:.6g}"))

    # Multi-panel residual figure (2026-08-19): row 1 keeps the historical
    # sci total residual (pred - observed); rows 2-4 show per-component
    # residuals (pred - recon(sci_true)) so a broadband deficit that lives
    # entirely in one component (e.g. moon spline) shows up separately from
    # a line-emission miss (mesospheric / atomic / ionospheric / O2).  The
    # legend is off; per-line hover shows row_idx + expnum from META.
    from plotly.subplots import make_subplots as _make_subplots_resid

    sci_resid_arr = np.vstack(sci_resid_rows) * FACTOR
    sci_moon_arr = np.vstack(sci_moon_resid_rows) * FACTOR
    sci_diffuse_arr = np.vstack(sci_diffuse_resid_rows) * FACTOR
    sci_lines_arr = np.vstack(sci_lines_resid_rows) * FACTOR
    wave_ref = sci_wave_rows[0]
    same_grid = all(
        (w.shape == wave_ref.shape) and np.allclose(w, wave_ref, rtol=0.0, atol=1e-8)
        for w in sci_wave_rows[1:]
    )

    fig_resid = _make_subplots_resid(
        rows=4, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.045,
        subplot_titles=(
            f"SCI residuals: pred - observed (n={n_use})",
            "Moon component: pred - recon(sci coef)",
            "Diffuse continuum (HO2 + FeO + O2ac): pred - recon(sci coef)",
            "Lines (OH + atom + ORC + O2): pred - recon(sci coef)",
        ),
    )

    def _expnum_str(i):
        if _expnum_all is None:
            return ""
        try:
            _e = int(_expnum_all[int(sel_rows[i])])
        except (IndexError, ValueError, TypeError):
            return ""
        return f" | expnum {_e}"

    _panels = [
        ("total",   sci_resid_arr),
        ("moon",    sci_moon_arr),
        ("diffuse", sci_diffuse_arr),
        ("lines",   sci_lines_arr),
    ]
    for _row_i, (_pname, _arr) in enumerate(_panels, start=1):
        if same_grid:
            for i in range(n_use):
                _rid = int(sel_rows[i])
                _hover = (
                    f"row {_rid}{_expnum_str(i)}<br>"
                    f"λ=%{{x:.1f}} Å<br>"
                    f"Δ_{_pname}=%{{y:.4g}}"
                    "<extra></extra>"
                )
                fig_resid.add_trace(
                    go.Scattergl(
                        x=wave_ref, y=_arr[i],
                        mode="lines",
                        line=dict(width=0.8),
                        opacity=0.7,
                        hovertemplate=_hover,
                        showlegend=False,
                    ),
                    row=_row_i, col=1,
                )
            _rms_band = np.sqrt(np.mean(_arr ** 2, axis=0))
            fig_resid.add_trace(
                go.Scatter(
                    x=wave_ref, y=-_rms_band,
                    mode="lines",
                    line=dict(color="rgba(120,120,120,0.6)", width=1.0),
                    hoverinfo="skip",
                    showlegend=False,
                ),
                row=_row_i, col=1,
            )
            fig_resid.add_trace(
                go.Scatter(
                    x=wave_ref, y=_rms_band,
                    mode="lines",
                    line=dict(color="rgba(120,120,120,0.6)", width=1.0),
                    fill="tonexty",
                    fillcolor="rgba(120,120,120,0.15)",
                    hoverinfo="skip",
                    showlegend=False,
                ),
                row=_row_i, col=1,
            )
        else:
            for i in range(n_use):
                _rid = int(sel_rows[i])
                _hover = (
                    f"row {_rid}{_expnum_str(i)}<br>"
                    f"λ=%{{x:.1f}} Å<br>"
                    f"Δ_{_pname}=%{{y:.4g}}"
                    "<extra></extra>"
                )
                fig_resid.add_trace(
                    go.Scattergl(
                        x=sci_wave_rows[i], y=_arr[i],
                        mode="lines",
                        line=dict(width=0.8),
                        opacity=0.7,
                        hovertemplate=_hover,
                        showlegend=False,
                    ),
                    row=_row_i, col=1,
                )
            _global_rms = float(np.sqrt(np.mean(_arr ** 2)))
            fig_resid.add_hrect(
                y0=-_global_rms, y1=_global_rms,
                fillcolor="rgba(120,120,120,0.15)",
                line_width=0, layer="above",
                row=_row_i, col=1,
            )
        fig_resid.add_hline(
            y=0.0, line=dict(color="rgba(0,0,0,0.5)", width=0.8, dash="dash"),
            row=_row_i, col=1,
        )

    fig_resid.update_xaxes(title_text="Wavelength [Å]", row=4, col=1)
    fig_resid.update_yaxes(title_text="pred - obs", row=1, col=1)
    fig_resid.update_yaxes(title_text="pred - recon(true) [moon]",    row=2, col=1)
    fig_resid.update_yaxes(title_text="pred - recon(true) [diffuse]", row=3, col=1)
    fig_resid.update_yaxes(title_text="pred - recon(true) [lines]",   row=4, col=1)
    fig_resid.update_layout(
        template="plotly_white",
        title=(f"SCI + per-component residuals (n={n_use} spectra, "
               f"display units x{FACTOR:.3g}). Hover any line for row_idx + expnum. "
               f"Gray band = ± RMS(λ) across rows."),
        height=1200,
        margin=dict(l=80, r=20, t=90, b=60),
        showlegend=False,
    )
    fig_resid.show()

    rmse_subset_results = {
        "row_positions": sel_pos,
        "row_indices": sel_rows,
        "near_rmse": near_rmse,
        "far_rmse": far_rmse,
        "sci_rmse": sci_rmse,
        "near_wrmse": near_wrmse,
        "far_wrmse": far_wrmse,
        "sci_wrmse": sci_wrmse,
        "pix_sigma_source": _pix_sigma_source,
        "summary": summary_df,
        "summary_display": summary_disp_df,
        "sci_residuals": sci_resid_arr,
    }


In [ ]:
# Plot worst-case reconstructions: SCI target plus sky1/sky2 input self-recon checks.
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from astropy.io import fits

required = [
    "rmse_subset_results",
    "reconstruct_with_lsf",
    "load_lsf_state_if_available",
    "load_o2_vector_if_available",
    "_infer_base_dir_for_reconstruction",
    "predict_sci_coefficients_default",
    "mlp_artifacts",
    "build_triplet_coef_dataset",
    "context_cols",
    "FACTOR",
    "_DECOMP_SUFFIX",
]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError("Run the batch RMSE subset cell first. Missing: " + ", ".join(missing))

n_worst = 15
row_pos = np.asarray(rmse_subset_results["row_positions"], dtype=int)
row_idx = np.asarray(rmse_subset_results["row_indices"], dtype=int)
sci_rmse = np.asarray(rmse_subset_results["sci_rmse"], dtype=np.float64)
near_rmse = np.asarray(rmse_subset_results.get("near_rmse", np.full_like(sci_rmse, np.nan)), dtype=np.float64)
far_rmse = np.asarray(rmse_subset_results.get("far_rmse", np.full_like(sci_rmse, np.nan)), dtype=np.float64)

finite = np.isfinite(sci_rmse)
if not finite.any():
    raise RuntimeError("No finite sci_rmse values in rmse_subset_results")
worst_local = np.flatnonzero(finite)[np.argsort(sci_rmse[finite])[::-1]][: min(n_worst, int(finite.sum()))]

if "coef_sci_pred" in globals() and np.asarray(coef_sci_pred).shape[0] == sci_rmse.shape[0]:
    coef_pred_subset = np.asarray(coef_sci_pred, dtype=np.float64)
else:
    _e10_stem = "spline_moon/lvmsframe_median_stack_1.2.1_p40_p70_every10"
    _e10_suffix = _DECOMP_SUFFIX
    EVERY10_INPUT = f"{_e10_stem}.fits"
    EVERY10_NEAR = f"{_e10_stem}_decomp_sky1{_e10_suffix}.fits"
    EVERY10_FAR = f"{_e10_stem}_decomp_sky2{_e10_suffix}.fits"
    EVERY10_SCI = f"{_e10_stem}_decomp_sci{_e10_suffix}.fits"
    e10_triplet_plot = build_triplet_coef_dataset(
        input_fits_path=EVERY10_INPUT,
        sky_near_decomp_fits_path=EVERY10_NEAR,
        sky_far_decomp_fits_path=EVERY10_FAR,
        sci_decomp_fits_path=EVERY10_SCI,
        context_columns=context_cols,
        return_chi2=True,
    )
    coef_pred_subset = predict_sci_coefficients_default(
        mlp_artifacts,
        coef_near_phys=e10_triplet_plot["coef_near"][row_pos],
        coef_far_phys=e10_triplet_plot["coef_far"][row_pos],
        ctx_near_phys=e10_triplet_plot["ctx_near"][row_pos],
        ctx_far_phys=e10_triplet_plot["ctx_far"][row_pos],
        ctx_sci_phys=e10_triplet_plot["ctx_sci"][row_pos],
    ).astype(np.float64)

if "coef_near_sel" in globals() and np.asarray(coef_near_sel).shape[0] == sci_rmse.shape[0]:
    coef_near_subset = np.asarray(coef_near_sel, dtype=np.float64)
else:
    coef_near_subset = np.asarray(e10_triplet_plot["coef_near"][row_pos], dtype=np.float64)

if "coef_far_sel" in globals() and np.asarray(coef_far_sel).shape[0] == sci_rmse.shape[0]:
    coef_far_subset = np.asarray(coef_far_sel, dtype=np.float64)
else:
    coef_far_subset = np.asarray(e10_triplet_plot["coef_far"][row_pos], dtype=np.float64)

# 2026-08-16: per-row WRMSE for the same subset.
# Two flavours are reported:
#   * wrmse_coef_subset   -- coefficient-space, weighted by decomposition COEF_ERR
#   * near_wrmse_pix / far_wrmse_pix / sci_wrmse_pix -- pixel-space, pulled from
#     rmse_subset_results (populated by the batch RMSE cell using per-pixel sigma
#     from FLUX_SIGMA_TOTAL when the new decompositions land, else propagated
#     from COEF_ERR on the fly).
try:
    weighted_rmse_per_row  # noqa: F821 -- defined by the WRMSE-helper cell
    _e10_wrmse_src = globals().get("e10_triplet_plot", globals().get("e10_triplet"))
    if _e10_wrmse_src is None:
        raise NameError("no e10 triplet available for sWRMSE_coef lookup")
    _coef_true_subset = np.asarray(_e10_wrmse_src["coef_sci"][row_pos], dtype=np.float64)
    _sigma_raw = _e10_wrmse_src.get("coef_err_sci", None)
    if _sigma_raw is None:
        _sigma_subset = np.full_like(_coef_true_subset, np.nan)
    else:
        _sigma_subset = np.asarray(_sigma_raw[row_pos], dtype=np.float64)
    _gidx_map = (_group_indices_compress if "_group_indices_compress" in globals()
                 else group_indices_sf)
    wrmse_coef_subset = weighted_rmse_per_row(
        _coef_true_subset, coef_pred_subset, _sigma_subset,
        _gidx_map, dict(DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP),
    ).astype(np.float64)
except (NameError, KeyError) as _wrmse_exc:
    wrmse_coef_subset = np.full(coef_pred_subset.shape[0], np.nan, dtype=np.float64)
    print(f"(coef-space wrmse unavailable: {_wrmse_exc}; reporting NaN)")

# Pixel-space WRMSE for the same rows.  Cell 25 populates these when it runs;
# if it has not been run this session, use NaN so the reporting stays graceful.
_pix_ok = ("rmse_subset_results" in globals()
           and all(k in rmse_subset_results for k in ("near_wrmse", "far_wrmse", "sci_wrmse")))
if _pix_ok:
    near_wrmse_pix = np.asarray(rmse_subset_results["near_wrmse"], dtype=np.float64)
    far_wrmse_pix  = np.asarray(rmse_subset_results["far_wrmse"], dtype=np.float64)
    sci_wrmse_pix  = np.asarray(rmse_subset_results["sci_wrmse"], dtype=np.float64)
    _pix_source_note = rmse_subset_results.get("pix_sigma_source", "unknown")
    print(f"pixel pWRMSE sigma source: {_pix_source_note}")
else:
    near_wrmse_pix = np.full_like(sci_rmse, np.nan)
    far_wrmse_pix  = np.full_like(sci_rmse, np.nan)
    sci_wrmse_pix  = np.full_like(sci_rmse, np.nan)
    print("(pixel pWRMSE unavailable: run the batch RMSE eval cell first)")

_e10_stem = "spline_moon/lvmsframe_median_stack_1.2.1_p40_p70_every10"
_e10_suffix = _DECOMP_SUFFIX
EVERY10_INPUT = f"{_e10_stem}.fits"
EVERY10_NEAR = f"{_e10_stem}_decomp_sky1{_e10_suffix}.fits"
EVERY10_FAR = f"{_e10_stem}_decomp_sky2{_e10_suffix}.fits"
EVERY10_SCI = f"{_e10_stem}_decomp_sci{_e10_suffix}.fits"

with fits.open(EVERY10_INPUT) as hdul:
    flux_near_all = np.asarray(hdul["FLUX_SKY_NEAR"].data, dtype=np.float64)
    flux_far_all = np.asarray(hdul["FLUX_SKY_FAR"].data, dtype=np.float64)
    flux_sci_all = np.asarray(hdul["FLUX_SCI"].data, dtype=np.float64)
    wave_arr = np.asarray(hdul["WAVE"].data, dtype=np.float64)
    lsf_sci_arr = np.asarray(hdul["LSF_SCI"].data, dtype=np.float64)

base_dir_guess = _infer_base_dir_for_reconstruction()

n_cases = len(worst_local)
fig = make_subplots(
    rows=n_cases,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.01,
    subplot_titles=[
        f"row {int(row_idx[j])} | sky1_pRMSE={float(near_rmse[j]):.3g}, sky2_pRMSE={float(far_rmse[j]):.3g}, sci_pRMSE={float(sci_rmse[j]):.3g}, sci_pWRMSE={float(sci_wrmse_pix[j]):.3g}, sci_sWRMSE={float(wrmse_coef_subset[j]):.3g}"
        for j in worst_local
    ],
)

for case_i, j in enumerate(worst_local):
    rr = int(row_idx[j])
    wave_row = wave_arr if wave_arr.ndim == 1 else np.asarray(wave_arr[rr], dtype=np.float64)
    lsf_row = lsf_sci_arr if lsf_sci_arr.ndim == 1 else np.asarray(lsf_sci_arr[rr], dtype=np.float64)

    lsf_sigma = lsf_row / 2.35

    lsf_state_near = load_lsf_state_if_available(EVERY10_NEAR, rr)
    lsf_state_far = load_lsf_state_if_available(EVERY10_FAR, rr)
    lsf_state_sci = load_lsf_state_if_available(EVERY10_SCI, rr)

    lsf_arg_near = lsf_state_near if lsf_state_near is not None else lsf_sigma
    lsf_arg_far = lsf_state_far if lsf_state_far is not None else lsf_sigma
    lsf_arg_sci = lsf_state_sci if lsf_state_sci is not None else lsf_sigma

    o2_near = load_o2_vector_if_available(EVERY10_NEAR, rr)
    o2_far = load_o2_vector_if_available(EVERY10_FAR, rr)
    o2_sci = load_o2_vector_if_available(EVERY10_SCI, rr)

    comps_near = reconstruct_with_lsf(
        wave=wave_row, coef=coef_near_subset[j], lsf=lsf_arg_near,
        n_spline_knots=25, base_dir=base_dir_guess, o2_vector=o2_near,
    )
    comps_far = reconstruct_with_lsf(
        wave=wave_row, coef=coef_far_subset[j], lsf=lsf_arg_far,
        n_spline_knots=25, base_dir=base_dir_guess, o2_vector=o2_far,
    )
    comps_sci = reconstruct_with_lsf(
        wave=wave_row, coef=coef_pred_subset[j], lsf=lsf_arg_sci,
        n_spline_knots=25, base_dir=base_dir_guess, o2_vector=o2_sci,
    )

    near_obs = np.asarray(flux_near_all[rr], dtype=np.float64) * FACTOR
    far_obs = np.asarray(flux_far_all[rr], dtype=np.float64) * FACTOR
    sci_obs = np.asarray(flux_sci_all[rr], dtype=np.float64) * FACTOR

    near_rec = np.asarray(comps_near["total"], dtype=np.float64)
    far_rec = np.asarray(comps_far["total"], dtype=np.float64)
    sci_rec = np.asarray(comps_sci["total"], dtype=np.float64)

    row_num = case_i + 1
    # All 6 spectra in one panel with different colors
    fig.add_trace(
        go.Scattergl(
            x=wave_row, y=near_obs, mode="lines", name="sky1_obs",
            line=dict(color="#1f77b4", width=1.2),
            showlegend=(row_num == 1),
            legendgroup="sky1_obs",
        ),
        row=row_num, col=1,
    )
    fig.add_trace(
        go.Scattergl(
            x=wave_row, y=near_rec, mode="lines", name="sky1_rec",
            line=dict(color="#1f77b4", width=0.6, dash="dash"),
            showlegend=(row_num == 1),
            legendgroup="sky1_rec",
        ),
        row=row_num, col=1,
    )
    fig.add_trace(
        go.Scattergl(
            x=wave_row, y=far_obs, mode="lines", name="sky2_obs",
            line=dict(color="#ff7f0e", width=1.2),
            showlegend=(row_num == 1),
            legendgroup="sky2_obs",
        ),
        row=row_num, col=1,
    )
    fig.add_trace(
        go.Scattergl(
            x=wave_row, y=far_rec, mode="lines", name="sky2_rec",
            line=dict(color="#ff7f0e", width=0.6, dash="dash"),
            showlegend=(row_num == 1),
            legendgroup="sky2_rec",
        ),
        row=row_num, col=1,
    )
    fig.add_trace(
        go.Scattergl(
            x=wave_row, y=sci_obs, mode="lines", name="sci_obs",
            line=dict(color="#2ca02c", width=1.2),
            showlegend=(row_num == 1),
            legendgroup="sci_obs",
        ),
        row=row_num, col=1,
    )
    fig.add_trace(
        go.Scattergl(
            x=wave_row, y=sci_rec, mode="lines", name="sci_rec",
            line=dict(color="#2ca02c", width=0.6, dash="dash"),
            showlegend=(row_num == 1),
            legendgroup="sci_rec",
        ),
        row=row_num, col=1,
    )

fig.update_xaxes(title_text="Wavelength [A]", row=n_cases, col=1)
fig.update_yaxes(title_text=f"Flux (x{FACTOR:.3g})", type='log')
fig.update_layout(
    template="plotly_white",
    title=f"Worst SCI reconstruction cases: All arms overlaid (top {n_cases})",
    height=max(540, 380 * n_cases),
    margin=dict(l=70, r=20, t=90, b=60),
    legend=dict(orientation="h", yanchor="bottom", y=1.01, xanchor="left", x=0.0),
)
fig.show()

print("Worst-case rows plotted (descending sci_pRMSE):")
print("  columns: sRMSE = per-row spectral RMSE (over pixels or coefficients);")
print("           pWRMSE = per-row pixel-space WRMSE;")
print("           sWRMSE_coef = per-row spectral WRMSE in coefficient space.")
for rank, j in enumerate(worst_local, start=1):
    print(
        f"  {rank:2d}. row={int(row_idx[j])} "
        f"sky1_pRMSE={float(near_rmse[j]):.6g} "
        f"sky1_pWRMSE={float(near_wrmse_pix[j]):.4g} "
        f"sky2_pRMSE={float(far_rmse[j]):.6g} "
        f"sky2_pWRMSE={float(far_wrmse_pix[j]):.4g} "
        f"sci_pRMSE={float(sci_rmse[j]):.6g} "
        f"sci_pWRMSE={float(sci_wrmse_pix[j]):.4g} "
        f"sci_sWRMSE={float(wrmse_coef_subset[j]):.4g}"
    )


In [ ]:
# Context variable analysis for worst reconstructions: SKY1 (near), SKY2 (far), and SCI arms.
# Compare all three telescope arms to identify cross-arm patterns.
import numpy as np
import pandas as pd

required = ['worst_local', 'row_pos', 'row_idx', 'sci_rmse', 
            'ctx_sci_all', 'ctx_near_all', 'ctx_far_all', 'ctx_names_all']
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError('Run the worst-case plotting cell first. Missing: ' + ', '.join(missing))

# Extract indices and context for worst rows
worst_indices = np.asarray(worst_local, dtype=int)
worst_row_positions = row_pos[worst_indices]
worst_sci_rmse = sci_rmse[worst_indices]

worst_ctx_sci = ctx_sci_all[worst_row_positions]
worst_ctx_near = ctx_near_all[worst_row_positions]
worst_ctx_far = ctx_far_all[worst_row_positions]

# ============================================================================
# SECTION 1: Raw Context Data for All Three Arms
# ============================================================================
print('\n' + '='*100)
print('WORST RECONSTRUCTION CASES: Multi-Arm Context Analysis (SKY1, SKY2, SCI)')
print('='*100)
print(f'\nAnalyzing {len(worst_indices)} worst SCI reconstruction cases by sci_pRMSE')
print('Displaying context from all three telescope arms (sky1/near, sky2/far, sci)\n')

ctx_cols = [str(c) for c in ctx_names_all]

for arm_label, ctx_data in [('SKY1 (NEAR)', worst_ctx_near), 
                             ('SKY2 (FAR)', worst_ctx_far),
                             ('SCI', worst_ctx_sci)]:
    print('\n' + '-'*100)
    print(f'{arm_label} ARM - Context Variables:')
    print('-'*100)
    arm_data = []
    for rank, (row_pos_val, rmse_val) in enumerate(zip(worst_row_positions, worst_sci_rmse), start=1):
        row_dict = {'rank': rank, 'row_pos': int(row_pos_val), 'sci_rmse': float(rmse_val)}
        for ctx_idx, ctx_name in enumerate(ctx_cols):
            row_dict[ctx_name] = float(ctx_data[rank-1, ctx_idx])
        arm_data.append(row_dict)
    arm_df = pd.DataFrame(arm_data)
    print(arm_df.to_string(index=False, float_format=lambda v: f'{v:.6g}'))

# ============================================================================
# SECTION 2: Statistical Comparison (Worst vs Full Dataset) - All Arms
# ============================================================================
print('\n' + '='*100)
print('STATISTICAL COMPARISON: Worst Cases vs Full Dataset (All Arms)')
print('='*100)

all_arms_stats = []
for arm_label, ctx_data_all, ctx_worst in [('sky1_near', ctx_near_all, worst_ctx_near),
                                             ('sky2_far', ctx_far_all, worst_ctx_far),
                                             ('sci', ctx_sci_all, worst_ctx_sci)]:
    arm_stats = []
    for ctx_idx, ctx_name in enumerate(ctx_cols):
        worst_vals = ctx_worst[:, ctx_idx]
        all_vals = ctx_data_all[:, ctx_idx]
        
        worst_mean = float(np.mean(worst_vals))
        all_mean = float(np.mean(all_vals))
        worst_std = float(np.std(worst_vals))
        all_std = float(np.std(all_vals))
        
        z_score = (worst_mean - all_mean) / max(all_std, 1e-10)
        percentile = 100.0 * np.mean(all_vals <= worst_mean)
        
        arm_stats.append({
            'arm': arm_label,
            'context': ctx_name,
            'worst_mean': worst_mean,
            'all_mean': all_mean,
            'delta': worst_mean - all_mean,
            'z_score': z_score,
            'percentile': percentile,
        })
    all_arms_stats.extend(arm_stats)

all_arms_df = pd.DataFrame(all_arms_stats)

for arm in ['sky1_near', 'sky2_far', 'sci']:
    arm_subset = all_arms_df[all_arms_df['arm'] == arm].copy()
    print(f'\n{arm.upper()}:')
    print(arm_subset[['context', 'worst_mean', 'all_mean', 'z_score', 'percentile']].to_string(
        index=False, float_format=lambda v: f'{v:.6g}'))

# ============================================================================
# SECTION 3: Cross-Arm Pattern Detection
# ============================================================================
print('\n' + '='*100)
print('CROSS-ARM PATTERN ANALYSIS')
print('='*100)

outlier_threshold = 1.5
print(f'\nLooking for context variables with |z-score| > {outlier_threshold}:\n')

for arm in ['sky1_near', 'sky2_far', 'sci']:
    arm_subset = all_arms_df[all_arms_df['arm'] == arm]
    outliers = arm_subset[np.abs(arm_subset['z_score']) > outlier_threshold].sort_values('z_score', ascending=False)
    
    if len(outliers) > 0:
        print(f'{arm.upper()}: {len(outliers)} significant patterns')
        for _, row in outliers.iterrows():
            direction = 'HIGH' if row['z_score'] > 0 else 'LOW'
            print(f"  • {row['context']:20s}: {direction:4s}  (z={row['z_score']:+6.2f}, pct={row['percentile']:.0f})")
    else:
        print(f'{arm.upper()}: No significant patterns (all |z| ≤ {outlier_threshold})')

# ============================================================================
# SECTION 4: Consensus Analysis - Variables that Show Patterns Across All Arms
# ============================================================================
print('\n' + '='*100)
print('CONSENSUS ANALYSIS: Common Patterns Across Arms')
print('='*100)

consensus_rows = []
for ctx_name in ctx_cols:
    sky1_row = all_arms_df[(all_arms_df['arm'] == 'sky1_near') & (all_arms_df['context'] == ctx_name)].iloc[0]
    sky2_row = all_arms_df[(all_arms_df['arm'] == 'sky2_far') & (all_arms_df['context'] == ctx_name)].iloc[0]
    sci_row = all_arms_df[(all_arms_df['arm'] == 'sci') & (all_arms_df['context'] == ctx_name)].iloc[0]
    
    # Check if pattern is consistent across arms
    z_scores = [sky1_row['z_score'], sky2_row['z_score'], sci_row['z_score']]
    signs = [1 if z > 0 else -1 for z in z_scores]
    magnitude = np.mean(np.abs(z_scores))
    consistency = np.sum([1 for s in signs if s == signs[0]]) / len(signs)
    
    consensus_rows.append({
        'context': ctx_name,
        'sky1_z': sky1_row['z_score'],
        'sky2_z': sky2_row['z_score'],
        'sci_z': sci_row['z_score'],
        'avg_magnitude': magnitude,
        'consistency': consistency,
    })

consensus_df = pd.DataFrame(consensus_rows).sort_values('avg_magnitude', ascending=False)

print('\nVariables ranked by cross-arm pattern strength:')
print('(consistency = % of arms showing same sign; avg_magnitude = mean |z|)\n')
print(consensus_df.to_string(index=False, float_format=lambda v: f'{v:.6g}'))

# Identify strongest cross-arm patterns
strong_consensus = consensus_df[
    (consensus_df['consistency'] >= 0.67) & 
    (consensus_df['avg_magnitude'] > 1.0)
].sort_values('avg_magnitude', ascending=False)

if len(strong_consensus) > 0:
    print('\n' + '='*100)
    print('STRONGEST CROSS-ARM PATTERNS (consistency ≥ 67%, avg |z| > 1.0):')
    print('='*100)
    for _, row in strong_consensus.iterrows():
        print(f"\n  {row['context']:20s}")
        print(f"    Sky1 z={row['sky1_z']:+6.2f}  Sky2 z={row['sky2_z']:+6.2f}  Sci z={row['sci_z']:+6.2f}")
        print(f"    Avg magnitude: {row['avg_magnitude']:.3f}  Consistency: {row['consistency']:.1%}")
else:
    print('\n' + '='*100)
    print('NO STRONG CROSS-ARM PATTERNS')
    print('='*100)
    print('→ Poor reconstructions do NOT show systematic context patterns across all arms.')

# ============================================================================
# SECTION 5: Summary Hypothesis
# ============================================================================
print('\n' + '='*100)
print('HYPOTHESIS & CONCLUSION')
print('='*100)

n_sci_outliers = len(all_arms_df[(all_arms_df['arm'] == 'sci') & 
                                  (np.abs(all_arms_df['z_score']) > outlier_threshold)])
n_sky1_outliers = len(all_arms_df[(all_arms_df['arm'] == 'sky1_near') & 
                                   (np.abs(all_arms_df['z_score']) > outlier_threshold)])
n_sky2_outliers = len(all_arms_df[(all_arms_df['arm'] == 'sky2_far') & 
                                   (np.abs(all_arms_df['z_score']) > outlier_threshold)])

if n_sci_outliers + n_sky1_outliers + n_sky2_outliers == 0:
    print('\n✗ NO SYSTEMATIC PATTERNS DETECTED')
    print('  Worst reconstructions appear randomly distributed across context space.')
    print('  This suggests issues are NOT context-dependent but rather:')
    print('    • Model overfitting to unobserved features')
    print('    • Rare coefficient patterns that are difficult to learn')
    print('    • Inherent data quality issues (noise/artifacts) in those rows')
elif len(strong_consensus) > 0:
    print('\n✓ STRONG CROSS-ARM PATTERNS IDENTIFIED')
    print(f'  {len(strong_consensus)} context variable(s) show consistent anomalies across arms.')
    print('  Worst cases cluster at specific context ranges across all telescopes.')
    print('  RECOMMENDATION: Focus model robustness improvement on these contexts.')
else:
    print('\n◑ PARTIAL PATTERNS DETECTED')
    print(f'  Sky1: {n_sky1_outliers} variables | Sky2: {n_sky2_outliers} | Sci: {n_sci_outliers}')
    print('  Patterns are arm-specific or inconsistent across telescopes.')
    print('  Suggests arm-dependent issues (e.g., alignment, calibration).')


In [ ]:
# --- diagnostic: check pipeline state after airmass filter ---
print('=== Pipeline state check ===')
print(f'filtered_triplet rows       : {filtered_triplet["coef_near"].shape[0]}')
print(f'  has compress_train_idx?   : {"compress_train_idx" in filtered_triplet}')
if 'compress_train_idx' in filtered_triplet:
    print(f'  compress_train_idx max    : {int(np.max(filtered_triplet["compress_train_idx"]))}')
    print(f'  compress_test_idx  max    : {int(np.max(filtered_triplet["compress_test_idx"]))}')

print()
print('=== mlp_artifacts (trained model) ===')
print(f'train_idx max               : {int(mlp_artifacts["train_idx"].max())}')
print(f'val_idx max                 : {int(mlp_artifacts["val_idx"].max())}')
print(f'test_idx max                : {int(mlp_artifacts["test_idx"].max())}')
print(f'group_score_dims            : {mlp_artifacts["group_score_dims"]}')
_gsd = mlp_artifacts['group_score_dims']
for g, n in _gsd.items():
    w = 1.0 / (float(n) ** 0.5)
    print(f'  loss weight w_{g:<12s} = 1/sqrt({n:>3d}) = {w:.4f}')

print()
print('=== group_compressors (fit state) ===')
for g, comp in group_compressors.items():
    print(f'  {g:<14s} n_input={comp["n_coef_group"]:>4d} n_kept={comp["kept"].size:>3d} transform={comp["kind"]:>8s}')

print()
print('=== index alignment check ===')
cur_n = filtered_triplet['coef_near'].shape[0]
mlp_max = int(mlp_artifacts['test_idx'].max())
if 'compress_train_idx' in filtered_triplet:
    ftr_max = int(np.max(filtered_triplet['compress_train_idx']))
    if ftr_max >= cur_n or mlp_max >= cur_n:
        print(f'  MISMATCH: filtered_triplet has {cur_n} rows but mlp_artifacts indices go up to {mlp_max}, filtered_triplet indices to {ftr_max}')
    else:
        print(f'  OK: mlp_artifacts test_idx.max={mlp_max} < filtered_triplet rows={cur_n}')
else:
    if mlp_max >= cur_n:
        print(f'  MISMATCH: current filtered_triplet has {cur_n} rows, but mlp_artifacts.test_idx.max={mlp_max}. Downstream ops using test_idx on filtered_triplet will index out of range.')
    else:
        print(f'  Model was trained on a different filtered_triplet (compress_train_idx lost). Current filtered_triplet has {cur_n} rows; model expects up to {mlp_max}.')

# Continuum-relevant group check
print()
print('=== continuum-related predictions (sanity check on median coef amplitude) ===')
_coef_true = np.asarray(filtered_triplet['coef_sci'], dtype=np.float32)
_coef_pred = predict_sci_coefficients_default(
    mlp_artifacts,
    coef_near_phys=np.asarray(filtered_triplet['coef_near'], dtype=np.float32),
    coef_far_phys=np.asarray(filtered_triplet['coef_far'], dtype=np.float32),
    ctx_near_phys=np.asarray(filtered_triplet['ctx_near'], dtype=np.float32),
    ctx_far_phys=np.asarray(filtered_triplet['ctx_far'], dtype=np.float32),
    ctx_sci_phys=np.asarray(filtered_triplet['ctx_sci'], dtype=np.float32),
).astype(np.float32)
_names = [str(n).lower() for n in filtered_triplet['coef_names']]
_gidx = _build_group_indices(filtered_triplet['coef_names'])
for gname, idx in _gidx.items():
    idx = np.asarray(idx, dtype=int)
    med_true = float(np.median(_coef_true[:, idx]))
    med_pred = float(np.median(_coef_pred[:, idx]))
    mean_true = float(np.mean(_coef_true[:, idx]))
    mean_pred = float(np.mean(_coef_pred[:, idx]))
    bias = mean_pred - mean_true
    print(f'  {gname:<12s} n={idx.size:>3d}  median true/pred = {med_true:.3g} / {med_pred:.3g}   mean true/pred = {mean_true:.3g} / {mean_pred:.3g}   mean bias = {bias:+.3g} ({100*bias/max(abs(mean_true), 1e-30):+.1f}%)')


In [ ]:
# Per-seed vs ensemble diagnostic reader (no retraining).
# Since 2026-08-11 the trainer cell above already fits the 4-seed ensemble; this
# cell reads mlp_artifacts['members'] to show whether individual seeds carry
# per-group biases the ensemble averages out (calibrated uncertainty for §11.7).

required = ['filtered_triplet', 'mlp_artifacts', 'predict_sci_coefficients_default',
            '_group_indices_compress', '_metric_row',
            'coef_near_all', 'coef_far_all', 'coef_sci_all',
            'ctx_near_all', 'ctx_far_all', 'ctx_sci_all', 'test_idx']
_missing = [k for k in required if k not in globals()]
if _missing:
    raise RuntimeError('Run the trainer cell first. Missing: ' + ', '.join(_missing))
if not mlp_artifacts.get('is_ensemble', False):
    raise RuntimeError('mlp_artifacts is not an ensemble; the trainer cell should build it.')

_members = mlp_artifacts['members']
_seeds = mlp_artifacts['seeds']
_te = np.asarray(test_idx, dtype=int)

# Per-seed predictions on the full filtered set for group-bias tables.
_per_seed_pred_all = {}
for _seed, _member in zip(_seeds, _members):
    _per_seed_pred_all[_seed] = predict_sci_coefficients_default(
        _member,
        coef_near_phys=coef_near_all, coef_far_phys=coef_far_all,
        ctx_near_phys=ctx_near_all, ctx_far_phys=ctx_far_all,
        ctx_sci_phys=ctx_sci_all).astype(np.float32)
_ensemble_all = np.mean(np.stack([_per_seed_pred_all[s] for s in _seeds]),
                        axis=0).astype(np.float32)

_bias_rows = []
for gname, idx in _group_indices_compress.items():
    idx = np.asarray(idx, dtype=int)
    mean_true = float(np.mean(coef_sci_all[:, idx]))
    row = {'group': gname, 'n': int(idx.size), 'mean_true': mean_true}
    for _seed in _seeds:
        _pred = _per_seed_pred_all[_seed]
        row[f'bias_s{_seed}_%'] = (
            100.0 * (float(np.mean(_pred[:, idx])) - mean_true)
            / max(abs(mean_true), 1e-30))
    row['bias_ens_%'] = (
        100.0 * (float(np.mean(_ensemble_all[:, idx])) - mean_true)
        / max(abs(mean_true), 1e-30))
    _bias_rows.append(row)

print(f'Per-group mean coefficient bias across {len(_seeds)} seeds + ensemble mean '
      f'(all {coef_sci_all.shape[0]} filtered rows):')
print(pd.DataFrame(_bias_rows).to_string(index=False, float_format=lambda v: f'{v:.3g}'))

_seed_max_biases = [max(abs(r[f'bias_s{s}_%']) for r in _bias_rows) for s in _seeds]
_ens_max_bias = max(abs(r['bias_ens_%']) for r in _bias_rows)
print(f'\nMax |bias| per single seed: min={min(_seed_max_biases):.2f}%, '
      f'max={max(_seed_max_biases):.2f}%, mean={float(np.mean(_seed_max_biases)):.2f}%')
print(f'Max |bias| of ensemble mean:  {_ens_max_bias:.2f}%')
if _ens_max_bias < 1.0:
    print(f'Verdict: ensemble drives max |bias| below 1% -- N={len(_seeds)} seeds are '
          f'sufficient for the group-level unbiasedness that §11.1 requires.')
elif _ens_max_bias < 0.7 * min(_seed_max_biases):
    print(f'Verdict: ensemble helps ({_ens_max_bias:.2f}% vs single-seed min '
          f'{min(_seed_max_biases):.2f}%) but does not fall below 1%. '
          f'Bumping ensemble_seeds to 8 in default_dual_group_config would help.')
else:
    print('Verdict: ensemble does not meaningfully reduce max |bias|; the bias is '
          'systematic (not seed variance). Investigate calibration / loss balance.')


In [ ]:
# Confidence check A: dense alpha_init grid at seed=42 only.
# Purpose: trace the shape of test mean_rmse as a function of alpha_init.
# If unimodal with a clear peak near 0.7, confidence is warranted. If flat
# or noisy, the earlier 3-point A/B was under-sampled.
import contextlib as _cA_contextlib
import io as _cA_io
import numpy as np, pandas as pd
import plotly.graph_objects as go

required = ['filtered_triplet', 'group_compressors', 'compress_geom_kwargs',
            '_group_indices_compress', 'train_compressed_group_mlp',
            'predict_sci_coefficients_default', '_metric_row',
            'default_dual_group_config',
            'coef_near_all', 'coef_far_all', 'coef_sci_all',
            'ctx_near_all', 'ctx_far_all', 'ctx_sci_all', 'test_idx']
_missing = [k for k in required if k not in globals()]
if _missing:
    raise RuntimeError('Run prerequisite cells first. Missing: ' + ', '.join(_missing))

RUN_ALPHA_GRID = False
if not RUN_ALPHA_GRID:
    print('Alpha-init grid check skipped. Set RUN_ALPHA_GRID = True to run.')
else:
    _grid_alphas = [0.40, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]
    _grid_seed = 42
    _grid_split = (
        filtered_triplet['compress_train_idx'],
        filtered_triplet['compress_val_idx'],
        filtered_triplet['compress_test_idx'],
    )
    _grid_kwargs = dict(
        n_epochs=int(default_dual_group_config['n_epochs']),
        batch_size=int(default_dual_group_config['batch_size']),
        lr=float(default_dual_group_config['lr']),
        encoder_dims=tuple(int(v) for v in default_dual_group_config['encoder_dims']),
        ctx_dims=tuple(int(v) for v in default_dual_group_config['ctx_dims']),
        trunk_dims=tuple(int(v) for v in default_dual_group_config['trunk_dims']),
        head_dim=int(default_dual_group_config['head_dim']),
        weight_decay=float(default_dual_group_config['weight_decay']),
        patience=int(default_dual_group_config['patience']),
        moon_group_weight=float(default_dual_group_config['moon_group_weight']),
        continuum_group_weight=float(default_dual_group_config['continuum_group_weight']),
        blend_optim='direct',
        seed=int(_grid_seed),
    )
    _te = np.asarray(test_idx, dtype=int)
    _y_te = coef_sci_all[_te]

    grid_alpha_results = {}
    print(f'Dense alpha_init grid at seed={_grid_seed}, blend_optim=direct')
    print(f'{"alpha_init":>10s}  {"best_ep":>7s}  {"stop_ep":>7s}  '
          f'{"val_loss":>10s}  {"mean_eRMSE":>10s}  {"median_corr":>11s}')
    for _a in _grid_alphas:
        with _cA_contextlib.redirect_stdout(_cA_io.StringIO()):
            _art = train_compressed_group_mlp(
                filtered_triplet, group_compressors, _group_indices_compress,
                compress_geom_kwargs,
                split_indices=_grid_split,
                blend_init_alpha=float(_a),
                **_grid_kwargs,
            )
        _pred = predict_sci_coefficients_default(
            _art,
            coef_near_phys=coef_near_all[_te], coef_far_phys=coef_far_all[_te],
            ctx_near_phys=ctx_near_all[_te], ctx_far_phys=ctx_far_all[_te],
            ctx_sci_phys=ctx_sci_all[_te]).astype(np.float32)
        _m = _metric_row(_y_te, _pred, f'grid_a{_a:.2f}')
        _final = {g: float(v) for g, v in _art['blend_history'][-1].items() if g != 'epoch'}
        grid_alpha_results[float(_a)] = {'artifacts': _art, 'metrics': _m, 'final_alpha': _final}
        print(f'{float(_a):>10.2f}  {int(_art["best_epoch"]):>7d}  '
              f'{int(_art["blend_history"][-1]["epoch"]):>7d}  '
              f'{float(_art["best_val_loss"]):>10.6f}  '
              f'{float(_m["mean_eRMSE"]):>10.4f}  {float(_m["median_corr"]):>11.4f}')

    _grid_df = pd.DataFrame([
        {'alpha_init': a, 'best_epoch': int(r['artifacts']['best_epoch']),
         'val_loss': float(r['artifacts']['best_val_loss']),
         **{k: float(v) for k, v in r['metrics'].items() if k in
            ('mean_eRMSE', 'median_eRMSE', 'mean_eMAE', 'mean_corr', 'median_corr')}}
        for a, r in grid_alpha_results.items()])

    _fig_grid = go.Figure()
    _fig_grid.add_trace(go.Scatter(
        x=_grid_df['alpha_init'], y=_grid_df['mean_eRMSE'],
        mode='lines+markers', name='test mean_eRMSE (seed=42)'))
    _fig_grid.add_hline(y=19.96, line_dash='dot', line_color='gray',
        annotation_text='pre-session baseline (sigmoid + alpha=0.7, 4-seed ensemble)')
    _fig_grid.update_layout(
        title=f'Dense alpha_init grid at seed={_grid_seed} (blend_optim=direct)',
        xaxis_title='alpha_init', yaxis_title='test mean_eRMSE',
        template='plotly_white', height=420)
    _fig_grid.show()

    _best_a = float(_grid_df.sort_values('median_eRMSE').iloc[0]['alpha_init'])
    _worst_a = float(_grid_df.sort_values('median_eRMSE').iloc[-1]['alpha_init'])
    _spread = float(_grid_df['mean_eRMSE'].max() - _grid_df['mean_eRMSE'].min())
    print(f'\nBest alpha_init at seed=42: {_best_a:.2f} (median_eRMSE={_grid_df.sort_values("median_eRMSE").iloc[0]["median_eRMSE"]:.4f}, mean_eRMSE={_grid_df.sort_values("mean_eRMSE").iloc[0]["mean_eRMSE"]:.4f})')
    print(f'Worst alpha_init at seed=42: {_worst_a:.2f} (median_eRMSE={_grid_df.sort_values("median_eRMSE").iloc[-1]["median_eRMSE"]:.4f}, mean_eRMSE={_grid_df.sort_values("mean_eRMSE").iloc[-1]["mean_eRMSE"]:.4f})')
    print(f'Spread across the grid: {_spread:.4f}')
    print('If spread < ~1.0 (single-seed noise scale), the curve is essentially flat '
          'and alpha_init selection is not resolved by a single seed.')


In [ ]:
# Confidence check B: 8-seed ensemble at alpha_init in {0.60, 0.70, 0.75}.
# Purpose: tighten error bars on the 4-seed A/B winner and check the neighbours
# at 0.6 and 0.75, which the coarse grid did not sample.
import contextlib as _cB_contextlib
import io as _cB_io
import numpy as np, pandas as pd

RUN_ALPHA_MULTISEED = False
if not RUN_ALPHA_MULTISEED:
    print('8-seed alpha-init check skipped. Set RUN_ALPHA_MULTISEED = True to run.')
else:
    _more_alphas = [0.60, 0.70, 0.75]
    _more_seeds = [42, 43, 44, 45, 46, 47, 48, 49]
    _more_split = (
        filtered_triplet['compress_train_idx'],
        filtered_triplet['compress_val_idx'],
        filtered_triplet['compress_test_idx'],
    )
    _more_kwargs = dict(
        n_epochs=int(default_dual_group_config['n_epochs']),
        batch_size=int(default_dual_group_config['batch_size']),
        lr=float(default_dual_group_config['lr']),
        encoder_dims=tuple(int(v) for v in default_dual_group_config['encoder_dims']),
        ctx_dims=tuple(int(v) for v in default_dual_group_config['ctx_dims']),
        trunk_dims=tuple(int(v) for v in default_dual_group_config['trunk_dims']),
        head_dim=int(default_dual_group_config['head_dim']),
        weight_decay=float(default_dual_group_config['weight_decay']),
        patience=int(default_dual_group_config['patience']),
        moon_group_weight=float(default_dual_group_config['moon_group_weight']),
        continuum_group_weight=float(default_dual_group_config['continuum_group_weight']),
        blend_optim='direct',
    )
    _te = np.asarray(test_idx, dtype=int)
    _y_te = coef_sci_all[_te]

    more_seed_results = {a: {} for a in _more_alphas}
    print(f'Multi-seed check at alpha_init in {_more_alphas}, seeds {_more_seeds}')
    for _a in _more_alphas:
        print(f'\n--- alpha_init = {_a:.2f} ---')
        _preds = {}
        for _seed in _more_seeds:
            with _cB_contextlib.redirect_stdout(_cB_io.StringIO()):
                _art = train_compressed_group_mlp(
                    filtered_triplet, group_compressors, _group_indices_compress,
                    compress_geom_kwargs,
                    split_indices=_more_split,
                    seed=int(_seed),
                    blend_init_alpha=float(_a),
                    **_more_kwargs,
                )
            more_seed_results[_a][_seed] = _art
            _preds[_seed] = predict_sci_coefficients_default(
                _art,
                coef_near_phys=coef_near_all[_te], coef_far_phys=coef_far_all[_te],
                ctx_near_phys=ctx_near_all[_te], ctx_far_phys=ctx_far_all[_te],
                ctx_sci_phys=ctx_sci_all[_te]).astype(np.float32)
            _m = _metric_row(_y_te, _preds[_seed], f'a{_a:.2f}_s{_seed}')
            print(f'  seed={_seed}: best_ep={_art["best_epoch"]:3d}  '
                  f'val_loss={_art["best_val_loss"]:.6f}  '
                  f'mean_eRMSE={float(_m["mean_eRMSE"]):.4f}')

    # --- Summary: per-seed mean_eRMSE and 8-seed ensemble ---
    _rows = []
    for _a in _more_alphas:
        _preds = {s: predict_sci_coefficients_default(
            more_seed_results[_a][s],
            coef_near_phys=coef_near_all[_te], coef_far_phys=coef_far_all[_te],
            ctx_near_phys=ctx_near_all[_te], ctx_far_phys=ctx_far_all[_te],
            ctx_sci_phys=ctx_sci_all[_te]).astype(np.float32) for s in _more_seeds}
        _per_seed = [float(_metric_row(_y_te, _preds[s], f'x')['mean_eRMSE']) for s in _more_seeds]
        _ens = np.mean(np.stack([_preds[s] for s in _more_seeds]), axis=0).astype(np.float32)
        _ens_m = _metric_row(_y_te, _ens, f'ens_a{_a:.2f}')
        _rows.append({'alpha_init': float(_a),
                      'seed_eRMSE_mean': float(np.mean(_per_seed)),
                      'seed_eRMSE_std': float(np.std(_per_seed, ddof=1)),
                      'seed_eRMSE_min': float(np.min(_per_seed)),
                      'seed_eRMSE_max': float(np.max(_per_seed)),
                      'ensemble_eRMSE': float(_ens_m['mean_eRMSE']),
                      'ensemble_median_corr': float(_ens_m['median_corr'])})
    more_seed_df = pd.DataFrame(_rows)
    print('\n8-seed summary:')
    print(more_seed_df.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

    # Report significance of the alpha_init differences.
    print('\nAre the ensemble differences bigger than the seed-to-seed std at each alpha?')
    for _r in _rows:
        print(f"  alpha_init={_r['alpha_init']:.2f}: seed std={_r['seed_eRMSE_std']:.3f}, "
              f"ensemble mean_eRMSE={_r['ensemble_eRMSE']:.3f}")
    _best = min(_rows, key=lambda r: r['ensemble_eRMSE'])
    _worst = max(_rows, key=lambda r: r['ensemble_eRMSE'])
    _gap = _worst['ensemble_eRMSE'] - _best['ensemble_eRMSE']
    _ref_std = np.mean([r['seed_eRMSE_std'] for r in _rows])
    print(f'\nGap between best and worst ensemble mean_eRMSE: {_gap:.3f}')
    print(f'Mean seed-to-seed std across alphas: {_ref_std:.3f}')
    if _gap > 2 * _ref_std / (len(_more_seeds) ** 0.5):
        print('Verdict: the gap is bigger than the ensemble-mean standard error; '
              f'best alpha_init = {_best["alpha_init"]:.2f} is a real preference.')
    else:
        print('Verdict: the gap is comparable to the ensemble-mean standard error; '
              'differences between alpha_init values are within noise.')


In [ ]:
# Confidence check C: does alpha drift toward a common attractor if given
# a much longer training budget? Runs alpha_init in {0.5, 0.7, 0.8} at
# n_epochs=200, patience=40 at seed=42 and compares final alpha values across
# inits. If they converge to similar per-group values, alpha has a real
# minimum and 50 epochs was just too short. If they stay locked near their
# init, alpha is genuinely a slow-varying prior (as suggested by the earlier
# A/B) and 0.7 is a choice of prior, not a discovered optimum.
import contextlib as _cC_contextlib
import io as _cC_io
import numpy as np, pandas as pd

RUN_ALPHA_LONGBUDGET = False
if not RUN_ALPHA_LONGBUDGET:
    print('Long-budget alpha check skipped. Set RUN_ALPHA_LONGBUDGET = True to run.')
else:
    _long_alphas = [0.50, 0.70, 0.80]
    _long_seed = 42
    _long_split = (
        filtered_triplet['compress_train_idx'],
        filtered_triplet['compress_val_idx'],
        filtered_triplet['compress_test_idx'],
    )
    _long_kwargs = dict(
        n_epochs=200, patience=40,
        batch_size=int(default_dual_group_config['batch_size']),
        lr=float(default_dual_group_config['lr']),
        encoder_dims=tuple(int(v) for v in default_dual_group_config['encoder_dims']),
        ctx_dims=tuple(int(v) for v in default_dual_group_config['ctx_dims']),
        trunk_dims=tuple(int(v) for v in default_dual_group_config['trunk_dims']),
        head_dim=int(default_dual_group_config['head_dim']),
        weight_decay=float(default_dual_group_config['weight_decay']),
        moon_group_weight=float(default_dual_group_config['moon_group_weight']),
        continuum_group_weight=float(default_dual_group_config['continuum_group_weight']),
        blend_optim='direct',
        seed=int(_long_seed),
    )
    _te = np.asarray(test_idx, dtype=int)
    _y_te = coef_sci_all[_te]

    long_budget_results = {}
    print(f'Long-budget check: n_epochs=200, patience=40, seed={_long_seed}, '
          f'alpha_init in {_long_alphas}')
    for _a in _long_alphas:
        print(f'\n--- alpha_init = {_a:.2f} ---')
        with _cC_contextlib.redirect_stdout(_cC_io.StringIO()):
            _art = train_compressed_group_mlp(
                filtered_triplet, group_compressors, _group_indices_compress,
                compress_geom_kwargs,
                split_indices=_long_split,
                blend_init_alpha=float(_a),
                **_long_kwargs,
            )
        _pred = predict_sci_coefficients_default(
            _art,
            coef_near_phys=coef_near_all[_te], coef_far_phys=coef_far_all[_te],
            ctx_near_phys=ctx_near_all[_te], ctx_far_phys=ctx_far_all[_te],
            ctx_sci_phys=ctx_sci_all[_te]).astype(np.float32)
        _m = _metric_row(_y_te, _pred, f'long_a{_a:.2f}')
        _final = {g: float(v) for g, v in _art['blend_history'][-1].items() if g != 'epoch'}
        _last_ep = int(_art['blend_history'][-1]['epoch'])
        long_budget_results[float(_a)] = {'artifacts': _art, 'metrics': _m,
                                           'final_alpha': _final, 'stop_ep': _last_ep}
        print(f'  best_ep={_art["best_epoch"]:3d}  stop_ep={_last_ep:3d}  '
              f'val_loss={_art["best_val_loss"]:.6f}  '
              f'mean_eRMSE={float(_m["mean_eRMSE"]):.4f}  '
              f'median_corr={float(_m["median_corr"]):.4f}')
        print(f'  final alpha: ' + '  '.join(f'{g}={_final[g]:.3f}' for g in _final))

    # --- Convergence table ---
    _groups = list(next(iter(long_budget_results.values()))['final_alpha'].keys())
    print('\nFinal alpha_g (last epoch) per (alpha_init, group) at 200-epoch budget:')
    print(f"  {'alpha_init':>10s}  {'stop_ep':>7s}  " + "  ".join(f"{g:>12s}" for g in _groups))
    for _a, _r in long_budget_results.items():
        _row = "  ".join(f"{_r['final_alpha'][g]:>12.3f}" for g in _groups)
        print(f"  {float(_a):>10.2f}  {int(_r['stop_ep']):>7d}  " + _row)

    # Per-group spread across the three inits.
    _spreads = {g: max(_r['final_alpha'][g] for _r in long_budget_results.values())
                 - min(_r['final_alpha'][g] for _r in long_budget_results.values())
                for g in _groups}
    print('\nPer-group alpha spread across inits at 200 epochs '
          '(near 0 = converged to common value; near 0.3 = locked near init):')
    for g, s in _spreads.items():
        print(f'  {g:>14s}: spread = {s:.3f}')
    if max(_spreads.values()) < 0.05:
        print('\nVerdict: alpha_g converges to a common value regardless of init. '
              'The alpha selection is not really about init; any near-optimum init works.')
    elif max(_spreads.values()) < 0.15:
        print('\nVerdict: partial convergence -- some groups converge, others remain '
              'init-dependent. alpha is a slow-varying prior for the sticky groups.')
    else:
        print('\nVerdict: alpha remains init-dependent even at 4x training budget. '
              'It is confirmed to be a prior, not a converged parameter.')


In [ ]:
# A/B: pre-fit alpha (closed-form OLS) vs the current 'direct' baseline.
# Pre-fit modes solve alpha_g = <s_near - s_far, s_sci - s_far> / ||s_near - s_far||^2
# per group in scaled score space on training rows, then either freeze it
# ('prefit_freeze') or use it as an init and let AdamW adjust ('prefit_warmstart').
# Compares against 'direct' + blend_init_alpha=0.7 (the current baseline) at 4 seeds.
import contextlib as _pa_contextlib
import io as _pa_io
import numpy as np, pandas as pd

required = ['filtered_triplet', 'group_compressors', 'compress_geom_kwargs',
            '_group_indices_compress', 'train_compressed_group_mlp',
            'predict_sci_coefficients_default', '_metric_row',
            'default_dual_group_config',
            'coef_near_all', 'coef_far_all', 'coef_sci_all',
            'ctx_near_all', 'ctx_far_all', 'ctx_sci_all', 'test_idx']
_missing = [k for k in required if k not in globals()]
if _missing:
    raise RuntimeError('Run prerequisite cells first. Missing: ' + ', '.join(_missing))

RUN_ALPHA_PREFIT_AB = False
if not RUN_ALPHA_PREFIT_AB:
    print('Alpha pre-fit A/B skipped. Set RUN_ALPHA_PREFIT_AB = False to run.')
else:
    _prefit_modes = ['direct', 'prefit_freeze', 'prefit_warmstart']
    _prefit_seeds = [42, 43, 44, 45]
    _prefit_split = (
        filtered_triplet['compress_train_idx'],
        filtered_triplet['compress_val_idx'],
        filtered_triplet['compress_test_idx'],
    )
    _prefit_kwargs = dict(
        n_epochs=int(default_dual_group_config['n_epochs']),
        batch_size=int(default_dual_group_config['batch_size']),
        lr=float(default_dual_group_config['lr']),
        encoder_dims=tuple(int(v) for v in default_dual_group_config['encoder_dims']),
        ctx_dims=tuple(int(v) for v in default_dual_group_config['ctx_dims']),
        trunk_dims=tuple(int(v) for v in default_dual_group_config['trunk_dims']),
        head_dim=int(default_dual_group_config['head_dim']),
        weight_decay=float(default_dual_group_config['weight_decay']),
        patience=int(default_dual_group_config['patience']),
        moon_group_weight=float(default_dual_group_config['moon_group_weight']),
        continuum_group_weight=float(default_dual_group_config['continuum_group_weight']),
        mesospheric_group_weight=float(default_dual_group_config['mesospheric_group_weight']),
        ionospheric_group_weight=float(default_dual_group_config['ionospheric_group_weight']),
        blend_init_alpha=0.7,
    )
    _te = np.asarray(test_idx, dtype=int)
    _y_te = coef_sci_all[_te]

    prefit_ab_results = {m: {} for m in _prefit_modes}
    for _mode in _prefit_modes:
        print(f'\n--- blend_optim = {_mode!r} ---')
        for _seed in _prefit_seeds:
            with _pa_contextlib.redirect_stdout(_pa_io.StringIO()):
                _art = train_compressed_group_mlp(
                    filtered_triplet, group_compressors, _group_indices_compress,
                    compress_geom_kwargs,
                    split_indices=_prefit_split,
                    seed=int(_seed),
                    blend_optim=_mode,
                    **_prefit_kwargs,
                )
            prefit_ab_results[_mode][_seed] = _art
            _pred = predict_sci_coefficients_default(
                _art,
                coef_near_phys=coef_near_all[_te], coef_far_phys=coef_far_all[_te],
                ctx_near_phys=ctx_near_all[_te], ctx_far_phys=ctx_far_all[_te],
                ctx_sci_phys=ctx_sci_all[_te]).astype(np.float32)
            _m = _metric_row(_y_te, _pred, f'{_mode}_s{_seed}')
            print(f'  seed={_seed}: best_ep={_art["best_epoch"]:3d}  '
                  f'val_loss={_art["best_val_loss"]:.6f}  '
                  f'mean_eRMSE={float(_m["mean_eRMSE"]):.4f}')

    # ---- Per-mode 4-seed ensemble summary ----
    _summary_rows = []
    _groups = list(prefit_ab_results['direct'][_prefit_seeds[0]]['group_score_dims'].keys())
    for _mode in _prefit_modes:
        _preds = {s: predict_sci_coefficients_default(
            prefit_ab_results[_mode][s],
            coef_near_phys=coef_near_all[_te], coef_far_phys=coef_far_all[_te],
            ctx_near_phys=ctx_near_all[_te], ctx_far_phys=ctx_far_all[_te],
            ctx_sci_phys=ctx_sci_all[_te]).astype(np.float32) for s in _prefit_seeds}
        _per_seed_rmse = [float(_metric_row(_y_te, _preds[s], 'x')['mean_eRMSE'])
                          for s in _prefit_seeds]
        _ens = np.mean(np.stack([_preds[s] for s in _prefit_seeds]),
                       axis=0).astype(np.float32)
        _ens_m = _metric_row(_y_te, _ens, f'ens_{_mode}')
        _summary_rows.append({
            'mode': _mode,
            'seed_mean_eRMSE': float(np.mean(_per_seed_rmse)),
            'seed_std_eRMSE': float(np.std(_per_seed_rmse, ddof=1)),
            'seed_min_eRMSE': float(np.min(_per_seed_rmse)),
            'seed_max_eRMSE': float(np.max(_per_seed_rmse)),
            'ensemble_eRMSE': float(_ens_m['mean_eRMSE']),
            'ensemble_median_corr': float(_ens_m['median_corr']),
        })
    prefit_ab_summary_df = pd.DataFrame(_summary_rows)
    print('\n4-seed ensemble summary:')
    print(prefit_ab_summary_df.to_string(index=False,
          float_format=lambda v: f'{v:.4f}'))

    # ---- Pre-fit alpha values (deterministic, identical across seeds) ----
    print('\nPre-fit alpha_g^* per group '
          '(closed-form OLS on training scores; deterministic, seed-independent):')
    _pf_art = prefit_ab_results['prefit_freeze'][_prefit_seeds[0]]
    _pf_alphas = _pf_art.get('blend_prefit_alphas')
    if _pf_alphas:
        print('  ' + '  '.join(f'{g}={float(_pf_alphas[g]):.3f}' for g in _groups))

    # ---- Sanity: alpha values across seeds for prefit_freeze must be identical ----
    print('\nPer-seed final alpha at best epoch (seed=42..45, per mode):')
    print(f"  {'mode':<18s} {'seed':>5s}  " + "  ".join(f"{g:>10s}" for g in _groups))
    for _mode in _prefit_modes:
        for _seed in _prefit_seeds:
            _art = prefit_ab_results[_mode][_seed]
            _bh = _art['blend_history']
            _at_best = _bh[int(_art['best_epoch'])]
            _at_best_no_epoch = {k: v for k, v in _at_best.items() if k != 'epoch'}
            print(f"  {_mode:<18s} {int(_seed):>5d}  "
                  + "  ".join(f"{float(_at_best_no_epoch[g]):>10.3f}" for g in _groups))

    # ---- Verdict vs baseline ----
    _direct_row = prefit_ab_summary_df[prefit_ab_summary_df['mode'] == 'direct'].iloc[0]
    _direct_ens = float(_direct_row['ensemble_eRMSE'])
    _direct_std = float(_direct_row['seed_std_eRMSE'])
    _direct_stderr = _direct_std / (len(_prefit_seeds) ** 0.5)
    print(f'\nBaseline direct ensemble mean_eRMSE: {_direct_ens:.3f}  '
          f'(seed std {_direct_std:.3f}, ensemble stderr ~ {_direct_stderr:.3f})')
    for _mode in ('prefit_freeze', 'prefit_warmstart'):
        _row = prefit_ab_summary_df[prefit_ab_summary_df['mode'] == _mode].iloc[0]
        _delta = float(_row['ensemble_eRMSE']) - _direct_ens
        _rel = 100.0 * _delta / max(_direct_ens, 1e-30)
        _sig = abs(_delta) / max(_direct_stderr, 1e-9)
        print(f'  {_mode} vs direct: {_delta:+.3f} ({_rel:+.1f}%), ~{_sig:.1f}sigma')


In [ ]:
# A/B: moon group compression level. The current compressor uses PCA + xarm>0.55
# and retains 28 of 29 spline coefficients, so the "PCA + cross-arm selection"
# path is essentially a no-op for moon. Historically the effective transferable
# rank was ~4. This cell first prints the current moon xarm profile as a
# diagnostic, then A/Bs several retention levels at 4 seeds each. Moon-only
# compressor is swapped; the other four groups keep their fitted state.
import contextlib as _mc_contextlib
import io as _mc_io
import copy as _mc_copy
import numpy as np, pandas as pd

required = ['filtered_triplet', 'group_compressors', 'compress_geom_kwargs',
            '_group_indices_compress', 'train_compressed_group_mlp',
            'predict_sci_coefficients_default', '_metric_row',
            'default_dual_group_config',
            'coef_near_all', 'coef_far_all', 'coef_sci_all',
            'ctx_near_all', 'ctx_far_all', 'ctx_sci_all', 'test_idx']
_missing = [k for k in required if k not in globals()]
if _missing:
    raise RuntimeError('Run prerequisite cells first. Missing: ' + ', '.join(_missing))

RUN_MOON_COMPRESSION_AB = False
if not RUN_MOON_COMPRESSION_AB:
    print('Moon-compression A/B skipped. Set RUN_MOON_COMPRESSION_AB = False to run.')
else:
    # --- 1. Diagnostic: current moon xarm correlation profile ---
    _base_moon = group_compressors['moon']
    if _base_moon.get('use_pca') and 'xarm_full' in _base_moon:
        _xarm_moon = np.abs(np.asarray(_base_moon['xarm_full'], dtype=np.float64))
        _order = np.argsort(_xarm_moon)[::-1]
        print(f'Moon xarm correlation profile '
              f'({len(_xarm_moon)} components, sorted by |xarm|):')
        print(f"  {'rank':>4s} {'|xarm|':>7s}  cutoff")
        for i, idx in enumerate(_order):
            _tag = ''
            if i + 1 == 4:
                _tag = '  <-- top-4 (historical target)'
            if _xarm_moon[idx] > 0.99 and (i + 1 == np.sum(_xarm_moon > 0.99)):
                _tag += '  <-- last of |xarm|>0.99'
            if _xarm_moon[idx] > 0.90 and (i + 1 == np.sum(_xarm_moon > 0.90)):
                _tag += '  <-- last of |xarm|>0.90'
            if _xarm_moon[idx] > 0.55 and (i + 1 == np.sum(_xarm_moon > 0.55)):
                _tag += '  <-- last of |xarm|>0.55 (current default)'
            print(f'  {i+1:>4d} {_xarm_moon[idx]:>7.3f}{_tag}')
        print()

    # --- 2. Build variant compressors (only 'moon' changes; other 4 groups reuse fits) ---
    def _make_moon_variant(mode, param):
        c = _mc_copy.deepcopy(_base_moon)
        if mode == 'no_pca':
            c['use_pca'] = False
            c['kept'] = np.arange(int(c['n_coef_group']), dtype=int)
            c['xarm_kept'] = np.full(int(c['n_coef_group']), np.nan)
            c['basis'] = np.eye(int(c['n_coef_group']))
        elif mode == 'xarm_threshold':
            xa = np.abs(np.asarray(c['xarm_full'], dtype=np.float64))
            c['kept'] = np.flatnonzero(xa > float(param))
            if c['kept'].size == 0:
                c['kept'] = np.argsort(xa)[::-1][:1]  # at least one
            c['xarm_kept'] = xa[c['kept']]
        elif mode == 'top_k':
            xa = np.abs(np.asarray(c['xarm_full'], dtype=np.float64))
            k = int(param)
            c['kept'] = np.argsort(xa)[::-1][:k]
            c['xarm_kept'] = xa[c['kept']]
        else:
            raise ValueError(f'unknown moon variant mode: {mode!r}')
        return c

    _variants = [
        ('no_pca_all_29',           _make_moon_variant('no_pca', None)),
        ('xarm_0.55_current',       _mc_copy.deepcopy(_base_moon)),
        ('xarm_0.90',               _make_moon_variant('xarm_threshold', 0.90)),
        ('xarm_0.99',               _make_moon_variant('xarm_threshold', 0.99)),
        ('top_4_by_xarm',           _make_moon_variant('top_k', 4)),
    ]

    print('Variant retention:')
    print(f"  {'variant':<24s} {'n_kept':>7s}  PCA?")
    for _lbl, _cv in _variants:
        _n = int(np.asarray(_cv['kept']).size)
        _pca_flag = 'yes' if _cv.get('use_pca') else 'no'
        print(f'  {_lbl:<24s} {_n:>7d}   {_pca_flag}')

    # --- 3. Retrain each variant at 4 seeds and evaluate on test split ---
    _seeds = [42, 43, 44, 45]
    _split = (
        filtered_triplet['compress_train_idx'],
        filtered_triplet['compress_val_idx'],
        filtered_triplet['compress_test_idx'],
    )
    _base_kwargs = dict(
        n_epochs=int(default_dual_group_config['n_epochs']),
        batch_size=int(default_dual_group_config['batch_size']),
        lr=float(default_dual_group_config['lr']),
        encoder_dims=tuple(int(v) for v in default_dual_group_config['encoder_dims']),
        ctx_dims=tuple(int(v) for v in default_dual_group_config['ctx_dims']),
        trunk_dims=tuple(int(v) for v in default_dual_group_config['trunk_dims']),
        head_dim=int(default_dual_group_config['head_dim']),
        weight_decay=float(default_dual_group_config['weight_decay']),
        patience=int(default_dual_group_config['patience']),
        moon_group_weight=float(default_dual_group_config['moon_group_weight']),
        continuum_group_weight=float(default_dual_group_config['continuum_group_weight']),
        mesospheric_group_weight=float(default_dual_group_config['mesospheric_group_weight']),
        ionospheric_group_weight=float(default_dual_group_config['ionospheric_group_weight']),
        blend_optim='direct',
        blend_init_alpha=0.7,
    )
    _te = np.asarray(test_idx, dtype=int)
    _y_te = coef_sci_all[_te]

    moon_ab_results = {}
    for _label, _moon_comp_v in _variants:
        moon_ab_results[_label] = {}
        _compressors_v = dict(group_compressors)
        _compressors_v['moon'] = _moon_comp_v
        print(f'\n--- {_label} (n_moon_kept={int(_moon_comp_v["kept"].size)}) ---')
        for _seed in _seeds:
            with _mc_contextlib.redirect_stdout(_mc_io.StringIO()):
                _art = train_compressed_group_mlp(
                    filtered_triplet, _compressors_v, _group_indices_compress,
                    compress_geom_kwargs,
                    split_indices=_split,
                    seed=int(_seed),
                    **_base_kwargs,
                )
            moon_ab_results[_label][_seed] = _art
            _pred = predict_sci_coefficients_default(
                _art,
                coef_near_phys=coef_near_all[_te], coef_far_phys=coef_far_all[_te],
                ctx_near_phys=ctx_near_all[_te], ctx_far_phys=ctx_far_all[_te],
                ctx_sci_phys=ctx_sci_all[_te]).astype(np.float32)
            _m = _metric_row(_y_te, _pred, f'{_label}_s{_seed}')
            print(f'  seed={_seed}: best_ep={_art["best_epoch"]:3d}  '
                  f'val_loss={_art["best_val_loss"]:.6f}  '
                  f'mean_eRMSE={float(_m["mean_eRMSE"]):.4f}')

    # --- 4. Per-variant 4-seed ensemble summary + per-group RMSE breakdown ---
    _moon_col_idx = np.asarray(_group_indices_compress['moon'], dtype=int)
    _summary_rows = []
    for _label, _moon_comp_v in _variants:
        _preds = {s: predict_sci_coefficients_default(
            moon_ab_results[_label][s],
            coef_near_phys=coef_near_all[_te], coef_far_phys=coef_far_all[_te],
            ctx_near_phys=ctx_near_all[_te], ctx_far_phys=ctx_far_all[_te],
            ctx_sci_phys=ctx_sci_all[_te]).astype(np.float32) for s in _seeds}
        _per_seed_rmse = [float(_metric_row(_y_te, _preds[s], 'x')['mean_eRMSE'])
                          for s in _seeds]
        _ens = np.mean(np.stack([_preds[s] for s in _seeds]),
                       axis=0).astype(np.float32)
        _ens_m = _metric_row(_y_te, _ens, f'ens_{_label}')
        # Per-group RMSE on the ensemble prediction.
        _rmse_by_grp = {}
        for _gname, _gidx in _group_indices_compress.items():
            _gidx = np.asarray(_gidx, dtype=int)
            _rmse_j = np.sqrt(np.mean(
                (_ens[:, _gidx].astype(np.float64)
                 - _y_te[:, _gidx].astype(np.float64)) ** 2, axis=0))
            _rmse_by_grp[f'ens_rmse_{_gname}'] = float(np.mean(_rmse_j))
        _summary_rows.append({
            'variant': _label,
            'n_moon_kept': int(_moon_comp_v['kept'].size),
            'seed_mean_eRMSE': float(np.mean(_per_seed_rmse)),
            'seed_std_eRMSE': float(np.std(_per_seed_rmse, ddof=1)),
            'ensemble_eRMSE': float(_ens_m['mean_eRMSE']),
            'ensemble_median_corr': float(_ens_m['median_corr']),
            **_rmse_by_grp,
        })
    moon_ab_summary_df = pd.DataFrame(_summary_rows)
    print('\n4-seed ensemble summary (per-variant):')
    print(moon_ab_summary_df.to_string(index=False,
          float_format=lambda v: f'{v:.4f}'))

    # --- 5. Verdict against the current default ---
    _baseline_row = moon_ab_summary_df[moon_ab_summary_df['variant']
                                       == 'xarm_0.55_current'].iloc[0]
    _base_ens = float(_baseline_row['ensemble_eRMSE'])
    _base_std = float(_baseline_row['seed_std_eRMSE'])
    _base_stderr = _base_std / (len(_seeds) ** 0.5)
    print(f'\nBaseline (xarm_0.55_current, n_kept={int(_baseline_row["n_moon_kept"])}): '
          f'ensemble mean_eRMSE = {_base_ens:.3f}, ensemble stderr ~ {_base_stderr:.3f}')
    for _lbl in ('no_pca_all_29', 'xarm_0.90', 'xarm_0.99', 'top_4_by_xarm'):
        _row = moon_ab_summary_df[moon_ab_summary_df['variant'] == _lbl].iloc[0]
        _delta = float(_row['ensemble_eRMSE']) - _base_ens
        _rel = 100.0 * _delta / max(_base_ens, 1e-30)
        _sig = abs(_delta) / max(_base_stderr, 1e-9)
        print(f"  {_lbl:<24s} (n_kept={int(_row['n_moon_kept']):>3d}): "
              f'{_delta:+.3f} ({_rel:+.1f}%), ~{_sig:.1f}sigma  '
              f"[moon_rmse={float(_row['ens_rmse_moon']):.3f}]")


In [ ]:
# Naive baselines vs the ML model on the same night-held-out test split (§11.1).
#
#   B0  copy_near        hat{c} = coef_near                                (no physics)
#   B1  near_geo         hat{c} = em_near * G_sci                          (near arm re-projected onto sci geometry)
#   B2  mean_geo         hat{c} = 0.5 * (em_near + em_far) * G_sci         (geometry-corrected symmetric average)
#   ML_default           coef_pred_det (single-seed model)
#   ML_ensemble          arithmetic-mean of the 4-seed ensemble (if multi-seed cell has run)
#
# em_arm = coef_arm / G_arm; G_arm = airglow_geometry_scale(ctx_arm).  G_arm is exactly 1 for
# the moon and other non-airglow groups, so B1's moon prediction reduces to `coef_near` and
# B2's moon prediction is the plain arithmetic mean of the two sky arms.

required = ['filtered_triplet', 'compress_geom_kwargs', '_group_indices_compress',
            'mlp_artifacts', 'cmp_df', '_metric_row',
            'coef_near_all', 'coef_far_all', 'coef_sci_all',
            'ctx_near_all', 'ctx_far_all', 'ctx_sci_all',
            'test_idx', 'coef_pred_det']
_missing = [k for k in required if k not in globals()]
if _missing:
    raise RuntimeError('Run prerequisite cells first. Missing: ' + ', '.join(_missing))

_te_bl = np.asarray(test_idx, dtype=int)
_ctx_near_te = np.asarray(ctx_near_all[_te_bl], dtype=np.float64)
_ctx_far_te  = np.asarray(ctx_far_all[_te_bl],  dtype=np.float64)
_ctx_sci_te  = np.asarray(ctx_sci_all[_te_bl],  dtype=np.float64)
_coef_near_te = np.asarray(coef_near_all[_te_bl], dtype=np.float64)
_coef_far_te  = np.asarray(coef_far_all[_te_bl],  dtype=np.float64)
_coef_sci_te  = np.asarray(coef_sci_all[_te_bl],  dtype=np.float64)

_G_near = airglow_geometry_scale(_ctx_near_te, **compress_geom_kwargs)
_G_far  = airglow_geometry_scale(_ctx_far_te,  **compress_geom_kwargs)
_G_sci  = airglow_geometry_scale(_ctx_sci_te,  **compress_geom_kwargs)

_em_near = _coef_near_te / _G_near
_em_far  = _coef_far_te  / _G_far

_all_preds = {
    'B0_copy_near': np.clip(_coef_near_te,                       0.0, None).astype(np.float32),
    'B1_near_geo':  np.clip(_em_near * _G_sci,                   0.0, None).astype(np.float32),
    'B2_mean_geo':  np.clip(0.5 * (_em_near + _em_far) * _G_sci, 0.0, None).astype(np.float32),
    # ML_default is the 4-seed ensemble prediction (see trainer cell + §12, 2026-08-11).
    'ML_default':   np.asarray(coef_pred_det, dtype=np.float32),
}

_y_te = _coef_sci_te.astype(np.float32)

# 2026-08-16: sigma-aware _metric_row -> mean/median/total WRMSE columns.
_sig_te = (coef_err_sci_all[test_idx].astype(np.float32)
           if 'coef_err_sci_all' in globals() else None)
_floor_te = (dict(DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP)
              if 'DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP' in globals() else None)
_rows = [{'variant': name,
          **_metric_row(_y_te, pred, name,
                        sigma=_sig_te,
                        group_indices=_group_indices_compress,
                        floor_by_group=_floor_te)}
         for name, pred in _all_preds.items()]
_summary_df = pd.DataFrame(_rows)

print('=' * 78)
print(f'Naive baselines vs ML on test split ({_te_bl.size} rows, {_y_te.shape[1]} coefficients)')
print('=' * 78)
print(_summary_df.to_string(index=False, float_format=lambda v: f'{v:.6g}'))

# Per-group mean coefficient bias.
_bias_rows = []
for gname, idx in _group_indices_compress.items():
    idx = np.asarray(idx, dtype=int)
    _mean_true = float(np.mean(_y_te[:, idx]))
    row = {'group': gname, 'n': int(idx.size), 'mean_true': _mean_true}
    for name, pred in _all_preds.items():
        row[f'bias_{name}_%'] = (
            100.0 * (float(np.mean(pred[:, idx])) - _mean_true)
            / max(abs(_mean_true), 1e-30))
    _bias_rows.append(row)
print()
print('Per-group mean coefficient bias on test rows (positive = over-predict, %):')
print(pd.DataFrame(_bias_rows).to_string(index=False, float_format=lambda v: f'{v:.3g}'))

# Per-group mean per-coefficient RMSE.
_rmse_group_rows = []
for gname, idx in _group_indices_compress.items():
    idx = np.asarray(idx, dtype=int)
    row = {'group': gname, 'n': int(idx.size)}
    for name, pred in _all_preds.items():
        _rmse_j = np.sqrt(np.mean(
            (pred[:, idx].astype(np.float64) - _y_te[:, idx].astype(np.float64)) ** 2,
            axis=0))
        row[f'eRMSE_{name}'] = float(np.mean(_rmse_j))
    _rmse_group_rows.append(row)
print()
print('Per-group mean per-coefficient eRMSE (lower is better):')
print(pd.DataFrame(_rmse_group_rows).to_string(index=False, float_format=lambda v: f'{v:.4g}'))

# Verdict against the best naive baseline.
# 2026-08-14 update: use median_eRMSE as the primary merit metric because
# mean_eRMSE is dominated by a small coefficient tail.
_bl_rows = _summary_df[~_summary_df['variant'].str.startswith('ML_')].copy()
_ml_row  = _summary_df[_summary_df['variant'] == 'ML_default'].iloc[0]
_best_bl = _bl_rows.sort_values(['median_eRMSE', 'mean_eRMSE']).iloc[0]

_ml_med_rmse = float(_ml_row['median_eRMSE'])
_best_bl_med_rmse = float(_best_bl['median_eRMSE'])
_gain_med_pct = 100.0 * (_best_bl_med_rmse - _ml_med_rmse) / max(_best_bl_med_rmse, 1e-30)

_ml_mean_rmse = float(_ml_row['mean_eRMSE'])
_best_bl_mean_rmse = float(_best_bl['mean_eRMSE'])
_gain_mean_pct = 100.0 * (_best_bl_mean_rmse - _ml_mean_rmse) / max(_best_bl_mean_rmse, 1e-30)

print()
print(f"Best naive baseline (by median_eRMSE): {_best_bl['variant']!r}")
print(f"  median_eRMSE={_best_bl_med_rmse:.4g}, mean_eRMSE={_best_bl_mean_rmse:.4g}, "
      f"median_corr={float(_best_bl['median_corr']):.4f}")
print("ML default:")
print(f"  median_eRMSE={_ml_med_rmse:.4g}, mean_eRMSE={_ml_mean_rmse:.4g}, "
      f"median_corr={float(_ml_row['median_corr']):.4f}")
print(f'ML improvement over best baseline: median_eRMSE {_gain_med_pct:+.1f}%  '
      f'(mean_eRMSE context: {_gain_mean_pct:+.1f}%)')

if _ml_med_rmse < _best_bl_med_rmse:
    print('Verdict (median_eRMSE-primary): the network earns its complexity on this test set.')
elif abs(_gain_med_pct) < 2.0:
    print('Verdict (median_eRMSE-primary): ML and best baseline are within noise; '
          'check per-group eRMSE and outlier-tail diagnostics for practical differences.')
else:
    print('Verdict (median_eRMSE-primary): the best naive baseline beats the ML model. '
          'Either the model overfits, the loss overfocuses a difficult tail, or '
          'the compressor discards directions the baseline preserves.')


In [ ]:
# Coefficient-vs-spectrum RMSE diagnostics: reconcile mean/median behavior.
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

required = ['coef_sci_all', 'coef_pred_det', 'test_idx']
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError('Run the training + prediction cells first. Missing: ' + ', '.join(missing))

# 1) Coefficient-space RMSE distribution on the night-held-out test split
_te = np.asarray(test_idx, dtype=int)
_y_true = np.asarray(coef_sci_all[_te], dtype=np.float64)
_y_pred = np.asarray(coef_pred_det, dtype=np.float64)
if _y_pred.shape != _y_true.shape:
    raise RuntimeError(f'coef_pred_det shape {_y_pred.shape} does not match y_true {_y_true.shape}')

rmse_coef = np.sqrt(np.mean((_y_pred - _y_true) ** 2, axis=0))
rmse_coef = rmse_coef[np.isfinite(rmse_coef)]
if rmse_coef.size == 0:
    raise RuntimeError('No finite eRMSE values')

# 2) Row-wise spectral RMSE + pixel-space WRMSE from the reconstruction subset cell
rmse_row_spec = None
wrmse_row_pix = None
if 'rmse_subset_results' in globals() and isinstance(rmse_subset_results, dict):
    _arr = np.asarray(rmse_subset_results.get('sci_rmse', []), dtype=np.float64)
    _arr = _arr[np.isfinite(_arr)]
    if _arr.size:
        rmse_row_spec = _arr
    _warr = np.asarray(rmse_subset_results.get('sci_wrmse', []), dtype=np.float64)
    _warr = _warr[np.isfinite(_warr)]
    if _warr.size:
        wrmse_row_pix = _warr

def _pct_table(arr, label):
    q = [0, 5, 25, 50, 75, 95, 99, 100]
    vals = np.percentile(arr, q)
    return pd.DataFrame({
        'metric': [label] * len(q),
        'percentile': q,
        'value': vals,
    })

tbl = [_pct_table(rmse_coef, 'eRMSE')]
if rmse_row_spec is not None:
    tbl.append(_pct_table(rmse_row_spec, 'sci_pRMSE'))
if wrmse_row_pix is not None:
    tbl.append(_pct_table(wrmse_row_pix, 'sci_pWRMSE'))
pct_df = pd.concat(tbl, ignore_index=True)

print('sRMSE / eRMSE / sWRMSE distribution summary:')
print(pct_df.to_string(index=False, float_format=lambda v: f'{v:.6g}'))

print('')
print('Per-coefficient (eRMSE) aggregate stats (Cell 33 now uses median_eRMSE as primary):')
print(f'  mean   = {float(np.mean(rmse_coef)):.6g}')
print(f'  median = {float(np.median(rmse_coef)):.6g}')
print(f'  p95    = {float(np.percentile(rmse_coef, 95)):.6g}')
print(f'  p99    = {float(np.percentile(rmse_coef, 99)):.6g}')
_tail_ratio = float(np.percentile(rmse_coef, 99) / max(np.median(rmse_coef), 1e-30))
print(f'  p99/median = {_tail_ratio:.3g}')
if _tail_ratio > 100.0:
    print('Loss note: extremely heavy-tailed coefficient errors. A pure MSE-style objective '
          'is likely over-weighting a small outlier set in model selection.')
elif _tail_ratio > 20.0:
    print('Loss note: heavy-tailed coefficient errors. MSE can still over-emphasize the tail; '
          'consider robust alternatives for selection or training.')
else:
    print('Loss note: tail is moderate; current loss likely not severely tail-dominated.')

if rmse_row_spec is not None:
    print('')
    print('Per-row pixel-space pRMSE stats (from rmse_subset_results["sci_rmse"]):')
    print(f'  mean   = {float(np.mean(rmse_row_spec)):.6g}')
    print(f'  median = {float(np.median(rmse_row_spec)):.6g}')
    print(f'  p95    = {float(np.percentile(rmse_row_spec, 95)):.6g}')
    print(f'  p99    = {float(np.percentile(rmse_row_spec, 99)):.6g}')
else:
    print('')
    print('Per-row sci_pRMSE not available: run the batch RMSE subset evaluation cell first.')

if wrmse_row_pix is not None:
    print('')
    print('Per-row pixel-space pWRMSE stats (from rmse_subset_results["sci_wrmse"]):')
    print(f'  mean   = {float(np.mean(wrmse_row_pix)):.6g}')
    print(f'  median = {float(np.median(wrmse_row_pix)):.6g}')
    print(f'  p95    = {float(np.percentile(wrmse_row_pix, 95)):.6g}')
    print(f'  p99    = {float(np.percentile(wrmse_row_pix, 99)):.6g}')
    _pix_src = (rmse_subset_results.get('pix_sigma_source', {})
                if isinstance(rmse_subset_results, dict) else {})
    if _pix_src:
        print(f'  sigma sources: {_pix_src}')
else:
    print('')
    print('Per-row sci_pWRMSE not available: batch RMSE subset cell has not populated sci_wrmse.')

# 2b) Worst coefficients by RMSE (largest errors)
n_worst_coef = 15
rmse_coef_full = np.sqrt(np.mean((_y_pred - _y_true) ** 2, axis=0))
coef_idx = np.arange(rmse_coef_full.size, dtype=int)

if 'coef_names_all' in globals() and len(coef_names_all) == rmse_coef_full.size:
    coef_names = [str(x) for x in coef_names_all]
elif 'filtered_triplet' in globals() and 'coef_names' in filtered_triplet and len(filtered_triplet['coef_names']) == rmse_coef_full.size:
    coef_names = [str(x) for x in filtered_triplet['coef_names']]
else:
    coef_names = [f'coef_{j}' for j in coef_idx]

worst_idx = np.argsort(rmse_coef_full)[::-1][:min(n_worst_coef, rmse_coef_full.size)]
worst_df = pd.DataFrame({
    'rank': np.arange(1, worst_idx.size + 1, dtype=int),
    'coef_index': coef_idx[worst_idx],
    'coef_name': [coef_names[j] for j in worst_idx],
    'rmse': rmse_coef_full[worst_idx],
    'true_mean': np.mean(_y_true[:, worst_idx], axis=0),
    'pred_mean': np.mean(_y_pred[:, worst_idx], axis=0),
    'mean_bias': np.mean(_y_pred[:, worst_idx] - _y_true[:, worst_idx], axis=0),
})
print('')
print(f'Top {worst_idx.size} worst coefficients by eRMSE (test split):')
print(worst_df.to_string(index=False, float_format=lambda v: f'{v:.6g}'))

# 3) Side-by-side histograms: coefficient RMSE, spectral RMSE, pixel WRMSE.
fig = make_subplots(rows=1, cols=3, subplot_titles=(
    'Per-coefficient eRMSE (test split)',
    'Per-row pixel pRMSE (subset)',
    'Per-row pixel pWRMSE (subset)',
))

fig.add_trace(
    go.Histogram(x=rmse_coef, nbinsx=80, name='eRMSE', marker_color='#1f77b4', opacity=0.85),
    row=1, col=1,
)
fig.add_vline(x=float(np.median(rmse_coef)), line=dict(color='#1f77b4', dash='dash'), row=1, col=1)
fig.add_vline(x=float(np.mean(rmse_coef)), line=dict(color='#d62728', dash='dot'), row=1, col=1)

if rmse_row_spec is not None:
    fig.add_trace(
        go.Histogram(x=rmse_row_spec, nbinsx=50, name='sci_pRMSE', marker_color='#2ca02c', opacity=0.85),
        row=1, col=2,
    )
    fig.add_vline(x=float(np.median(rmse_row_spec)), line=dict(color='#2ca02c', dash='dash'), row=1, col=2)
    fig.add_vline(x=float(np.mean(rmse_row_spec)), line=dict(color='#d62728', dash='dot'), row=1, col=2)
else:
    fig.add_annotation(x=0.5, y=0.5, xref='x2 domain', yref='y2 domain',
                       text='Run RMSE subset cell to populate this panel', showarrow=False)

if wrmse_row_pix is not None:
    fig.add_trace(
        go.Histogram(x=wrmse_row_pix, nbinsx=50, name='sci_pWRMSE',
                     marker_color='#ff7f0e', opacity=0.85),
        row=1, col=3,
    )
    fig.add_vline(x=float(np.median(wrmse_row_pix)), line=dict(color='#ff7f0e', dash='dash'), row=1, col=3)
    fig.add_vline(x=float(np.mean(wrmse_row_pix)), line=dict(color='#d62728', dash='dot'), row=1, col=3)
else:
    fig.add_annotation(x=0.5, y=0.5, xref='x3 domain', yref='y3 domain',
                       text='Run RMSE subset cell to populate this panel',
                       showarrow=False)

fig.update_layout(
    template='plotly_white',
    barmode='overlay',
    height=460,
    title='Error diagnostics: coefficient-space eRMSE, pixel-space pRMSE, pixel-space pWRMSE',
    margin=dict(l=60, r=20, t=70, b=55),
)
fig.update_xaxes(title_text='eRMSE value', row=1, col=1)
fig.update_xaxes(title_text='sRMSE value', row=1, col=2)
fig.update_xaxes(title_text='sWRMSE value', row=1, col=3)
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.update_yaxes(title_text='Count', row=1, col=2)
fig.update_yaxes(title_text='Count', row=1, col=3)
fig.show()


In [ ]:
# Stability check: are the current top-15 worst coefficients always bad?
import numpy as np
import pandas as pd
import plotly.graph_objects as go

required = ['coef_sci_all', 'test_idx']
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError('Run the training + prediction cells first. Missing: ' + ', '.join(missing))

N_EXPERIMENTS = 10
N_WORST = 15
RNG_SEED = 12345

y_true_all = np.asarray(coef_sci_all, dtype=np.float64)
n_rows, n_coef = y_true_all.shape

# Prefer full-dataset predictions if available so random splits are cheap.
if 'coef_pred_all' in globals() and np.asarray(coef_pred_all).shape == y_true_all.shape:
    y_pred_all = np.asarray(coef_pred_all, dtype=np.float64)
else:
    req_pred = ['predict_sci_coefficients_default', 'mlp_artifacts',
                'coef_near_all', 'coef_far_all', 'ctx_near_all', 'ctx_far_all', 'ctx_sci_all']
    miss_pred = [k for k in req_pred if k not in globals()]
    if miss_pred:
        raise RuntimeError('Need predictions on all rows for randomized experiments. Missing: ' + ', '.join(miss_pred))
    y_pred_all = predict_sci_coefficients_default(
        mlp_artifacts,
        coef_near_phys=np.asarray(coef_near_all, dtype=np.float32),
        coef_far_phys=np.asarray(coef_far_all, dtype=np.float32),
        ctx_near_phys=np.asarray(ctx_near_all, dtype=np.float32),
        ctx_far_phys=np.asarray(ctx_far_all, dtype=np.float32),
        ctx_sci_phys=np.asarray(ctx_sci_all, dtype=np.float32),
    ).astype(np.float64)

if y_pred_all.shape != y_true_all.shape:
    raise RuntimeError(f'Prediction shape mismatch: y_pred_all={y_pred_all.shape}, y_true_all={y_true_all.shape}')

test_idx_arr = np.asarray(test_idx, dtype=int)
if test_idx_arr.size == 0:
    raise RuntimeError('test_idx is empty')
n_test = int(test_idx_arr.size)

# Baseline top-15 worst coefficients from the current held-out test split.
rmse_base = np.sqrt(np.mean((y_pred_all[test_idx_arr] - y_true_all[test_idx_arr]) ** 2, axis=0))
base_worst_idx = np.argsort(rmse_base)[::-1][:min(N_WORST, n_coef)]

if 'coef_names_all' in globals() and len(coef_names_all) == n_coef:
    coef_names = [str(x) for x in coef_names_all]
elif 'filtered_triplet' in globals() and 'coef_names' in filtered_triplet and len(filtered_triplet['coef_names']) == n_coef:
    coef_names = [str(x) for x in filtered_triplet['coef_names']]
else:
    coef_names = [f'coef_{j}' for j in range(n_coef)]

rng = np.random.default_rng(RNG_SEED)
all_top_sets = []
all_top_ranks = []

for exp in range(N_EXPERIMENTS):
    idx = rng.choice(n_rows, size=n_test, replace=False)
    rmse = np.sqrt(np.mean((y_pred_all[idx] - y_true_all[idx]) ** 2, axis=0))
    top_idx = np.argsort(rmse)[::-1][:min(N_WORST, n_coef)]
    all_top_sets.append(set(int(j) for j in top_idx))
    all_top_ranks.append({int(j): (r + 1) for r, j in enumerate(top_idx)})

rows = []
for j in base_worst_idx:
    appears = [j in s for s in all_top_sets]
    count = int(np.sum(appears))
    ranks = [d[j] for d in all_top_ranks if j in d]
    rows.append({
        'coef_index': int(j),
        'coef_name': coef_names[int(j)],
        'baseline_rmse': float(rmse_base[int(j)]),
        'appear_count': count,
        'appear_frac': float(count / N_EXPERIMENTS),
        'always_bad': bool(count == N_EXPERIMENTS),
        'mean_rank_when_present': (float(np.mean(ranks)) if ranks else np.nan),
    })

stability_df = (pd.DataFrame(rows)
                .sort_values(['appear_count', 'baseline_rmse'], ascending=[False, False])
                .reset_index(drop=True))

print(f'Randomized test-index stability check: N_EXPERIMENTS={N_EXPERIMENTS}, N_WORST={N_WORST}, n_test={n_test}')
print('Baseline set = top-15 worst coefficients from the current held-out test split.')
print('')
print(stability_df.to_string(index=False, float_format=lambda v: f'{v:.6g}'))
print('')
_n_always = int(stability_df['always_bad'].sum())
print(f'Always bad (present in top-{N_WORST} for all {N_EXPERIMENTS} experiments): {_n_always}/{len(stability_df)}')

# Also report any coefficients that were consistently bad even if not in the baseline top-15.
global_counts = np.zeros(n_coef, dtype=int)
for s in all_top_sets:
    for j in s:
        global_counts[j] += 1
always_global = np.flatnonzero(global_counts == N_EXPERIMENTS)
if always_global.size:
    _base_set = set(int(x) for x in base_worst_idx.tolist())
    global_df = pd.DataFrame({
        'coef_index': always_global.astype(int),
        'coef_name': [coef_names[int(j)] for j in always_global],
        'appear_count': global_counts[always_global],
        'baseline_in_top15': [bool(int(j) in _base_set) for j in always_global],
    }).sort_values('coef_index').reset_index(drop=True)
    print('')
    print('Coefficients that are in top-15 worst for ALL randomized experiments:')
    print(global_df.to_string(index=False))
else:
    print('')
    print('No coefficient is in the top-15 worst set for all randomized experiments.')

# Quick visual: appearance count for the baseline top-15 coefficients.
fig = go.Figure()
fig.add_trace(go.Bar(
    x=stability_df['coef_name'],
    y=stability_df['appear_count'],
    marker_color=['#1f77b4' if not a else '#d62728' for a in stability_df['always_bad']],
    text=stability_df['appear_count'],
    textposition='outside',
))
fig.update_layout(
    template='plotly_white',
    title=f'How often baseline top-{N_WORST} coefficients stay in top-{N_WORST} (N={N_EXPERIMENTS} randomized tests)',
    xaxis_title='Coefficient',
    yaxis_title='Appearance count across experiments',
    yaxis=dict(range=[0, N_EXPERIMENTS + 1]),
    height=420,
    margin=dict(l=60, r=20, t=70, b=120),
)
fig.show()

rmse_worst_stability_results = {
    'N_EXPERIMENTS': N_EXPERIMENTS,
    'N_WORST': N_WORST,
    'n_test': n_test,
    'baseline_worst_idx': base_worst_idx,
    'stability_df': stability_df,
    'global_counts': global_counts,
}


In [ ]:
# Per-lunation bias drift on the test split (item 14 of the review).
# Aerosol optical depth varies on ~lunation timescales; the fitted k_eff (§4.5)
# uses one value averaged over the training window, so a systematic per-lunation
# drift in the per-group mean coefficient bias would flag that k_eff is stale
# on the deployment nights and motivate a rolling-window refit (§11.3 open).

required = ['filtered_triplet', 'mlp_artifacts', 'coef_pred_det',
            '_group_indices_compress', 'coef_sci_all', 'test_idx']
_missing = [k for k in required if k not in globals()]
if _missing:
    raise RuntimeError('Run prerequisite cells first. Missing: ' + ', '.join(_missing))

_te_ll = np.asarray(test_idx, dtype=int)
_mjd_te = np.asarray(filtered_triplet['obstime_mjd'], dtype=np.float64)[_te_ll]
_LUNATION_DAYS = 29.53058867
_lun_bin = np.floor(_mjd_te / _LUNATION_DAYS).astype(int)

_y_te_ll = coef_sci_all[_te_ll].astype(np.float64)
_pred_te_ll = np.asarray(coef_pred_det, dtype=np.float64)

# Small bins are dominated by shot noise on the bias estimate; require enough rows
# per bin that a 3% group-mean bias is resolvable above sampling variance.
_MIN_ROWS_PER_LUN = 30
_lun_rows = []
for _lid in np.unique(_lun_bin):
    _mask = _lun_bin == _lid
    if int(_mask.sum()) < _MIN_ROWS_PER_LUN:
        continue
    _row = {
        'lunation': int(_lid),
        'mjd_mid': float((float(_lid) + 0.5) * _LUNATION_DAYS),
        'n_rows': int(_mask.sum()),
    }
    _y_lun = _y_te_ll[_mask]
    _p_lun = _pred_te_ll[_mask]
    for _gname, _idx in _group_indices_compress.items():
        _idx = np.asarray(_idx, dtype=int)
        _mt = float(np.mean(_y_lun[:, _idx]))
        _mp = float(np.mean(_p_lun[:, _idx]))
        _row[f'bias_{_gname}_%'] = (100.0 * (_mp - _mt)
                                    / max(abs(_mt), 1e-30))
    _lun_rows.append(_row)

lunation_bias_df = (pd.DataFrame(_lun_rows).sort_values('lunation')
                     .reset_index(drop=True))
print(f'Per-lunation bias drift on test split ({_te_ll.size} rows total, '
      f'{len(lunation_bias_df)} lunation bins with >= {_MIN_ROWS_PER_LUN} rows):')
print(lunation_bias_df.to_string(index=False, float_format=lambda v: f'{v:.3g}'))

_bias_cols = [c for c in lunation_bias_df.columns if c.startswith('bias_')]
_group_ord = [c[len('bias_'):-len('_%')] for c in _bias_cols]

_fig_lun = go.Figure()
for _g in _group_ord:
    _fig_lun.add_trace(go.Scatter(
        x=lunation_bias_df['mjd_mid'],
        y=lunation_bias_df[f'bias_{_g}_%'],
        mode='lines+markers', name=_g))
_fig_lun.add_hline(y=0.0, line=dict(color='rgba(0,0,0,0.4)', width=0.8, dash='dash'))
_fig_lun.update_layout(
    template='plotly_white',
    title='Per-group mean coefficient bias vs lunation on the test split '
          '(item 14: k_eff staleness diagnostic)',
    xaxis_title='mid-lunation MJD',
    yaxis_title='per-group mean coefficient bias (%)',
    height=460,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0.0),
)
_fig_lun.show()

# Linear-trend summary: significant slope means k_eff is drifting through the
# test window; a large range without a slope means episodic aerosol events.
print()
print('Per-group linear trend of bias(%) vs mid-lunation MJD:')
print(f"  {'group':<14s} {'slope [%/day]':>14s} {'r':>7s} "
      f"{'range [%]':>10s} {'max |bias| [%]':>15s}")
_xt = np.asarray(lunation_bias_df['mjd_mid'], dtype=np.float64)
_absmax_over_all = 0.0
for _g in _group_ord:
    _yt = np.asarray(lunation_bias_df[f'bias_{_g}_%'], dtype=np.float64)
    if _yt.size < 3 or float(np.std(_xt)) < 1e-6:
        continue
    _slope = float(np.polyfit(_xt, _yt, 1)[0])
    _r = (float(np.corrcoef(_xt, _yt)[0, 1])
          if float(np.std(_yt)) > 1e-12 else 0.0)
    _rng = float(_yt.max() - _yt.min())
    _absmax = float(np.max(np.abs(_yt)))
    _absmax_over_all = max(_absmax_over_all, _absmax)
    print(f'  {_g:<14s} {_slope:>+14.3g} {_r:>+7.3f} '
          f'{_rng:>10.2f} {_absmax:>15.2f}')

print()
if _absmax_over_all > 3.0:
    print(f'Verdict: at least one group hits per-lunation |bias| > 3% '
          f'(max = {_absmax_over_all:.2f}%). Refit k_eff on a rolling '
          f'~lunation window (§4.5, §11.3 open) before deploying on nights '
          f'outside the training epoch.')
else:
    print(f'Verdict: per-lunation |bias| stays below 3% across all groups '
          f'(max = {_absmax_over_all:.2f}%). k_eff appears stable across the '
          f'test window; no rolling refit needed at this level.')


In [ ]:
# Per-context-slice test metrics (item 15 of the review). A single aggregate
# mean_eRMSE hides the cases the sky-subtraction stage cares about most --
# bright time, moon-up, high-airmass. This cell splits the test set by
# moon_phase quartile, moon_alt sign, and airmass threshold and reports
# mean_eRMSE / median_corr / max per-group |bias| for each slice. Uses the
# ensemble prediction (coef_pred_det) already computed in the trainer cell.

required = ['filtered_triplet', 'mlp_artifacts', 'coef_pred_det',
            '_group_indices_compress', 'coef_sci_all', 'ctx_sci_all',
            'test_idx', '_metric_row', '_moon_phase_deg_from_ctx']
_missing = [k for k in required if k not in globals()]
if _missing:
    raise RuntimeError('Run prerequisite cells first. Missing: ' + ', '.join(_missing))

_te_slice = np.asarray(test_idx, dtype=int)
_ctx_names_ll = list(filtered_triplet['ctx_names'])
_ctx_sci_te = np.asarray(ctx_sci_all[_te_slice], dtype=np.float64)
_y_te_ll = coef_sci_all[_te_slice].astype(np.float32)
_pred_te_ll = np.asarray(coef_pred_det, dtype=np.float32)

# Decoded moon_phase in [0, 360). Reuses the helper the split code uses so the
# slice quartiles match the training split's phase-quantile convention.
_moon_phase_te = _moon_phase_deg_from_ctx(
    {'ctx_sci': _ctx_sci_te, 'ctx_names': _ctx_names_ll})
_moon_alt_te = _ctx_sci_te[:, _ctx_names_ll.index('moon_alt')]
_airmass_te = _ctx_sci_te[:, _ctx_names_ll.index('airmass')]


def _slice_row(name, mask):
    n = int(mask.sum())
    if n < 20:
        return {'slice': name, 'n_rows': n, 'mean_eRMSE': np.nan,
                'median_eRMSE': np.nan, 'median_corr': np.nan,
                'max_abs_bias_%': np.nan, 'worst_bias_group': ''}
    _y = _y_te_ll[mask]
    _p = _pred_te_ll[mask]
    _s = (coef_err_sci_all[test_idx][mask]
          if 'coef_err_sci_all' in globals() else None)
    _floor_g = (dict(DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP)
                 if 'DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP' in globals() else None)
    _m = _metric_row(_y, _p, name,
                     sigma=_s,
                     group_indices=_group_indices_compress,
                     floor_by_group=_floor_g)
    _max_abs = -np.inf
    _worst = ''
    for _gname, _idx in _group_indices_compress.items():
        _idx = np.asarray(_idx, dtype=int)
        _mt = float(np.mean(_y[:, _idx]))
        _mp = float(np.mean(_p[:, _idx]))
        _b = 100.0 * (_mp - _mt) / max(abs(_mt), 1e-30)
        if abs(_b) > _max_abs:
            _max_abs = abs(_b)
            _worst = _gname
    return {'slice': name, 'n_rows': n,
            'mean_eRMSE': float(_m['mean_eRMSE']),
            'median_eRMSE': float(_m['median_eRMSE']),
            'median_corr': float(_m['median_corr']),
            'max_abs_bias_%': float(_max_abs),
            'worst_bias_group': _worst}


_slices = [_slice_row('all_test', np.ones(_te_slice.size, dtype=bool))]

# Moon-phase quartiles across the actual test-set phase distribution.
_q = np.quantile(_moon_phase_te, [0.0, 0.25, 0.5, 0.75, 1.0])
for _k in range(4):
    _lo, _hi = float(_q[_k]), float(_q[_k + 1])
    _mask = ((_moon_phase_te >= _lo)
             & (_moon_phase_te <= _hi if _k == 3 else _moon_phase_te < _hi))
    _slices.append(_slice_row(
        f'moon_phase Q{_k + 1} [{_lo:.0f}-{_hi:.0f} deg]', _mask))

# Moon-alt sign: moon above/below horizon at the science pointing.
_slices.append(_slice_row('moon_alt > 0 (moon up)',    _moon_alt_te > 0.0))
_slices.append(_slice_row('moon_alt <= 0 (moon down)', _moon_alt_te <= 0.0))

# Airmass threshold: 1.5 corresponds to z ~ 48 deg, a common science limit.
_slices.append(_slice_row('airmass <= 1.5 (low)', _airmass_te <= 1.5))
_slices.append(_slice_row('airmass > 1.5 (high)', _airmass_te > 1.5))

slice_metrics_df = pd.DataFrame(_slices)
print(f'Per-context-slice test metrics on the ensemble prediction '
      f'({_te_slice.size} test rows total):')
print(slice_metrics_df.to_string(index=False, float_format=lambda v: f'{v:.4g}'))

# Compact deployment view: relative degradation vs the aggregate, with a flag
# on any slice that carries >20% higher mean_eRMSE or >2% |bias|.
_all = slice_metrics_df.iloc[0]
print(f"\nAggregate reference: n={int(_all['n_rows'])}, "
      f"mean_eRMSE={float(_all['mean_eRMSE']):.3f}, "
      f"median_corr={float(_all['median_corr']):.4f}, "
      f"max_abs_bias={float(_all['max_abs_bias_%']):.2f}% "
      f"({_all['worst_bias_group']})")
print('Slice deviations (>20% mean_eRMSE gap or >2% bias flagged):')
for _, r in slice_metrics_df.iloc[1:].iterrows():
    if r['n_rows'] < 20 or not np.isfinite(r['mean_eRMSE']):
        print(f"  {r['slice']:<40s} n={int(r['n_rows']):>4d}  "
              f'(too few rows, skipped)')
        continue
    _rel = float(r['mean_eRMSE']) / max(float(_all['mean_eRMSE']), 1e-9) - 1.0
    _flag = ''
    if abs(_rel) > 0.2 or float(r['max_abs_bias_%']) > 2.0:
        _flag = '  <-- flagged'
    print(f"  {r['slice']:<40s} n={int(r['n_rows']):>4d}  "
          f"rmse={float(r['mean_eRMSE']):.3f} ({100.0 * _rel:+.1f}%)  "
          f"median_corr={float(r['median_corr']):.4f}  "
          f"max_abs_bias={float(r['max_abs_bias_%']):.2f}% "
          f"({r['worst_bias_group']}){_flag}")


In [ ]:
# Item 6 (predictive uncertainty from ensemble spread). For each row + coefficient
# sigma_row = std_across_seeds(pred). z = (y - mean_pred) / sigma is the standardized
# residual: mean 0 / std 1 under a well-calibrated ensemble. §11.7 wants this before
# any downstream code trusts a sigma. Reports var(z), coverage, PIT chi2; a var(z) >> 1
# means the 4-seed spread only captures init noise, not epistemic + aleatoric error.

from scipy.special import erf as _erf_vec

required = ['filtered_triplet', 'mlp_artifacts', 'coef_sci_all',
            'coef_near_all', 'coef_far_all', 'ctx_sci_all', 'ctx_near_all',
            'ctx_far_all', 'val_idx', 'test_idx',
            '_group_indices_compress', 'predict_sci_coefficients_default']
_missing = [k for k in required if k not in globals()]
if _missing:
    raise RuntimeError('Run prerequisite cells first. Missing: ' + ', '.join(_missing))
if not mlp_artifacts.get('is_ensemble', False):
    raise RuntimeError('mlp_artifacts is not an ensemble; item 6 requires N>=2 members.')

_members = mlp_artifacts['members']
_seeds = list(mlp_artifacts['seeds'])
print(f'Item 6 -- predictive uncertainty from ensemble spread '
      f'(N={len(_members)} seeds: {_seeds}).')


def _predict_all_seeds(idx):
    _stack = np.stack([predict_sci_coefficients_default(
        _m,
        coef_near_phys=coef_near_all[idx], coef_far_phys=coef_far_all[idx],
        ctx_near_phys=ctx_near_all[idx], ctx_far_phys=ctx_far_all[idx],
        ctx_sci_phys=ctx_sci_all[idx],
    ).astype(np.float32) for _m in _members], axis=0)
    return _stack   # (N_seed, N_row, N_coef)


_val_idx_arr = np.asarray(val_idx, dtype=int)
_test_idx_arr = np.asarray(test_idx, dtype=int)

# Coefficients that are ~0 in the training data collapse sigma to ~0; floor per
# group at 1e-3 x MAD(coef_train) so a handful of dead coefficients don't push
# a few |z| values into 1e9 and blow up mean/std of z.
_train_idx_arr = np.asarray(train_idx, dtype=int)
_sigma_floor_by_group = {}
for _gname, _idx in _group_indices_compress.items():
    _idx = np.asarray(_idx, dtype=int)
    _train_coef = coef_sci_all[_train_idx_arr][:, _idx].astype(np.float32)
    _mad = np.median(np.abs(_train_coef - np.median(_train_coef, axis=0)), axis=0)
    _sigma_floor_by_group[_gname] = np.maximum(1e-3 * _mad, 1e-12).astype(np.float32)

_val_stack = _predict_all_seeds(_val_idx_arr)
_val_mean = _val_stack.mean(axis=0)
_val_sigma = _val_stack.std(axis=0, ddof=1)
for _gname, _idx in _group_indices_compress.items():
    _idx = np.asarray(_idx, dtype=int)
    _val_sigma[:, _idx] = np.maximum(_val_sigma[:, _idx], _sigma_floor_by_group[_gname])
_y_val = coef_sci_all[_val_idx_arr].astype(np.float32)
_z_val = (_y_val - _val_mean) / _val_sigma

_rows = []
for _gname, _idx in _group_indices_compress.items():
    _idx = np.asarray(_idx, dtype=int)
    _zg = _z_val[:, _idx].ravel()
    _sg = _val_sigma[:, _idx].ravel()
    _rg = (_y_val - _val_mean)[:, _idx].ravel()
    _true_rms = float(np.sqrt(np.mean(_rg ** 2)))
    _pred_rms = float(np.sqrt(np.mean(_sg ** 2)))
    _rows.append({
        'group': _gname,
        'n_pts': int(_zg.size),
        'median|z|': float(np.median(np.abs(_zg))),
        'MAD_z/0.6745': float(np.median(np.abs(_zg - np.median(_zg))) / 0.6745),
        'sigma_true/pred': _true_rms / max(_pred_rms, 1e-30),
        'cov_|z|<1.96': float(np.mean(np.abs(_zg) < 1.96)),
        'cov_|z|<1.00': float(np.mean(np.abs(_zg) < 1.00)),
    })
_z_all = _z_val.ravel()
_r_all = (_y_val - _val_mean).ravel()
_s_all = _val_sigma.ravel()
_rows.append({
    'group': 'all_groups',
    'n_pts': int(_z_all.size),
    'median|z|': float(np.median(np.abs(_z_all))),
    'MAD_z/0.6745':
        float(np.median(np.abs(_z_all - np.median(_z_all))) / 0.6745),
    'sigma_true/pred':
        float(np.sqrt(np.mean(_r_all ** 2))
              / max(np.sqrt(np.mean(_s_all ** 2)), 1e-30)),
    'cov_|z|<1.96': float(np.mean(np.abs(_z_all) < 1.96)),
    'cov_|z|<1.00': float(np.mean(np.abs(_z_all) < 1.00)),
})
ensemble_uncertainty_val_df = pd.DataFrame(_rows)
print(f'\nPer-group predictive-uncertainty calibration on val '
      f'(n_val = {_val_idx_arr.size} rows):')
print(ensemble_uncertainty_val_df.to_string(
    index=False, float_format=lambda v: f'{v:.4g}'))

# PIT histogram: u = Phi(z) should be U[0,1] under a well-calibrated Gaussian ensemble.
_u = 0.5 * (1.0 + _erf_vec(_z_all / np.sqrt(2.0)))
_hist_edges = np.linspace(0.0, 1.0, 21)
_hist, _ = np.histogram(_u, bins=_hist_edges)
_expected = _u.size / (len(_hist_edges) - 1)
_pit_chi2 = float(np.sum((_hist - _expected) ** 2 / max(_expected, 1e-30)))
print(f'\nPIT histogram (u = Phi(z), 20 bins on [0,1]; expected {_expected:.0f}/bin):')
_bar_max = max(int(_hist.max()), 1)
for _lo, _hi, _c in zip(_hist_edges[:-1], _hist_edges[1:], _hist):
    _bar = '#' * int(round(50.0 * _c / _bar_max))
    print(f'  [{_lo:.2f}, {_hi:.2f})  n={int(_c):>6d}  {_bar}')
print(f'PIT chi2 (20 bins): {_pit_chi2:.1f}  '
      f'(uniform target chi2_19 = 19 +/- 6; large values -> uncalibrated).')

# Headline transfer check on test (same per-group floor computed on train).
_test_stack = _predict_all_seeds(_test_idx_arr)
_test_mean = _test_stack.mean(axis=0)
_test_sigma = _test_stack.std(axis=0, ddof=1)
for _gname, _idx in _group_indices_compress.items():
    _idx = np.asarray(_idx, dtype=int)
    _test_sigma[:, _idx] = np.maximum(_test_sigma[:, _idx], _sigma_floor_by_group[_gname])
_y_test = coef_sci_all[_test_idx_arr].astype(np.float32)
_z_test = (_y_test - _test_mean) / _test_sigma
_r_test = _y_test - _test_mean
_true_rmse_test = float(np.sqrt(np.mean(_r_test ** 2)))
_pred_rmse_test = float(np.sqrt(np.mean(_test_sigma ** 2)))
_sigma_ratio = _true_rmse_test / max(_pred_rmse_test, 1e-30)

print(f'\nHeadline calibration on test (n_test = {_test_idx_arr.size} rows):')
_robust_std_val = float(np.median(np.abs(_z_val - np.median(_z_val))) / 0.6745)
_robust_std_test = float(np.median(np.abs(_z_test - np.median(_z_test))) / 0.6745)
print(f'  robust std(z) on val  (MAD/0.6745) = {_robust_std_val:.3f}')
print(f'  robust std(z) on test (MAD/0.6745) = {_robust_std_test:.3f}')
print(f'  true pooled RMSE (y - mean_pred)         = {_true_rmse_test:.4f}')
print(f'  ensemble-predicted sigma RMS           = {_pred_rmse_test:.4f}')
print(f'  sigma under-estimation factor (true/pred) = {_sigma_ratio:.2f}x')
print(f'  coverage |z|<1.96 (nominal 0.95) on test = '
      f'{float(np.mean(np.abs(_z_test) < 1.96)):.3f}')

if _sigma_ratio > 1.2:
    print('  Verdict: ensemble spread UNDER-estimates true error (factor > 1.2x). '
          'A per-group sigma_scale calibration or a deeper ensemble (bootstrap + '
          'longer training-init spread) is required before shipping sigma downstream. '
          'See item 7 (spectrum-space validation) and §11.7.')
elif _sigma_ratio < 0.8:
    print('  Verdict: ensemble spread OVER-estimates true error (factor < 0.8x). '
          'Sigma is safe as an upper bound but overly conservative.')
else:
    print('  Verdict: ensemble spread is within 20% of the true error scale. '
          'A single scalar sigma_scale per group should suffice for §11.7.')

# Save per-group sigma_scale factors (val -> test transfer) so downstream code
# can bolt them onto sigma_row with one multiplication.
_sigma_scale_by_group = {}
for _r in _rows:
    if _r['group'] == 'all_groups':
        continue
    _sigma_scale_by_group[_r['group']] = float(_r['sigma_true/pred'])
mlp_artifacts.setdefault('predictive_uncertainty', {}).update({
    'sigma_scale_by_group_val': _sigma_scale_by_group,
    'sigma_floor_by_group_train': {
        _g: _v.tolist() for _g, _v in _sigma_floor_by_group.items()},
    'val_robust_std_z': _robust_std_val,
    'test_robust_std_z': _robust_std_test,
    'test_sigma_ratio': _sigma_ratio,
    'pit_chi2_val': _pit_chi2,
})
print(f'\nStored per-group sigma_scale in '
      f"mlp_artifacts['predictive_uncertainty']['sigma_scale_by_group_val'].")


In [ ]:
# |residual| / sigma per coefficient group -- calibration test for the
# LSF-aware decomposition sigma (COEF_ERR HDU, aleatoric branch).  For each
# coefficient j: z_j = |coef_pred - coef_true| / sigma_j, aggregated per group.
# Under a well-calibrated Gaussian sigma the |z| distribution is half-N(1)
# with median 0.6745, mean 0.7979, p95 = 1.96.  Median|z| systematically above
# 1 means sigma is under-estimated; below 0.5 means over-estimated.
#
# Compares against the item-6 ensemble-spread sigma (epistemic branch) when
# available and reports the combined predictive sigma
# sigma_pred = sqrt(sigma_aleatoric^2 + sigma_epistemic^2) as the third source.
import numpy as np
import plotly.graph_objects as go
import pandas as pd
from plotly.subplots import make_subplots
from scipy.stats import halfnorm

required = ['coef_pred_det', 'coef_sci_all', 'test_idx',
            '_group_indices_compress', 'coef_err_sci_all']
_missing = [k for k in required if k not in globals()]
if _missing:
    raise RuntimeError('Run trainer + coef_err loading first. Missing: '
                       + ', '.join(_missing))

_te = np.asarray(test_idx, dtype=int)
_y_true = np.asarray(coef_sci_all[_te], dtype=np.float64)
_y_pred = np.asarray(coef_pred_det, dtype=np.float64)
_resid = _y_pred - _y_true

# Aleatoric sigma from the decomposition COEF_ERR HDU.
_sigma_alea = np.asarray(coef_err_sci_all[_te], dtype=np.float64)

# Epistemic sigma from item-6 (ensemble spread across seeds), if the cell ran.
_sigma_epi = None
if ('mlp_artifacts' in globals()
        and mlp_artifacts.get('is_ensemble', False)
        and 'predict_sci_coefficients_default' in globals()):
    _members = mlp_artifacts['members']
    _stack = np.stack([
        predict_sci_coefficients_default(
            _m,
            coef_near_phys=coef_near_all[_te], coef_far_phys=coef_far_all[_te],
            ctx_near_phys=ctx_near_all[_te], ctx_far_phys=ctx_far_all[_te],
            ctx_sci_phys=ctx_sci_all[_te],
        ).astype(np.float32) for _m in _members], axis=0)
    _sigma_epi = _stack.std(axis=0, ddof=1).astype(np.float64)

# Combined predictive sigma = sqrt(aleatoric^2 + epistemic^2)
_sigma_combined = None
if _sigma_epi is not None:
    _valid_mask = np.isfinite(_sigma_alea) & np.isfinite(_sigma_epi)
    _sigma_combined = np.sqrt(
        np.where(_valid_mask, _sigma_alea, 0.0) ** 2
        + np.where(_valid_mask, _sigma_epi, 0.0) ** 2
    )
    _sigma_combined[~_valid_mask] = np.nan

_sources = [('COEF_ERR (aleatoric)', _sigma_alea, '#1f77b4')]
if _sigma_epi is not None:
    _sources.append(('ensemble spread (epistemic)', _sigma_epi, '#ff7f0e'))
    _sources.append(('combined (quadrature)', _sigma_combined, '#2ca02c'))
print(f'sigma sources exercised: {[s[0] for s in _sources]}')


def _z_per_group(sigma_arr, gidx):
    r = _resid[:, gidx]
    s = sigma_arr[:, gidx]
    ok = np.isfinite(r) & np.isfinite(s) & (s > 0.0)
    return np.abs(r[ok]) / s[ok] if ok.any() else np.array([], dtype=np.float64)


_groups = list(_group_indices_compress.keys())
_n_groups = len(_groups)
_n_cols = min(3, _n_groups)
_n_rows = (_n_groups + _n_cols - 1) // _n_cols

fig = make_subplots(
    rows=_n_rows, cols=_n_cols,
    subplot_titles=[f'{g} (n_coef={len(_group_indices_compress[g])})'
                    for g in _groups],
    vertical_spacing=0.10, horizontal_spacing=0.08,
)

_z_by_source = {label: {} for label, _s, _c in _sources}
for panel_i, gname in enumerate(_groups):
    r = panel_i // _n_cols + 1
    c = panel_i % _n_cols + 1
    gidx = np.asarray(_group_indices_compress[gname], dtype=int)

    _panel_max = 3.0
    for label, sigma_arr, color in _sources:
        zvals = _z_per_group(sigma_arr, gidx)
        _z_by_source[label][gname] = zvals
        if zvals.size == 0:
            continue
        _panel_max = max(_panel_max, float(np.percentile(zvals, 95)) * 1.5)
        fig.add_trace(go.Histogram(
            x=zvals, nbinsx=60, name=label,
            marker_color=color, opacity=0.55,
            histnorm='probability density',
            showlegend=(panel_i == 0),
        ), row=r, col=c)

    # Half-N(1) target overlay.
    _x = np.linspace(0.0, _panel_max, 200)
    fig.add_trace(go.Scatter(
        x=_x, y=halfnorm.pdf(_x), mode='lines',
        line=dict(color='#d62728', width=2.2, dash='dash'),
        name='half-N(1) target', showlegend=(panel_i == 0),
    ), row=r, col=c)
    # Reference vertical: 0.6745 = target median.
    fig.add_vline(x=0.6745, line=dict(color='#d62728', dash='dot', width=1),
                  row=r, col=c)
    fig.update_xaxes(range=[0.0, min(6.0, _panel_max)], row=r, col=c,
                     title_text='|residual|/sigma' if r == _n_rows else '')
    fig.update_yaxes(title_text='density' if c == 1 else '', row=r, col=c)

fig.update_layout(
    template='plotly_white',
    barmode='overlay',
    height=280 * _n_rows + 100,
    title=('|residual| / sigma per coefficient group. '
           'Well-calibrated sigma -> histogram follows half-N(1) (red dashed); '
           'red dotted line at 0.6745 is the target median.'),
    margin=dict(l=60, r=30, t=90, b=60),
)
fig.show()

# Tabular summary per (source, group).
_summary_rows = []
for label, zdict in _z_by_source.items():
    for gname, zvals in zdict.items():
        if zvals.size == 0:
            _summary_rows.append({
                'sigma_source': label, 'group': gname, 'n_valid': 0,
                'median|z|': np.nan, 'mean|z|': np.nan, 'p95|z|': np.nan,
                'sigma_scale_hint': np.nan,
            })
            continue
        _p50 = float(np.median(zvals))
        _p95 = float(np.percentile(zvals, 95))
        _summary_rows.append({
            'sigma_source': label,
            'group': gname,
            'n_valid': int(zvals.size),
            'median|z|': _p50,
            'mean|z|': float(np.mean(zvals)),
            'p95|z|': _p95,
            # If sigma is off by scalar factor f, median|z| = 0.6745 * f, so
            # f = median|z|/0.6745 gives the per-group sigma-scale correction.
            'sigma_scale_hint': _p50 / 0.6745,
        })

_pd_df = pd.DataFrame(_summary_rows)
print()
print('Per-group |z| calibration diagnostic '
      '(half-N target: median|z|=0.6745, mean|z|=0.7979, p95|z|=1.96):')
print(_pd_df.to_string(index=False, float_format=lambda v: f'{v:.4g}'))

# Verdict per source.
print()
for label in [s[0] for s in _sources]:
    _rows = [r for r in _summary_rows if r['sigma_source'] == label and r['n_valid'] > 0]
    if not _rows:
        continue
    _hints = np.array([r['sigma_scale_hint'] for r in _rows], dtype=np.float64)
    _mean_hint = float(np.nanmean(_hints))
    _max_dev = float(np.nanmax(np.abs(_hints - 1.0)))
    _verdict = ('CALIBRATED' if _max_dev < 0.25
                else 'DRIFT' if _max_dev < 0.5
                else 'UNCALIBRATED')
    print(f'  {label}: sigma_scale hint {_mean_hint:.2f}x on average, '
          f'max per-group drift {_max_dev * 100:.1f}%  -> {_verdict}')


In [ ]:
# mean|residual| AND median|residual| vs COEF_ERR sigma per decile.
#
# The mean-vs-median comparison directly tests whether the σ over-estimation
# seen in cell 41 is driven by heavy-tailed outliers or by the bulk of the
# distribution. For a well-calibrated Gaussian σ:
#     E[|r|] / σ = √(2/π)  ≈ 0.7979
#     median|r| / σ        = 0.6745
#     E[|r|] / median|r|   = 0.7979 / 0.6745 ≈ 1.183
# A per-decile mean/median ratio well above 1.183 means outliers inflate the
# mean; if both mean and median lines sit far below the Gaussian references,
# even the bulk is calibrated wrong (σ is genuinely too large).
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

required = ['coef_pred_det', 'coef_sci_all', 'test_idx',
            '_group_indices_compress', 'coef_err_sci_all',
            'coef_near_all', 'coef_far_all', 'ctx_sci_all',
            'ctx_near_all', 'ctx_far_all',
            'predict_sci_coefficients_default', 'mlp_artifacts']
_missing = [k for k in required if k not in globals()]
if _missing:
    raise RuntimeError('Run trainer + coef_err loading first. Missing: '
                       + ', '.join(_missing))

_variants = {}
for _name in ('mlp_artifacts', 'art_pergroup', 'art_global'):
    _obj = globals().get(_name)
    if isinstance(_obj, dict) and _obj.get('is_ensemble', False):
        _variants[_name] = _obj

_te = np.asarray(test_idx, dtype=int)
_y_true = np.asarray(coef_sci_all[_te], dtype=np.float64)
_sig = np.asarray(coef_err_sci_all[_te], dtype=np.float64)

_pred_by_variant = {}
for _name, _arts in _variants.items():
    if (_name == 'mlp_artifacts' and 'coef_pred_det' in globals()
            and np.asarray(coef_pred_det).shape == _y_true.shape):
        _pred_by_variant[_name] = np.asarray(coef_pred_det, dtype=np.float64)
    else:
        _pred_by_variant[_name] = predict_sci_coefficients_default(
            _arts,
            coef_near_phys=coef_near_all[_te], coef_far_phys=coef_far_all[_te],
            ctx_near_phys=ctx_near_all[_te], ctx_far_phys=ctx_far_all[_te],
            ctx_sci_phys=ctx_sci_all[_te],
        ).astype(np.float64)

_variant_colors = {
    'mlp_artifacts': '#2ca02c',
    'art_pergroup':  '#1f77b4',
    'art_global':    '#ff7f0e',
}

_groups = list(_group_indices_compress.keys())
_n_groups = len(_groups)
fig = make_subplots(
    rows=1, cols=_n_groups,
    subplot_titles=[f'{g}  (n_coef={len(_group_indices_compress[g])})'
                    for g in _groups],
    horizontal_spacing=0.05,
)

_n_bins = 10
_summary_rows = []
_GAUSS_MEAN = float(np.sqrt(2.0 / np.pi))   # 0.7979
_GAUSS_MEDIAN = 0.6744897501960817          # scipy.stats.halfnorm.median()

for panel_i, gname in enumerate(_groups):
    col = panel_i + 1
    gidx = np.asarray(_group_indices_compress[gname], dtype=int)

    _sig_g_full = _sig[:, gidx].ravel()
    _ok = np.isfinite(_sig_g_full) & (_sig_g_full > 0.0)
    _sig_g = _sig_g_full[_ok]
    if _sig_g.size < _n_bins * 2:
        continue

    _edges = np.quantile(_sig_g, np.linspace(0.0, 1.0, _n_bins + 1))
    _bin_center = np.sqrt(_edges[:-1] * _edges[1:])
    _bin_id = np.digitize(_sig_g, _edges[1:-1], right=False)

    _sig_x = np.geomspace(max(_sig_g.min(), 1e-30), _sig_g.max(), 200)
    fig.add_trace(go.Scatter(
        x=_sig_x, y=_sig_x, mode='lines',
        line=dict(color='black', width=1.4, dash='dash'),
        name='y = σ (slope 1)', showlegend=(panel_i == 0),
    ), row=1, col=col)
    fig.add_trace(go.Scatter(
        x=_sig_x, y=_GAUSS_MEAN * _sig_x, mode='lines',
        line=dict(color='#d62728', width=1.4, dash='dot'),
        name=f'mean target: √(2/π) σ = {_GAUSS_MEAN:.4f} σ',
        showlegend=(panel_i == 0),
    ), row=1, col=col)
    fig.add_trace(go.Scatter(
        x=_sig_x, y=_GAUSS_MEDIAN * _sig_x, mode='lines',
        line=dict(color='#9467bd', width=1.4, dash='dot'),
        name=f'median target: 0.6745 σ',
        showlegend=(panel_i == 0),
    ), row=1, col=col)

    for _var_name, _pred in _pred_by_variant.items():
        _abs_resid = np.abs((_pred - _y_true)[:, gidx].ravel()[_ok])
        _mean_r = np.array([
            float(_abs_resid[_bin_id == b].mean())
            if np.any(_bin_id == b) else np.nan
            for b in range(_n_bins)
        ], dtype=np.float64)
        _median_r = np.array([
            float(np.median(_abs_resid[_bin_id == b]))
            if np.any(_bin_id == b) else np.nan
            for b in range(_n_bins)
        ], dtype=np.float64)

        _fin_mean = np.isfinite(_mean_r) & (_mean_r > 0.0) & (_bin_center > 0.0)
        _slope_mean = (np.polyfit(np.log10(_bin_center[_fin_mean]),
                                  np.log10(_mean_r[_fin_mean]), 1)[0]
                       if _fin_mean.sum() >= 3 else np.nan)
        _fin_med = np.isfinite(_median_r) & (_median_r > 0.0) & (_bin_center > 0.0)
        _slope_median = (np.polyfit(np.log10(_bin_center[_fin_med]),
                                    np.log10(_median_r[_fin_med]), 1)[0]
                         if _fin_med.sum() >= 3 else np.nan)

        _color = _variant_colors.get(_var_name, '#7f7f7f')
        fig.add_trace(go.Scatter(
            x=_bin_center, y=_mean_r, mode='lines+markers',
            marker=dict(size=6, symbol='circle'),
            line=dict(color=_color, width=2),
            name=f'{_var_name}  mean  slope={_slope_mean:.2f}',
            showlegend=True,
        ), row=1, col=col)
        fig.add_trace(go.Scatter(
            x=_bin_center, y=_median_r, mode='lines+markers',
            marker=dict(size=6, symbol='square'),
            line=dict(color=_color, width=2, dash='dash'),
            name=f'{_var_name}  median  slope={_slope_median:.2f}',
            showlegend=True,
        ), row=1, col=col)

        _mid = len(_mean_r) // 2
        _ratio_mean_mid = (float(_mean_r[_mid] / max(_bin_center[_mid], 1e-30))
                           if np.isfinite(_mean_r[_mid]) else np.nan)
        _ratio_median_mid = (float(_median_r[_mid] / max(_bin_center[_mid], 1e-30))
                             if np.isfinite(_median_r[_mid]) else np.nan)
        # Outlier proxy: per-bin mean/median, then median across bins so we
        # aren't fooled by one anomalous decile.
        _both = _fin_mean & _fin_med
        _outlier_ratio = (float(np.median(_mean_r[_both] / _median_r[_both]))
                          if _both.any() else np.nan)
        _summary_rows.append({
            'group': gname,
            'variant': _var_name,
            'slope_mean': _slope_mean,
            'slope_median': _slope_median,
            'mean/σ@med_bin': _ratio_mean_mid,
            'median/σ@med_bin': _ratio_median_mid,
            'mean/median ratio': _outlier_ratio,
            'n_pts': int(_sig_g.size),
        })

    fig.update_xaxes(type='log', title_text='σ (COEF_ERR, native units)',
                     row=1, col=col)
    fig.update_yaxes(type='log',
                     title_text=('|residual|' if panel_i == 0 else ''),
                     row=1, col=col)

fig.update_layout(
    template='plotly_white',
    height=520,
    width=280 * _n_groups + 100,
    title=(f'COEF_ERR σ-calibration: |residual| vs σ per decile '
           f'(mean = circles, median = squares; test split, n_row={_te.size})'),
    margin=dict(l=60, r=30, t=90, b=100),
    legend=dict(orientation='h', y=-0.22, x=0.0),
)
fig.show()

print()
print(f'Gaussian targets: mean/σ = √(2/π) = {_GAUSS_MEAN:.4f},  '
      f'median/σ = {_GAUSS_MEDIAN:.4f},  '
      f'expected mean/median = {_GAUSS_MEAN/_GAUSS_MEDIAN:.3f}')
print()
_df = pd.DataFrame(_summary_rows)
print(_df.to_string(index=False, float_format=lambda v: f'{v:.3g}'))
print()
print('Interpretation:')
print('  slope=1 for BOTH mean and median  -> σ scales correctly across deciles')
print('  slopes differ  -> outliers shift the mean trace relative to the median trace')
print('  mean/median ratio  ≈ 1.18  -> Gaussian bulk (mean gap explained by tail);')
print('                     >> 1.18 -> heavy outliers inflate mean|resid|')
print('  If median|residual| is still ≪ 0.6745·σ, σ over-estimation is real,')
print('    not just an outlier artefact of the mean statistic.')


In [ ]:
# Item 11 (physical-space cap). The trainer now stores a per-group per-coefficient
# upper bound = 3 x max(coef_sci_train, axis=0). expand_scores_to_coefs applies it
# after the >= 0 clip. This cell verifies (a) no legitimate train/val/test prediction
# comes close to the bound (max ratio << 1) and (b) the cap fires zero times on our
# splits so it acts purely as a runaway guard, not a legitimate constraint.

required = ['mlp_artifacts', 'coef_sci_all', 'coef_near_all', 'coef_far_all',
            'ctx_sci_all', 'ctx_near_all', 'ctx_far_all',
            'train_idx', 'val_idx', 'test_idx',
            '_group_indices_compress', 'predict_sci_coefficients_default']
_missing = [k for k in required if k not in globals()]
if _missing:
    raise RuntimeError('Run prerequisite cells first. Missing: ' + ', '.join(_missing))

_bounds = mlp_artifacts.get('coef_upper_bound')
if _bounds is None and mlp_artifacts.get('is_ensemble'):
    _bounds = mlp_artifacts['members'][0].get('coef_upper_bound')
if _bounds is None:
    raise RuntimeError("mlp_artifacts['coef_upper_bound'] missing; re-run the trainer cell.")


def _summarize_split(name, idx):
    _idx = np.asarray(idx, dtype=int)
    _pred = predict_sci_coefficients_default(
        mlp_artifacts,
        coef_near_phys=coef_near_all[_idx], coef_far_phys=coef_far_all[_idx],
        ctx_near_phys=ctx_near_all[_idx], ctx_far_phys=ctx_far_all[_idx],
        ctx_sci_phys=ctx_sci_all[_idx],
    ).astype(np.float32)
    _true = coef_sci_all[_idx].astype(np.float32)
    rows = []
    for _gname, _gidx in _group_indices_compress.items():
        _gidx = np.asarray(_gidx, dtype=int)
        _ub = np.asarray(_bounds[_gname], dtype=np.float64)
        _p = _pred[:, _gidx].astype(np.float64)
        _t = _true[:, _gidx].astype(np.float64)
        _bnd = np.broadcast_to(_ub[None, :], _p.shape)
        _n_clipped = int(np.sum(_p >= _bnd - 1e-30))
        _pred_ratio = float(np.max(_p / np.maximum(_bnd, 1e-30)))
        _true_ratio = float(np.max(_t / np.maximum(_bnd, 1e-30)))
        rows.append({
            'split': name, 'group': _gname,
            'n_pts': int(_p.size),
            'bound_median': float(np.median(_ub)),
            'bound_max': float(np.max(_ub)),
            'max(pred/bound)': _pred_ratio,
            'max(true/bound)': _true_ratio,
            'n_clipped': _n_clipped,
        })
    return rows


_out = []
for _name, _idx in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
    _out.extend(_summarize_split(_name, _idx))

item11_clip_df = pd.DataFrame(_out)
print('Item 11 -- per-group upper-cap activation on train/val/test '
      '(bound = 3 x max(coef_sci_train)):')
print(item11_clip_df.to_string(index=False, float_format=lambda v: f'{v:.4g}'))

_total_train = int(item11_clip_df.loc[item11_clip_df['split'] == 'train', 'n_clipped'].sum())
_total_val = int(item11_clip_df.loc[item11_clip_df['split'] == 'val', 'n_clipped'].sum())
_total_test = int(item11_clip_df.loc[item11_clip_df['split'] == 'test', 'n_clipped'].sum())
_total_pts = int(item11_clip_df['n_pts'].sum())
_all_clips = _total_train + _total_val + _total_test
_clip_frac = _all_clips / max(_total_pts, 1)
print(f'\nClip activations : train={_total_train}  val={_total_val}  test={_total_test}  '
      f'(total {_all_clips} of {_total_pts} points, {100.0 * _clip_frac:.4g}%)')
if _all_clips == 0:
    print('Verdict: no prediction reaches the bound; the cap is a pure runaway guard '
          "for production. Deployed by default via mlp_artifacts['coef_upper_bound'].")
elif _clip_frac < 1e-4:
    print(f'Verdict: cap activates on {_all_clips} of {_total_pts} points '
          f'({100.0 * _clip_frac:.4g}%), all on the sparse tail of mesospheric OH lines '
          'with near-zero training envelopes. This is exactly the runaway-guard use '
          "case item 11 asks for; deployed by default via mlp_artifacts['coef_upper_bound'].")
else:
    print('Verdict: cap activated on a non-negligible fraction of rows. Investigate whether '
          'the model is genuinely predicting brighter than 3x training max or the cap '
          'should be raised.')


In [ ]:
# WRMSE vs galactic coordinates for all observations.
# Uses build_triplet_coef_dataset (no region filter) so LMC/SMC rows are also
# predicted and shown; membership in train / val-test / originally-excluded
# controls only the marker opacity and the color-scale reference range.
import plotly.graph_objects as go
from plotly.colors import sample_colorscale
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.table import Table

required = ['filtered_triplet', 'train_idx', 'val_idx', 'test_idx',
            'mlp_artifacts', 'predict_sci_coefficients_default',
            'build_triplet_coef_dataset']
_missing = [k for k in required if k not in globals()]
if _missing:
    raise RuntimeError(
        'Missing kernel state: ' + ', '.join(_missing)
        + '. Run all prior cells (data loading + model training + split '
        'assignment) before executing this cell.'
    )

INPUT_FITS = 'spline_moon/lvmsframe_median_stack_1.2.1_p40_p70.fits'
NEAR_FITS = 'spline_moon/lvmsframe_median_stack_1.2.1_p40_p70_decomp_sky1_lsf_surface_iterative.fits'
FAR_FITS = 'spline_moon/lvmsframe_median_stack_1.2.1_p40_p70_decomp_sky2_lsf_surface_iterative.fits'
SCI_FITS = 'spline_moon/lvmsframe_median_stack_1.2.1_p40_p70_decomp_sci_lsf_surface_iterative.fits'
META_ONLY_FITS = 'spline_moon/lvmsframe_median_stack_1.2.1_p40_p70_meta_only.fits'

with fits.open(META_ONLY_FITS) as hdul:
    meta_tbl_all = Table(hdul['META'].data)

n_all_total = len(meta_tbl_all)
ra_col = next((c for c in ['sci_ra', 'ra', 'RA'] if c in meta_tbl_all.colnames), None)
dec_col = next((c for c in ['sci_dec', 'dec', 'DEC'] if c in meta_tbl_all.colnames), None)
if ra_col is None or dec_col is None:
    raise ValueError('RA/DEC not found in metadata')

ra_all = np.asarray(meta_tbl_all[ra_col], dtype=np.float64)
dec_all = np.asarray(meta_tbl_all[dec_col], dtype=np.float64)
coords_icrs = SkyCoord(ra=ra_all * u.deg, dec=dec_all * u.deg, frame='icrs')

lmc_cfg = globals().get('LMC_EXCLUSION', {'ra_deg': 80.894, 'dec_deg': -69.756, 'radius_deg': 10.0})
smc_cfg = globals().get('SMC_EXCLUSION', {'ra_deg': 13.187, 'dec_deg': -72.829, 'radius_deg': 10.0})

is_region_excluded = np.zeros(n_all_total, dtype=bool)
for cfg in (lmc_cfg, smc_cfg):
    center = SkyCoord(ra=float(cfg['ra_deg']) * u.deg,
                      dec=float(cfg['dec_deg']) * u.deg, frame='icrs')
    is_region_excluded |= (coords_icrs.separation(center).deg <= float(cfg['radius_deg']))

filtered_row_indices = np.asarray(filtered_triplet['row_index'], dtype=int)
is_train = np.zeros(n_all_total, dtype=bool)
is_valtest = np.zeros(n_all_total, dtype=bool)
train_idx_arr = np.asarray(train_idx, dtype=int)
valtest_idx_arr = np.unique(np.concatenate([np.asarray(val_idx, dtype=int),
                                            np.asarray(test_idx, dtype=int)]))
is_train[filtered_row_indices[train_idx_arr]] = True
is_valtest[filtered_row_indices[valtest_idx_arr]] = True
is_other = ~(is_train | is_valtest | is_region_excluded)

print(f'Row counts: total={n_all_total} '
      f'train={is_train.sum()} valtest={is_valtest.sum()} '
      f'lmcsmc={is_region_excluded.sum()} other={is_other.sum()}')

# Full aligned triplet without region filter -> includes LMC/SMC rows.
# ECLIPTIC-CTX-V1: ecliptic features are attached post-hoc via
# _augment_triplet_with_ecliptic; strip them from the loader's context_columns
# so _load_decomp_with_row_index doesn't try to read them from META.
context_columns = list(globals().get('ctx_names_all', filtered_triplet['ctx_names']))
_ecl_names = set(globals().get('ECLIPTIC_FEATURE_NAMES', []))
context_columns = [c for c in context_columns if c not in _ecl_names]
triplet_full = build_triplet_coef_dataset(
    input_fits_path=INPUT_FITS,
    sky_near_decomp_fits_path=NEAR_FITS,
    sky_far_decomp_fits_path=FAR_FITS,
    sci_decomp_fits_path=SCI_FITS,
    context_columns=context_columns,
    return_chi2=False,
)
# ECLIPTIC-CTX-V1: match training-time ctx layout on the full triplet.
if '_augment_triplet_with_ecliptic' in globals():
    _augment_triplet_with_ecliptic(triplet_full, meta_fits_path=INPUT_FITS)

full_rows = np.asarray(triplet_full['row_index'], dtype=int)
coef_true_full = np.asarray(triplet_full['coef_sci'], dtype=np.float64)
coef_pred_full = predict_sci_coefficients_default(
    mlp_artifacts,
    coef_near_phys=triplet_full['coef_near'],
    coef_far_phys=triplet_full['coef_far'],
    ctx_near_phys=triplet_full['ctx_near'],
    ctx_far_phys=triplet_full['ctx_far'],
    ctx_sci_phys=triplet_full['ctx_sci'],
).astype(np.float64)

# Per-row WRMSE for the WRMSE map + downstream correlation/scatter cells.
# Uses coef_err_sci from the full triplet if COEF_ERR was loaded; else
# falls back to NaN sigmas (which the helper treats as floor-only weights).
_sigma_full = triplet_full.get('coef_err_sci', None)
if _sigma_full is None:
    _sigma_full = np.full_like(coef_true_full, np.nan)
_sigma_full = np.asarray(_sigma_full, dtype=np.float64)
_gidx_map = (_group_indices_compress if '_group_indices_compress' in globals()
             else group_indices_sf)
wrmse_full = weighted_rmse_per_row(
    coef_true_full, coef_pred_full, _sigma_full,
    _gidx_map, dict(DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP),
).astype(np.float32)

wrmse_all = np.full(n_all_total, np.nan, dtype=np.float32)
valid_rows = (full_rows >= 0) & (full_rows < n_all_total)
wrmse_all[full_rows[valid_rows]] = wrmse_full[valid_rows]

print(f'Finite sWRMSE_coef: total={np.isfinite(wrmse_all).sum()} '
      f'train={np.isfinite(wrmse_all[is_train]).sum()} '
      f'valtest={np.isfinite(wrmse_all[is_valtest]).sum()} '
      f'lmcsmc={np.isfinite(wrmse_all[is_region_excluded]).sum()} '
      f'other={np.isfinite(wrmse_all[is_other]).sum()}')

gl_all = coords_icrs.galactic.l.deg
gb_all = coords_icrs.galactic.b.deg
field_diameter_arcmin = 30.0
field_radius_deg = (field_diameter_arcmin / 2.0) / 60.0

def make_circle(center_l, center_b, radius_deg, n_points=32):
    a = np.linspace(0.0, 2.0 * np.pi, n_points)
    return center_l + radius_deg * np.cos(a), center_b + radius_deg * np.sin(a)

# Color scale fixed to train+val/test in-domain range so LMC/SMC out-of-domain
# behaviour cannot compress the visible dynamic range for the training set.
wrmse_in_domain = wrmse_all[(is_train | is_valtest) & np.isfinite(wrmse_all)]
if len(wrmse_in_domain) == 0:
    raise RuntimeError('No finite sWRMSE_coef available for in-domain (train+val/test) rows.')
wrmse_vmin, wrmse_vmax = np.percentile(wrmse_in_domain, [1, 99])
_wrmse_denom = max(wrmse_vmax - wrmse_vmin, 1e-30)
# Lower WRMSE -> lighter red (good); higher -> darker red (bad).
wrmse_norm = np.clip((wrmse_all - wrmse_vmin) / _wrmse_denom, 0, 1)

colors_list = ['rgb(235,235,235)'] * n_all_total
finite_norm = np.isfinite(wrmse_norm)
if np.any(finite_norm):
    sampled = sample_colorscale('Reds', wrmse_norm[finite_norm].tolist())
    for idx_i, color in zip(np.where(finite_norm)[0], sampled):
        colors_list[idx_i] = color

fig = go.Figure()
for i in range(n_all_total):
    cl, cb = make_circle(gl_all[i], gb_all[i], field_radius_deg)
    if is_train[i]:
        opacity, region = 0.88, 'Train'
    elif is_valtest[i]:
        opacity, region = 0.45, 'Val/Test'
    elif is_region_excluded[i]:
        opacity, region = 0.35, 'Originally excluded (LMC/SMC)'
    else:
        opacity, region = 0.10, 'Other'

    color_rgba = colors_list[i].replace('rgb(', 'rgba(').replace(')', f', {opacity})')
    wrmse_str = f'{wrmse_all[i]:.4g}' if np.isfinite(wrmse_all[i]) else 'N/A'
    fig.add_trace(go.Scatter(
        x=cl, y=cb, mode='lines', fill='toself',
        fillcolor=color_rgba, line=dict(color=color_rgba, width=0.5),
        hoverinfo='text',
        text=f'L={gl_all[i]:.2f} deg, B={gb_all[i]:.2f} deg<br>sWRMSE_coef={wrmse_str}<br>{region}',
        hovertemplate='%{text}<extra></extra>', showlegend=False,
    ))

fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='markers',
    marker=dict(colorscale='Reds', showscale=True,
                colorbar=dict(title='sWRMSE_coef', thickness=15, len=0.7,
                              tickvals=[0, 0.25, 0.5, 0.75, 1.0],
                              ticktext=[f'{wrmse_vmin + (wrmse_vmax - wrmse_vmin) * t:.2g}'
                                        for t in [0, 0.25, 0.5, 0.75, 1.0]]),
                cmin=0, cmax=1, size=0),
    hoverinfo='none', showlegend=False,
))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
    marker=dict(size=10, color='rgba(200,100,100,0.88)'), name='Train', showlegend=True))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
    marker=dict(size=10, color='rgba(200,100,100,0.45)'), name='Val/Test', showlegend=True))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
    marker=dict(size=10, color='rgba(200,100,100,0.35)'), name='Originally excluded (LMC/SMC)', showlegend=True))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
    marker=dict(size=10, color='rgba(180,180,180,0.10)'), name='Other', showlegend=True))
fig.update_layout(
    title='sWRMSE_coef vs Galactic Coordinates (Train / Val-Test / LMC-SMC Excluded)',
    xaxis_title='Galactic Longitude [deg]', yaxis_title='Galactic Latitude [deg]',
    xaxis=dict(scaleanchor='y', scaleratio=1),
    yaxis=dict(scaleanchor='x', scaleratio=1),
    width=1200, height=900, template='plotly_white', hovermode='closest',
)
fig.show()

print('sWRMSE_coef summary by region:')
for name, mask in [('Train', is_train),
                   ('Val/Test', is_valtest),
                   ('Originally excluded (LMC/SMC)', is_region_excluded),
                   ('Other', is_other)]:
    vals = wrmse_all[mask & np.isfinite(wrmse_all)]
    if len(vals) == 0:
        print(f'  {name} (n={int(mask.sum())}): no finite sWRMSE_coef available')
    else:
        print(f'  {name} (n={int(mask.sum())}): '
              f'{vals.min():.4g} to {vals.max():.4g}, '
              f'median={np.median(vals):.4g}, finite={len(vals)}')


In [ ]:
# Correlation of prediction quality (WRMSE) with sci-pointing context variables.
# Reports Pearson (linear) and Spearman (rank) for the full row set and
# separately for train, val/test and LMC/SMC so the same axes can be compared
# in and out of the training domain.
import pandas as pd
from scipy.stats import pearsonr, spearmanr

required = ['triplet_full', 'wrmse_full', 'full_rows', 'is_train',
            'is_valtest', 'is_region_excluded']
_missing = [k for k in required if k not in globals()]
if _missing:
    raise RuntimeError(
        'Missing kernel state: ' + ', '.join(_missing)
        + '. Run the sWRMSE_coef map cell above first.'
    )

ctx_sci_full = np.asarray(triplet_full['ctx_sci'], dtype=np.float64)
ctx_names_full = list(triplet_full['ctx_names'])
wrmse_arr = np.asarray(wrmse_full, dtype=np.float64)  # 2026-08-16: WRMSE

train_mask_t = is_train[full_rows]
valtest_mask_t = is_valtest[full_rows]
lmc_mask_t = is_region_excluded[full_rows]

subsets = [
    ('all', np.ones_like(train_mask_t, dtype=bool)),
    ('train', train_mask_t),
    ('valtest', valtest_mask_t),
    ('lmcsmc', lmc_mask_t),
]

rows = []
for j, cname in enumerate(ctx_names_full):
    x = ctx_sci_full[:, j]
    row = {'ctx': cname}
    for label, submask in subsets:
        m = submask & np.isfinite(x) & np.isfinite(wrmse_arr)
        n_pts = int(m.sum())
        row[f'n_{label}'] = n_pts
        if n_pts < 20 or np.std(x[m]) == 0.0 or np.std(wrmse_arr[m]) == 0.0:
            row[f'pearson_{label}'] = np.nan
            row[f'spearman_{label}'] = np.nan
        else:
            r_p, _ = pearsonr(x[m], wrmse_arr[m])
            r_s, _ = spearmanr(x[m], wrmse_arr[m])
            row[f'pearson_{label}'] = float(r_p)
            row[f'spearman_{label}'] = float(r_s)
    rows.append(row)

wrmse_ctx_corr_df = pd.DataFrame(rows).sort_values(
    by='pearson_all', key=lambda s: s.abs(), ascending=False
).reset_index(drop=True)

col_order = ['ctx',
             'pearson_all', 'spearman_all', 'n_all',
             'pearson_train', 'spearman_train', 'n_train',
             'pearson_valtest', 'spearman_valtest', 'n_valtest',
             'pearson_lmcsmc', 'spearman_lmcsmc', 'n_lmcsmc']
wrmse_ctx_corr_df = wrmse_ctx_corr_df[col_order]

print('Correlation of sWRMSE_coef vs sci-pointing context columns '
      '(sorted by |Pearson (all)|):')
with pd.option_context('display.max_rows', None,
                       'display.max_columns', None,
                       'display.width', 240,
                       'display.float_format', lambda v: f'{v: .3f}'):
    print(wrmse_ctx_corr_df.to_string(index=False))


In [ ]:
# Scatter of WRMSE vs each sci-pointing context column, colored by subset.
import math
import plotly.graph_objects as go
from plotly.subplots import make_subplots

required = ['ctx_sci_full', 'ctx_names_full', 'wrmse_arr',
            'train_mask_t', 'valtest_mask_t', 'lmc_mask_t']
_missing = [k for k in required if k not in globals()]
if _missing:
    raise RuntimeError(
        'Missing kernel state: ' + ', '.join(_missing)
        + '. Run the sWRMSE_coef correlation cell above first.'
    )

subsets = [
    ('Train', train_mask_t, 'rgba(31,119,180,0.30)'),
    ('Val/Test', valtest_mask_t, 'rgba(44,160,44,0.55)'),
    ('LMC/SMC', lmc_mask_t, 'rgba(214,39,40,0.50)'),
]

useful = [(j, cname) for j, cname in enumerate(ctx_names_full)
          if np.any(np.isfinite(ctx_sci_full[:, j])) and np.nanstd(ctx_sci_full[:, j]) > 0]

n_cols_grid = 4
n_rows_grid = math.ceil(len(useful) / n_cols_grid)

fig = make_subplots(rows=n_rows_grid, cols=n_cols_grid,
                    subplot_titles=[c for _, c in useful],
                    horizontal_spacing=0.04, vertical_spacing=0.06)

for panel_i, (j, cname) in enumerate(useful):
    r = panel_i // n_cols_grid + 1
    c = panel_i % n_cols_grid + 1
    x_full = ctx_sci_full[:, j]
    for label, mask, color in subsets:
        m = mask & np.isfinite(x_full) & np.isfinite(wrmse_arr)
        if not np.any(m):
            continue
        fig.add_trace(go.Scattergl(
            x=x_full[m], y=wrmse_arr[m], mode='markers',
            marker=dict(color=color, size=3),
            name=label, legendgroup=label,
            showlegend=(panel_i == 0),
            hoverinfo='skip',
        ), row=r, col=c)

# Y-range from in-domain 0.5-99.5 percentile so extreme tail WRMSE does not
# compress the visible range.  WRMSE is unbounded above, so no ceiling clamp.
in_domain = wrmse_arr[(train_mask_t | valtest_mask_t) & np.isfinite(wrmse_arr)]
y_lo = float(np.percentile(in_domain, 0.5))
y_hi = float(np.percentile(in_domain, 99.5))
y_pad = 0.05 * max(y_hi - y_lo, 1e-6)
y_range = [max(0.0, y_lo - y_pad), y_hi + y_pad]
for panel_i in range(len(useful)):
    fig.update_yaxes(range=y_range,
                     row=panel_i // n_cols_grid + 1,
                     col=panel_i % n_cols_grid + 1)

fig.update_layout(
    title='sWRMSE_coef vs sci-pointing context columns (colored by subset)',
    width=1500, height=260 * n_rows_grid + 100,
    template='plotly_white',
    legend=dict(orientation='h', y=-0.02),
)
fig.show()


> **2026-08-11 cleanup status.** The notebook now contains only the deployed model path (`mlp_artifacts` at cell 18) and the 4-seed ensemble (`_ensemble_test` at cell 26). Nine historical A/B and experiment cells were removed (ion-head, two-stage sweep, verify cell, alpha diagnostic, `blend_optim` A/B, multi-seed top-K rerun, `drop_vanrhijn` A/B, weight A/B, training-curve diagnostic). Guarded confidence-check templates (dense sweep, three alpha checks, pre-fit A/B, moon compression A/B) were kept for future re-verification. Items resolved by this pass: **5** (ensemble packaged), **10** (α=0.7 confirmed by dense grid + 8-seed + long-budget), **21** (patience=16 confirmed by training-curve diagnostic — kept as-is), **22** (RUN_SWEEP deleted; RUN_DENSE_SWEEP kept as a guarded template), **25** (Weight A/B cell removed). Items still open below are unchanged.

---

Below is a review pass focused on what would actually change before packaging. Ordered by severity, not enthusiasm. §-numbers refer to the notebook's own methods cell.

---

## Tier 1 — deployment blockers

1. **Serialize `mlp_artifacts` to disk.** Right now the "artifact" is a live Python dict containing the PyTorch model, two `RobustScaler`s, five compressors (with basis matrices), `geom_kwargs`, `jensen_corrections`, and a config. There is no save/load path. Before shipping you need:
   - A single file (`.pt` for weights + `.npz` / JSON sidecar for scalers, compressors, config) with a schema version tag.
   - `load_artifacts()` that fails loudly (not silently mispredicts) when `context_cols` / `COEF_SCHEMA` / group set / compressor kinds don't match the running notebook's version.
   - Re-decide `device` at load time (an artifact trained on `mps` should still run on `cpu` at LCO).

---

## Tier 3 — diagnostics to add before shipping

12. **Round-trip test on the compressor.** For every group, encode a random subset of training rows, decode, and check `max(|coef_out - coef_in|) < tol`. Catches basis / mean / sd bugs deterministically. Would have caught the `y_train_max` blowup in one line.

13. **Near ↔ far swap test.** Predict with `(near, far)` and with `(far, near)` swapped on the same row. The learned α is per-group symmetric-with-sign, so the predictions should be:
    - identical when the sci pointing is equidistant from both arms;
    - structured by `sci_sep_near - sci_sep_far` otherwise.
    Plot the delta as a function of that geometric asymmetry; if it's random noise, the model isn't using the near/far ordering as designed.

16. **Feature ablation — retrain without `f107` / `f107_81d` / `kp`.** §9.2 says only ionospheric sees group-wide F10.7 signal (r ~+0.17). Run one seed with those three features zeroed at inference and measure delta on the ionospheric group. Confirms the feature is doing what it's supposed to.

17. **Feature ablation — `sci_sep` for the science arm.** `sci_sep_sci = 0` by construction; the encoder sees a constant. Confirm it isn't accidentally being used as a bias term by measuring test RMSE with that column zeroed vs randomized.

---

## Tier 4 — nice-to-haves

21. Fold the `patience=16` back to something like 8 — no experiment used past epoch 20 and the extra budget just extends sweep time.
22. `RUN_SWEEP` / `RUN_DENSE_SWEEP` cells should be deleted or moved to a separate notebook before this file becomes a deployed artifact; they're 15 % of the source and never run in production.
23. `moon_smooth_lambda` is still a docstring artefact in the retired-path notes — clean up before external eyes read the file.
24. `_group_amplitude` reducer defaults to `sum` for the structure-function cell. For the ionospheric group (n=4) the sum is fine, but if you ever hit a group with all-zeros the log clip at 1e-8 pins the whole structure function to a constant. Guard by returning NaN when `sum <= 0` and dropping those rows from the diagnostic.
25. Cell 22 (`Weight A/B`) has dead code: `_blend = {g: float(...) for g in ()}` overwrites nothing but is confusing. Delete.
26. There are two `import contextlib as _..._contextlib` blocks that could share one alias.


In [ ]:
# --- A/B: global floor 0.05 vs per-group floor (2026-08-16 refinement) --
# Self-contained: reloads the trainer + helpers from disk so kernel state
# matches the on-disk per-group-floor patch, then runs two 15-epoch,
# single-seed trainings that differ only in floor policy.
import ast, json, time
import numpy as np
from pathlib import Path

_nb_path = Path("notebook_sky_interpolation_triplet_dual_encoder_group_mlp.ipynb")
_nb = json.loads(_nb_path.read_text())


def _slice_defs(src, wanted, keep_module_assigns=()):
    tree = ast.parse(src)
    lines = src.splitlines()
    keep = []
    for node in tree.body:
        name = getattr(node, "name", None)
        if name in wanted:
            keep.append("\n".join(lines[node.lineno - 1:node.end_lineno]))
        elif isinstance(node, ast.Assign):
            tgt = node.targets[0]
            aname = getattr(tgt, "id", None)
            if aname in keep_module_assigns:
                keep.append("\n".join(lines[node.lineno - 1:node.end_lineno]))
    return "\n\n".join(keep)


_helper_src = None
_trainer_src = None
for _c in _nb["cells"]:
    _s = "".join(_c.get("source", []))
    if "def compress_coef_err_to_score_sigma" in _s and _helper_src is None:
        _helper_src = _slice_defs(_s, {"compress_coef_err_to_score_sigma"})
    if "def train_compressed_group_mlp" in _s and _trainer_src is None:
        _trainer_src = _slice_defs(
            _s,
            {"DualEncoderGroupHeadMLPCompressed",
             "train_compressed_group_mlp",
             "predict_sci_coefficients_default"},
            keep_module_assigns=("DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP",),
        )
assert _helper_src and _trainer_src

exec(compile(_helper_src, "<cell17_helper>", "exec"), globals())
exec(compile(_trainer_src, "<cell18_trainer>", "exec"), globals())

import inspect
_sig = inspect.signature(train_compressed_group_mlp)
_default_floor = _sig.parameters["coef_err_sigma_floor_rel"].default
print("reloaded trainer.  floor default =", _default_floor)
print("DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP =",
      DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP)

if "coef_err_sci" not in filtered_triplet:
    from astropy.io import fits
    from astropy.table import Table
    with fits.open("spline_moon/lvmsframe_median_stack_1.2.1_p40_p70_decomp_sci_lsf_surface_iterative.fits") as _h:
        _err_tbl = Table(_h["COEF_ERR"].data)
        _err_full = np.column_stack([
            np.asarray(_err_tbl[c], dtype=np.float32)
            for c in filtered_triplet["coef_names"]
        ])
    _row_idx = np.asarray(filtered_triplet["row_index"], dtype=int)
    filtered_triplet["coef_err_sci"] = _err_full[_row_idx]
    print(f"attached filtered_triplet[coef_err_sci] shape={_err_full[_row_idx].shape}")

_common_kwargs = dict(
    n_epochs=15,
    batch_size=256,
    lr=7e-4,
    encoder_dims=(384, 192),
    trunk_dims=(320, 160),
    head_dim=192,
    weight_decay=1e-4,
    grad_clip=1.0,
    patience=4,
    seed=42,
    moon_group_weight=3.5,
    continuum_group_weight=1.5,
    mesospheric_group_weight=3.0,
    ionospheric_group_weight=1.0,
    blend_init_alpha=0.7,
    blend_optim="direct",
    moon_alt_conditional_alpha=False,
    high_airmass_boost=1.0,
)
_split = (train_idx, val_idx, test_idx)
_gidx = (globals().get("group_indices_sf", None)
         or globals().get("_group_indices_compress", None))
assert _gidx is not None

print("\n=== A: global scalar floor 0.05 (previous default) ===")
_t0 = time.time()
_art_global = train_compressed_group_mlp(
    filtered_triplet,
    group_compressors,
    _gidx,
    compress_geom_kwargs,
    split_indices=_split,
    use_coef_err_weights=True,
    coef_err_sigma_floor_rel=0.05,
    **_common_kwargs,
)
_t_global = time.time() - _t0
print(f'A best_val_loss={_art_global["best_val_loss"]:.6f}  elapsed={_t_global:.1f}s')

print("\n=== B: per-group floor (moon/continuum/ionospheric=0.05, "
      "mesospheric/atomic=0.20) ===")
_t0 = time.time()
_art_pergroup = train_compressed_group_mlp(
    filtered_triplet,
    group_compressors,
    _gidx,
    compress_geom_kwargs,
    split_indices=_split,
    use_coef_err_weights=True,
    coef_err_sigma_floor_rel=DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP,
    **_common_kwargs,
)
_t_pergroup = time.time() - _t0
print(f'B best_val_loss={_art_pergroup["best_val_loss"]:.6f}  elapsed={_t_pergroup:.1f}s')

_pred_A = predict_sci_coefficients_default(
    _art_global,
    coef_near_phys=coef_near_all[test_idx], coef_far_phys=coef_far_all[test_idx],
    ctx_near_phys=ctx_near_all[test_idx], ctx_far_phys=ctx_far_all[test_idx],
    ctx_sci_phys=ctx_sci_all[test_idx],
).astype(np.float32)
_pred_B = predict_sci_coefficients_default(
    _art_pergroup,
    coef_near_phys=coef_near_all[test_idx], coef_far_phys=coef_far_all[test_idx],
    ctx_near_phys=ctx_near_all[test_idx], ctx_far_phys=ctx_far_all[test_idx],
    ctx_sci_phys=ctx_sci_all[test_idx],
).astype(np.float32)
_y_te = coef_sci_all[test_idx].astype(np.float32)

print(f"\n=== per-group test-split eRMSE (n={test_idx.size}) ===")
print(f'{"group":<14s} {"n_coef":>7s} {"eRMSE_global":>12s} {"eRMSE_pergrp":>12s} {"rel_delta":>10s}')
for _g, _idx in _gidx.items():
    _idx = np.asarray(_idx, dtype=int)
    if _idx.size == 0:
        continue
    _rmse_A = float(np.sqrt(np.mean((_pred_A[:, _idx] - _y_te[:, _idx]) ** 2)))
    _rmse_B = float(np.sqrt(np.mean((_pred_B[:, _idx] - _y_te[:, _idx]) ** 2)))
    _rel = (_rmse_B - _rmse_A) / max(_rmse_A, 1e-30)
    _sign = "+" if _rel > 0 else ""
    print(f'{_g:<14s} {_idx.size:>7d} {_rmse_A:>12.4g} {_rmse_B:>12.4g} '
          f"{_sign}{100.0*_rel:>9.2f}%")
_rmse_A_g = float(np.sqrt(np.mean((_pred_A - _y_te) ** 2)))
_rmse_B_g = float(np.sqrt(np.mean((_pred_B - _y_te) ** 2)))
_rel_g = (_rmse_B_g - _rmse_A_g) / max(_rmse_A_g, 1e-30)
_sign = "+" if _rel_g > 0 else ""
print(f'{"TOTAL":<14s} {"":>7s} {_rmse_A_g:>12.4g} {_rmse_B_g:>12.4g} '
      f"{_sign}{100.0*_rel_g:>9.2f}%")

print("\n(previous unweighted baseline on this split: eRMSE_total = 263.3;\n"
      " previous global-floor-0.05 baseline: eRMSE_total = 290.8.\n"
      " See the 2026-08-16 changelog entry for details.)")


In [ ]:
# --- One-shot LR + weight-decay sweep at the flat-weight defaults --------
# The loss surface changed after removing the hand-tuned per-group
# multipliers and airmass boost.  We do a log-scale sweep of lr and
# weight_decay around the previous baseline (lr=7e-4, wd=1e-4).  All runs
# use per-group floor coef_err weighting, flat group weights, airmass=1,
# seed=42.  Patience is bumped to 8 and n_epochs to 25 so we can see
# whether the 4-patience default was cutting training too early.
import ast, json, time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Reload trainer from disk to match the reverted per-group-floor code.
_nb_path = Path("notebook_sky_interpolation_triplet_dual_encoder_group_mlp.ipynb")
_nb = json.loads(_nb_path.read_text())


def _slice_defs(src, wanted, keep_module_assigns=()):
    tree = ast.parse(src)
    lines = src.splitlines()
    keep = []
    for node in tree.body:
        name = getattr(node, "name", None)
        if name in wanted:
            keep.append("\n".join(lines[node.lineno - 1:node.end_lineno]))
        elif isinstance(node, ast.Assign):
            tgt = node.targets[0]
            aname = getattr(tgt, "id", None)
            if aname in keep_module_assigns:
                keep.append("\n".join(lines[node.lineno - 1:node.end_lineno]))
    return "\n\n".join(keep)


_helper_src = _trainer_src = None
for _c in _nb["cells"]:
    _s = "".join(_c.get("source", []))
    if "def compress_coef_err_to_score_sigma" in _s and _helper_src is None:
        _helper_src = _slice_defs(_s, {"compress_coef_err_to_score_sigma"})
    if "def train_compressed_group_mlp" in _s and _trainer_src is None:
        _trainer_src = _slice_defs(
            _s,
            {"DualEncoderGroupHeadMLPCompressed",
             "train_compressed_group_mlp",
             "predict_sci_coefficients_default"},
            keep_module_assigns=("DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP",),
        )
exec(compile(_helper_src, "<cell17_helper>", "exec"), globals())
exec(compile(_trainer_src, "<cell18_trainer>", "exec"), globals())

if "coef_err_sci" not in filtered_triplet:
    from astropy.io import fits
    from astropy.table import Table
    with fits.open("spline_moon/lvmsframe_median_stack_1.2.1_p40_p70_decomp_sci_lsf_surface_iterative.fits") as _h:
        _err_tbl = Table(_h["COEF_ERR"].data)
        _err_full = np.column_stack([
            np.asarray(_err_tbl[c], dtype=np.float32)
            for c in filtered_triplet["coef_names"]
        ])
    _row_idx = np.asarray(filtered_triplet["row_index"], dtype=int)
    filtered_triplet["coef_err_sci"] = _err_full[_row_idx]

_gidx = (globals().get("group_indices_sf", None)
         or globals().get("_group_indices_compress", None))
assert _gidx is not None
_split = (train_idx, val_idx, test_idx)

_LR0, _WD0 = 7.0e-4, 1.0e-4

_configs = [
    ("lr/3", dict(lr=_LR0 / 3.0, weight_decay=_WD0)),
    ("lr",   dict(lr=_LR0,       weight_decay=_WD0)),
    ("3*lr", dict(lr=_LR0 * 3.0, weight_decay=_WD0)),
    ("wd/3", dict(lr=_LR0,       weight_decay=_WD0 / 3.0)),
    ("3*wd", dict(lr=_LR0,       weight_decay=_WD0 * 3.0)),
]

_N_EPOCHS = 25
_PATIENCE = 8

_base_kwargs = dict(
    n_epochs=_N_EPOCHS, batch_size=256,
    encoder_dims=(384, 192), trunk_dims=(320, 160), head_dim=192,
    grad_clip=1.0, patience=_PATIENCE, seed=42,
    moon_group_weight=1.0, continuum_group_weight=1.0,
    mesospheric_group_weight=1.0, ionospheric_group_weight=1.0,
    high_airmass_boost=1.0,
    blend_init_alpha=0.7, blend_optim="direct",
    moon_alt_conditional_alpha=False,
    use_coef_err_weights=True,
    coef_err_sigma_floor_rel=DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP,
)

_arts_sweep = {}
_y = coef_sci_all[test_idx].astype(np.float32)

for _label, _cfg in _configs:
    print(f"\n=== {_label}  lr={_cfg['lr']:.3e}  wd={_cfg['weight_decay']:.3e} ===")
    _t0 = time.time()
    _art = train_compressed_group_mlp(
        filtered_triplet, group_compressors, _gidx, compress_geom_kwargs,
        split_indices=_split,
        **{**_base_kwargs, **_cfg},
    )
    _dt = time.time() - _t0
    _hist = _art.get("history", [])
    _n_ep = len(_hist)
    _best_ep = (int(np.argmin([h["val_loss"] for h in _hist])) + 1) if _hist else -1
    _arts_sweep[_label] = _art
    print(f'  best_val_loss={_art["best_val_loss"]:.6f}  '
          f'best_epoch={_best_ep}/{_n_ep}  early_stop={_n_ep < _N_EPOCHS}  '
          f'elapsed={_dt:.1f}s')

# --- Val curves --------------------------------------------------------
_fig, _ax = plt.subplots(1, 1, figsize=(9, 5))
for _label, _art in _arts_sweep.items():
    _h = _art.get("history", [])
    if not _h:
        continue
    _eps = [x["epoch"] for x in _h]
    _val = [x["val_loss"] for x in _h]
    _line, = _ax.plot(_eps, _val, marker="o", markersize=3, linewidth=1.2,
                      label=_label)
    _best_ep = int(np.argmin(_val)) + 1
    _ax.axvline(_best_ep, color=_line.get_color(), linestyle=":", alpha=0.4)
_ax.set_yscale("log")
_ax.set_xlabel("epoch")
_ax.set_ylabel("val loss (log)")
_ax.set_title(f"lr / wd sweep val curves  (patience={_PATIENCE}, n_epochs={_N_EPOCHS})")
_ax.grid(True, which="both", alpha=0.3)
_ax.legend()
plt.tight_layout()
plt.show()

# --- Test summary ------------------------------------------------------
def _predict(art):
    return predict_sci_coefficients_default(
        art,
        coef_near_phys=coef_near_all[test_idx], coef_far_phys=coef_far_all[test_idx],
        ctx_near_phys=ctx_near_all[test_idx], ctx_far_phys=ctx_far_all[test_idx],
        ctx_sci_phys=ctx_sci_all[test_idx],
    ).astype(np.float32)


print(f"\n=== test-split summary (n={test_idx.size}) ===")
print(f'{"config":<7s} {"lr":>10s} {"wd":>10s} {"val*":>10s} '
      f'{"best_ep":>8s} {"n_ep":>5s} {"early":>6s} {"eRMSE_tot":>10s}')
_summary_rows = []
for _label, _cfg in _configs:
    _art = _arts_sweep[_label]
    _p = _predict(_art)
    _h = _art.get("history", [])
    _n_ep = len(_h)
    _best_ep = (int(np.argmin([x["val_loss"] for x in _h])) + 1) if _h else -1
    _early = _n_ep < _N_EPOCHS
    _rmse_tot = float(np.sqrt(np.mean((_p - _y) ** 2)))
    _summary_rows.append((_label, _art["best_val_loss"], _best_ep, _n_ep, _rmse_tot))
    print(f'{_label:<7s} {_cfg["lr"]:>10.3e} {_cfg["weight_decay"]:>10.3e} '
          f'{_art["best_val_loss"]:>10.6f} {_best_ep:>8d} {_n_ep:>5d} '
          f'{str(_early):>6s} {_rmse_tot:>10.4g}')

_best = min(_summary_rows, key=lambda r: r[4])
print(f"\nBest by test eRMSE_total: {_best[0]}  (eRMSE_tot={_best[4]:.4g}, "
      f"best_epoch={_best[2]}/{_best[3]})")
print("\n(prev on this split:\n"
      "   unweighted:                                                eRMSE_total = 263.3\n"
      "   per-group floor + flat weights + patience=4 (prev):        eRMSE_total = 278.7)")